In [ ]:
# ════════════════════════════════════════════════════════════════
# BA & PO COPILOT
# ════════════════════════════════════════════════════════════════

import threading, webbrowser, time, re, io, atexit, os, json, base64, difflib
from datetime import datetime
import pandas as pd
import xml.sax.saxutils as saxutils

from google import genai
from google.genai import types

import dash
from dash import dcc, html, Input, Output, State, callback_context, ALL, MATCH
import dash_bootstrap_components as dbc
import plotly.express as px

import sqlite3
from contextlib import closing
from pathlib import Path
import uuid

import requests
from requests.auth import HTTPBasicAuth

import hashlib

from docx import Document
from docx.shared import Pt, RGBColor
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

try:
    from groq import Groq as GroqClient
    GROQ_AVAILABLE = True
except ImportError:
    GROQ_AVAILABLE = False
    print("ℹ️  pip install groq")

try:
    from pypdf import PdfReader
    PYPDF_OK = True
except ImportError:
    PYPDF_OK = False

# ════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════

# API keys are loaded from Windows User Environment Variables.
# They are NOT stored in the source code.

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# Ordered Gemini failover pool.
GEMINI_PREFERRED_MODELS = (
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash",
    "gemini-2.5-pro",
)

OPENROUTER_API_BASE = "https://openrouter.ai/api/v1"

OPENROUTER_PREFERRED_MODELS = (
    "nvidia/nemotron-3.5-lightning:free",
    "nvidia/nemotron-3-ultra-550b-a55b:free",
    "google/gemma-4-26b-a4b-it:free",
    "poolside/laguna-s-2.1:free",
    "poolside/laguna-xs-2.1:free",
    "openrouter/free",
)

GROQ_PREFERRED_MODELS = (
    "llama-3.3-70b-versatile",
    "groq/compound-mini",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "qwen/qwen3.6-27b",
    "groq/compound",
)

GROQ_JSON_MODELS = (
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "qwen/qwen3.6-27b",
    "groq/compound",
    "groq/compound-mini",
)

OPENROUTER_MODEL_CACHE_SECONDS = 300
OPENROUTER_MAX_DISCOVERED_FALLBACKS = 12

OLLAMA_QWEN_MODEL = "qwen3.8:27b-mlx"

# ════════════════════════════════════════════════════════════════
# INTEGRATION STATUS
# ════════════════════════════════════════════════════════════════

def _check_environment_variables():
    """Check whether required credentials are configured."""

    integrations = {
        "Gemini": GEMINI_API_KEY,
        "Groq": GROQ_API_KEY,
        "OpenRouter": OPENROUTER_API_KEY,
    }

    print("\n🔧 API Configuration Status")
    print("─" * 45)

    for name, value in integrations.items():
        if value:
            print(f"✅ {name:<15} API key detected")
        else:
            print(f"❌ {name:<15} API key missing")

    print("─" * 45)


_check_environment_variables()


# ════════════════════════════════════════════════════════════════
# AI CLIENT INITIALIZATION
# ════════════════════════════════════════════════════════════════

gemini_client = None

if GEMINI_API_KEY:
    try:
        gemini_client = genai.Client(api_key=GEMINI_API_KEY)
        print("✅ Gemini connected")
    except Exception as e:
        print(f"⚠️ Gemini: {e}")
else:
    print("⚠️ Gemini skipped — API key not configured")


groq_client = None

if GROQ_AVAILABLE and GROQ_API_KEY:
    try:
        groq_client = GroqClient(api_key=GROQ_API_KEY)
        print("✅ Groq connected")
    except Exception as e:
        print(f"⚠️ Groq: {e}")
elif not GROQ_API_KEY:
    print("⚠️ Groq skipped — API key not configured")


# ════════════════════════════════════════════════════════════════
# JIRA CONFIGURATION
# ════════════════════════════════════════════════════════════════

JIRA_BASE_URL = os.getenv(
    "JIRA_BASE_URL",
    ""
)

JIRA_EMAIL = os.getenv("JIRA_EMAIL")
JIRA_API_TOKEN = os.getenv("JIRA_API_TOKEN")


# ════════════════════════════════════════════════════════════════
# SQLITE CONFIGURATION
# ════════════════════════════════════════════════════════════════
DB_PATH = Path(os.getenv("BA_PO_DB_PATH", "ba_po_workspace.db"))

def init_db():
    # JIRA sync tracking — separate table
    def init_jira_db():
        with closing(sqlite3.connect(DB_PATH)) as conn:
            conn.executescript("""
                CREATE TABLE IF NOT EXISTS jira_sync (
                    id         INTEGER PRIMARY KEY AUTOINCREMENT,
                    chat_id    TEXT NOT NULL,
                    card_id    TEXT NOT NULL,
                    jira_key   TEXT,
                    jira_url   TEXT,
                    status     TEXT DEFAULT 'pending',
                    synced_at  TEXT,
                    UNIQUE(chat_id, card_id)
                );
            """)
            cols = [r[1] for r in conn.execute("PRAGMA table_info(jira_sync)").fetchall()]
            if "content_hash" not in cols:
                conn.execute("ALTER TABLE jira_sync ADD COLUMN content_hash TEXT")
            conn.commit()

    init_jira_db()

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        conn.executescript("""
            CREATE TABLE IF NOT EXISTS projects (
                id         TEXT PRIMARY KEY,
                name       TEXT NOT NULL DEFAULT 'Uncategorised',
                created_at TEXT NOT NULL,
                is_default INTEGER NOT NULL DEFAULT 0
            );
            CREATE TABLE IF NOT EXISTS chats (
                id             TEXT PRIMARY KEY,
                title          TEXT NOT NULL DEFAULT 'New Chat',
                role           TEXT NOT NULL DEFAULT 'ba',
                mode           TEXT NOT NULL DEFAULT 'general',
                created_at     TEXT NOT NULL,
                updated_at     TEXT NOT NULL,
                is_pinned      INTEGER NOT NULL DEFAULT 0,
                is_favorite    INTEGER NOT NULL DEFAULT 0,
                is_archived    INTEGER NOT NULL DEFAULT 0,
                project_id     TEXT REFERENCES projects(id),
                active_submode TEXT NOT NULL DEFAULT 'text',
                chat_summary       TEXT NOT NULL DEFAULT '',
                summary_through_id INTEGER NOT NULL DEFAULT 0
            );
            CREATE TABLE IF NOT EXISTS messages (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                chat_id    TEXT NOT NULL REFERENCES chats(id) ON DELETE CASCADE,
                role       TEXT NOT NULL,
                content    TEXT NOT NULL,
                mode       TEXT NOT NULL DEFAULT 'general',
                created_at TEXT NOT NULL
            );
            CREATE TABLE IF NOT EXISTS workspace_state (
                chat_id         TEXT PRIMARY KEY REFERENCES chats(id) ON DELETE CASCADE,
                doc_content     TEXT    DEFAULT '',
                doc_mode        TEXT    DEFAULT 'general',
                document_type   TEXT    DEFAULT 'BRD',
                process_submode TEXT    DEFAULT 'text',
                bpmn_xml        TEXT    DEFAULT '',
                workspace_open  INTEGER DEFAULT 0,
                updated_at      TEXT
            );
            CREATE TABLE IF NOT EXISTS workspace_artifacts (
                id                TEXT PRIMARY KEY,
                chat_id           TEXT NOT NULL REFERENCES chats(id) ON DELETE CASCADE,
                message_id        INTEGER UNIQUE REFERENCES messages(id) ON DELETE CASCADE,
                artifact_type     TEXT NOT NULL DEFAULT 'document',
                role              TEXT NOT NULL DEFAULT 'ba',
                mode              TEXT NOT NULL DEFAULT 'general',
                document_type     TEXT NOT NULL DEFAULT '',
                process_submode   TEXT NOT NULL DEFAULT 'text',
                doc_content       TEXT DEFAULT '',
                bpmn_xml          TEXT DEFAULT '',
                version           INTEGER NOT NULL DEFAULT 1,
                parent_artifact_id TEXT,
                created_at        TEXT NOT NULL
            );
            CREATE INDEX IF NOT EXISTS idx_workspace_artifacts_chat
                ON workspace_artifacts(chat_id, version);
            CREATE INDEX IF NOT EXISTS idx_workspace_artifacts_message
                ON workspace_artifacts(message_id);
            CREATE TABLE IF NOT EXISTS artifact_quality_verifications (
                artifact_id             TEXT PRIMARY KEY,
                source_artifact_id      TEXT,
                baseline_regulatory     TEXT DEFAULT '',
                baseline_redteam        TEXT DEFAULT '',
                current_regulatory      TEXT DEFAULT '',
                current_redteam         TEXT DEFAULT '',
                verified_at             TEXT NOT NULL
            );
        """)

        def init_ripple_db():
            with closing(sqlite3.connect(DB_PATH)) as conn:
                conn.executescript("""
                    CREATE TABLE IF NOT EXISTS artifact_links (
                        id         INTEGER PRIMARY KEY AUTOINCREMENT,
                        chat_id_a  TEXT NOT NULL,
                        chat_id_b  TEXT NOT NULL,
                        relation   TEXT NOT NULL DEFAULT 'related',
                        created_at TEXT NOT NULL,
                        UNIQUE(chat_id_a, chat_id_b, relation)
                    );
                    CREATE TABLE IF NOT EXISTS content_snapshots (
                        chat_id      TEXT PRIMARY KEY,
                        content_hash TEXT,
                        updated_at   TEXT
                    );
                    CREATE TABLE IF NOT EXISTS ripple_flags (
                        id             INTEGER PRIMARY KEY AUTOINCREMENT,
                        chat_id        TEXT NOT NULL,
                        source_chat_id TEXT NOT NULL,
                        reason         TEXT,
                        created_at     TEXT NOT NULL,
                        acknowledged   INTEGER DEFAULT 0
                    );
                """)
                conn.commit()

        init_ripple_db()

        def init_ripple_dismissed_table():
            with closing(sqlite3.connect(DB_PATH)) as conn:
                conn.execute("""
                CREATE TABLE IF NOT EXISTS ripple_dismissed (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                chat_id_a TEXT NOT NULL,
                chat_id_b TEXT NOT NULL,
                created_at TEXT NOT NULL,
                UNIQUE(chat_id_a, chat_id_b)
            );
        """)
                cols = [r[1] for r in conn.execute("PRAGMA table_info(content_snapshots)").fetchall()]
                if "content_len" not in cols:
                    conn.execute("ALTER TABLE content_snapshots ADD COLUMN content_len INTEGER DEFAULT 0")
                    conn.commit()
        
        init_ripple_dismissed_table()

        # Safe migrations for existing DBs
        chat_cols = [r[1] for r in conn.execute("PRAGMA table_info(chats)").fetchall()]
        for col, defn in [
            ("is_archived",    "INTEGER NOT NULL DEFAULT 0"),
            ("project_id",     "TEXT"),
            ("active_submode", "TEXT NOT NULL DEFAULT 'text'"),
            ("chat_summary", "TEXT NOT NULL DEFAULT ''"),
            ("summary_through_id", "INTEGER NOT NULL DEFAULT 0"),
        ]:
            if col not in chat_cols:
                conn.execute(f"ALTER TABLE chats ADD COLUMN {col} {defn}")

        ws_cols = [r[1] for r in conn.execute("PRAGMA table_info(workspace_state)").fetchall()]
        for col, defn in [
            ("doc_content",     "TEXT DEFAULT ''"),
            ("doc_mode",        "TEXT DEFAULT 'general'"),
            ("document_type",   "TEXT DEFAULT ''"),
            ("process_submode", "TEXT DEFAULT 'text'"),
            ("bpmn_xml",        "TEXT DEFAULT ''"),
            ("workspace_open",  "INTEGER DEFAULT 0"),
            ("updated_at",      "TEXT"),
        ]:
            if col not in ws_cols and ws_cols:
                conn.execute(f"ALTER TABLE workspace_state ADD COLUMN {col} {defn}")

        project_cols = [r[1] for r in conn.execute("PRAGMA table_info(projects)").fetchall()]
        if "is_collapsed" not in project_cols:
            conn.execute("ALTER TABLE projects ADD COLUMN is_collapsed INTEGER NOT NULL DEFAULT 0")

        if not conn.execute("SELECT id FROM projects WHERE is_default=1").fetchone():
            conn.execute(
                "INSERT INTO projects (id,name,created_at,is_default,is_collapsed) VALUES (?,?,?,?,0)",
                (str(uuid.uuid4()), "Uncategorised", datetime.now().isoformat()))

        # Backfill immutable artifacts for existing chats. Historical document
        # responses are already stored in messages, so this preserves their
        # individual workspace versions without changing existing chat data.
        artifact_rows = conn.execute(
            """
            SELECT
                m.id,
                m.chat_id,
                m.mode,
                m.content,
                c.role
            FROM messages AS m
            JOIN chats AS c ON c.id = m.chat_id
            WHERE m.role = 'assistant'
              AND m.mode <> 'general'
            ORDER BY m.chat_id, m.id
            """
        ).fetchall()

        for message_id, artifact_chat_id, artifact_mode, artifact_content, artifact_role in artifact_rows:
            exists = conn.execute(
                "SELECT 1 FROM workspace_artifacts WHERE message_id=?",
                (message_id,),
            ).fetchone()
            if exists:
                continue

            artifact_type = "bpmn" if (artifact_content or "").startswith("[BPMN]") else (
                "cards" if artifact_mode in ("stories", "backlog", "prioritize", "pi-planning",
                                              "sprint", "roadmap", "vision", "acceptance",
                                              "raid", "changelog")
                else "document"
            )

            detected_type = ""
            if artifact_mode == "brd":
                detected_type = infer_document_type_from_content(artifact_content)

            version_row = conn.execute(
                "SELECT COALESCE(MAX(version), 0) FROM workspace_artifacts WHERE chat_id=?",
                (artifact_chat_id,),
            ).fetchone()
            version = int(version_row[0] or 0) + 1
            parent_row = conn.execute(
                "SELECT id FROM workspace_artifacts WHERE chat_id=? ORDER BY version DESC LIMIT 1",
                (artifact_chat_id,),
            ).fetchone()
            parent_id = parent_row[0] if parent_row else None

            conn.execute(
                """
                INSERT INTO workspace_artifacts
                    (
                        id, chat_id, message_id, artifact_type, role, mode,
                        document_type, process_submode, doc_content, bpmn_xml,
                        version, parent_artifact_id, created_at
                    )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    str(uuid.uuid4()),
                    artifact_chat_id,
                    message_id,
                    artifact_type,
                    artifact_role or "ba",
                    artifact_mode or "general",
                    detected_type,
                    "bpmn" if artifact_type == "bpmn" else "text",
                    artifact_content or "",
                    "",
                    version,
                    parent_id,
                    datetime.now().isoformat(),
                ),
            )

        # Existing BPMN XML is only safely available for the current/latest
        # workspace state; attach it to the latest BPMN artifact.
        conn.execute(
            """
            UPDATE workspace_artifacts
            SET bpmn_xml = (
                SELECT ws.bpmn_xml
                FROM workspace_state AS ws
                WHERE ws.chat_id = workspace_artifacts.chat_id
            )
            WHERE artifact_type='bpmn'
              AND version = (
                  SELECT MAX(a2.version)
                  FROM workspace_artifacts AS a2
                  WHERE a2.chat_id = workspace_artifacts.chat_id
                    AND a2.artifact_type='bpmn'
              )
              AND EXISTS (
                  SELECT 1
                  FROM workspace_state AS ws2
                  WHERE ws2.chat_id = workspace_artifacts.chat_id
                    AND COALESCE(ws2.bpmn_xml, '') <> ''
              )
            """
        )

        conn.commit()

init_db()


# ════════════════════════════════════════════════════════════════
# WORKSPACE STATE MANAGER
# ════════════════════════════════════════════════════════════════
WORKSPACE_DOCUMENT_TITLES = {
    "brd": "Business Requirements Document",
    "reqs": "Requirements Document",
    "srs": "Software Requirements Specification",
    "rtm": "Requirements Traceability Matrix",
    "analytics": "Business Analysis — Analytics",
    "process": "Business Process",
    "email": "Stakeholder Communication",
    "scope": "Scope Management",
    "charter": "Project Charter",
    "plan": "Project Management Plan",
    "raid": "RAID Log",
    "status": "Project Status Report",
    "governance": "Governance & Controls",
    "changelog": "Change Log",
    "closure": "Project Closure Report",
    "general": "Document",
}

DOCUMENT_TYPE_CONFIG = {
    "BRD": {
        "title": "Business Requirements Document", "short_title": "BRD",
        "description": "Business objectives, business problem, scope, stakeholders, business requirements, business rules, assumptions, constraints, and measurable success criteria.",
        "focus": ["Business objectives", "Business problem", "Scope and out-of-scope", "Stakeholders", "Business requirements", "Business rules", "Assumptions", "Constraints", "Success criteria"],
    },
    "FRD": {
        "title": "Functional Requirements Document", "short_title": "FRD",
        "description": "Detailed functional requirements, functional workflows, business rules, inputs, outputs, validations, exceptions, dependencies, and functional acceptance criteria.",
        "focus": ["Functional requirements", "Functional workflows", "Business rules", "Inputs and outputs", "Validations", "Exceptions and error handling", "Acceptance criteria", "Dependencies"],
    },
    "PRD": {
        "title": "Product Requirements Document", "short_title": "PRD",
        "description": "Product vision, problem statement, users/personas, product goals, features, user stories, acceptance criteria, prioritization, success metrics, roadmap, dependencies, and risks.",
        "focus": ["Product vision", "Problem statement", "Users/personas", "Product goals", "Features and capabilities", "User stories/use cases", "Acceptance criteria", "Prioritization", "Success metrics", "Roadmap"],
    },
    "SRS": {
        "title": "Software Requirements Specification", "short_title": "SRS",
        "description": "System-level software requirements covering functional and non-functional behavior, interfaces, data, security, performance, availability, error handling, constraints, dependencies, assumptions, and traceability.",
        "focus": ["Introduction", "System overview", "Functional requirements", "Non-functional requirements", "System interfaces", "External interfaces", "Data requirements", "Security requirements", "Performance requirements", "Availability", "Error handling", "Constraints", "Dependencies", "Assumptions", "Traceability"],
    },
    "FSD": {
        "title": "Functional Specification Document", "short_title": "FSD",
        "description": "Implementation-oriented functional specifications covering functional behavior, workflows, inputs/outputs, business rules, validations, integrations, API behavior, error handling, data handling, dependencies, and system behavior.",
        "focus": ["Functional overview", "Detailed functional specifications", "System workflows", "Inputs and outputs", "Business rules", "Validations", "Interfaces and integrations", "API behavior where applicable", "Error handling", "Data handling", "Dependencies", "System behavior"],
    },
    "SOP": {
        "title": "Standard Operating Procedure", "short_title": "SOP",
        "description": "Repeatable operational procedure describing purpose, scope, roles, prerequisites, inputs, systems, ordered procedure steps, decision points, exceptions, controls, outputs, quality checks, records, and related documents.",
        "focus": ["Purpose", "Scope", "Roles and responsibilities", "Prerequisites", "Required inputs", "Tools and systems", "Procedure steps", "Decision points and business rules", "Exceptions and escalation", "Controls and compliance considerations", "Outputs", "Quality checks", "Records and documentation"],
    },
    "SOW": {
        "title": "Scope of Work", "short_title": "SOW",
        "description": "Delivery-oriented scope definition covering objectives, in-scope and out-of-scope work, deliverables, activities, responsibilities, assumptions, dependencies, constraints, milestones, acceptance criteria, change control, risks, governance, and completion/sign-off.",
        "focus": ["Purpose and background", "Objectives", "In scope", "Out of scope", "Deliverables", "Activities and work packages", "Roles and responsibilities", "Assumptions", "Dependencies", "Constraints", "Milestones and schedule", "Acceptance criteria", "Change control", "Risks and issues", "Governance and reporting", "Completion and sign-off"],
    },
}
SUPPORTED_REQUIREMENTS_DOCUMENT_TYPES = tuple(DOCUMENT_TYPE_CONFIG.keys())


def detect_document_type(user_message):
    """Detect the requested requirements document type from explicit language."""
    text = (user_message or "").strip().upper()

    conversion_patterns = (
        r"\b(?:CONVERT|CHANGE|TRANSFORM|TURN|REWRITE|REFACTOR|UPDATE|ALIGN)\b"
        r".{0,180}?\b(?:TO|INTO|AS)\s+(BRD|FRD|PRD|SRS|FSD|SOP|SOW)\b",
        r"\bFROM\s+(BRD|FRD|PRD|SRS|FSD|SOP|SOW)\b"
        r".{0,120}?\b(?:TO|INTO)\s+(BRD|FRD|PRD|SRS|FSD|SOP|SOW)\b",
    )

    for pattern in conversion_patterns:
        matches = list(re.finditer(pattern, text, flags=re.DOTALL))
        if matches and matches[-1].lastindex:
            target = matches[-1].group(matches[-1].lastindex).upper()
            if target in DOCUMENT_TYPE_CONFIG:
                return target

    aliases = {
        "STANDARD OPERATING PROCEDURE": "SOP",
        "STATEMENT OF PROCEDURE": "SOP",
        "SCOPE OF WORK": "SOW",
        "FUNCTIONAL SPECIFICATION DOCUMENT": "FSD",
        "SOFTWARE REQUIREMENTS SPECIFICATION": "SRS",
        "FUNCTIONAL REQUIREMENTS DOCUMENT": "FRD",
        "PRODUCT REQUIREMENTS DOCUMENT": "PRD",
        "BUSINESS REQUIREMENTS DOCUMENT": "BRD",
    }

    candidates = []

    for phrase, document_type in aliases.items():
        match = re.search(rf"\b{re.escape(phrase)}\b", text)
        if match:
            candidates.append((match.start(), document_type))

    for document_type in SUPPORTED_REQUIREMENTS_DOCUMENT_TYPES:
        match = re.search(rf"\b{re.escape(document_type)}\b", text)
        if match:
            candidates.append((match.start(), document_type))

    if not candidates:
        return "BRD"

    candidates.sort(key=lambda item: item[0])
    return candidates[0][1]


def infer_document_type_from_content(doc_content):
    """Infer BRD/FRD/PRD from an artifact, preferring its actual title."""
    text = (doc_content or "")[:4000].upper()

    title_patterns = (
        ("SOP", r"(?:^|\n)\s*#{0,6}\s*STANDARD OPERATING PROCEDURE\b"),
        ("SOW", r"(?:^|\n)\s*#{0,6}\s*SCOPE OF WORK\b"),
        ("FSD", r"(?:^|\n)\s*#{0,6}\s*FUNCTIONAL SPECIFICATION DOCUMENT\b"),
        ("SRS", r"(?:^|\n)\s*#{0,6}\s*SOFTWARE REQUIREMENTS SPECIFICATION\b"),
        ("FRD", r"(?:^|\n)\s*#{0,6}\s*FUNCTIONAL REQUIREMENTS DOCUMENT\b"),
        ("PRD", r"(?:^|\n)\s*#{0,6}\s*PRODUCT REQUIREMENTS DOCUMENT\b"),
        ("BRD", r"(?:^|\n)\s*#{0,6}\s*BUSINESS REQUIREMENTS DOCUMENT\b"),
    )

    for document_type, pattern in title_patterns:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return document_type

    patterns = (
        ("SOP", r"\b(STANDARD OPERATING PROCEDURE|STATEMENT OF PROCEDURE|SOP)\b"),
        ("SOW", r"\b(SCOPE OF WORK|SOW)\b"),
        ("FSD", r"\b(FUNCTIONAL SPECIFICATION DOCUMENT|FSD)\b"),
        ("SRS", r"\b(SOFTWARE REQUIREMENTS SPECIFICATION|SRS)\b"),
        ("FRD", r"\b(FUNCTIONAL REQUIREMENTS DOCUMENT|FUNCTIONAL REQUIREMENTS\s*\(FRD\)|FRD)\b"),
        ("PRD", r"\b(PRODUCT REQUIREMENTS DOCUMENT|PRODUCT REQUIREMENTS\s*\(PRD\)|PRD)\b"),
        ("BRD", r"\b(BUSINESS REQUIREMENTS DOCUMENT|BUSINESS REQUIREMENTS\s*\(BRD\)|BRD)\b"),
    )

    for document_type, pattern in patterns:
        if re.search(pattern, text):
            return document_type

    return "BRD"


def resolve_artifact_document_type(artifact):
    """Resolve the display type while correcting stale metadata from old artifacts."""
    if not artifact:
        return "BRD"

    role = (artifact.get("role") or "ba").lower()
    mode = (artifact.get("mode") or "general").lower()

    if role != "ba" or mode != "brd":
        return ""

    stored = (artifact.get("document_type") or "").upper()
    inferred = infer_document_type_from_content(
        artifact.get("doc_content") or ""
    )

    # A strong title in the artifact is more trustworthy than stale metadata
    # created before dynamic BRD/FRD/PRD conversion was fixed.
    title_text = (artifact.get("doc_content") or "")[:1200].upper()
    strong_title = (
        re.search(
            r"(?:^|\n)\s*#{0,6}\s*FUNCTIONAL REQUIREMENTS DOCUMENT\b",
            title_text,
        )
        or re.search(
            r"(?:^|\n)\s*#{0,6}\s*PRODUCT REQUIREMENTS DOCUMENT\b",
            title_text,
        )
        or re.search(
            r"(?:^|\n)\s*#{0,6}\s*BUSINESS REQUIREMENTS DOCUMENT\b",
            title_text,
        )
        or re.search(
            r"(?:^|\n)\s*#{0,6}\s*STANDARD OPERATING PROCEDURE\b",
            title_text,
        )
        or re.search(
            r"(?:^|\n)\s*#{0,6}\s*SCOPE OF WORK\b",
            title_text,
        )
        or re.search(
            r"(?:^|\n)\s*#{0,6}\s*FUNCTIONAL SPECIFICATION DOCUMENT\b",
            title_text,
        )
        or re.search(
            r"(?:^|\n)\s*#{0,6}\s*SOFTWARE REQUIREMENTS SPECIFICATION\b",
            title_text,
        )
    )

    if inferred in {"SOP", "SOW", "FSD", "SRS"}:
        return inferred

    if strong_title and inferred in DOCUMENT_TYPE_CONFIG:
        return inferred

    return stored if stored in DOCUMENT_TYPE_CONFIG else inferred


def get_document_type_label(document_type):
    """Return the short display label for a document type."""
    document_type = (document_type or "BRD").upper()
    return DOCUMENT_TYPE_CONFIG.get(
        document_type,
        DOCUMENT_TYPE_CONFIG["BRD"],
    )["short_title"]


def get_document_type_title(document_type):
    """Return the full display title for a document type."""
    document_type = (document_type or "BRD").upper()
    return DOCUMENT_TYPE_CONFIG.get(
        document_type,
        DOCUMENT_TYPE_CONFIG["BRD"],
    )["title"]


def get_workspace_display_mode(role, mode, document_type=""):
    """Return the user-facing artifact label without changing the internal mode."""
    if role == "ba" and mode == "brd":
        normalized_type = (document_type or "").upper()
        if normalized_type in DOCUMENT_TYPE_CONFIG:
            return DOCUMENT_TYPE_CONFIG[normalized_type]["short_title"]
        return "BRD"

    if role == "pm" and mode == "scope":
        return "Scope Management"

    return (mode or "general").upper()


def get_workspace_document_title(role, mode):
    """Return the legacy workspace title for non-dynamic artifact modes."""
    return WORKSPACE_DOCUMENT_TITLES.get(
        mode,
        f"{role.upper()} Document",
    )


def build_workspace_document(
    doc_content,
    role,
    mode,
    document_type="BRD",
):
    """Render a saved text artifact using its persisted document type."""
    document_type = (document_type or "").upper()

    if role == "ba" and mode == "brd" and document_type in DOCUMENT_TYPE_CONFIG:
        title = get_document_type_title(document_type)
        short_title = get_document_type_label(document_type)
    else:
        title = get_workspace_document_title(role, mode)
        short_title = mode.upper()

    return html.Div(
        [
            html.Div(
                [
                    html.Div(
                        [
                            html.Span(
                                "📄",
                                style={
                                    "fontSize": "20px",
                                    "marginRight": "8px",
                                },
                            ),
                            html.Span(
                                title,
                                style={
                                    "fontSize": "18px",
                                    "fontWeight": "700",
                                    "color": "#202124",
                                },
                            ),
                        ],
                        style={
                            "display": "flex",
                            "alignItems": "center",
                        },
                    ),
                    html.Div(
                        f"{role.upper()} · {short_title}",
                        style={
                            "fontSize": "10px",
                            "fontWeight": "600",
                            "color": "#888",
                            "marginTop": "5px",
                            "letterSpacing": "0.4px",
                        },
                    ),
                ],
                style={
                    "padding": "16px 30px",
                    "borderBottom": "1px solid #e8eaed",
                    "position": "sticky",
                    "top": "0",
                    "background": "white",
                    "zIndex": "2",
                },
            ),
            html.Div(
                dcc.Markdown(
                    doc_content,
                    className="markdown-body",
                    style={
                        "fontSize": "13px",
                        "lineHeight": "1.7",
                        "color": "#202124",
                        "maxWidth": "100%",
                        "paddingBottom": "40px",
                    },
                ),
                style={
                    "padding": "22px 30px 40px",
                },
            ),
        ],
        style={
            "background": "white",
            "minHeight": "100%",
        },
    )


def build_workspace_chart(doc_content, role, mode):
    """Rebuild chart artifacts during initial generation and sidebar reload."""
    should_chart = (
        (role == "ba" and mode == "analytics")
        or (role == "pm" and mode == "status")
    )

    if not should_chart:
        return []

    chart_data = extract_chart_data(doc_content)

    if not chart_data:
        return []

    fig = px.bar(
        x=[item[0] for item in chart_data],
        y=[item[1] for item in chart_data],
        title="Key Metrics",
    )

    fig.update_layout(
        height=280,
        margin=dict(l=20, r=20, t=40, b=20),
    )

    return [
        dcc.Graph(
            figure=fig,
            config={"displayModeBar": False},
        )
    ]


class WorkspaceState:
    DEFAULTS = {
        "doc_content": "",
        "doc_mode": "general",
        "document_type": "BRD",
        "process_submode": "text",
        "bpmn_xml": "",
        "workspace_open": False,
    }

    @classmethod
    def save(cls, chat_id, **kwargs):
        """Persist workspace state without destroying unspecified fields."""
        if not chat_id:
            return

        with closing(sqlite3.connect(DB_PATH)) as conn:
            conn.row_factory = sqlite3.Row

            row = conn.execute(
                """
                SELECT
                    doc_content,
                    doc_mode,
                    document_type,
                    process_submode,
                    bpmn_xml,
                    workspace_open
                FROM workspace_state
                WHERE chat_id = ?
                """,
                (chat_id,),
            ).fetchone()

            if row:
                current = {
                    "doc_content": row["doc_content"] or "",
                    "doc_mode": row["doc_mode"] or "general",
                    "document_type": row["document_type"] or "BRD",
                    "process_submode": row["process_submode"] or "text",
                    "bpmn_xml": row["bpmn_xml"] or "",
                    "workspace_open": bool(row["workspace_open"]),
                }
            else:
                current = dict(cls.DEFAULTS)

            state = {
                **current,
                **{
                    key: value
                    for key, value in kwargs.items()
                    if key in cls.DEFAULTS
                },
            }

            now = datetime.now().isoformat()

            conn.execute(
                """
                INSERT INTO workspace_state
                    (
                        chat_id,
                        doc_content,
                        doc_mode,
                        document_type,
                        process_submode,
                        bpmn_xml,
                        workspace_open,
                        updated_at
                    )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(chat_id) DO UPDATE SET
                    doc_content = excluded.doc_content,
                    doc_mode = excluded.doc_mode,
                    document_type = excluded.document_type,
                    process_submode = excluded.process_submode,
                    bpmn_xml = excluded.bpmn_xml,
                    workspace_open = excluded.workspace_open,
                    updated_at = excluded.updated_at
                """,
                (
                    chat_id,
                    state["doc_content"],
                    state["doc_mode"],
                    state["document_type"],
                    state["process_submode"],
                    state["bpmn_xml"],
                    1 if state["workspace_open"] else 0,
                    now,
                ),
            )

            conn.commit()

    @classmethod
    def load(cls, chat_id):
        if not chat_id:
            return dict(cls.DEFAULTS)

        with closing(sqlite3.connect(DB_PATH)) as conn:
            conn.row_factory = sqlite3.Row
            row = conn.execute(
                "SELECT * FROM workspace_state WHERE chat_id = ?",
                (chat_id,),
            ).fetchone()

        if not row:
            return dict(cls.DEFAULTS)

        keys = set(row.keys())
        doc_content = row["doc_content"] or ""
        stored_document_type = (
            row["document_type"]
            if "document_type" in keys and row["document_type"]
            else ""
        )

        document_type = stored_document_type.upper() if stored_document_type else ""

        # Existing databases created before dynamic document types may have
        # no persisted type. Infer it once from the saved document content.
        if (
            document_type not in DOCUMENT_TYPE_CONFIG
            or (
                document_type == "BRD"
                and doc_content
                and not stored_document_type
            )
        ):
            document_type = infer_document_type_from_content(doc_content)

        return {
            "doc_content": doc_content,
            "doc_mode": row["doc_mode"] or "general",
            "document_type": document_type or "BRD",
            "process_submode": row["process_submode"] or "text",
            "bpmn_xml": row["bpmn_xml"] or "",
            "workspace_open": bool(row["workspace_open"]),
        }

    @classmethod
    def build_content(cls, state, role, chat_id=None):
        """Build workspace children from persisted artifact state."""
        bpmn_xml = state.get("bpmn_xml", "")
        doc_content = state.get("doc_content", "")
        doc_mode = (state.get("doc_mode", "") or "").strip().lower()
        document_type = (state.get("document_type") or "BRD").upper()

        if bpmn_xml:
            ws_children = html.Div(
                [
                    html.Div(
                        "🔷 BPMN Diagram",
                        style={
                            "fontWeight": "700",
                            "color": "#00695c",
                            "fontSize": "15px",
                            "marginBottom": "6px",
                        },
                    ),
                    html.Div(
                        "Rendered in canvas below — use Export to download",
                        style={
                            "fontSize": "12px",
                            "color": "#888",
                        },
                    ),
                ],
                style={
                    "padding": "24px 32px",
                    "textAlign": "center",
                },
            )
            return ws_children, bpmn_xml, []

        if not doc_content:
            return (
                html.Div(
                    [
                        html.Div(
                            "📄",
                            style={
                                "fontSize": "30px",
                                "marginBottom": "8px",
                            },
                        ),
                        html.Div(
                            "No document saved for this chat.",
                            style={
                                "color": "#999",
                                "fontSize": "13px",
                            },
                        ),
                    ],
                    style={
                        "textAlign": "center",
                        "marginTop": "70px",
                    },
                ),
                "",
                [],
            )

        if doc_mode == "general" and chat_id:
            with closing(sqlite3.connect(DB_PATH)) as conn:
                row = conn.execute(
                    "SELECT role, mode FROM chats WHERE id = ?",
                    (chat_id,),
                ).fetchone()

            if row and row[1] != "general":
                doc_mode = row[1]

        if role == "pm" and doc_mode == "scope":
            try:
                parsed_scope = parse_po_json(doc_content)
            except Exception:
                parsed_scope = None

            if isinstance(parsed_scope, dict):
                jira_map = load_jira_sync(chat_id) if chat_id else {}
                return render_scope_management(parsed_scope, jira_map=jira_map), "", []

        if is_structured_card_mode(role, doc_mode):
            parsed = None

            try:
                parsed = parse_po_json(doc_content)
            except Exception:
                parsed = None

            if parsed and isinstance(parsed, (dict, list)):
                jira_map = load_jira_sync(chat_id) if chat_id else {}

                return (
                    render_jira_cards(
                        parsed,
                        jira_map=jira_map,
                    ),
                    "",
                    [],
                )

        return (
            build_workspace_document(
                doc_content,
                role,
                doc_mode,
                document_type=document_type,
            ),
            "",
            build_workspace_chart(
                doc_content,
                role,
                doc_mode,
            ),
        )


# ════════════════════════════════════════════════════════════════
# IMMUTABLE WORKSPACE ARTIFACTS
# ════════════════════════════════════════════════════════════════
def create_workspace_artifact(
    chat_id,
    message_id,
    role,
    mode,
    doc_content="",
    document_type="",
    process_submode="text",
    bpmn_xml="",
    parent_artifact_id=None,
):
    """Create an immutable workspace snapshot for one assistant response."""
    if not chat_id or not message_id:
        return None

    artifact_type = "bpmn" if bpmn_xml else "document"
    if not bpmn_xml:
        try:
            if role == "pm" and mode == "scope":
                artifact_type = "document"
            elif is_structured_card_mode(role, mode) and parse_po_json(doc_content):
                artifact_type = "cards"
        except Exception:
            pass

    document_type = (
        (document_type or "").upper()
        if role == "ba" and mode == "brd"
        else ""
    )

    artifact_id = str(uuid.uuid4())
    now = datetime.now().isoformat()

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        latest = conn.execute(
            """
            SELECT id, version
            FROM workspace_artifacts
            WHERE chat_id=?
            ORDER BY version DESC
            LIMIT 1
            """,
            (chat_id,),
        ).fetchone()

        version = int(latest["version"] or 0) + 1 if latest else 1
        parent_id = (
            parent_artifact_id
            if parent_artifact_id
            else (latest["id"] if latest else None)
        )

        conn.execute(
            """
            INSERT INTO workspace_artifacts
                (
                    id, chat_id, message_id, artifact_type, role, mode,
                    document_type, process_submode, doc_content, bpmn_xml,
                    version, parent_artifact_id, created_at
                )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                artifact_id,
                chat_id,
                message_id,
                artifact_type,
                role or "ba",
                mode or "general",
                document_type,
                process_submode or "text",
                doc_content or "",
                bpmn_xml or "",
                version,
                parent_id,
                now,
            ),
        )
        conn.commit()

    return artifact_id


def get_workspace_artifact(artifact_id):
    """Load one immutable workspace artifact by its ID."""
    if not artifact_id:
        return None

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(
            "SELECT * FROM workspace_artifacts WHERE id=?",
            (artifact_id,),
        ).fetchone()

    return dict(row) if row else None


def get_workspace_artifact_by_message_id(message_id):
    """Resolve the exact workspace snapshot generated by one assistant message."""
    if not message_id:
        return None

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(
            "SELECT * FROM workspace_artifacts WHERE message_id=?",
            (message_id,),
        ).fetchone()

    return dict(row) if row else None


def get_latest_workspace_artifact(chat_id):
    """Load the newest artifact without being affected by a selected older artifact."""
    if not chat_id:
        return None

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(
            """
            SELECT *
            FROM workspace_artifacts
            WHERE chat_id=?
            ORDER BY version DESC
            LIMIT 1
            """,
            (chat_id,),
        ).fetchone()

    return dict(row) if row else None


def delete_workspace_artifacts(chat_id):
    """Delete immutable workspace history when a chat is explicitly reset."""
    if not chat_id:
        return

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute(
            "DELETE FROM workspace_artifacts WHERE chat_id=?",
            (chat_id,),
        )
        conn.commit()


def workspace_artifact_to_state(artifact):
    """Convert an immutable artifact into the workspace rendering state shape."""
    if not artifact:
        return dict(WorkspaceState.DEFAULTS)

    return {
        "doc_content": artifact.get("doc_content") or "",
        "doc_mode": artifact.get("mode") or "general",
        "document_type": resolve_artifact_document_type(artifact),
        "process_submode": artifact.get("process_submode") or "text",
        "bpmn_xml": artifact.get("bpmn_xml") or "",
        "workspace_open": True,
    }


def render_workspace_artifact(artifact):
    """Render one immutable artifact exactly as it was generated."""
    if not artifact:
        return html.Div(
            "⚠️ Workspace version not found.",
            style={
                "color": "#c62828",
                "textAlign": "center",
                "marginTop": "40px",
            },
        ), "", []

    state = workspace_artifact_to_state(artifact)
    role = artifact.get("role") or "ba"
    chat_id = artifact.get("chat_id")
    return WorkspaceState.build_content(state, role, chat_id)


# ════════════════════════════════════════════════════════════════
# SQLITE HELPER FUNCTIONS
# ════════════════════════════════════════════════════════════════
def save_jira_sync(chat_id, card_id, jira_key, jira_url, status, content_hash=None):
    now = datetime.now().isoformat()
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("""
            INSERT INTO jira_sync
                (chat_id,card_id,jira_key,jira_url,status,synced_at,content_hash)
            VALUES (?,?,?,?,?,?,?)
            ON CONFLICT(chat_id,card_id) DO UPDATE SET
                jira_key = excluded.jira_key,
                jira_url = excluded.jira_url,
                status = excluded.status,
                synced_at = excluded.synced_at,
                content_hash = excluded.content_hash
        """, (chat_id, card_id, jira_key, jira_url, status, now, content_hash))
        conn.commit()

def load_jira_sync(chat_id):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            "SELECT * FROM jira_sync WHERE chat_id=?", (chat_id,)).fetchall()
    return {r["card_id"]: dict(r) for r in rows}
    
def _get_default_pid():
    with closing(sqlite3.connect(DB_PATH)) as conn:
        r = conn.execute("SELECT id FROM projects WHERE is_default=1").fetchone()
        return r[0] if r else None
        
def _strip_model_thinking(text):
    """Remove common reasoning wrappers without discarding a later JSON object."""
    value = (text or "").strip()
    if not value:
        return ""

    value = re.sub(
        r"<think>.*?</think>",
        "",
        value,
        flags=re.IGNORECASE | re.DOTALL,
    )
    value = re.sub(
        r"<analysis>.*?</analysis>",
        "",
        value,
        flags=re.IGNORECASE | re.DOTALL,
    )
    value = re.sub(
        r"^\s*(?:here(?:'s| is)\s+(?:my\s+)?(?:thinking|analysis|reasoning)\s*:?)",
        "",
        value,
        flags=re.IGNORECASE,
    )
    return value.strip()


def _extract_balanced_json_objects(text):
    """Yield balanced JSON object candidates embedded in arbitrary model text."""
    value = _strip_model_thinking(text)
    if not value:
        return

    starts = [
        match.start()
        for match in re.finditer(r"\{", value)
    ]

    for start in starts:
        depth = 0
        in_string = False
        escaped = False

        for index in range(start, len(value)):
            char = value[index]

            if in_string:
                if escaped:
                    escaped = False
                elif char == "\\":
                    escaped = True
                elif char == '"':
                    in_string = False
                continue

            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
                if depth == 0:
                    yield value[start:index + 1]
                    break


def _extract_regulatory_json(raw):
    """Parse regulatory JSON despite reasoning, Markdown, or surrounding prose."""
    if not raw:
        return None

    decoder = json.JSONDecoder()
    value = _strip_model_thinking(raw)

    candidates = [
        value,
        re.sub(
            r"^\s*```(?:json)?\s*",
            "",
            value,
            flags=re.IGNORECASE,
        ).strip(),
    ]

    for candidate in candidates:
        candidate = re.sub(
            r"\s*```\s*$",
            "",
            candidate,
            flags=re.IGNORECASE,
        ).strip()

        if not candidate:
            continue

        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except (TypeError, ValueError):
            pass

        try:
            parsed, _ = decoder.raw_decode(candidate)
            if isinstance(parsed, dict):
                return parsed
        except (TypeError, ValueError):
            pass

    for candidate in _extract_balanced_json_objects(value):
        try:
            parsed = json.loads(candidate)
        except (TypeError, ValueError):
            continue

        if isinstance(parsed, dict):
            return parsed

    return None


def _repair_regulatory_json(raw):
    """Repair a malformed scan using a dedicated structured-output provider."""
    repair_messages = [
        {
            "role": "system",
            "content": (
                "You are a JSON repair service. "
                "Return exactly one valid JSON object and absolutely nothing else. "
                "Do not explain your reasoning. "
                "Do not use Markdown. "
                "Do not invent findings."
            ),
        },
        {
            "role": "user",
            "content": (
                "Convert the following regulatory scan response into the exact "
                "JSON schema below. Preserve only information present in the "
                "response.\n\n"
                'Schema: {"detected_domain":"string",'
                '"detected_jurisdiction":"string","regulations":[],'
                '"overall_risk":"Low|Medium|High",'
                '"risk_rationale":"string"}\n\n'
                "Each regulation must contain: name, authority, applicable, "
                "rationale, gaps. Each gap must be a string.\n\n"
                "SOURCE RESPONSE:\n"
                f"{(raw or '')[:12000]}"
            ),
        },
    ]

    try:
        repaired_raw, _, _ = AIProvider.call_json(
            repair_messages,
            max_tokens=2500,
        )
    except Exception:
        return None

    return _extract_regulatory_json(
        repaired_raw,
    )


def _normalize_regulatory_result(result):
    """Normalize a parsed scan into the schema expected by the UI."""
    if not isinstance(result, dict):
        return None

    regulations = result.get("regulations")
    if not isinstance(regulations, list):
        return None

    normalized = {
        "detected_domain": str(
            result.get("detected_domain") or "Unknown"
        ),
        "detected_jurisdiction": str(
            result.get("detected_jurisdiction") or "Unknown"
        ),
        "regulations": [],
        "overall_risk": str(
            result.get("overall_risk") or "Medium"
        ).title(),
        "risk_rationale": str(
            result.get("risk_rationale") or ""
        ),
    }

    if normalized["overall_risk"] not in {
        "Low",
        "Medium",
        "High",
    }:
        normalized["overall_risk"] = "Medium"

    for regulation in regulations:
        if not isinstance(regulation, dict):
            continue

        name = str(
            regulation.get("name")
            or "Regulatory requirement"
        )
        authority = str(
            regulation.get("authority")
            or ""
        )
        applicable = bool(
            regulation.get("applicable", True)
        )
        rationale = str(
            regulation.get("rationale")
            or ""
        )

        raw_gaps = regulation.get("gaps")
        if not isinstance(raw_gaps, list):
            raw_gaps = []

        gaps = [
            str(gap).strip()
            for gap in raw_gaps
            if str(gap).strip()
        ]

        normalized["regulations"].append(
            {
                "name": name,
                "authority": authority,
                "applicable": applicable,
                "rationale": rationale,
                "gaps": gaps,
            }
        )

    return normalized


def _regulatory_scan_messages(doc_content):
    """Build strict structured-output messages for Regulatory Radar."""
    source = str(doc_content or "").strip()
    if len(source) > 30000:
        source = (
            source[:24000]
            + "\n\n[Middle of artifact omitted for model context; "
              "preserve findings supported by the supplied content.]\n\n"
            + source[-6000:]
        )

    return [
        {
            "role": "system",
            "content": (
                REGULATORY_RADAR_SYSTEM
                + "\n\nSTRICT MACHINE-READABLE OUTPUT CONTRACT:"
                "\n- Return exactly one JSON object."
                "\n- Do not output reasoning, analysis, thinking, Markdown, "
                  "code fences, or prose."
                "\n- The response must begin with '{' and end with '}'."
                "\n- Every gap must be a plain string."
                "\n- Do not invent regulations, citations, or findings."
            ),
        },
        {
            "role": "user",
            "content": (
                "Analyze this workspace artifact and return the required "
                "Regulatory Radar JSON object only.\n\n"
                f"ARTIFACT:\n{source}"
            ),
        },
    ]


def _call_regulatory_provider(messages, max_tokens=3500):
    """Call structured providers, then a tolerant generic fallback."""
    errors = []

    try:
        raw, provider_name, banner = AIProvider.call_json(
            messages,
            max_tokens=max_tokens,
        )
        parsed = _extract_regulatory_json(raw)
        if parsed is not None:
            return parsed, provider_name, banner, raw
        errors.append(
            f"{provider_name or 'structured provider'} returned "
            "unparseable JSON"
        )
    except Exception as exc:
        errors.append(f"structured providers: {str(exc)[:180]}")

    try:
        raw, provider_name, banner = AIProvider.call(
            messages,
            max_tokens=max_tokens,
        )
        parsed = _extract_regulatory_json(raw)
        if parsed is not None:
            return parsed, provider_name, banner, raw
        errors.append(
            f"{provider_name or 'generic provider'} returned "
            "no extractable JSON"
        )
    except Exception as exc:
        errors.append(f"generic provider chain: {str(exc)[:180]}")

    raise RuntimeError(" | ".join(errors))


def _repair_regulatory_result(raw):
    """Repair malformed Regulatory Radar output using a tolerant provider."""
    source = str(raw or "").strip()
    if not source:
        return None

    repair_messages = [
        {
            "role": "system",
            "content": (
                "You are a strict JSON repair service. Return exactly one "
                "valid JSON object and nothing else. Never output reasoning "
                "or Markdown. Preserve only facts contained in the source."
            ),
        },
        {
            "role": "user",
            "content": (
                "Convert the source response into this exact schema. "
                "Do not invent findings.\n\n"
                '{"detected_domain":"string",'
                '"detected_jurisdiction":"string",'
                '"regulations":[{"name":"string","authority":"string",'
                '"applicable":true,"rationale":"string","gaps":["string"]}],'
                '"overall_risk":"Low|Medium|High",'
                '"risk_rationale":"string"}\n\n'
                f"SOURCE RESPONSE:\n{source[:16000]}"
            ),
        },
    ]

    try:
        parsed, _, _, _ = _call_regulatory_provider(
            repair_messages,
            max_tokens=3000,
        )
        return parsed
    except Exception:
        return None


def run_regulatory_scan(doc_content):
    """Run Regulatory Radar with tolerant parsing and provider fallback."""
    if not doc_content or not str(doc_content).strip():
        raise ValueError(
            "No workspace content is available for Regulatory Radar."
        )

    messages = _regulatory_scan_messages(doc_content)
    raw = ""
    provider_name = ""

    try:
        result, provider_name, _, raw = _call_regulatory_provider(
            messages,
            max_tokens=3500,
        )
    except Exception as first_error:
        result = None
        provider_name = ""
        raw = ""
        first_error_text = str(first_error)

    if result is None:
        result = _repair_regulatory_result(raw)

    result = _normalize_regulatory_result(result)

    if result is None:
        preview = re.sub(
            r"\s+",
            " ",
            (raw or "")[:700],
        ).strip()

        details = (
            first_error_text
            if "first_error_text" in locals()
            else "No structured result was returned."
        )

        raise ValueError(
            "Regulatory Radar could not obtain valid structured results. "
            f"Provider: {provider_name or 'unknown'}. "
            f"Details: {details[:500]}. "
            f"AI response preview: {preview or '[empty response]'}"
        )

    return result


def create_chat(role="ba", mode="general", title="New Chat"):
    cid = str(uuid.uuid4()); now = datetime.now().isoformat()
    pid = _get_default_pid()
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        conn.execute(
            "INSERT INTO chats (id,title,role,mode,created_at,updated_at,project_id) VALUES (?,?,?,?,?,?,?)",
            (cid, title, role, mode, now, now, pid))
        conn.commit()
    return cid

def link_artifacts(chat_id_a, chat_id_b, relation="sibling"):
    if chat_id_a == chat_id_b: return
    now = datetime.now().isoformat()
    with closing(sqlite3.connect(DB_PATH)) as conn:
        for a, b in [(chat_id_a, chat_id_b), (chat_id_b, chat_id_a)]:
            conn.execute("""
                INSERT OR IGNORE INTO artifact_links (chat_id_a,chat_id_b,relation,created_at)
                VALUES (?,?,?,?)
            """, (a, b, relation, now))
        conn.commit()

def unlink_artifacts(chat_id_a, chat_id_b):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("DELETE FROM artifact_links WHERE chat_id_a=? AND chat_id_b=?", (chat_id_a, chat_id_b))
        conn.execute("DELETE FROM artifact_links WHERE chat_id_a=? AND chat_id_b=?", (chat_id_b, chat_id_a))
        conn.commit()

def dismiss_suggestion(chat_id_a, chat_id_b):
    now = datetime.now().isoformat()
    with closing(sqlite3.connect(DB_PATH)) as conn:
        for a, b in [(chat_id_a, chat_id_b), (chat_id_b, chat_id_a)]:
            conn.execute("INSERT OR IGNORE INTO ripple_dismissed (chat_id_a,chat_id_b,created_at) VALUES (?,?,?)", (a, b, now))
        conn.commit()

def load_dismissed_pairs(chat_id):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        return {r["chat_id_b"] for r in conn.execute(
            "SELECT chat_id_b FROM ripple_dismissed WHERE chat_id_a=?", (chat_id,)).fetchall()}

def get_linked_chats(chat_id):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(
            "SELECT * FROM artifact_links WHERE chat_id_a=?", (chat_id,)).fetchall()]

def save_content_snapshot(chat_id, content_hash, content_len=0):
    now = datetime.now().isoformat()
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("""
            INSERT INTO content_snapshots (chat_id,content_hash,content_len,updated_at) VALUES (?,?,?,?)
            ON CONFLICT(chat_id) DO UPDATE SET content_hash=excluded.content_hash, content_len=excluded.content_len, updated_at=excluded.updated_at
        """, (chat_id, content_hash, content_len, now))
        conn.commit()

def load_content_snapshot(chat_id):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        r = conn.execute("SELECT content_hash, content_len FROM content_snapshots WHERE chat_id=?", (chat_id,)).fetchone()
        return (r["content_hash"], r["content_len"]) if r else (None, 0)

def flag_ripple(chat_id, source_chat_id, reason):
    now = datetime.now().isoformat()
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("""
            INSERT INTO ripple_flags (chat_id,source_chat_id,reason,created_at,acknowledged)
            VALUES (?,?,?,?,0)
        """, (chat_id, source_chat_id, reason, now))
        conn.commit()

def load_unacknowledged_ripples():
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(
            "SELECT * FROM ripple_flags WHERE acknowledged=0 ORDER BY created_at DESC").fetchall()]

def acknowledge_ripple(ripple_id):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("UPDATE ripple_flags SET acknowledged=1 WHERE id=?", (ripple_id,))
        conn.commit()

def check_and_flag_ripple(chat_id, new_content):
    new_content = new_content or ""
    new_hash = hashlib.sha256(new_content.encode("utf-8")).hexdigest()[:16]
    prior_hash, prior_len = load_content_snapshot(chat_id)
    save_content_snapshot(chat_id, new_hash, len(new_content))
    if prior_hash and prior_hash != new_hash:
        with closing(sqlite3.connect(DB_PATH)) as conn:
            conn.row_factory = sqlite3.Row
            row = conn.execute("SELECT title FROM chats WHERE id=?", (chat_id,)).fetchone()
        title = row["title"] if row else "a linked artifact"
        delta = len(new_content) - prior_len
        delta_note = f" ({'+' if delta>=0 else ''}{delta} chars)" if prior_len else ""
        for link in get_linked_chats(chat_id):
            flag_ripple(link["chat_id_b"], chat_id, f"'{title}' was updated{delta_note} — review for impact")

def find_related_chats(chat_id, threshold=0.25, max_results=5):
    base_words = _doc_keywords(chat_id)
    if not base_words: return []
    excluded = {l["chat_id_b"] for l in get_linked_chats(chat_id)} | load_dismissed_pairs(chat_id)
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        others = conn.execute("SELECT id, title FROM chats WHERE id != ? AND is_archived=0", (chat_id,)).fetchall()
    scored = []
    for o in others:
        if o["id"] in excluded: continue
        other_words = _doc_keywords(o["id"])
        if not other_words: continue
        union = base_words | other_words
        score = len(base_words & other_words)/len(union) if union else 0
        if score >= threshold:
            scored.append({"chat_id": o["id"], "title": o["title"], "score": round(score,2)})
    scored.sort(key=lambda x: -x["score"])
    return scored[:max_results]

def build_ripple_modal_content(current_chat_id):
    flags = load_unacknowledged_ripples()
    label = f"🌊 Ripple ({len(flags)})" if flags else "🌊 Ripple"

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        titles = {r["id"]: r["title"] for r in conn.execute("SELECT id, title FROM chats").fetchall()}

    def clickable_title(chat_id):
        return html.Span(titles.get(chat_id, chat_id),
            id={"type":"sidebar-item","index":chat_id}, n_clicks=0,
            style={"fontWeight":"600","fontSize":"13px","color":"#1a73e8","cursor":"pointer","textDecoration":"underline"})

    sections = [html.Div("⚠️ Flags needing review", style={"fontWeight":"700","fontSize":"13px","marginBottom":"6px"})]
    if flags:
        for f in flags:
            sections.append(html.Div([
                html.Div(clickable_title(f["chat_id"])),
                html.Div(f["reason"], style={"fontSize":"12px","color":"#666","marginBottom":"6px"}),
                dbc.Button("✅ Mark Reviewed", id={"type":"ripple-ack-btn","index":f["id"]},
                           size="sm", color="light", outline=True, n_clicks=0)
            ], style={"borderBottom":"1px solid #eee","padding":"8px 0"}))
    else:
        sections.append(html.Div("No pending flags — everything's in sync.", style={"color":"#888","marginBottom":"12px","fontSize":"12px"}))

    if current_chat_id:
        linked = get_linked_chats(current_chat_id)
        sections.append(html.Div("🔗 Linked to this chat", style={"fontWeight":"700","fontSize":"13px","marginTop":"16px","marginBottom":"6px"}))
        if linked:
            for l in linked:
                badge_color = "#1a73e8" if l["relation"]=="meeting-sibling" else "#9c27b0"
                sections.append(html.Div([
                    clickable_title(l["chat_id_b"]),
                    html.Span(f" [{l['relation']}]", style={"fontSize":"10px","color":badge_color,"marginLeft":"6px"}),
                    dbc.Button("Unlink", id={"type":"ripple-unlink-btn","chat_a":current_chat_id,"chat_b":l["chat_id_b"]},
                               size="sm", color="light", outline=True, n_clicks=0, style={"marginLeft":"10px","fontSize":"10px"})
                ], style={"padding":"4px 0","display":"flex","alignItems":"center"}))
        else:
            sections.append(html.Div("No linked artifacts yet.", style={"color":"#888","fontSize":"12px"}))

        suggestions = find_related_chats(current_chat_id)
        if suggestions:
            sections.append(html.Div("🔎 Possibly related (not linked)", style={"fontWeight":"700","fontSize":"13px","marginTop":"16px","marginBottom":"6px"}))
            for s in suggestions:
                sections.append(html.Div([
                    clickable_title(s["chat_id"]),
                    html.Span(f" ({int(s['score']*100)}% overlap)", style={"fontSize":"11px","color":"#999","marginLeft":"6px"}),
                    dbc.Button("🔗 Link", id={"type":"ripple-link-btn","index":s["chat_id"]},
                               size="sm", color="info", outline=True, n_clicks=0, style={"marginLeft":"10px","fontSize":"10px"}),
                    dbc.Button("✕ Dismiss", id={"type":"ripple-dismiss-btn","index":s["chat_id"]},
                               size="sm", color="light", outline=True, n_clicks=0, style={"marginLeft":"4px","fontSize":"10px"}),
                ], style={"padding":"4px 0","display":"flex","alignItems":"center"}))

    return sections, label

def save_message(chat_id, speaker, content, mode="general"):
    now = datetime.now().isoformat()
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        cursor = conn.execute(
            "INSERT INTO messages (chat_id,role,content,mode,created_at) VALUES (?,?,?,?,?)",
            (chat_id, speaker, content, mode, now))
        conn.execute("UPDATE chats SET updated_at=? WHERE id=?", (now, chat_id))
        conn.commit()
        return cursor.lastrowid

def load_messages(chat_id):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            "SELECT id,role,content,mode,created_at FROM messages WHERE chat_id=? ORDER BY id ASC",
            (chat_id,)).fetchall()
    return [dict(r) for r in rows]

def load_chat_summary(chat_id):
    """Load the persisted rolling summary and its coverage marker."""
    if not chat_id:
        return "", 0

    with closing(sqlite3.connect(DB_PATH)) as conn:
        row = conn.execute(
            "SELECT chat_summary, summary_through_id FROM chats WHERE id=?",
            (chat_id,),
        ).fetchone()

    if not row:
        return "", 0

    return row[0] or "", int(row[1] or 0)


def save_chat_summary(chat_id, summary, through_id=0):
    """Persist the rolling conversation summary and coverage marker."""
    if not chat_id:
        return

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute(
            """
            UPDATE chats
            SET chat_summary=?,
                summary_through_id=?,
                updated_at=?
            WHERE id=?
            """,
            (
                summary or "",
                int(through_id or 0),
                datetime.now().isoformat(),
                chat_id,
            ),
        )
        conn.commit()


def clear_chat_summary(chat_id):
    """Clear persisted conversation memory for a chat."""
    save_chat_summary(chat_id, "", 0)


def maybe_refresh_chat_summary(chat_id, role, mode):
    """Compact older turns into durable context once a chat becomes long."""
    if not chat_id:
        return

    try:
        messages = load_messages(chat_id)

        if len(messages) <= 16:
            return

        summary, through_id = load_chat_summary(chat_id)

        eligible = [
            item for item in messages[:-12]
            if item.get("id", 0) > through_id
            and item["role"] in ("user", "assistant")
        ]

        if len(eligible) < 8:
            return

        transcript_parts = []
        for item in eligible:
            speaker = "User" if item["role"] == "user" else "Assistant"
            transcript_parts.append(
                f"{speaker}: {item['content']}"
            )

        previous = (
            f"Existing summary:\n{summary}\n\n"
            if summary
            else ""
        )

        prompt = (
            previous
            + "New conversation turns to incorporate:\n"
            + "\n\n".join(transcript_parts)
        )

        summary_messages = [
            {
                "role": "system",
                "content": (
                    "You maintain durable memory for a professional BA/PO/PM "
                    "conversation. Summarize the supplied conversation into "
                    "compact factual context for future turns. Preserve: "
                    "user goals, requirements, decisions, constraints, "
                    "stakeholders, terminology, numbers, unresolved questions, "
                    "preferences, and important corrections. Remove greetings, "
                    "repetition, and transient wording. Do not invent facts. "
                    "Use concise headings and bullets. This is internal context, "
                    "not a response to the user."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]

        new_summary, _, _ = AIProvider.call(
            summary_messages,
            max_tokens=2500,
        )

        last_id = max(item.get("id", 0) for item in eligible)

        save_chat_summary(
            chat_id,
            new_summary.strip(),
            last_id,
        )

    except Exception as exc:
        print(f"⚠️ Conversation summary refresh skipped: {exc}")


def load_chats(limit=50):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            "SELECT * FROM chats WHERE is_archived=0 ORDER BY is_pinned DESC, updated_at DESC LIMIT ?",
            (limit,)).fetchall()
    return [dict(r) for r in rows]

def rename_chat(chat_id, new_title):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("UPDATE chats SET title=? WHERE id=?", (new_title, chat_id))
        conn.commit()

def delete_chat(chat_id):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("PRAGMA foreign_keys = ON")
        conn.execute("DELETE FROM chats WHERE id=?", (chat_id,))
        conn.commit()

def update_chat_context(chat_id, role, mode, active_submode="text"):
    if not chat_id:
        return
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute(
            "UPDATE chats SET role=?, mode=?, active_submode=? WHERE id=?",
            (role, mode, active_submode, chat_id))
        conn.commit()

def auto_title(msg, n=40):
    t = msg.strip().split("\n")[0]
    return t[:n] + ("…" if len(t) > n else "")

# Project helpers
def load_projects():
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(
            "SELECT * FROM projects ORDER BY is_default DESC, name ASC").fetchall()]

def create_project(name):
    pid = str(uuid.uuid4())
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute(
            "INSERT INTO projects (id,name,created_at,is_default,is_collapsed) VALUES (?,?,?,?,0)",
            (pid, name.strip(), datetime.now().isoformat()))
        conn.commit()
    return pid

def rename_project(pid, new_name):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("UPDATE projects SET name=? WHERE id=? AND is_default=0",
                     (new_name.strip(), pid))
        conn.commit()

def delete_project(pid):
    dpid = _get_default_pid()
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("UPDATE chats SET project_id=? WHERE project_id=?", (dpid, pid))
        conn.execute("DELETE FROM projects WHERE id=? AND is_default=0", (pid,))
        conn.commit()

def load_chats_for_project(pid):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(
            "SELECT * FROM chats WHERE project_id=? AND is_archived=0 ORDER BY is_pinned DESC, updated_at DESC",
            (pid,)).fetchall()]

def assign_chat_project(chat_id, pid):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("UPDATE chats SET project_id=? WHERE id=?", (pid, chat_id))
        conn.commit()

def set_project_collapsed(project_id, collapsed):
    """Persist project expand/collapse state without changing chat membership."""
    if not project_id:
        return
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute(
            "UPDATE projects SET is_collapsed=? WHERE id=?",
            (1 if collapsed else 0, project_id),
        )
        conn.commit()

def set_all_projects_collapsed(collapsed):
    """Persist the global project collapse state."""
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute(
            "UPDATE projects SET is_collapsed=?",
            (1 if collapsed else 0,),
        )
        conn.commit()

# Archive helpers
def archive_chat(chat_id):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("UPDATE chats SET is_archived=1 WHERE id=?", (chat_id,))
        conn.commit()

def unarchive_chat(chat_id):
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("UPDATE chats SET is_archived=0 WHERE id=?", (chat_id,))
        conn.commit()

def load_archived_chats():
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(
            "SELECT * FROM chats WHERE is_archived=1 ORDER BY updated_at DESC LIMIT 100"
        ).fetchall()]

# ════════════════════════════════════════════════════════════════
# FILE EXTRACTION
# ════════════════════════════════════════════════════════════════
def extract_file_text(filename, content_b64):
    try:
        raw = content_b64.split(",")[1] if "," in content_b64 else content_b64
        bts = base64.b64decode(raw)
        ext = filename.lower().rsplit(".", 1)[-1] if "." in filename else ""

        if ext in ("txt","md","py","sql","json","html","css","js","xml","yaml","yml"):
            return bts.decode("utf-8", errors="replace")[:8000], f"📄 {filename}"
        if ext == "csv":
            df = pd.read_csv(io.BytesIO(bts))
            preview = f"CSV: {filename}\nRows:{len(df)}, Cols:{list(df.columns)}\n\n{df.head(20).to_string()}"
            return preview[:6000], f"📊 {filename} ({len(df)} rows)"
        if ext in ("xlsx","xls"):
            df = pd.read_excel(io.BytesIO(bts))
            preview = f"Excel:{filename}\nRows:{len(df)}, Cols:{list(df.columns)}\n\n{df.head(20).to_string()}"
            return preview[:6000], f"📊 {filename}"
        if ext == "pdf":
            if PYPDF_OK:
                reader = PdfReader(io.BytesIO(bts))
                text = "\n\n".join(p.extract_text() or "" for p in reader.pages[:15])
                return f"PDF:{filename} ({len(reader.pages)} pages)\n\n{text[:6000]}", f"📑 {filename}"
            return f"[PDF:{filename} — pip install pypdf]", f"📑 {filename}"
        if ext == "docx":
            d = Document(io.BytesIO(bts))
            text = "\n".join(p.text for p in d.paragraphs if p.text.strip())
            return f"DOCX:{filename}\n\n{text[:6000]}", f"📝 {filename}"
        if ext in ("png","jpg","jpeg","gif","webp","bmp"):
            return f"[Image file: {filename}]", f"🖼️ {filename}"
        return bts.decode("utf-8", errors="replace")[:4000], f"📎 {filename}"
    except Exception as e:
        return f"[Error reading {filename}: {e}]", f"⚠️ {filename}"

# ════════════════════════════════════════════════════════════════
# CENTRALIZED AI PROVIDER
# ════════════════════════════════════════════════════════════════
# ════════════════════════════════════════════════════════════════
# CENTRALIZED AI PROVIDER
# ════════════════════════════════════════════════════════════════

class AIProvider:
    """Centralized AI provider abstraction with resilient failover.

    Normal generation:
        Gemini
            ↓
        configured Groq model pool
            ↓
        OpenRouter text models
            ↓
        Ollama

    Structured generation:
        Gemini native JSON
            ↓
        configured Groq JSON model pool
            ↓
        OpenRouter structured-capable text models
            ↓
        Generic provider JSON extraction
            ↓
        Ollama JSON

    Model IDs are centralized in the configuration section so provider
    behavior can be updated without changing the AIProvider architecture.
    """

    OPENROUTER_TIMEOUT = (10, 75)
    OPENROUTER_DISCOVERY_TIMEOUT = (10, 20)
    OLLAMA_TIMEOUT = (10, 120)
    GROQ_TIMEOUT = 90

    MODEL_FAILURE_COOLDOWN_SECONDS = 300

    _gemini_failed_models = {}
    _groq_failed_models = {}
    _openrouter_failed_models = {}

    _openrouter_model_cache = {
        "expires_at": 0.0,
        "models": [],
    }

    # ------------------------------------------------------------------
    # General helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _is_configured(value):
        """Return whether a provider configuration value is usable."""

        if value is None:
            return False

        value = str(value).strip()

        return value not in {
            "",
            "api",
            "API",
            "OPENROUTER_API_KEY",
            "GROQ_API_KEY",
            "GEMINI_API_KEY",
        }

    @staticmethod
    def _safe_error(exc, limit=350):
        """Return a compact single-line error message."""

        message = str(exc or "")
        message = re.sub(r"\s+", " ", message).strip()

        if len(message) > limit:
            return message[:limit] + "..."

        return message

    @staticmethod
    def _strip_markdown_code_fence(text):
        """Remove a surrounding Markdown code fence."""

        if not text:
            return ""

        value = str(text).strip()

        if value.startswith("```"):
            value = re.sub(
                r"^```(?:json|JSON)?\s*",
                "",
                value,
                count=1,
            )

            value = re.sub(
                r"\s*```$",
                "",
                value,
                count=1,
            )

        return value.strip()

    # ------------------------------------------------------------------
    # JSON parsing
    # ------------------------------------------------------------------

    @classmethod
    def _extract_json_object(cls, text):
        """Extract one valid JSON object from model output."""

        if not text or not isinstance(text, str):
            raise ValueError(
                "Empty structured AI response."
            )

        value = cls._strip_markdown_code_fence(text)

        try:
            parsed = json.loads(value)

            if isinstance(parsed, dict):
                return parsed

        except (TypeError, ValueError):
            pass

        start = value.find("{")

        if start < 0:
            raise ValueError(
                "No JSON object found in AI response."
            )

        depth = 0
        in_string = False
        escaped = False

        for index in range(start, len(value)):
            char = value[index]

            if in_string:
                if escaped:
                    escaped = False
                elif char == "\\":
                    escaped = True
                elif char == '"':
                    in_string = False

                continue

            if char == '"':
                in_string = True

            elif char == "{":
                depth += 1

            elif char == "}":
                depth -= 1

                if depth == 0:
                    candidate = value[
                        start:index + 1
                    ]

                    try:
                        parsed = json.loads(
                            candidate
                        )
                    except (TypeError, ValueError):
                        continue

                    if isinstance(parsed, dict):
                        return parsed

        raise ValueError(
            "AI response contained an incomplete "
            "or malformed JSON object."
        )

    @classmethod
    def _normalize_json_response(cls, text):
        """Return canonical JSON text after extraction."""

        parsed = cls._extract_json_object(text)

        return json.dumps(
            parsed,
            ensure_ascii=False,
            indent=2,
        )

    # ------------------------------------------------------------------
    # Model cooldown
    # ------------------------------------------------------------------

    @classmethod
    def _mark_model_failed(
        cls,
        provider,
        model_id,
    ):
        """Temporarily remove a failing model from rotation."""

        expires_at = (
            time.time()
            + cls.MODEL_FAILURE_COOLDOWN_SECONDS
        )

        if provider == "gemini":
            cls._gemini_failed_models[
                model_id
            ] = expires_at

        elif provider == "groq":
            cls._groq_failed_models[
                model_id
            ] = expires_at

        elif provider == "openrouter":
            cls._openrouter_failed_models[
                model_id
            ] = expires_at

    @classmethod
    def _is_model_temporarily_failed(
        cls,
        provider,
        model_id,
    ):
        """Return whether a model is currently cooling down."""

        if provider == "gemini":
            failures = cls._gemini_failed_models
        elif provider == "groq":
            failures = cls._groq_failed_models
        else:
            failures = cls._openrouter_failed_models

        expires_at = failures.get(model_id)

        if not expires_at:
            return False

        if expires_at <= time.time():
            failures.pop(model_id, None)
            return False

        return True

    @classmethod
    def _clear_model_failure(
        cls,
        provider,
        model_id,
    ):
        """Remove a model from the failure cooldown."""

        if provider == "gemini":
            failures = cls._gemini_failed_models
        elif provider == "groq":
            failures = cls._groq_failed_models
        else:
            failures = cls._openrouter_failed_models

        failures.pop(model_id, None)

    # ------------------------------------------------------------------
    # Gemini
    # ------------------------------------------------------------------

    @classmethod
    def _configured_gemini_models(cls):
        """Return the configured Gemini model pool in priority order."""

        models = globals().get("GEMINI_PREFERRED_MODELS", ())

        if isinstance(models, str):
            models = (item.strip() for item in models.split(","))

        normalized = []
        for model in models or ():
            model_id = str(model).strip()
            if model_id and model_id not in normalized:
                normalized.append(model_id)

        if not normalized:
            raise RuntimeError(
                "GEMINI_PREFERRED_MODELS is empty or missing."
            )

        return tuple(normalized)

    @classmethod
    def _gemini(cls, messages, max_tokens, model_id):
        """Generate normal text through one Gemini model."""

        if not gemini_client:
            raise RuntimeError("Gemini not configured.")

        system_text = next(
            (str(message["content"]) for message in messages
             if message["role"] == "system"),
            "",
        )

        chat = [
            message for message in messages
            if message["role"] in ("user", "assistant")
        ]

        if not chat:
            raise RuntimeError("No user message supplied to Gemini.")

        contents = [
            types.Content(
                role=("user" if message["role"] == "user" else "model"),
                parts=[types.Part(text=str(message["content"]))],
            )
            for message in chat
        ]

        try:
            response = gemini_client.models.generate_content(
                model=model_id,
                config=types.GenerateContentConfig(
                    system_instruction=system_text,
                    max_output_tokens=max_tokens,
                ),
                contents=contents,
            )
        except Exception as exc:
            raise RuntimeError(
                f"Gemini {model_id} request failed: {cls._safe_error(exc)}"
            ) from exc

        content = response.text or ""
        if not content:
            raise RuntimeError(
                f"Gemini {model_id} returned an empty response."
            )

        return content

    @classmethod
    def _gemini_fallback(cls, messages, max_tokens):
        """Try Gemini models sequentially with per-model cooldown."""

        candidates = cls._configured_gemini_models()
        errors = []
        attempted = 0

        for model_id in candidates:
            if cls._is_model_temporarily_failed("gemini", model_id):
                errors.append(f"{model_id}: temporarily cooling down")
                continue

            attempted += 1
            try:
                reply = cls._gemini(messages, max_tokens, model_id)
                cls._clear_model_failure("gemini", model_id)
                return reply, model_id
            except Exception as exc:
                cls._mark_model_failed("gemini", model_id)
                errors.append(f"{model_id}: {cls._safe_error(exc)}")

        if attempted == 0:
            raise RuntimeError(
                "All configured Gemini models are currently in cooldown: "
                + " | ".join(errors)
            )

        raise RuntimeError(
            "Configured Gemini models exhausted: " + " | ".join(errors)
        )

    # ------------------------------------------------------------------
    # Groq configuration
    # ------------------------------------------------------------------

    @classmethod
    def _configured_groq_models(cls):
        """Return the configured normal Groq model pool."""

        models = globals().get(
            "GROQ_PREFERRED_MODELS",
            (),
        )

        if isinstance(models, str):
            models = (
                item.strip()
                for item in models.split(",")
            )

        normalized = []

        for model in models or ():
            model_id = str(model).strip()

            if (
                model_id
                and model_id not in normalized
            ):
                normalized.append(model_id)

        if not normalized:
            raise RuntimeError(
                "GROQ_PREFERRED_MODELS is empty or missing."
            )

        return tuple(normalized)

    @classmethod
    def _configured_groq_json_models(cls):
        """Return the dedicated structured-output Groq pool."""

        configured = globals().get(
            "GROQ_JSON_MODELS",
            (),
        )

        preferred = globals().get(
            "GROQ_PREFERRED_MODELS",
            (),
        )

        if isinstance(configured, str):
            configured = (
                item.strip()
                for item in configured.split(",")
            )

        if isinstance(preferred, str):
            preferred = (
                item.strip()
                for item in preferred.split(",")
            )

        configured_ids = [
            str(item).strip()
            for item in configured or ()
            if str(item).strip()
        ]

        preferred_ids = [
            str(item).strip()
            for item in preferred or ()
            if str(item).strip()
        ]

        normalized = []

        for model_id in configured_ids + preferred_ids:
            if model_id not in normalized:
                normalized.append(model_id)

        if not normalized:
            raise RuntimeError(
                "No Groq structured-output models are configured."
            )

        return tuple(normalized)

    @classmethod
    def _groq_candidates(cls):
        """Return available normal Groq models."""

        if not groq_client:
            return []

        try:
            models = cls._configured_groq_models()
        except RuntimeError:
            return []

        return [
            model_id
            for model_id in models
            if not cls._is_model_temporarily_failed(
                "groq",
                model_id,
            )
        ]

    @classmethod
    def _groq_json_candidates(cls):
        """Return available Groq models suitable for JSON work."""

        if not groq_client:
            return []

        try:
            models = cls._configured_groq_json_models()
        except RuntimeError:
            return []

        return [
            model_id
            for model_id in models
            if not cls._is_model_temporarily_failed(
                "groq",
                model_id,
            )
        ]

    # ------------------------------------------------------------------
    # Groq
    # ------------------------------------------------------------------

    @classmethod
    def _groq(
        cls,
        messages,
        max_tokens,
        model,
        json_mode=False,
    ):
        """Call one Groq model."""

        if not groq_client:
            raise RuntimeError(
                "Groq not configured."
            )

        request_kwargs = {
            "model": model,
            "messages": [
                {
                    "role": message["role"],
                    "content": str(
                        message["content"]
                    ),
                }
                for message in messages
                if message["role"]
                in (
                    "system",
                    "user",
                    "assistant",
                )
            ],
            "max_tokens": min(
                max_tokens,
                32768,
            ),
        }

        if json_mode:
            request_kwargs["response_format"] = {
                "type": "json_object",
            }

        response = (
            groq_client
            .chat
            .completions
            .create(
                **request_kwargs
            )
        )

        choices = response.choices or []

        if not choices:
            raise RuntimeError(
                f"Groq {model} returned no choices."
            )

        choice = choices[0]

        content = (
            choice.message.content
            or ""
        )

        if not content:
            raise RuntimeError(
                f"Groq {model} returned empty content."
            )

        return content

    @classmethod
    def _groq_fallback(
        cls,
        messages,
        max_tokens,
    ):
        """Try configured Groq models sequentially."""

        candidates = cls._groq_candidates()

        if not candidates:
            raise RuntimeError(
                "No configured Groq models are currently available."
            )

        errors = []

        for model_id in candidates:
            try:
                reply = cls._groq(
                    messages,
                    max_tokens,
                    model_id,
                )

                cls._clear_model_failure(
                    "groq",
                    model_id,
                )

                return (
                    reply,
                    model_id,
                )

            except Exception as exc:
                cls._mark_model_failed(
                    "groq",
                    model_id,
                )

                errors.append(
                    f"{model_id}: "
                    f"{cls._safe_error(exc)}"
                )

        raise RuntimeError(
            "Configured Groq models exhausted: "
            + " | ".join(errors)
        )

    # ------------------------------------------------------------------
    # OpenRouter
    # ------------------------------------------------------------------

    @classmethod
    def _openrouter_headers(
        cls,
        title="BA & PO Copilot",
    ):
        """Build OpenRouter request headers."""

        if not cls._is_configured(
            OPENROUTER_API_KEY
        ):
            raise RuntimeError(
                "OpenRouter not configured."
            )

        return {
            "Authorization": (
                f"Bearer {OPENROUTER_API_KEY}"
            ),
            "Content-Type": "application/json",
            "X-Title": title,
        }

    @classmethod
    def _openrouter(
        cls,
        messages,
        max_tokens,
        model,
        json_mode=False,
    ):
        """Call one OpenRouter model."""

        payload = {
            "model": model,
            "messages": messages,
            "max_tokens": max_tokens,
        }

        if json_mode:
            payload["response_format"] = {
                "type": "json_object",
            }

        response = requests.post(
            f"{OPENROUTER_API_BASE}/chat/completions",
            headers=cls._openrouter_headers(),
            json=payload,
            timeout=cls.OPENROUTER_TIMEOUT,
        )

        if response.status_code >= 300:
            raise RuntimeError(
                f"OpenRouter {model} error "
                f"({response.status_code}): "
                f"{response.text[:450]}"
            )

        try:
            data = response.json()
        except ValueError as exc:
            raise RuntimeError(
                f"OpenRouter {model} returned invalid HTTP JSON."
            ) from exc

        error_data = data.get("error")

        if error_data:
            if isinstance(
                error_data,
                dict,
            ):
                error_message = (
                    error_data.get("message")
                    or error_data.get("code")
                    or str(error_data)
                )
            else:
                error_message = str(
                    error_data
                )

            raise RuntimeError(
                f"OpenRouter {model}: "
                f"{error_message}"
            )

        choices = data.get("choices") or []

        if not choices:
            raise RuntimeError(
                f"OpenRouter {model} returned no choices."
            )

        message = (
            choices[0].get("message")
            or {}
        )

        content = message.get("content")

        if isinstance(content, list):
            content = "".join(
                str(part.get("text", ""))
                for part in content
                if isinstance(part, dict)
            )

        if not content:
            raise RuntimeError(
                f"OpenRouter {model} returned empty content."
            )

        return str(content)

    @classmethod
    def _discover_openrouter_free_models(cls):
        """Discover free OpenRouter text-generation models."""

        now = time.time()
        cache = cls._openrouter_model_cache

        if cache["expires_at"] > now:
            return list(
                cache["models"]
            )

        response = requests.get(
            f"{OPENROUTER_API_BASE}/models",
            headers=cls._openrouter_headers(
                "BA & PO Copilot · Model Discovery"
            ),
            timeout=cls.OPENROUTER_DISCOVERY_TIMEOUT,
        )

        if response.status_code >= 300:
            raise RuntimeError(
                "OpenRouter model discovery failed "
                f"({response.status_code}): "
                f"{response.text[:300]}"
            )

        try:
            data = response.json()
        except ValueError as exc:
            raise RuntimeError(
                "OpenRouter model discovery returned invalid JSON."
            ) from exc

        models = data.get("data") or []

        candidates = []

        blocked_tokens = (
            "embedding",
            "embed",
            "rerank",
            "moderation",
            "prompt-guard",
            "guard",
            "whisper",
            "tts",
            "speech",
            "audio-only",
            "image-only",
            "video-only",
            "lyria",
        )

        for model in models:
            if not isinstance(model, dict):
                continue

            model_id = str(
                model.get("id") or ""
            ).strip()

            if not model_id:
                continue

            if model_id == "openrouter/free":
                continue

            if model.get("deprecation"):
                continue

            lowered = model_id.lower()

            if any(
                token in lowered
                for token in blocked_tokens
            ):
                continue

            pricing = model.get("pricing") or {}

            try:
                prompt_price = float(
                    pricing.get("prompt") or 1
                )
                completion_price = float(
                    pricing.get("completion") or 1
                )
            except (
                TypeError,
                ValueError,
            ):
                continue

            if (
                prompt_price != 0.0
                or completion_price != 0.0
            ):
                continue

            architecture = (
                model.get("architecture")
                or {}
            )

            input_modalities = (
                architecture.get(
                    "input_modalities"
                )
                or []
            )

            output_modalities = (
                architecture.get(
                    "output_modalities"
                )
                or []
            )

            if (
                "text" not in input_modalities
                or "text" not in output_modalities
            ):
                continue

            try:
                context_length = int(
                    model.get(
                        "context_length"
                    )
                    or 0
                )
            except (
                TypeError,
                ValueError,
            ):
                context_length = 0

            candidates.append(
                {
                    "id": model_id,
                    "context_length": context_length,
                }
            )

        preferred = [
            str(model).strip()
            for model in OPENROUTER_PREFERRED_MODELS
            if str(model).strip()
            and str(model).strip() != "openrouter/free"
        ]

        preferred_ids = set(preferred)

        candidates.sort(
            key=lambda item: (
                item["id"] not in preferred_ids,
                -item["context_length"],
                item["id"],
            )
        )

        discovered = [
            item["id"]
            for item in candidates
            if item["id"] not in preferred_ids
        ]

        cache["models"] = discovered
        cache["expires_at"] = (
            now
            + OPENROUTER_MODEL_CACHE_SECONDS
        )

        return list(discovered)

    @classmethod
    def _openrouter_candidates(
        cls,
        structured=False,
    ):
        """Build safe OpenRouter candidates."""

        candidates = []

        preferred = [
            str(model).strip()
            for model in OPENROUTER_PREFERRED_MODELS
            if str(model).strip()
        ]

        for model_id in preferred:
            lowered = model_id.lower()

            if structured and model_id == "openrouter/free":
                continue

            if structured and any(
                token in lowered
                for token in (
                    "lyria",
                    "audio",
                    "tts",
                    "speech",
                    "whisper",
                    "image",
                    "video",
                    "embedding",
                    "rerank",
                    "moderation",
                    "guard",
                )
            ):
                continue

            if model_id in candidates:
                continue

            if cls._is_model_temporarily_failed(
                "openrouter",
                model_id,
            ):
                continue

            candidates.append(model_id)

        try:
            discovered = (
                cls._discover_openrouter_free_models()
            )
        except Exception:
            discovered = []

        for model_id in discovered:
            model_id = str(model_id).strip()

            if not model_id:
                continue

            if model_id in candidates:
                continue

            if cls._is_model_temporarily_failed(
                "openrouter",
                model_id,
            ):
                continue

            candidates.append(model_id)

            if len(candidates) >= (
                len(preferred)
                + OPENROUTER_MAX_DISCOVERED_FALLBACKS
            ):
                break

        return candidates

    @classmethod
    def _openrouter_fallback(
        cls,
        messages,
        max_tokens,
    ):
        """Try normal OpenRouter models sequentially."""

        candidates = cls._openrouter_candidates(
            structured=False,
        )

        if not candidates:
            raise RuntimeError(
                "No OpenRouter models are currently available."
            )

        errors = []

        for model_id in candidates:
            try:
                reply = cls._openrouter(
                    messages,
                    max_tokens,
                    model_id,
                    json_mode=False,
                )

                cls._clear_model_failure(
                    "openrouter",
                    model_id,
                )

                return (
                    reply,
                    model_id,
                )

            except Exception as exc:
                cls._mark_model_failed(
                    "openrouter",
                    model_id,
                )

                errors.append(
                    f"{model_id}: "
                    f"{cls._safe_error(exc)}"
                )

        raise RuntimeError(
            "OpenRouter models exhausted: "
            + " | ".join(errors)
        )

    # ------------------------------------------------------------------
    # Ollama
    # ------------------------------------------------------------------

    @classmethod
    def _ollama(
        cls,
        messages,
        max_tokens,
        model=None,
    ):
        """Generate text through local Ollama."""

        model = model or OLLAMA_QWEN_MODEL

        system_text = next(
            (
                str(message["content"])
                for message in messages
                if message["role"] == "system"
            ),
            "",
        )

        chat_messages = [
            {
                "role": message["role"],
                "content": str(
                    message["content"]
                ),
            }
            for message in messages
            if message["role"]
            in (
                "user",
                "assistant",
            )
        ]

        payload_messages = []

        if system_text:
            payload_messages.append(
                {
                    "role": "system",
                    "content": system_text,
                }
            )

        payload_messages.extend(
            chat_messages
        )

        response = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": model,
                "messages": payload_messages,
                "stream": False,
                "options": {
                    "num_predict": max_tokens,
                },
            },
            timeout=cls.OLLAMA_TIMEOUT,
        )

        if response.status_code >= 300:
            raise RuntimeError(
                f"Ollama {model} error "
                f"({response.status_code}): "
                f"{response.text[:300]}"
            )

        try:
            data = response.json()
        except ValueError as exc:
            raise RuntimeError(
                "Ollama returned invalid JSON."
            ) from exc

        message = data.get("message") or {}
        content = message.get("content")

        if not content:
            raise RuntimeError(
                f"Ollama {model} returned empty content."
            )

        return str(content)

    # ------------------------------------------------------------------
    # Provider chain
    # ------------------------------------------------------------------

    @classmethod
    def _provider_chain(cls):
        """Build the normal generation provider chain."""

        providers = []

        if gemini_client:
            providers.append((
                "gemini",
                "Gemini",
                lambda messages, tokens: cls._gemini_fallback(
                    messages, tokens
                ),
            ))

        if groq_client:
            providers.append((
                "groq",
                "Groq",
                lambda messages, tokens: cls._groq_fallback(
                    messages, tokens
                )[0],
            ))

        if cls._is_configured(OPENROUTER_API_KEY):
            providers.append((
                "openrouter",
                "OpenRouter",
                lambda messages, tokens: cls._openrouter_fallback(
                    messages, tokens
                ),
            ))

        providers.append((
            "qwen",
            "Qwen3.8 27B",
            lambda messages, tokens: cls._ollama(messages, tokens),
        ))

        return providers

    @classmethod
    def call(cls, messages, max_tokens=20480, bpmn=False):
        """Call normal-generation providers with automatic failover."""

        providers = cls._provider_chain()
        if not providers:
            raise RuntimeError("No AI providers are configured.")

        primary_key = providers[0][0]
        errors = []

        for key, name, function in providers:
            try:
                result = function(messages, max_tokens)

                if key == "gemini":
                    reply, model_id = result
                    display_name = f"Gemini · {model_id}"
                    configured = cls._configured_gemini_models()
                    if configured and model_id == configured[0]:
                        banner = None
                    else:
                        banner = (
                            "⚡ Switched to "
                            f"{display_name} — "
                            "previous Gemini model was unavailable or quota-limited"
                        )

                elif key == "groq":
                    reply = result
                    display_name = "Groq"
                    banner = (
                        None if key == primary_key else
                        "⚡ Switched to Groq — previous AI provider was unavailable or quota-limited"
                    )

                elif key == "openrouter":
                    reply, model_id = result
                    display_name = f"OpenRouter · {model_id}"
                    banner = (
                        None if key == primary_key else
                        "⚡ Switched to " + display_name +
                        " — previous AI provider was unavailable or quota-limited"
                    )

                else:
                    reply = result
                    display_name = name
                    banner = (
                        None if key == primary_key else
                        "⚡ Switched to " + display_name +
                        " — previous AI providers were unavailable or quota-limited"
                    )

                if not reply:
                    raise RuntimeError(
                        f"{display_name} returned empty content."
                    )

                return reply, display_name, banner

            except Exception as exc:
                errors.append(
                    f"{name}: {cls._safe_error(exc)}"
                )

        raise RuntimeError(
            "❌ All AI providers exhausted:\n" + "\n".join(errors)
        )

    # ------------------------------------------------------------------
    # Structured JSON
    # ------------------------------------------------------------------

    @classmethod
    def _gemini_json(cls, messages, max_tokens, model_id):
        """Generate JSON using Gemini native JSON mode for one model."""

        if not gemini_client:
            raise RuntimeError("Gemini not configured.")

        system_text = next(
            (str(message["content"]) for message in messages
             if message["role"] == "system"),
            "",
        )

        chat = [
            message for message in messages
            if message["role"] in ("user", "assistant")
        ]

        if not chat:
            raise RuntimeError("No user message supplied.")

        contents = [
            types.Content(
                role=("user" if message["role"] == "user" else "model"),
                parts=[types.Part(text=str(message["content"]))],
            )
            for message in chat
        ]

        try:
            response = gemini_client.models.generate_content(
                model=model_id,
                config=types.GenerateContentConfig(
                    system_instruction=system_text,
                    max_output_tokens=max_tokens,
                    response_mime_type="application/json",
                ),
                contents=contents,
            )
        except Exception as exc:
            raise RuntimeError(
                f"Gemini {model_id} JSON request failed: {cls._safe_error(exc)}"
            ) from exc

        content = response.text or ""
        if not content:
            raise RuntimeError(
                f"Gemini {model_id} returned empty structured output."
            )

        return cls._normalize_json_response(content)

    @classmethod
    def _gemini_json_fallback(cls, messages, max_tokens):
        """Try Gemini models sequentially for structured JSON."""

        candidates = cls._configured_gemini_models()
        errors = []
        attempted = 0

        for model_id in candidates:
            if cls._is_model_temporarily_failed("gemini", model_id):
                errors.append(f"{model_id}: temporarily cooling down")
                continue

            attempted += 1
            try:
                raw = cls._gemini_json(
                    messages, max_tokens, model_id
                )
                cls._clear_model_failure("gemini", model_id)

                banner = None
                if model_id != candidates[0]:
                    banner = (
                        "⚡ Switched to "
                        f"Gemini · {model_id} — "
                        "previous Gemini model was unavailable or quota-limited"
                    )

                return raw, f"Gemini · {model_id}", banner

            except Exception as exc:
                cls._mark_model_failed("gemini", model_id)
                errors.append(f"{model_id}: {cls._safe_error(exc)}")

        if attempted == 0:
            raise RuntimeError(
                "All configured Gemini JSON models are currently in cooldown: "
                + " | ".join(errors)
            )

        raise RuntimeError(
            "Gemini JSON models exhausted: " + " | ".join(errors)
        )

    @classmethod
    def _make_json_prompt_messages(
        cls,
        messages,
    ):
        """Add strict JSON instructions without mutating input."""

        copied = [
            {
                "role": message["role"],
                "content": str(
                    message["content"]
                ),
            }
            for message in messages
        ]

        instruction = (
            "\n\nIMPORTANT OUTPUT CONTRACT:\n"
            "Return ONLY one valid JSON object.\n"
            "Do not return Markdown.\n"
            "Do not use code fences.\n"
            "Do not provide reasoning.\n"
            "Do not provide a preamble.\n"
            "Do not provide explanations before or after JSON.\n"
            "The first character must be '{'.\n"
            "The final character must be '}'.\n"
            "All strings must be valid JSON strings."
        )

        for index in range(
            len(copied) - 1,
            -1,
            -1,
        ):
            if copied[index]["role"] == "user":
                copied[index]["content"] += instruction
                return copied

        copied.append(
            {
                "role": "user",
                "content": (
                    "Return ONLY one valid JSON object."
                    + instruction
                ),
            }
        )

        return copied

    @classmethod
    def _groq_json_fallback(
        cls,
        messages,
        max_tokens,
    ):
        """Generate structured JSON through configured Groq models."""

        candidates = cls._groq_json_candidates()

        if not candidates:
            raise RuntimeError(
                "No configured Groq JSON models are currently available."
            )

        strict_messages = (
            cls._make_json_prompt_messages(
                messages
            )
        )

        errors = []

        for model_id in candidates:
            # Native JSON.
            try:
                raw = cls._groq(
                    messages,
                    max_tokens,
                    model_id,
                    json_mode=True,
                )

                normalized = (
                    cls._normalize_json_response(
                        raw
                    )
                )

                cls._clear_model_failure(
                    "groq",
                    model_id,
                )

                return (
                    normalized,
                    f"Groq · {model_id}",
                    None,
                )

            except Exception as native_exc:
                errors.append(
                    f"{model_id} native JSON: "
                    f"{cls._safe_error(native_exc)}"
                )

            # Strict prompt fallback.
            try:
                raw = cls._groq(
                    strict_messages,
                    max_tokens,
                    model_id,
                    json_mode=False,
                )

                normalized = (
                    cls._normalize_json_response(
                        raw
                    )
                )

                cls._clear_model_failure(
                    "groq",
                    model_id,
                )

                return (
                    normalized,
                    f"Groq · {model_id}",
                    None,
                )

            except Exception as prompt_exc:
                errors.append(
                    f"{model_id} strict JSON: "
                    f"{cls._safe_error(prompt_exc)}"
                )

                cls._mark_model_failed(
                    "groq",
                    model_id,
                )

        raise RuntimeError(
            "Configured Groq JSON models exhausted: "
            + " | ".join(errors)
        )

    @classmethod
    def _openrouter_json_fallback(
        cls,
        messages,
        max_tokens,
    ):
        """Generate structured JSON through safe OpenRouter models."""

        if not cls._is_configured(
            OPENROUTER_API_KEY
        ):
            raise RuntimeError(
                "OpenRouter not configured."
            )

        candidates = cls._openrouter_candidates(
            structured=True,
        )

        if not candidates:
            raise RuntimeError(
                "No suitable OpenRouter structured-output "
                "models are currently available."
            )

        strict_messages = (
            cls._make_json_prompt_messages(
                messages
            )
        )

        errors = []

        for model_id in candidates:
            # Native JSON.
            try:
                raw = cls._openrouter(
                    messages,
                    max_tokens,
                    model_id,
                    json_mode=True,
                )

                normalized = (
                    cls._normalize_json_response(
                        raw
                    )
                )

                cls._clear_model_failure(
                    "openrouter",
                    model_id,
                )

                return (
                    normalized,
                    f"OpenRouter · {model_id}",
                    None,
                )

            except Exception as native_exc:
                errors.append(
                    f"{model_id} native JSON: "
                    f"{cls._safe_error(native_exc)}"
                )

            # Strict prompt without response_format.
            try:
                raw = cls._openrouter(
                    strict_messages,
                    max_tokens,
                    model_id,
                    json_mode=False,
                )

                normalized = (
                    cls._normalize_json_response(
                        raw
                    )
                )

                cls._clear_model_failure(
                    "openrouter",
                    model_id,
                )

                return (
                    normalized,
                    f"OpenRouter · {model_id}",
                    None,
                )

            except Exception as prompt_exc:
                errors.append(
                    f"{model_id} strict JSON: "
                    f"{cls._safe_error(prompt_exc)}"
                )

                cls._mark_model_failed(
                    "openrouter",
                    model_id,
                )

        raise RuntimeError(
            "OpenRouter JSON models exhausted: "
            + " | ".join(errors)
        )

    @classmethod
    def _generic_json_fallback(
        cls,
        messages,
        max_tokens,
    ):
        """Try normal providers and extract JSON."""

        errors = []

        try:
            raw, provider_name, banner = cls.call(
                messages,
                max_tokens=max_tokens,
            )

            normalized = (
                cls._normalize_json_response(
                    raw
                )
            )

            return (
                normalized,
                provider_name,
                banner,
            )

        except Exception as exc:
            errors.append(
                f"generic provider chain: "
                f"{cls._safe_error(exc)}"
            )

        try:
            raw = cls._ollama(
                cls._make_json_prompt_messages(
                    messages
                ),
                max_tokens,
            )

            normalized = (
                cls._normalize_json_response(
                    raw
                )
            )

            return (
                normalized,
                "Qwen3.8 27B",
                "⚡ Switched to Qwen3.8 27B — "
                "structured cloud providers unavailable",
            )

        except Exception as exc:
            errors.append(
                f"Ollama JSON: "
                f"{cls._safe_error(exc)}"
            )

        raise RuntimeError(
            "Generic JSON fallback exhausted: "
            + " | ".join(errors)
        )

    @classmethod
    def call_json(
        cls,
        messages,
        max_tokens=12000,
    ):
        """Generate valid JSON through the structured failover chain."""

        errors = []

        if gemini_client:
            try:
                return cls._gemini_json_fallback(
                    messages,
                    max_tokens,
                )

            except Exception as exc:
                errors.append(
                    "Gemini JSON: "
                    + cls._safe_error(exc)
                )

        if groq_client:
            try:
                return cls._groq_json_fallback(
                    messages,
                    max_tokens,
                )

            except Exception as exc:
                errors.append(
                    "Groq JSON: "
                    + cls._safe_error(exc)
                )

        if cls._is_configured(
            OPENROUTER_API_KEY
        ):
            try:
                return cls._openrouter_json_fallback(
                    messages,
                    max_tokens,
                )

            except Exception as exc:
                errors.append(
                    "OpenRouter JSON: "
                    + cls._safe_error(exc)
                )

        try:
            return cls._generic_json_fallback(
                messages,
                max_tokens,
            )

        except Exception as exc:
            errors.append(
                "Generic JSON fallback: "
                + cls._safe_error(exc)
            )

        raise RuntimeError(
            "❌ All structured AI providers exhausted:\n"
            + "\n".join(errors)
        )

    # ------------------------------------------------------------------
    # Chat construction
    # ------------------------------------------------------------------

    @classmethod
    def build_messages_and_call(
        cls,
        message,
        role,
        mode,
        file_context="",
        chat_id=None,
    ):
        """Build a context-aware request and call the provider chain."""

        if role == "ba":
            modes = BA_MODES
            system_prompt = BA_SYSTEM

        elif role == "pm":
            modes = PM_MODES
            system_prompt = PM_SYSTEM

        else:
            modes = PO_MODES
            system_prompt = PO_SYSTEM

        mode_prompt = modes.get(
            mode,
            "",
        )

        full_system = system_prompt

        if mode_prompt:
            full_system += (
                "\n\n"
                + mode_prompt
            )

        if (
            role == "ba"
            and mode == "brd"
        ):
            document_type = (
                resolve_requirements_document_type(
                    chat_id,
                    message,
                )
            )

            full_system += (
                "\n\n"
                + build_document_type_instruction(
                    document_type
                )
            )

        messages = [
            {
                "role": "system",
                "content": full_system,
            }
        ]

        if chat_id:
            conversation = load_messages(
                chat_id
            )

            summary, _ = load_chat_summary(
                chat_id
            )

            if summary:
                messages.append(
                    {
                        "role": "system",
                        "content": (
                            "Persisted conversation summary "
                            "from earlier turns. Use it as "
                            "context, but prefer the latest "
                            "messages when they conflict.\n\n"
                            + summary
                        ),
                    }
                )

            recent_messages = conversation[-12:]

            if (
                recent_messages
                and recent_messages[-1]["role"]
                == "user"
            ):
                recent_messages = (
                    recent_messages[:-1]
                )

            for item in recent_messages:
                if item["role"] in (
                    "user",
                    "assistant",
                ):
                    messages.append(
                        {
                            "role": item["role"],
                            "content": item["content"],
                        }
                    )

        if file_context:
            user_content = (
                "[Attached File]\n"
                + file_context
                + "\n\n---\n"
                + str(message)
            )
        else:
            user_content = str(message)

        messages.append(
            {
                "role": "user",
                "content": user_content,
            }
        )

        return cls.call(
            messages,
            max_tokens=20480,
        )

    # ------------------------------------------------------------------
    # BPMN
    # ------------------------------------------------------------------

    @classmethod
    def bpmn(cls, message):
        """Generate BPMN JSON through structured failover."""

        messages = [
            {
                "role": "system",
                "content": BPMN_SCHEMA_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    str(message)
                    + "\n\n"
                    "IMPORTANT: Return ONLY one valid JSON object. "
                    "Do not return Markdown. "
                    "Do not return reasoning. "
                    "Do not return explanations. "
                    "The response must start with '{' "
                    "and end with '}'."
                ),
            },
        ]

        raw, ai_name, banner = cls.call_json(
            messages,
            max_tokens=20480,
        )

        normalized = cls._normalize_json_response(
            raw
        )

        return (
            normalized,
            ai_name,
            banner,
        )
        
# ════════════════════════════════════════════════════════════════
# JIRA CLIENT (Cloud REST API v3)
# ════════════════════════════════════════════════════════════════
class JiraClient:
    def __init__(self, base_url, email, api_token, project_key):
        self.base_url = base_url.rstrip("/")
        self.auth     = HTTPBasicAuth(email, api_token)
        self.project_key = project_key
        self.headers  = {"Content-Type": "application/json"}

    def _adf(self, text):
        content, bullets = [], []
        def flush_bullets():
            if bullets:
                content.append({"type":"bulletList","content":[
                    {"type":"listItem","content":[{"type":"paragraph",
                        "content":[{"type":"text","text":b}]}]} for b in bullets]})
                bullets.clear()
        for line in (text or "").split("\n"):
            line = line.strip()
            if not line:
                continue
            if line.startswith(("- ", "* ", "▸ ")):
                bullets.append(line.lstrip("-*▸ ").strip())
                continue
            flush_bullets()
            content.append({"type":"paragraph","content":[{"type":"text","text":line}]})
        flush_bullets()
        if not content:
            content = [{"type":"paragraph","content":[{"type":"text","text":" "}]}]
        return {"type":"doc","version":1,"content":content}

    def create_issue(self, summary, description="", issue_type="Task", labels=None):
            url = f"{self.base_url}/rest/api/3/issue"
            payload = {"fields": {
                "project":   {"key": self.project_key},
                "summary":   (summary or "(untitled)")[:255],
                "issuetype": {"name": issue_type},
                "description": self._adf(description),
                "labels": labels or [],
            }}
            r = requests.post(url, json=payload, auth=self.auth, headers=self.headers, timeout=20)
            if r.status_code >= 300:
                raise RuntimeError(f"JIRA create failed ({r.status_code}) in space '{self.project_key}' "
                                    f"[type={issue_type}]: {r.text[:300]}")
            try:
                data = r.json()
            except ValueError:
                raise RuntimeError(f"JIRA returned a non-JSON response (status {r.status_code}): {r.text[:300]}")
            if "key" not in data:
                raise RuntimeError(f"JIRA response missing issue key: {r.text[:300]}")
            key = data["key"]
            return key, f"{self.base_url}/browse/{key}"

    def link_parent(self, issue_key, parent_key):
        url = f"{self.base_url}/rest/api/3/issue/{issue_key}"
        r = requests.put(url, json={"fields": {"parent": {"key": parent_key}}},
                          auth=self.auth, headers=self.headers, timeout=20)
        if r.status_code >= 300:
            raise RuntimeError(f"epic link failed ({r.status_code}): {r.text[:200]}")
        return True

    def create_subtask(self, parent_key, summary, description="", issue_type="Sub-task"):
        url = f"{self.base_url}/rest/api/3/issue"
        payload = {"fields": {
            "project":   {"key": self.project_key},
            "parent":    {"key": parent_key},
            "summary":   (summary or "(untitled)")[:255],
            "issuetype": {"name": issue_type},
            "description": self._adf(description),
        }}
        r = requests.post(url, json=payload, auth=self.auth, headers=self.headers, timeout=20)
        if r.status_code >= 300:
            raise RuntimeError(f"JIRA sub-task create failed ({r.status_code}) under {parent_key}: {r.text[:300]}")
        try:
            data = r.json()
        except ValueError:
            raise RuntimeError(f"JIRA returned non-JSON for sub-task (status {r.status_code}): {r.text[:300]}")
        key = data.get("key")
        if not key:
            raise RuntimeError(f"JIRA sub-task response missing key: {r.text[:300]}")
        return key, f"{self.base_url}/browse/{key}"

    def search_issues(self, jql, max_results=100):
        """Search Jira Cloud issues using the enhanced JQL endpoint."""
        url = f"{self.base_url}/rest/api/3/search/jql"
        payload = {
            "jql": jql,
            "maxResults": max(1, min(int(max_results), 100)),
            "fields": ["summary", "description", "status", "issuetype", "labels"],
        }
        r = requests.post(
            url,
            json=payload,
            auth=self.auth,
            headers=self.headers,
            timeout=20,
        )
        if r.status_code >= 300:
            raise RuntimeError(
                f"JIRA search failed ({r.status_code}): {r.text[:300]}"
            )
        data = r.json()
        return data.get("issues", [])

    def update_issue(self, issue_key, summary=None, description=None):
        url = f"{self.base_url}/rest/api/3/issue/{issue_key}"
        fields = {}
        if summary is not None:
            fields["summary"] = summary[:255]
        if description is not None:
            fields["description"] = self._adf(description)
        if not fields:
            return True
        r = requests.put(url, json={"fields": fields}, auth=self.auth,
                          headers=self.headers, timeout=20)
        if r.status_code >= 300:
            raise RuntimeError(f"JIRA update failed ({r.status_code}): {r.text[:200]}")
        return True
        
# ════════════════════════════════════════════════════════════════
# BA / PO MODES & CONSTANTS
# ════════════════════════════════════════════════════════════════
BA_MODES = {
    "general":   "",
    "brd":       """
The user is requesting a Requirements Document.

The document type is determined dynamically from the user's request:
- BRD = Business Requirements Document
- FRD = Functional Requirements Document
- PRD = Product Requirements Document
- SRS = Software Requirements Specification
- FSD = Functional Specification Document
- SOP = Standard Operating Procedure
- SOW = Scope of Work

If the user explicitly mentions a supported document type, produce that type.
If no type is explicitly mentioned, default to BRD.

Never silently substitute one document type for another.

BRD focus:
- Business context and problem
- Business objectives
- Scope
- Stakeholders
- Business requirements
- Business rules
- Appropriate functional and non-functional requirements
- Assumptions
- Constraints
- Success criteria

FRD focus:
- Detailed functional requirements
- System behavior
- Workflows and user/system interactions
- Business rules
- Inputs and outputs
- Validations
- Exceptions and error handling
- Data requirements
- Integration requirements
- Security requirements
- Functional acceptance criteria

PRD focus:
- Product vision
- User problem
- Target users/personas
- Product goals and business outcomes
- Product scope
- Features and capabilities
- User stories/use cases
- Product requirements
- UX considerations
- Success metrics
- Release considerations
- Dependencies and risks

SRS focus:
- Introduction
- System overview
- Functional requirements
- Non-functional requirements
- System and external interfaces
- Data requirements
- Security requirements
- Performance and availability
- Error handling
- Constraints and dependencies
- Assumptions
- Traceability

FSD focus:
- Functional overview
- Detailed functional specifications
- System workflows
- Inputs and outputs
- Business rules
- Validations
- Interfaces and integrations
- API behavior where applicable
- Error handling
- Data handling
- Dependencies
- System behavior

SOP focus:
- Purpose
- Scope
- Roles and responsibilities
- Prerequisites
- Required inputs
- Tools and systems
- Procedure steps
- Decision points and business rules
- Exceptions and escalation
- Controls and compliance considerations
- Outputs
- Quality checks
- Records and documentation

SOW focus:
- Purpose and background
- Objectives
- In scope
- Out of scope
- Deliverables
- Activities and work packages
- Roles and responsibilities
- Assumptions
- Dependencies
- Constraints
- Milestones and schedule
- Acceptance criteria
- Change control
- Risks and issues
- Governance and reporting
- Completion and sign-off

Use the terminology and section structure appropriate to the selected document type.
Do not label one document type as another.
""",
    "stories":   'Output ONLY valid JSON:\n{"mode":"stories","title":"<title>","summary":"<1 sentence>","cards":[{"type":"epic","id":"EP-001","title":"<epic title>","priority":"High","status":"Draft","description":"<description>","sub_items":["Story: <story 1>"]},{"type":"story","id":"US-001","title":"As a <persona>, I want <goal>, so that <benefit>","priority":"High","status":"Draft","points":5,"tags":["Epic: EP-001"],"description":"<description>","sub_items":["Given <context> When <action> Then <outcome>"]}]}',
    "analytics": "Analyse and return: Top 3 Insights, Anomalies/Risks, Root Cause Hypotheses, Recommended Actions. ALSO output a markdown table titled 'CHART_DATA' with two columns Label and Value.",
    "process":   "Describe As-Is and To-Be processes. Identify bottlenecks, gaps, handoff points, automation opportunities.",
    "rtm":       "Output a Requirements Traceability Matrix: Req ID | Description | Source | Priority (MoSCoW) | Status | Test Case.",
    "email":     "Draft a professional stakeholder email or meeting summary with decisions, action items, and owners."
}
# Point 5: "Requirements Docs" label for brd button only (key stays "brd")
BA_LABELS = {
    "general":"general", "brd":"Requirements Docs", "stories":"stories",
    "analytics":"analytics", "process":"process", "rtm":"rtm", "email":"email"
}
BA_ICONS  = {"general":"💬","brd":"📄","stories":"📝","analytics":"📊","process":"🔄","rtm":"📋","email":"📧"}
BA_COLORS = {"general":"#1a73e8","brd":"#9c27b0","stories":"#2e7d32","analytics":"#e65100","process":"#00695c","rtm":"#c62828","email":"#4527a0"}

PM_MODES = {
    "general":    "",
    "scope": """Act as the Project Manager responsible for Scope Management.

Return ONLY valid JSON using exactly this structure:
{
  "mode": "scope",
  "title": "<scope management title>",
  "summary": "<one sentence>",
  "scope_baseline": {
    "version": "1.0",
    "scope_statement": "<approved/proposed scope statement>",
    "in_scope": ["<item>"],
    "out_of_scope": ["<item>"],
    "assumptions": ["<item>"],
    "constraints": ["<item>"],
    "dependencies": ["<item>"],
    "acceptance_authority": "<role or TBD>",
    "approval_status": "Draft"
  },
  "wbs": [
    {
      "id": "WBS-001",
      "parent_id": null,
      "title": "<work package>",
      "deliverable": "<deliverable>",
      "acceptance_criteria": ["<criterion>"]
    }
  ],
  "traceability": [
    {
      "requirement_id": "REQ-001",
      "wbs_id": "WBS-001",
      "deliverable": "<deliverable>",
      "coverage": "Covered"
    }
  ],
  "cards": [
    {
      "type": "epic",
      "id": "EP-001",
      "title": "<scope epic>",
      "priority": "High",
      "status": "Draft",
      "tags": ["WBS: WBS-001", "Scope Baseline: 1.0"],
      "description": "<description>",
      "acceptance_criteria": ["<criterion>"],
      "sub_items": []
    },
    {
      "type": "story",
      "id": "US-001",
      "title": "As a <persona>, I want <goal>, so that <benefit>",
      "priority": "High",
      "status": "Draft",
      "tags": ["WBS: WBS-001", "Epic: EP-001"],
      "description": "<description>",
      "acceptance_criteria": ["Given <context> When <action> Then <outcome>"],
      "sub_items": []
    },
    {
      "type": "task",
      "id": "TSK-001",
      "title": "<implementation task>",
      "priority": "Medium",
      "status": "Draft",
      "tags": ["WBS: WBS-001", "Epic: EP-001"],
      "description": "<description>",
      "acceptance_criteria": [],
      "sub_items": []
    }
  ],
  "change_requests": [],
  "scope_creep_findings": []
}

Rules:
- Every Epic, Story, and Task MUST map to a WBS ID in tags.
- Stories and Tasks MUST identify their parent Epic where applicable.
- Do not create work for anything explicitly Out of Scope.
- Preserve the Scope Baseline as the governing boundary.
- If the user is asking for a scope change, populate change_requests with a change request containing requested_change, baseline_impact, scope_impact, schedule_impact, cost_impact, risk_impact, recommendation, approval_status, and a JIRA-ready card.
- If the user is asking for scope-creep analysis, populate scope_creep_findings with baseline item, observed item, reason_flagged, severity, recommendation, and linked_jira_key when available.
- Do not invent dates, costs, approvals, commitments, or JIRA keys. Use TBD where necessary.
- Treat Scope Management as a control discipline, not merely a document.""",
    "charter":    "Write a Project Charter including: Business Case, Project Purpose & Justification, Objectives, Success Criteria, High-Level Scope, Key Stakeholders, High-Level Risks, Milestones, and a Sponsor Approval section.",
    "plan":       "Write a Project Management Plan covering: Scope Statement, Work Breakdown Structure (WBS) as a hierarchical list or table, Schedule approach and key milestones, Budget/Cost baseline approach, Quality standards and controls, Resource Plan, Communication Plan, and how the chosen approach (Agile/Waterfall/Lean/Hybrid) shapes execution.",
    "raid":       'Output ONLY valid JSON:\n{"mode":"raid","title":"<title>","summary":"<1 sentence>","cards":[{"type":"risk","id":"R-001","title":"<risk title>","priority":"High","status":"Open","description":"<description>","sub_items":["Mitigation: <action>","Owner: <name/role>","Impact: <High/Medium/Low>","Probability: <High/Medium/Low>"]},{"type":"issue","id":"I-001","title":"<issue title>","priority":"High","status":"Open","description":"<description>","sub_items":["Resolution plan: <action>","Owner: <name/role>"]},{"type":"assumption","id":"A-001","title":"<assumption title>","priority":"Medium","status":"Open","description":"<description>","sub_items":["Validation approach: <how it will be confirmed>"]},{"type":"dependency","id":"D-001","title":"<dependency title>","priority":"Medium","status":"Open","description":"<description>","sub_items":["Depends on: <team/system/vendor>","Target resolution: <date/milestone>"]}]}',
    "status":     "Write a Project Status Report including: Overall Health (Green/Amber/Red) with rationale, Progress Summary, Key Accomplishments this period, Upcoming Milestones, Schedule Variance, Cost/Budget Variance, Top Risks & Issues, Blockers, and Actions Needed from stakeholders. ALSO output a markdown table titled 'CHART_DATA' with two columns Label and Value representing key project metrics (e.g. % Complete, Budget Utilized, Milestones Achieved).",
    "governance": "Write a Governance & Controls document including: Governance Structure (PMO, Sponsor, Steering Committee), Decision Rights & Approval Matrix (as a table: Decision Type | Approver | Escalation Path), Stage Gates, Change Control Process, Reporting Cadence & Hierarchy, and Compliance/Audit Trail approach.",
    "changelog":  'Output ONLY valid JSON:\n{"mode":"changelog","title":"<title>","summary":"<1 sentence>","cards":[{"type":"change_request","id":"CR-001","title":"<change title>","priority":"High","status":"Proposed","description":"<what is changing and why>","sub_items":["Impact on Scope: <description>","Impact on Schedule: <description>","Impact on Cost: <description>","Approver: <role>"]}]}',
    "closure":    "Write a Project Closure Report including: Final Deliverables Summary, Objectives Met vs Planned, Budget Actual vs Planned, Schedule Actual vs Planned, Benefits Realized, Lessons Learned (What Went Well / What Could Improve), Outstanding Items & Transition Plan, and Formal Sign-off section.",
}
PM_LABELS = {"general":"general","scope":"Scope Management","charter":"Charter","plan":"PM Plan","raid":"RAID Log","status":"Status Report","governance":"Governance","changelog":"Change Log","closure":"Closure"}
PM_ICONS  = {"general":"💬","scope":"🎯","charter":"📜","plan":"🗂️","raid":"⚠️","status":"📈","governance":"🏛️","changelog":"🔀","closure":"🏁"}
PM_COLORS = {"general":"#1a73e8","scope":"#1565c0","charter":"#5e35b1","plan":"#00695c","raid":"#c62828","status":"#0277bd","governance":"#6a1b9a","changelog":"#e65100","closure":"#2e7d32"}

PM_SYSTEM = """You are an expert Project Manager with deep knowledge across the full Project Management discipline. Reason using this conceptual model, keeping each dimension distinct — never conflate them:

- LIFECYCLE (always applies, independent of approach): Initiation → Planning → Execution → Monitoring & Control → Closure.
- APPROACH/METHODOLOGY (how the lifecycle is executed): Agile, Waterfall, Lean, or Hybrid — chosen based on project context, complexity, risk, timeline, and stakeholder needs. Do not assume Agile by default; infer or ask, and be explicit about which approach a given recommendation assumes.
- KNOWLEDGE AREAS: Scope, Schedule, Cost, Quality, Resource, Risk, Communications, Procurement, Stakeholder, and Integration Management.
- PROCESSES & CONTROLS: planning, execution tracking, scope/schedule/cost/quality/resource control, risk management, issue management, dependency management, change control, and performance monitoring.
- GOVERNANCE: PMO, Sponsor, Steering Committee, stage gates, approval matrix, decision rights, escalation, reporting hierarchy, compliance, audit trail.
- METRICS & VALUE: schedule variance (SV), cost variance (CV), budget utilization, milestone achievement, resource utilization, risk exposure, issue aging, delivery predictability, ROI, benefits realization, business value.
- ROLES: Project Manager, Sponsor, PMO, Business Analyst, Product Owner, Scrum Master, Developers, Designers, QA/Testers, SMEs, Vendors, and other stakeholders — name the right role for each responsibility rather than defaulting to "the team."

Be industry-agnostic by default. Adapt practices to a specific domain (BFSI, Insurance, Healthcare, SaaS, IT, Digital Transformation, Software Development, Consulting) only when the user's context makes that domain clear.
Structure outputs with headers, bullets, and tables where useful. Be concise and avoid unnecessary preamble."""

pm_histories = {m: [] for m in PM_MODES}

PO_MODES = {
    "general":     "",
    "backlog":     'Output ONLY valid JSON:\n{"mode":"backlog","title":"<title>","summary":"<1 sentence>","cards":[{"type":"epic","id":"EP-001","title":"<epic title>","priority":"High","status":"Draft","description":"<description>","sub_items":["Story: <story 1>"]},{"type":"story","id":"US-001","title":"As a <persona>, I want <goal>, so that <benefit>","priority":"High","status":"Draft","points":5,"tags":["Epic: EP-001"],"description":"<description>","sub_items":["Given <context> When <action> Then <outcome>"]}]}',
    "prioritize":  'Output ONLY valid JSON:\n{"mode":"prioritize","title":"<title>","summary":"<1 sentence>","cards":[{"type":"item","id":"P-001","title":"<feature>","priority":"High","status":"Draft","points":8,"tags":["Must Have","WSJF: 8"],"description":"<justification>","sub_items":["Business Value: High","Time Criticality: Medium"]}]}',
    "pi-planning": 'Output ONLY valid JSON:\n{"mode":"pi-planning","title":"<title>","summary":"<1 sentence>","cards":[{"type":"objective","id":"PI-001","title":"<PI objective>","priority":"High","status":"Committed","tags":["ART: <name>","PI: <number>"],"description":"<description>","sub_items":["Feature: <feature>","Dependency: <team>","Risk: <risk>","Business Value: <score>/10"]}]}',
    "sprint":      'Output ONLY valid JSON:\n{"mode":"sprint","title":"<title>","summary":"<1 sentence>","cards":[{"type":"task","id":"SP-001","title":"<task>","priority":"High","status":"To Do","points":3,"tags":["Sprint <n>"],"description":"<desc>","sub_items":["Owner: <name>","AC: <criteria>"]}]}',
    "roadmap":     'Output ONLY valid JSON:\n{"mode":"roadmap","title":"<title>","summary":"<1 sentence>","cards":[{"type":"milestone","id":"RM-001","title":"<milestone>","priority":"High","status":"Planned","tags":["Q<n> <year>"],"description":"<desc>","sub_items":["Feature: <feature>","Metric: <metric>"]}]}',
    "vision":      'Output ONLY valid JSON:\n{"mode":"vision","title":"<title>","summary":"<1 sentence>","cards":[{"type":"vision","id":"V-001","title":"<vision statement>","priority":"High","status":"Draft","tags":["OKR","Product Goal"],"description":"<expanded>","sub_items":["Objective: <obj>","KR1: <kr>","KR2: <kr>"]}]}',
    "acceptance":  'Output ONLY valid JSON:\n{"mode":"acceptance","title":"<title>","summary":"<1 sentence>","cards":[{"type":"criteria","id":"AC-001","title":"<story>","priority":"High","status":"Draft","tags":["DoR","DoD"],"description":"As a <persona>, I want <goal>","sub_items":["Given <ctx> When <act> Then <out>","DoR: <check>","DoD: <check>"]}]}'
}
PO_ICONS  = {"general":"💬","backlog":"📦","prioritize":"🎯","pi-planning":"🗓️","sprint":"🏃","roadmap":"🗺️","vision":"🔭","acceptance":"✅"}
PO_COLORS = {"general":"#1a73e8","backlog":"#0277bd","prioritize":"#6a1b9a","pi-planning":"#1b5e20","sprint":"#e65100","roadmap":"#00695c","vision":"#c62828","acceptance":"#4527a0"}
CARD_COLORS = {"epic":"#9c27b0","story":"#1a73e8","task":"#e65100","objective":"#2e7d32","milestone":"#00695c","vision":"#c62828","criteria":"#4527a0","item":"#0277bd",
                "risk":"#c62828","issue":"#e65100","assumption":"#6a1b9a","dependency":"#00695c","change_request":"#1a73e8"}
CARD_ICONS  = {"epic":"📦","story":"📝","task":"🏃","objective":"🎯","milestone":"🗺️","vision":"🔭","criteria":"✅","item":"⚖️",
                "risk":"⚠️","issue":"🐞","assumption":"❓","dependency":"🔗","change_request":"🔀"}
JIRA_ISSUE_TYPE_MAP = {
    "epic":"Epic", "story":"Story", "task":"Task", "item":"Task",
    "objective":"Task", "milestone":"Task", "vision":"Task", "criteria":"Task",
    "risk":"Risk", "issue":"Task", "assumption":"Task", "dependency":"Task", "change_request":"Task",
}

def is_structured_card_mode(role, mode):
    """Which (role, mode) combinations produce card JSON rather than free-form text.
    Single source of truth — used by WorkspaceState.build_content, send_message, and
    is_doc_request so a new structured mode only needs to be registered here once."""
    if role == "po": return True  # every PO mode is JSON-schema-forced
    if role == "ba": return mode == "stories"
    if role == "pm": return mode in {"raid", "changelog", "scope"}
    return False

def parse_scope_payload(doc_content):
    """Return a validated Scope Management JSON payload or None."""
    parsed = parse_po_json(doc_content or "")
    if not isinstance(parsed, dict) or parsed.get("mode") != "scope":
        return None
    parsed.setdefault("scope_baseline", {})
    parsed.setdefault("wbs", [])
    parsed.setdefault("traceability", [])
    parsed.setdefault("cards", [])
    parsed.setdefault("change_requests", [])
    parsed.setdefault("scope_creep_findings", [])
    return parsed


def _scope_terms(value):
    return set(
        re.findall(
            r"[a-z0-9]{4,}",
            str(value or "").lower(),
        )
    )


def scope_change_request_analysis(scope_payload, request_text):
    """Assess a requested scope change against the current baseline."""
    baseline = scope_payload.get("scope_baseline", {})
    in_scope = baseline.get("in_scope", [])
    out_scope = baseline.get("out_of_scope", [])
    wbs = scope_payload.get("wbs", [])

    request_terms = _scope_terms(request_text)
    in_matches = [
        item for item in in_scope
        if request_terms & _scope_terms(item)
    ]
    out_matches = [
        item for item in out_scope
        if request_terms & _scope_terms(item)
    ]
    wbs_matches = [
        item for item in wbs
        if request_terms & _scope_terms(item.get("title"))
    ]

    if out_matches:
        recommendation = "Formal change control required: the request overlaps explicit Out of Scope items."
        classification = "Outside Baseline"
    elif wbs_matches or in_matches:
        recommendation = "Assess as a controlled scope clarification/change and confirm acceptance criteria before implementation."
        classification = "Potential Baseline Change"
    else:
        recommendation = "Treat as a formal scope change until the Change Control Board confirms that it is already covered."
        classification = "Unmapped Scope"

    card_id = f"CR-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    card = {
        "type": "change_request",
        "id": card_id,
        "title": f"Scope Change Request: {request_text[:180]}",
        "priority": "High",
        "status": "Proposed",
        "tags": [
            f"Scope Baseline: {baseline.get('version', 'TBD')}",
            "Scope Change",
        ],
        "description": request_text,
        "acceptance_criteria": [
            "Impact assessment completed",
            "Scope baseline decision recorded",
            "Required approvals obtained before implementation",
        ],
        "sub_items": [
            f"Classification: {classification}",
            f"Baseline matches: {len(in_matches) + len(wbs_matches)}",
            f"Out-of-scope matches: {len(out_matches)}",
            "Impact on Schedule: TBD",
            "Impact on Cost: TBD",
            "Impact on Risk: TBD",
            "Approver: Change Control Board / Sponsor (TBD)",
        ],
    }

    return {
        "classification": classification,
        "recommendation": recommendation,
        "baseline_version": baseline.get("version", "TBD"),
        "in_scope_matches": in_matches,
        "out_of_scope_matches": out_matches,
        "wbs_matches": [
            {"id": item.get("id"), "title": item.get("title")}
            for item in wbs_matches
        ],
        "card": card,
    }


def _select_scope_jira_project(scope_payload):
    spaces = jira_fetch_projects()
    if not spaces:
        return None
    baseline = scope_payload.get("scope_baseline", {})
    text = " ".join(
        [baseline.get("scope_statement", "")]
        + baseline.get("in_scope", [])
        + [item.get("title", "") for item in scope_payload.get("wbs", [])]
    )
    matched = match_jira_space(text, spaces)
    return matched["key"] if matched else spaces[0]["key"]


def fetch_scope_jira_issues(scope_payload, max_results=100):
    """Fetch JIRA issues tagged as belonging to this Scope Management stream."""
    if not (JIRA_BASE_URL and JIRA_EMAIL and JIRA_API_TOKEN):
        return {"project_key": None, "issues": [], "error": "JIRA is not configured."}

    project_key = _select_scope_jira_project(scope_payload)
    if not project_key:
        return {"project_key": None, "issues": [], "error": "No accessible JIRA project found."}

    client = JiraClient(JIRA_BASE_URL, JIRA_EMAIL, JIRA_API_TOKEN, project_key)
    try:
        issues = client.search_issues(
            f'project = "{project_key}" AND labels = "copilot-scope" ORDER BY updated DESC',
            max_results=max_results,
        )
    except Exception as exc:
        return {"project_key": project_key, "issues": [], "error": str(exc)}

    normalized = []
    for issue in issues:
        fields = issue.get("fields", {})
        issue_type = (fields.get("issuetype") or {}).get("name", "")
        status = (fields.get("status") or {}).get("name", "")
        normalized.append({
            "key": issue.get("key", ""),
            "summary": fields.get("summary", ""),
            "description": fields.get("description", ""),
            "issue_type": issue_type,
            "status": status,
            "labels": fields.get("labels", []),
        })

    return {"project_key": project_key, "issues": normalized, "error": None}


def detect_scope_creep(scope_payload, jira_context):
    """Compare scoped Jira work against the approved baseline and WBS."""
    baseline = scope_payload.get("scope_baseline", {})
    in_scope = baseline.get("in_scope", [])
    out_scope = baseline.get("out_of_scope", [])
    wbs = scope_payload.get("wbs", [])

    wbs_terms = [
        (item.get("id", ""), _scope_terms(item.get("title")))
        for item in wbs
    ]
    baseline_terms = _scope_terms(" ".join(in_scope))
    findings = []

    for issue in jira_context.get("issues", []):
        issue_terms = _scope_terms(
            f"{issue.get('summary', '')} {issue.get('description', '')}"
        )
        out_hits = [
            item for item in out_scope
            if issue_terms & _scope_terms(item)
        ]
        wbs_hits = [
            wbs_id for wbs_id, terms in wbs_terms
            if terms and issue_terms & terms
        ]

        if out_hits:
            severity = "High"
            reason = "JIRA work overlaps an explicit Out of Scope item."
        elif not wbs_hits and not (issue_terms & baseline_terms):
            severity = "Medium"
            reason = "JIRA work could not be traced to the approved scope baseline or WBS."
        else:
            continue

        findings.append({
            "severity": severity,
            "linked_jira_key": issue.get("key"),
            "observed_item": issue.get("summary"),
            "baseline_item": ", ".join(out_hits) if out_hits else "No matching baseline/WBS item",
            "reason_flagged": reason,
            "recommendation": (
                "Hold or route through formal scope change control before implementation."
                if severity == "High"
                else "Confirm WBS traceability and baseline coverage."
            ),
        })

    return findings


def push_cards_to_jira(chat_id, cards, source_mode, project_key=None):
    if not (JIRA_BASE_URL and JIRA_EMAIL and JIRA_API_TOKEN):
        return [{"card_id": c.get("id","?"), "title": c.get("title",""), "status":"error",
                  "error":"JIRA not configured — set JIRA_BASE_URL / EMAIL / API_TOKEN"}
                 for c in cards]

    if not project_key:
        spaces = jira_fetch_projects()
        combined = " ".join(f"{c.get('title','')} {c.get('description','')}" for c in cards)
        matched  = match_jira_space(combined, spaces)
        project_key = matched["key"] if matched else (spaces[0]["key"] if spaces else None)

    if not project_key:
        return [{"card_id": c.get("id","?"), "title": c.get("title",""), "status":"error",
                  "error":"No JIRA space could be determined — no spaces are accessible to this token"}
                 for c in cards]

    all_types    = jira_fetch_issue_types(project_key)
    normal_types = [t["name"] for t in all_types if not t["subtask"]]
    subtask_type = next((t["name"] for t in all_types if t["subtask"]), None)
    client   = JiraClient(JIRA_BASE_URL, JIRA_EMAIL, JIRA_API_TOKEN, project_key)
    existing = load_jira_sync(chat_id)
    results  = []
    epic_map = {}

    def push_one(c):
        card_id   = c.get("id") or (c.get("title","") or "")[:40]
        title     = c.get("title","(untitled)")
        ac_items  = c.get("acceptance_criteria", [])
        sub_items = c.get("sub_items", [])
        card_type = c.get("type","task")

        desc_lines = [c.get("description","")]
        if ac_items:
            desc_lines.append("Acceptance Criteria:")
            desc_lines += [f"- {a}" for a in ac_items]
        if not subtask_type and sub_items:
            desc_lines.append("Implementation Tasks:")
            desc_lines += [f"- {s}" for s in sub_items]
        description = "\n".join(l for l in desc_lines if l)
        issue_type  = resolve_jira_issue_type(card_type, normal_types)

        try:
            prior = existing.get(card_id)
            if prior and prior.get("jira_key"):
                client.update_issue(prior["jira_key"], summary=title, description=description)
                jkey, jurl, status = prior["jira_key"], prior["jira_url"], "updated"
            else:
                jira_labels = [f"copilot-{source_mode}"]
                for tag in c.get("tags", []):
                    normalized = re.sub(r"[^a-zA-Z0-9_-]+", "-", str(tag)).strip("-").lower()
                    if normalized:
                        jira_labels.append(normalized[:255])
                jkey, jurl = client.create_issue(
                    title,
                    description,
                    issue_type,
                    labels=list(dict.fromkeys(jira_labels)),
                )
                status = "created"
            save_jira_sync(chat_id, card_id, jkey, jurl, status, content_hash=_card_content_hash(c))
            results.append({"card_id":card_id,"title":title,"status":status,
                             "jira_key":jkey,"jira_url":jurl,"space":project_key,
                             "issue_type":issue_type,"subtasks":[],"epic_link":None})

            if card_type == "epic":
                epic_map[card_id] = jkey

            if subtask_type and sub_items and card_type != "epic":
                for idx, sub_text in enumerate(sub_items):
                    sub_card_id = f"{card_id}::sub{idx}"
                    sub_title   = sub_text.strip()[:255] or f"Subtask {idx+1}"
                    try:
                        sub_prior = existing.get(sub_card_id)
                        if sub_prior and sub_prior.get("jira_key"):
                            client.update_issue(sub_prior["jira_key"], summary=sub_title)
                            skey, surl, sstatus = sub_prior["jira_key"], sub_prior["jira_url"], "updated"
                        else:
                            skey, surl = client.create_subtask(jkey, sub_title, issue_type=subtask_type)
                            sstatus = "created"
                        save_jira_sync(chat_id, sub_card_id, skey, surl, sstatus)
                        results[-1]["subtasks"].append({"title":sub_title,"status":sstatus,
                                                         "jira_key":skey,"jira_url":surl})
                    except Exception as e:
                        results[-1]["subtasks"].append({"title":sub_title,"status":"error",
                                                         "error":str(e)[:150]})

            if card_type != "epic":
                epic_ref = extract_epic_ref(c.get("tags", []))
                epic_key = epic_map.get(epic_ref) if epic_ref else None
                if epic_key:
                    try:
                        client.link_parent(jkey, epic_key)
                        results[-1]["epic_link"] = epic_key
                    except Exception:
                        results[-1]["epic_link"] = "unsupported"

        except Exception as e:
            results.append({"card_id":card_id,"title":title,"status":"error",
                             "error":str(e)[:200],"space":project_key})

    for c in cards:
        if c.get("type") == "epic": push_one(c)
    for c in cards:
        if c.get("type") != "epic": push_one(c)

    return results

def jira_fetch_projects():
    """Fetch every Jira space accessible to this token, across all pages."""
    if not (JIRA_BASE_URL and JIRA_EMAIL and JIRA_API_TOKEN):
        return []
    all_spaces, start_at = [], 0
    try:
        while True:
            r = requests.get(f"{JIRA_BASE_URL}/rest/api/3/project/search",
                              auth=HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN),
                              headers={"Accept":"application/json"},
                              params={"maxResults": 100, "startAt": start_at}, timeout=15)
            r.raise_for_status()
            data = r.json()
            all_spaces += [{"id":p["id"],"key":p["key"],"name":p["name"]}
                            for p in data.get("values",[])]
            if data.get("isLast", True) or not data.get("values"):
                break
            start_at += len(data["values"])
        return all_spaces
    except Exception as e:
        print(f"⚠️ JIRA space fetch failed: {e}")
        return []

def jira_fetch_issue_types(project_key):
    """Return every work type in this space, tagged with Jira's own subtask flag."""
    try:
        r = requests.get(f"{JIRA_BASE_URL}/rest/api/3/issue/createmeta/{project_key}/issuetypes",
                          auth=HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN),
                          headers={"Accept":"application/json"}, timeout=15)
        r.raise_for_status()
        return [{"name": it["name"], "subtask": bool(it.get("subtask", False))}
                for it in r.json().get("issueTypes", [])]
    except Exception as e:
        print(f"⚠️ JIRA issue-type fetch failed for {project_key}: {e}")
        return []

def resolve_jira_issue_type(card_type, available_type_names):
    desired = JIRA_ISSUE_TYPE_MAP.get(card_type, "Task")
    if not available_type_names:
        return desired
    for candidate in (desired, "Story", "Task", "Idea"):
        if candidate in available_type_names:
            return candidate
    return available_type_names[0]

def extract_epic_ref(tags):
    for t in tags or []:
        m = re.match(r'^\s*epic\s*:\s*(.+)$', t, re.IGNORECASE)
        if m: return m.group(1).strip()
    return None

def _card_content_hash(card):
    basis = (card.get("title","") + "|" + card.get("description","") + "|" +
             "|".join(card.get("sub_items", [])))
    return hashlib.sha256(basis.encode("utf-8")).hexdigest()[:16]

def match_jira_space(text, spaces):
    """Best-guess space for a batch of cards, by keyword overlap with space names."""
    text_l = (text or "").lower()
    best, best_score = None, 0
    for s in spaces:
        words = set(re.findall(r"[a-z]{4,}", s["name"].lower()))
        score = sum(1 for w in words if w in text_l)
        if score > best_score:
            best, best_score = s, score
    return best

def compute_project_health(rows_for_project):
    open_high = sum(1 for r in rows_for_project
                     if r["card_type"] in ("risk","issue") and r["status"]=="Open" and r["priority"]=="High")
    open_med  = sum(1 for r in rows_for_project
                     if r["card_type"] in ("risk","issue") and r["status"]=="Open" and r["priority"]=="Medium")
    if open_high > 0: return "Red"
    if open_med  > 0: return "Amber"
    return "Green"

def save_jira_sync(chat_id, card_id, jira_key, jira_url, status, content_hash=None):
    now = datetime.now().isoformat()
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute("""
            INSERT INTO jira_sync (chat_id,card_id,jira_key,jira_url,status,synced_at,content_hash)
            VALUES (?,?,?,?,?,?,?)
            ON CONFLICT(chat_id,card_id) DO UPDATE SET
                jira_key     = excluded.jira_key,
                jira_url     = excluded.jira_url,
                status       = excluded.status,
                synced_at    = excluded.synced_at,
                content_hash = excluded.content_hash
        """, (chat_id, card_id, jira_key, jira_url, status, now, content_hash))
        conn.commit()
    
PRIORITY_COLORS = {"High":"#c62828","Medium":"#e65100","Low":"#2e7d32","Committed":"#1b5e20","Planned":"#0277bd"}
STATUS_COLORS   = {"Draft":"#888","To Do":"#1a73e8","In Progress":"#e65100","Done":"#2e7d32","Planned":"#9c27b0","Committed":"#1b5e20"}

BA_SYSTEM = """You are an expert Business Analyst.
Be industry-agnostic by default. Infer the domain (e.g., BFSI, Insurance, Healthcare, SaaS, Retail, IT, Consulting) only from concrete signals actually present in the user's request or attached context — never assume or default to a specific industry, region, or regulation set unless the user's own words point there.
Structure outputs with headers, bullets, tables. Be concise and avoid unnecessary preamble."""

PO_SYSTEM = """You are an expert Product Owner with SAFe Agile expertise.
Be industry-agnostic by default. Infer the domain (e.g., BFSI, Insurance, Healthcare, SaaS, Retail, IT, Consulting) only from concrete signals actually present in the user's request or attached context — never assume or default to a specific industry, region, or regulation set unless the user's own words point there.
For document/artifact requests: output ONLY valid JSON as instructed.
For conversational questions: respond in plain text."""

BPMN_SCHEMA_PROMPT = """Output ONLY valid JSON (no markdown, no extra text) describing a FULL BPMN 2.0 process model:
{"title":"<process name>","pools":[{"id":"Pool_1","name":"<participant>","lanes":[{"id":"Lane_1","name":"<role>","elements":[{"id":"StartEvent_1","type":"startEvent","name":"<name>"},{"id":"Task_1","type":"userTask","name":"<name>"},{"id":"Gateway_1","type":"exclusiveGateway","name":"<question?>"},{"id":"EndEvent_1","type":"endEvent","name":"<name>"}]}]}],"sequenceFlows":[{"id":"Flow_1","source":"StartEvent_1","target":"Task_1","name":""},{"id":"Flow_2","source":"Task_1","target":"Gateway_1","name":""},{"id":"Flow_3","source":"Gateway_1","target":"EndEvent_1","name":"Yes"}],"messageFlows":[]}
Rules:
- Output ONLY the JSON object. No explanation before or after.
- Every id in sequenceFlows/messageFlows MUST exist in a lane's elements.
- Valid types: startEvent, endEvent, task, userTask, serviceTask, exclusiveGateway, parallelGateway, inclusiveGateway, subProcess, intermediateTimerEvent, intermediateCatchEvent.
- messageFlows only between DIFFERENT pools.
- Minimum 8 elements. Use realistic BFSI/Insurance domain content."""

REDTEAM_PERSONAS = {
    "compliance": {"label":"⚖️ Compliance Officer",
        "prompt":"You are a strict Compliance Officer. First, determine from the document's actual content what regulatory/compliance domain applies (e.g., healthcare — HIPAA/ACA; banking — GLBA/PSD2/local banking regulator; insurance — state DOI/Solvency II; general software/PM — data privacy, contractual, audit-trail concerns; etc.). Do NOT assume a specific industry unless the document gives concrete evidence of it. Focus on regulatory gaps, missing audit trails, unclear accountability, and data privacy exposure relevant to the domain you identified. Be skeptical and specific, and name which domain you inferred and why."},
    "member": {"label":"🧑 End User",
        "prompt":"You are an end user of whatever this document describes, with no technical background, reviewing it for clarity and usability. Infer who the realistic end user actually is from the document's content (e.g., a beneficiary, a customer, an internal team member, a stakeholder) rather than assuming a fixed persona. Flag confusing language, missing user-facing considerations, accessibility gaps, and anything a real person in that role would find frustrating or unclear."},
    "auditor": {"label":"🔍 Auditor",
        "prompt":"You are an external auditor reviewing this for traceability, evidence, and control gaps. Flag anything that can't be independently verified, lacks a clear audit trail, or is ambiguous enough to cause disputes later. This applies regardless of industry — focus on structural traceability, not domain-specific rules."},
    "ops": {"label":"📋 Operations Specialist",
        "prompt":"You are an operations/delivery specialist reviewing this for real-world feasibility. First infer what kind of operations this document actually involves (e.g., claims processing, software delivery, project execution, customer service) from its content. Flag edge cases, exception handling gaps, and anything that would be unclear or unworkable in daily execution of that specific kind of work."},
}

DOCUMENT_REDTEAM_PERSONAS = {
    "BRD": ["sponsor", "product_manager", "business_user", "operations", "compliance"],
    "FRD": ["business_analyst", "developer", "qa", "security", "operations"],
    "PRD": ["product_manager", "ux", "end_user", "business", "compliance"],
    "FSD": ["architect", "developer", "qa", "security", "devops", "operations"],
    "SRS": ["architect", "developer", "qa", "security", "devops", "compliance"],
    "SOP": ["process_owner", "operations", "business_user", "quality", "compliance"],
    "SOW": ["project_manager", "business_owner", "delivery_lead", "technical_lead", "procurement", "compliance"],
}

REDTEAM_PERSONAS.update({
    "sponsor": {"label": "👤 Business Sponsor", "prompt": "You are the business sponsor. Review business outcomes, scope, decision rights, measurable success, stakeholder accountability, and unresolved business risks."},
    "product_manager": {"label": "🧭 Product Manager", "prompt": "You are a Product Manager. Review product/business value, prioritization, user outcomes, scope, dependencies, and measurable success."},
    "business_user": {"label": "🧑‍💼 Business User", "prompt": "You are a representative business user. Review usability, clarity, operational fit, missing business scenarios, and acceptance expectations."},
    "business_analyst": {"label": "📊 Business Analyst", "prompt": "You are a Business Analyst. Review completeness, ambiguity, traceability, business rules, requirements quality, and testability."},
    "developer": {"label": "👨‍💻 Developer", "prompt": "You are a Developer. Review implementability, ambiguity, system behavior, data handling, interfaces, error handling, and technical gaps."},
    "qa": {"label": "🧪 QA", "prompt": "You are a QA engineer. Review testability, acceptance criteria, negative paths, validations, edge cases, and observable outcomes."},
    "security": {"label": "🔐 Security", "prompt": "You are a Security reviewer. Review security requirements, access control, data protection, threats, auditability, and security-related gaps."},
    "architect": {"label": "🏗️ Solution Architect", "prompt": "You are a Solution Architect. Review architecture, interfaces, dependencies, system boundaries, scalability, resilience, and technical feasibility."},
    "devops": {"label": "⚙️ DevOps / SRE", "prompt": "You are a DevOps/SRE reviewer. Review deployment, observability, availability, reliability, recovery, operational readiness, and supportability."},
    "operations": {"label": "📋 Operations", "prompt": "You are an Operations specialist. Review day-to-day execution, handoffs, exceptions, controls, ownership, operational readiness, and feasibility."},
    "compliance": REDTEAM_PERSONAS.get("compliance", {"label": "⚖️ Compliance Officer", "prompt": "Review compliance and auditability."}),
    "ux": {"label": "🎨 UX", "prompt": "You are a UX specialist. Review user journeys, usability, accessibility, information clarity, interaction requirements, and user-facing gaps."},
    "end_user": REDTEAM_PERSONAS.get("member", {"label": "🧑 End User", "prompt": "Review the artifact from the realistic end-user perspective."}),
    "business": {"label": "🏢 Business Stakeholder", "prompt": "You are a business stakeholder. Review business value, scope, process fit, stakeholder impact, and decision clarity."},
    "process_owner": {"label": "🧩 Process Owner", "prompt": "You own the operational process. Review procedure completeness, ownership, controls, handoffs, exceptions, and measurable process outcomes."},
    "quality": {"label": "✅ Quality / QA", "prompt": "You are a quality specialist. Review consistency, control points, quality checks, evidence, acceptance, and repeatability."},
    "project_manager": {"label": "📅 Project Manager", "prompt": "You are a Project Manager. Review scope, deliverables, milestones, responsibilities, dependencies, risks, governance, and acceptance."},
    "business_owner": {"label": "👤 Business Owner", "prompt": "You are the business owner. Review objectives, scope boundaries, business outcomes, responsibilities, acceptance, and stakeholder commitments."},
    "delivery_lead": {"label": "🚚 Delivery Lead", "prompt": "You are the delivery lead. Review work packages, sequencing, dependencies, deliverables, resources, acceptance, and execution risks."},
    "technical_lead": {"label": "🛠️ Technical Lead", "prompt": "You are the technical delivery lead. Review technical deliverables, implementation assumptions, interfaces, dependencies, and technical acceptance."},
    "procurement": {"label": "🤝 Procurement / Vendor Management", "prompt": "You are a procurement/vendor-management reviewer. Review supplier responsibilities, deliverables, acceptance, dependencies, assumptions, governance, and scope ambiguity."},
})

def get_document_redteam_personas(document_type):
    """Return the recommended stakeholder suite for the current document type."""
    document_type = (document_type or "BRD").upper()
    return DOCUMENT_REDTEAM_PERSONAS.get(
        document_type,
        DOCUMENT_REDTEAM_PERSONAS["BRD"],
    )

def build_custom_redteam_persona(stakeholder):
    """Construct a stakeholder perspective without adding it to global configuration."""
    name = (stakeholder or "").strip()
    if not name:
        return None
    return {
        "key": "custom",
        "label": f"👥 {name}",
        "prompt": (
            f"You are an independent reviewer acting as a {name}. "
            "Infer the relevant responsibilities of this stakeholder from "
            "the artifact and identify role-specific operational risks, "
            "missing requirements, process gaps, controls, dependencies, "
            "and implementation concerns."
        ),
    }

REGULATORY_RADAR_SYSTEM = """You are a regulatory compliance analyst.

Do not reveal or output reasoning. Do not provide a thinking process. Return only the final JSON object.

Analyze ONLY the supplied artifact. Identify the most likely business domain and jurisdiction, then identify only regulations that are concretely relevant to the artifact.

Keep the response SMALL so it can always be parsed reliably:
- Maximum 8 applicable regulations.
- Maximum 3 gaps per applicable regulation.
- Each gap must be one concise sentence.
- Keep rationale to one sentence.
- Keep risk_rationale to one sentence.
- Do not reproduce the artifact.
- Do not provide prose before or after the JSON.

Return ONLY one valid JSON object. Do not use Markdown fences, comments, trailing commas, or extra text.

Required shape:
{
  "detected_domain": "Banking",
  "detected_jurisdiction": "India",
  "regulations": [
    {
      "name": "Regulation name",
      "authority": "Authority",
      "applicable": true,
      "rationale": "Why this regulation is relevant to the artifact.",
      "gaps": ["Specific missing control or requirement."]
    }
  ],
  "overall_risk": "Low",
  "risk_rationale": "One concise sentence explaining the overall risk."
}

Rules:
- Only include gaps for regulations where applicable=true.
- Use [] for gaps when no gap is identified.
- Do not invent exact regulatory clause numbers or citations.
- Be conservative: a regulation must have textual support in the artifact or an explicit domain/jurisdiction basis.
"""

REDTEAM_SYSTEM = """You are role-playing a specific stakeholder persona reviewing a business document.
Critique honestly and specifically — do not be diplomatic or generic. Structure your response as:
## Concerns
## Missing Requirements
## Questions for the Author
## Risk Rating (Low/Medium/High) with 1-sentence justification
Stay strictly in character as described. Base every point on the actual document content provided — do not invent unrelated issues."""

REDTEAM_MODE_FOCUS = {
    "brd":        "This is a Business Requirements Document. Focus on: completeness of scope/out-of-scope boundaries, stakeholder coverage, functional vs non-functional clarity, and whether success criteria are measurable.",
    "srs":        "This is a Software Requirements Specification. Focus on functional/non-functional completeness, interfaces, data, security, performance, availability, error handling, constraints, dependencies, assumptions, and traceability.",
    "fsd":        "This is a Functional Specification Document. Focus on detailed system behavior, workflows, interfaces, validations, API behavior, error handling, data handling, dependencies, and implementation clarity.",
    "sop":        "This is a Standard Operating Procedure. Focus on repeatability, prerequisites, roles, ordered steps, decision points, exceptions, controls, escalation, outputs, quality checks, and records.",
    "sow":        "This is a Scope of Work. Focus on scope boundaries, deliverables, work packages, responsibilities, assumptions, dependencies, milestones, acceptance, change control, risks, governance, and sign-off.",
    "scope":       "This is a Scope Management artifact. Focus on the approved scope baseline, in-scope/out-of-scope boundaries, WBS completeness, deliverable acceptance, requirements-to-scope traceability, validation, change control, scope creep, assumptions, constraints, dependencies, governance, and measurable scope controls.",
    "stories":    "This is a set of User Stories with acceptance criteria. Focus on: whether acceptance criteria (Given/When/Then) are testable, missing edge cases and negative paths, story sizing/independence, and missing personas.",
    "process":    "This is a BPMN process description. Focus on: missing handoff points, unclear decision gateways, unmodeled exception/error paths, and whether lane ownership is correctly assigned.",
    "rtm":        "This is a Requirements Traceability Matrix. Focus on: whether every requirement has verifiable test coverage, MoSCoW consistency, and traceability gaps back to source requirements.",
    "analytics":  "This is an analytics/insights report. Focus on: whether conclusions are actually supported by the data shown, missing caveats, and whether recommended actions are specific and actionable.",
    "email":      "This is a stakeholder email/meeting summary. Focus on: whether action items have clear owners and deadlines, ambiguous language, and decisions implied but not explicitly stated.",
    "backlog":    "This is a product backlog. Focus on: prioritization justification, dependency gaps between epics/stories, and whether definition-of-ready is met.",
    "prioritize": "This is a prioritization exercise. Focus on: whether the scoring (WSJF/MoSCoW) is justified and consistent, and whether business-value claims are substantiated.",
}

MEETING_EXTRACT_SYSTEM = """You are a Business Analyst extracting structured requirements from a raw meeting transcript.
Read the transcript and identify: stated requirements, decisions made, open questions, stakeholders mentioned, and any process steps described.
Output ONLY valid JSON, no markdown fences:
{"meeting_title":"<short descriptive title>",
 "key_decisions":["<decision 1>", "..."],
 "requirements_summary":"<2-3 sentence plain-English summary of what needs to be built>",
 "stakeholders":["<name/role>", "..."],
 "process_steps":["<step 1 in order>", "..."],
 "open_questions":["<unresolved question>", "..."]}
If the transcript has no clear process flow, return an empty process_steps list."""

# ── In-memory histories (unchanged) ─────────────────────────────
ba_histories = {m: [] for m in BA_MODES}
po_histories = {m: [] for m in PO_MODES}
session_log  = []
SAVE_PATH    = "BA_PO_PM_Workspace_History.xlsx"
_bpmn_store  = {"xml":"","ts":0}

# ════════════════════════════════════════════════════════════════
# CORE HELPERS
# ════════════════════════════════════════════════════════════════
def resolve_requirements_document_type(chat_id, message):
    """Resolve the active requirements type without being affected by an older selected artifact."""
    if re.search(
        r"\b(BRD|FRD|PRD|SRS|FSD|SOP|SOW)\b"
        r"|\bSTANDARD OPERATING PROCEDURE\b"
        r"|\bSCOPE OF WORK\b"
        r"|\bSOFTWARE REQUIREMENTS SPECIFICATION\b"
        r"|\bFUNCTIONAL SPECIFICATION DOCUMENT\b",
        (message or "").upper(),
    ):
        return detect_document_type(message)

    if chat_id:
        latest = get_latest_workspace_artifact(chat_id)
        if latest:
            previous = (latest.get("document_type") or "").upper()
            if previous in DOCUMENT_TYPE_CONFIG:
                return previous

        state = WorkspaceState.load(chat_id)
        previous = (state.get("document_type") or "").upper()
        if previous in DOCUMENT_TYPE_CONFIG:
            return previous

    return "BRD"


def build_document_type_instruction(document_type):
    """Build a strict model instruction for the selected document type."""
    document_type = (document_type or "BRD").upper()
    config = DOCUMENT_TYPE_CONFIG.get(
        document_type,
        DOCUMENT_TYPE_CONFIG["BRD"],
    )

    return (
        "IMPORTANT DOCUMENT TYPE CONTROL:\n"
        f"The required document type for this turn is {document_type}.\n"
        f"Document title: {config['title']}.\n"
        f"Definition: {config['description']}\n"
        f"You MUST produce a {document_type}.\n"
        "Do not substitute one supported document type for another.\n"
        "Use the correct document title and terminology throughout the output."
    )


def is_doc_request(role, mode, msg):
    """Return whether the active mode represents a workspace artifact."""
    return mode != "general"

def parse_po_json(reply):
    try:
        c = reply.strip()
        if "```" in c:
            c = re.sub(r'^```[a-z]*\n?','',c); c = re.sub(r'\n?```$','',c)
        start = c.find('{')
        if start < 0:
            return None
        # Balanced-brace scan: isolate exactly one JSON object, ignoring any
        # duplicated/garbage content appended after it (can happen if a
        # continuation round re-emits the object).
        depth, in_string, escape, end = 0, False, False, None
        for i in range(start, len(c)):
            ch = c[i]
            if in_string:
                if escape: escape = False
                elif ch == '\\': escape = True
                elif ch == '"': in_string = False
                continue
            if ch == '"': in_string = True
            elif ch == '{': depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break
        if end is None:
            return None
        return json.loads(c[start:end])
    except:
        return None

def extract_chart_data(reply):
    if "CHART_DATA" not in reply: return None
    sec  = reply.split("CHART_DATA",1)[1]
    rows = re.findall(r'\|\s*([^\|\n]+?)\s*\|\s*([\d.,%]+)\s*\|', sec)
    data = []
    for l, v in rows:
        l = l.strip()
        if l.lower() in ("label","---",""): continue
        try: data.append((l, float(v.replace(",","").replace("%",""))))
        except: continue
    return data or None

def extract_meeting_requirements(transcript_text):
    msgs = [{"role":"system","content": MEETING_EXTRACT_SYSTEM},
            {"role":"user","content": transcript_text[:12000]}]
    raw, ai_name, banner = AIProvider.call(msgs, max_tokens=4096)
    parsed = parse_po_json(raw)
    return parsed, banner
    
def run_meeting_pipeline(transcript_text):
    """Extracts requirements, then generates BRD + Stories (+ BPMN if a process was described).
    Returns a dict of created chat_ids and a status summary per artifact."""
    extracted, banner = extract_meeting_requirements(transcript_text)
    if not extracted:
        return {"error": "Could not extract structured requirements from this transcript."}

    title_base = extracted.get("meeting_title","Meeting")[:40]
    context = (f"Meeting: {extracted.get('meeting_title','')}\n"
               f"Summary: {extracted.get('requirements_summary','')}\n"
               f"Key decisions: {'; '.join(extracted.get('key_decisions',[]))}\n"
               f"Stakeholders: {', '.join(extracted.get('stakeholders',[]))}")

    results = {"extracted": extracted, "artifacts": []}

def run_meeting_pipeline(transcript_text):
    """Extracts requirements, then generates BRD + Stories (+ BPMN if a process was described).
    Returns a dict of created chat_ids and a status summary per artifact."""
    extracted, banner = extract_meeting_requirements(transcript_text)
    if not extracted:
        return {"error": "Could not extract structured requirements from this transcript."}

    title_base = extracted.get("meeting_title","Meeting")[:40]
    context = (f"Meeting: {extracted.get('meeting_title','')}\n"
               f"Summary: {extracted.get('requirements_summary','')}\n"
               f"Key decisions: {'; '.join(extracted.get('key_decisions',[]))}\n"
               f"Stakeholders: {', '.join(extracted.get('stakeholders',[]))}")

    results = {"extracted": extracted, "artifacts": []}

    # ── BRD ──────────────────────────────────────────────────────
    try:
        brd_chat = create_chat(role="ba", mode="brd", title=f"BRD — {title_base}")
        brd_msg  = f"Create a BRD based on this meeting:\n\n{context}"
        reply, ai_name, brd_banner = AIProvider.build_messages_and_call(brd_msg, "ba", "brd")
        save_message(brd_chat, "user", brd_msg, "brd")
        save_message(brd_chat, "assistant", reply, "brd")
        WorkspaceState.save(brd_chat, doc_content=reply, doc_mode="brd", document_type="BRD", workspace_open=True)
        update_chat_context(brd_chat, "ba", "brd")
        results["artifacts"].append({"type":"BRD","chat_id":brd_chat,"status":"created"})
    except Exception as e:
        results["artifacts"].append({"type":"BRD","status":"error","error":str(e)[:150]})

    # ── User Stories ─────────────────────────────────────────────
    try:
        st_chat = create_chat(role="ba", mode="stories", title=f"Stories — {title_base}")
        st_msg  = f"Create user stories based on this meeting:\n\n{context}"
        reply, ai_name, st_banner = AIProvider.build_messages_and_call(st_msg, "ba", "stories")
        save_message(st_chat, "user", st_msg, "stories")
        save_message(st_chat, "assistant", reply, "stories")
        WorkspaceState.save(st_chat, doc_content=reply, doc_mode="stories", workspace_open=True)
        update_chat_context(st_chat, "ba", "stories")
        parsed_stories = parse_po_json(reply)
        results["artifacts"].append({"type":"Stories","chat_id":st_chat,"status":"created",
                                      "cards": parsed_stories.get("cards", []) if parsed_stories else []})
    except Exception as e:
        results["artifacts"].append({"type":"Stories","status":"error","error":str(e)[:150]})

    # ── BPMN (only if a process was actually described) ─────────
    if extracted.get("process_steps"):
        try:
            bpmn_chat = create_chat(role="ba", mode="process", title=f"BPMN — {title_base}")
            bpmn_msg  = ("Generate a BPMN model for this process:\n" +
                         "\n".join(f"- {s}" for s in extracted["process_steps"]))
            bpmn_json_raw, ai_name, bpmn_banner = AIProvider.bpmn(bpmn_msg)
            bpmn_data = parse_po_json(bpmn_json_raw)
            bpmn_xml  = build_bpmn_xml(bpmn_data) if bpmn_data else ""
            save_message(bpmn_chat, "user", bpmn_msg, "process")
            save_message(bpmn_chat, "assistant",
                (f"[BPMN] {bpmn_json_raw[:300]}…" if bpmn_xml else f"[BPMN-FAILED] {bpmn_json_raw[:300]}…"), "process")
            WorkspaceState.save(bpmn_chat, doc_content=bpmn_json_raw, doc_mode="process",
                                 process_submode="bpmn", bpmn_xml=bpmn_xml, workspace_open=bool(bpmn_xml))
            update_chat_context(bpmn_chat, "ba", "process", "bpmn")
            results["artifacts"].append({"type":"BPMN","chat_id":bpmn_chat,
                                          "status":"created" if bpmn_xml else "error",
                                          "error": None if bpmn_xml else "BPMN parse failed"})
        except Exception as e:
            results["artifacts"].append({"type":"BPMN","status":"error","error":str(e)[:150]})

    return results
    created_ids = [a["chat_id"] for a in results["artifacts"] if a.get("status")=="created" and a.get("chat_id")]
    for i in range(len(created_ids)):
        for j in range(i+1, len(created_ids)):
            link_artifacts(created_ids[i], created_ids[j], relation="meeting-sibling")

    # ── BRD ──────────────────────────────────────────────────────
    try:
        brd_chat = create_chat(role="ba", mode="brd", title=f"BRD — {title_base}")
        brd_msg  = f"Create a BRD based on this meeting:\n\n{context}"
        reply, ai_name, brd_banner = AIProvider.build_messages_and_call(brd_msg, "ba", "brd")
        save_message(brd_chat, "user", brd_msg, "brd")
        save_message(brd_chat, "assistant", reply, "brd")
        WorkspaceState.save(brd_chat, doc_content=reply, doc_mode="brd", document_type="BRD", workspace_open=True)
        update_chat_context(brd_chat, "ba", "brd")
        results["artifacts"].append({"type":"BRD","chat_id":brd_chat,"status":"created"})
    except Exception as e:
        results["artifacts"].append({"type":"BRD","status":"error","error":str(e)[:150]})

    # ── User Stories ─────────────────────────────────────────────
    try:
        st_chat = create_chat(role="ba", mode="stories", title=f"Stories — {title_base}")
        st_msg  = f"Create user stories based on this meeting:\n\n{context}"
        reply, ai_name, st_banner = AIProvider.build_messages_and_call(st_msg, "ba", "stories")
        save_message(st_chat, "user", st_msg, "stories")
        save_message(st_chat, "assistant", reply, "stories")
        WorkspaceState.save(st_chat, doc_content=reply, doc_mode="stories", workspace_open=True)
        update_chat_context(st_chat, "ba", "stories")
        parsed_stories = parse_po_json(reply)
        results["artifacts"].append({"type":"Stories","chat_id":st_chat,"status":"created",
                                      "cards": parsed_stories.get("cards", []) if parsed_stories else []})
    except Exception as e:
        results["artifacts"].append({"type":"Stories","status":"error","error":str(e)[:150]})

    # ── BPMN (only if a process was actually described) ─────────
    if extracted.get("process_steps"):
        try:
            bpmn_chat = create_chat(role="ba", mode="process", title=f"BPMN — {title_base}")
            bpmn_msg  = ("Generate a BPMN model for this process:\n" +
                         "\n".join(f"- {s}" for s in extracted["process_steps"]))
            bpmn_json_raw, ai_name, bpmn_banner = AIProvider.bpmn(bpmn_msg)
            bpmn_data = parse_po_json(bpmn_json_raw)
            bpmn_xml  = build_bpmn_xml(bpmn_data) if bpmn_data else ""
            save_message(bpmn_chat, "user", bpmn_msg, "process")
            save_message(bpmn_chat, "assistant",
                (f"[BPMN] {bpmn_json_raw[:300]}…" if bpmn_xml else f"[BPMN-FAILED] {bpmn_json_raw[:300]}…"), "process")
            WorkspaceState.save(bpmn_chat, doc_content=bpmn_json_raw, doc_mode="process",
                                 process_submode="bpmn", bpmn_xml=bpmn_xml, workspace_open=bool(bpmn_xml))
            update_chat_context(bpmn_chat, "ba", "process", "bpmn")
            results["artifacts"].append({"type":"BPMN","chat_id":bpmn_chat,
                                          "status":"created" if bpmn_xml else "error",
                                          "error": None if bpmn_xml else "BPMN parse failed"})
        except Exception as e:
            results["artifacts"].append({"type":"BPMN","status":"error","error":str(e)[:150]})

    return results

# ════════════════════════════════════════════════════════════════
# BPMN XML BUILDER
# ════════════════════════════════════════════════════════════════
def esc(s):
    return saxutils.escape(str(s)) if s else ""

_BPMN_TAGMAP = {
    "startEvent":"startEvent","endEvent":"endEvent","task":"task",
    "userTask":"userTask","serviceTask":"serviceTask",
    "exclusiveGateway":"exclusiveGateway","parallelGateway":"parallelGateway",
    "inclusiveGateway":"inclusiveGateway",
    "intermediateCatchEvent":"intermediateCatchEvent",
    "intermediateThrowEvent":"intermediateThrowEvent",
}

def _bpmn_element_xml(eid, etype, ename):
    e = esc(ename)
    if etype == "subProcess":
        return f'<bpmn:subProcess id="{eid}" name="{e}"/>'
    if etype == "intermediateTimerEvent":
        return f'<bpmn:intermediateCatchEvent id="{eid}" name="{e}"><bpmn:timerEventDefinition/></bpmn:intermediateCatchEvent>'
    tag = _BPMN_TAGMAP.get(etype, "task")
    return f'<bpmn:{tag} id="{eid}" name="{e}"/>'

def _bpmn_shape_xml(eid, x, y, w, h, is_subprocess=False):
    ea = ' isExpanded="false"' if is_subprocess else ''
    return (f'<bpmndi:BPMNShape id="{eid}_di" bpmnElement="{eid}"{ea}>'
            f'<dc:Bounds x="{x:.0f}" y="{y:.0f}" width="{w:.0f}" height="{h:.0f}"/>'
            f'</bpmndi:BPMNShape>')

def _bpmn_lane_shape_xml(lid, x, y, w, h):
    return (f'<bpmndi:BPMNShape id="{lid}_di" bpmnElement="{lid}" isHorizontal="true">'
            f'<dc:Bounds x="{x:.0f}" y="{y:.0f}" width="{w:.0f}" height="{h:.0f}"/>'
            f'</bpmndi:BPMNShape>')

def _bpmn_pool_shape_xml(pid, x, y, w, h):
    return (f'<bpmndi:BPMNShape id="{pid}_participant_di" bpmnElement="{pid}_participant" isHorizontal="true">'
            f'<dc:Bounds x="{x:.0f}" y="{y:.0f}" width="{w:.0f}" height="{h:.0f}"/>'
            f'</bpmndi:BPMNShape>')

def build_bpmn_xml(data):
    if not data or not data.get("pools"): return ""
    LANE_H, GAP_X, START_X, POOL_GAP = 160, 150, 200, 80
    TW, TH, EW, EH, GW, GH = 100, 80, 36, 36, 50, 50
    pools = data.get("pools",[]); seq_flows = data.get("sequenceFlows",[]); msg_flows = data.get("messageFlows",[])
    proc_parts, participants, shapes, edges = [], [], [], []
    elem_pos, elem_proc = {}, {}
    pool_y = 0
    for p_i, pool in enumerate(pools):
        pid, proc_id = pool.get("id") or f"Pool_{p_i+1}", f"Process_{pool.get('id','Pool_'+str(p_i+1))}"
        lanes = pool.get("lanes",[])
        lane_xml, elems_xml, lane_shapes = [], [], []
        pool_h = max_x = 0
        for l_i, lane in enumerate(lanes):
            lid = lane.get("id") or f"{pid}_Lane_{l_i+1}"
            elems = lane.get("elements",[]); refs = []; x = START_X
            lt = pool_y + l_i * LANE_H; yc = lt + LANE_H / 2
            for e in elems:
                eid,etype,ename = e["id"],e.get("type","task"),e.get("name","")
                elem_proc[eid] = proc_id; refs.append(eid)
                is_e = etype in ("startEvent","endEvent") or "Event" in etype
                is_g = "Gateway" in etype
                w,h = (EW,EH) if is_e else (GW,GH) if is_g else (TW,TH)
                elem_pos[eid] = (x, yc-h/2, w, h)
                elems_xml.append(_bpmn_element_xml(eid,etype,ename))
                shapes.append(_bpmn_shape_xml(eid,x,yc-h/2,w,h,is_subprocess=(etype=="subProcess")))
                x += w + GAP_X
            max_x = max(max_x, x)
            refs_xml = "".join(f'<bpmn:flowNodeRef>{r}</bpmn:flowNodeRef>' for r in refs)
            lane_xml.append(f'<bpmn:lane id="{lid}" name="{esc(lane.get("name",""))}">{refs_xml}</bpmn:lane>')
            lane_shapes.append((lid, lt)); pool_h += LANE_H
        if pool_h == 0: pool_h = LANE_H
        ls = f'<bpmn:laneSet id="{pid}_LaneSet">{"".join(lane_xml)}</bpmn:laneSet>' if lanes else ""
        proc_parts.append({"id":proc_id,"xml":f'<bpmn:process id="{proc_id}" name="{esc(pool.get("name",""))}" isExecutable="false">{ls}{"".join(elems_xml)}'})
        participants.append((pid, pool.get("name",""), proc_id))
        pw = max_x + 60
        for lid, lt in lane_shapes:
            shapes.append(_bpmn_lane_shape_xml(lid, 30, lt, pw-30, LANE_H))
        shapes.append(_bpmn_pool_shape_xml(pid, 0, pool_y, pw, pool_h))
        pool_y += pool_h + POOL_GAP
    seq_by_proc = {}
    for f in seq_flows:
        src,tgt = f.get("source"),f.get("target")
        if src not in elem_pos or tgt not in elem_pos: continue
        proc = elem_proc.get(src) or elem_proc.get(tgt)
        seq_by_proc.setdefault(proc,[]).append(f)
        sx1,sy1,sw1,sh1 = elem_pos[src]; sx2,sy2,sw2,sh2 = elem_pos[tgt]
        edges.append(f'<bpmndi:BPMNEdge id="{f["id"]}_di" bpmnElement="{f["id"]}"><di:waypoint x="{sx1+sw1:.0f}" y="{sy1+sh1/2:.0f}"/><di:waypoint x="{sx2:.0f}" y="{sy2+sh2/2:.0f}"/></bpmndi:BPMNEdge>')
    final_procs = []
    for part in proc_parts:
        flows = seq_by_proc.get(part["id"],[])
        fx = "".join(f'<bpmn:sequenceFlow id="{f["id"]}" name="{esc(f.get("name",""))}" sourceRef="{f["source"]}" targetRef="{f["target"]}"/>' for f in flows)
        final_procs.append(part["xml"] + fx + "</bpmn:process>")
    mf_xml = ""
    if msg_flows and len(pools) > 1:
        mparts = []
        for m in msg_flows:
            src,tgt = m.get("source"),m.get("target")
            if src not in elem_pos or tgt not in elem_pos: continue
            mparts.append(f'<bpmn:messageFlow id="{m["id"]}" name="{esc(m.get("name",""))}" sourceRef="{src}" targetRef="{tgt}"/>')
            sx1,sy1,sw1,sh1 = elem_pos[src]; sx2,sy2,sw2,sh2 = elem_pos[tgt]
            edges.append(f'<bpmndi:BPMNEdge id="{m["id"]}_di" bpmnElement="{m["id"]}"><di:waypoint x="{sx1+sw1/2:.0f}" y="{sy1+sh1:.0f}"/><di:waypoint x="{sx2+sw2/2:.0f}" y="{sy2:.0f}"/></bpmndi:BPMNEdge>')
        mf_xml = "".join(mparts)
    part_xml = "".join(f'<bpmn:participant id="{pid}_participant" name="{esc(pn)}" processRef="{pr}"/>' for pid,pn,pr in participants)
    collab = f'<bpmn:collaboration id="Collaboration_1">{part_xml}{mf_xml}</bpmn:collaboration>'
    return ('<?xml version="1.0" encoding="UTF-8"?>\n'
            '<bpmn:definitions xmlns:bpmn="http://www.omg.org/spec/BPMN/20100524/MODEL" '
            'xmlns:bpmndi="http://www.omg.org/spec/BPMN/20100524/DI" '
            'xmlns:dc="http://www.omg.org/spec/DD/20100524/DC" '
            'xmlns:di="http://www.omg.org/spec/DD/20100524/DI" '
            'id="Definitions_1" targetNamespace="http://bpmn.io/schema/bpmn">\n'
            f'{collab}\n{"".join(final_procs)}\n'
            '<bpmndi:BPMNDiagram id="BPMNDiagram_1">'
            '<bpmndi:BPMNPlane id="BPMNPlane_1" bpmnElement="Collaboration_1">'
            f'{"".join(shapes)}{"".join(edges)}'
            '</bpmndi:BPMNPlane></bpmndi:BPMNDiagram>\n</bpmn:definitions>')

# ════════════════════════════════════════════════════════════════
# JIRA CARD RENDERER
# ════════════════════════════════════════════════════════════════
def render_scope_management_dashboard():
    """Render the Scope Management landing dashboard before generation."""
    return html.Div([
        html.Div([
            html.Div("🎯", style={"fontSize": "34px"}),
            html.H3("Scope Management", style={"fontWeight": "700", "fontSize": "21px", "margin": "4px 0"}),
            html.P(
                "Define, baseline, trace, control and govern project scope — then connect approved scope to JIRA delivery.",
                style={"fontSize": "13px", "color": "#666", "maxWidth": "760px", "margin": "0 auto"},
            ),
        ], style={"textAlign": "center", "padding": "28px 20px 18px"}),
        dbc.Row([
            dbc.Col(dbc.Card(dbc.CardBody([
                html.Div("📄", style={"fontSize": "24px"}),
                html.H6("Scope Document", style={"fontWeight": "700"}),
                html.P("Persistent baseline, WBS, deliverables, traceability, change register and creep register.",
                       style={"fontSize": "11px", "color": "#666"}),
            ]), style={"height": "100%"}), md=4),
            dbc.Col(dbc.Card(dbc.CardBody([
                html.Div("🚀", style={"fontSize": "24px"}),
                html.H6("JIRA Delivery", style={"fontWeight": "700"}),
                html.P("Generate Epics, Stories and Tasks with WBS and Scope Baseline traceability.",
                       style={"fontSize": "11px", "color": "#666"}),
            ]), style={"height": "100%"}), md=4),
            dbc.Col(dbc.Card(dbc.CardBody([
                html.Div("🔎", style={"fontSize": "24px"}),
                html.H6("Scope Control", style={"fontWeight": "700"}),
                html.P("Submit scope changes and scan JIRA for potential scope creep against the baseline.",
                       style={"fontSize": "11px", "color": "#666"}),
            ]), style={"height": "100%"}), md=4),
        ], className="g-3", style={"maxWidth": "1000px", "margin": "0 auto"}),
        html.Div([
            html.H5("Recommended starting prompt", style={"fontWeight": "700", "fontSize": "14px"}),
            html.Div(
                "Create a Scope Management baseline including in-scope/out-of-scope boundaries, WBS, "
                "deliverables, acceptance criteria, requirements traceability and JIRA Epic/Story/Task mapping.",
                style={"fontSize": "12px", "color": "#444", "background": "#f8fbff",
                       "border": "1px solid #dbe7f5", "borderRadius": "8px", "padding": "12px"},
            ),
        ], style={"maxWidth": "1000px", "margin": "18px auto 0"}),
    ], style={"padding": "12px 20px"})


def render_scope_management(data, jira_map=None):
    """Render the persistent Scope Management document and JIRA delivery view."""
    if not isinstance(data, dict):
        return html.Div("Invalid Scope Management data.", style={"color": "#c62828"})

    baseline = data.get("scope_baseline") or {}
    wbs = data.get("wbs") or []
    traceability = data.get("traceability") or []
    cards = data.get("cards") or []
    change_requests = data.get("change_requests") or []
    creep_findings = data.get("scope_creep_findings") or []

    def section(title, content):
        return html.Div([
            html.H5(title, style={"fontWeight": "700", "fontSize": "15px", "marginBottom": "8px"}),
            content,
        ], style={
            "background": "#fff", "border": "1px solid #e2e8f0",
            "borderRadius": "8px", "padding": "14px", "marginBottom": "12px",
        })

    def list_block(title, values):
        if not values:
            return html.Div()
        return html.Div([
            html.Div(title, style={"fontWeight": "700", "fontSize": "12px", "marginTop": "8px"}),
            html.Ul([
                html.Li(str(value), style={"fontSize": "12px", "marginBottom": "3px"})
                for value in values
            ], style={"paddingLeft": "20px", "marginBottom": "4px"}),
        ])

    baseline_document = html.Div([
        html.H4(
            data.get("title", "Scope Management"),
            style={"fontWeight": "700", "fontSize": "18px", "color": "#1565c0", "marginBottom": "4px"},
        ),
        html.P(
            data.get("summary", ""),
            style={"fontSize": "12px", "color": "#666", "marginBottom": "12px"},
        ),
        section("1. Scope Baseline", html.Div([
            html.Div(
                f"Version {baseline.get('version', 'TBD')} · "
                f"Status: {baseline.get('approval_status', 'Draft')} · "
                f"Acceptance Authority: {baseline.get('acceptance_authority', 'TBD')}",
                style={"fontSize": "11px", "color": "#666", "marginBottom": "8px"},
            ),
            html.P(
                baseline.get("scope_statement", "No scope statement provided."),
                style={"fontSize": "13px", "lineHeight": "1.5"},
            ),
            list_block("In Scope", baseline.get("in_scope", [])),
            list_block("Out of Scope", baseline.get("out_of_scope", [])),
            list_block("Assumptions", baseline.get("assumptions", [])),
            list_block("Constraints", baseline.get("constraints", [])),
            list_block("Dependencies", baseline.get("dependencies", [])),
        ])),
        section("2. Work Breakdown Structure (WBS)", html.Div([
            dbc.Table([
                html.Thead(html.Tr([
                    html.Th("WBS"), html.Th("Parent"), html.Th("Work Package"),
                    html.Th("Deliverable"), html.Th("Acceptance Criteria"),
                ])),
                html.Tbody([
                    html.Tr([
                        html.Td(item.get("id", ""), style={"fontWeight": "600"}),
                        html.Td(item.get("parent_id") or "—"),
                        html.Td(item.get("title", "")),
                        html.Td(item.get("deliverable", "")),
                        html.Td("; ".join(item.get("acceptance_criteria", []))),
                    ]) for item in wbs
                ]),
            ], bordered=True, hover=True, size="sm", responsive=True)
            if wbs else html.Div("No WBS items defined.", style={"fontSize": "12px", "color": "#999"}),
        ])),
        section("3. Requirements-to-Scope Traceability", html.Div([
            dbc.Table([
                html.Thead(html.Tr([
                    html.Th("Requirement"), html.Th("WBS"),
                    html.Th("Deliverable"), html.Th("Coverage"),
                ])),
                html.Tbody([
                    html.Tr([
                        html.Td(item.get("requirement_id", "")),
                        html.Td(item.get("wbs_id", "")),
                        html.Td(item.get("deliverable", "")),
                        html.Td(item.get("coverage", "")),
                    ]) for item in traceability
                ]),
            ], bordered=True, hover=True, size="sm", responsive=True)
            if traceability else html.Div("No traceability records defined.", style={"fontSize": "12px", "color": "#999"}),
        ])),
        section("4. Deliverables & JIRA Traceability", html.Div([
            html.P(
                "The Scope Document is the control record. JIRA Epics, Stories and Tasks are delivery records linked back to the baseline and WBS.",
                style={"fontSize": "12px", "color": "#666"},
            ),
            html.Ul([
                html.Li(
                    f"{item.get('id', '')} · {item.get('title', '')} · "
                    f"{'; '.join(item.get('tags', []))}",
                    style={"fontSize": "12px", "marginBottom": "4px"},
                ) for item in cards
            ]) if cards else html.Div("No JIRA-ready work items defined.", style={"fontSize": "12px", "color": "#999"}),
        ])),
        section("5. Scope Change Register", html.Div([
            html.Div([
                html.Div(
                    f"{item.get('id', 'CR-TBD')} · {item.get('requested_change', item.get('title', ''))}",
                    style={"fontWeight": "700", "fontSize": "12px"},
                ),
                html.Div(
                    f"Status: {item.get('approval_status', item.get('status', 'Proposed'))} · "
                    f"Classification: {item.get('classification', 'TBD')}",
                    style={"fontSize": "11px", "color": "#666", "marginTop": "2px"},
                ),
                html.Div(
                    item.get("recommendation", ""),
                    style={"fontSize": "11px", "marginTop": "4px"},
                ),
            ], style={
                "borderLeft": "3px solid #f9a825", "padding": "7px 10px",
                "marginBottom": "7px", "background": "#fffaf0",
            }) for item in change_requests
        ] if change_requests else [
            html.Div("No scope change requests recorded.", style={"fontSize": "12px", "color": "#999"})
        ])),
        section("6. Scope Creep Register", html.Div([
            html.Div([
                html.Div(
                    f"{item.get('severity', 'TBD')} · {item.get('linked_jira_key', 'JIRA-TBD')} · "
                    f"{item.get('observed_item', '')}",
                    style={"fontWeight": "700", "fontSize": "12px"},
                ),
                html.Div(item.get("reason_flagged", ""), style={"fontSize": "11px", "color": "#666", "marginTop": "3px"}),
                html.Div(
                    f"Recommendation: {item.get('recommendation', '')}",
                    style={"fontSize": "11px", "marginTop": "3px"},
                ),
            ], style={
                "borderLeft": "3px solid #c62828", "padding": "7px 10px",
                "marginBottom": "7px", "background": "#fff7f7",
            }) for item in creep_findings
        ] if creep_findings else [
            html.Div("No scope-creep findings recorded.", style={"fontSize": "12px", "color": "#999"})
        ])),
    ], style={"padding": "16px 20px"})

    jira_view = html.Div([
        html.H4("🚀 Scope → JIRA Delivery Backlog", style={
            "fontWeight": "700", "fontSize": "17px", "color": "#1565c0", "marginBottom": "4px",
        }),
        html.P(
            "Epics, Stories and Tasks generated from the Scope Baseline and WBS.",
            style={"fontSize": "12px", "color": "#666"},
        ),
        render_jira_cards({
            "title": "JIRA Delivery Backlog",
            "summary": "Every work item remains traceable to the Scope Baseline and WBS.",
            "cards": cards,
        }, jira_map=jira_map),
    ], style={"padding": "16px 20px"})

    return dbc.Tabs([
        dbc.Tab(baseline_document, label="📄 Scope Document", tab_id="scope-document"),
        dbc.Tab(jira_view, label="🚀 JIRA Delivery", tab_id="scope-jira"),
    ], active_tab="scope-document")


def render_jira_cards(data, jira_map=None):
    if not data or "cards" not in data:
        return html.Div("No cards.", style={"color":"#bbb","textAlign":"center","marginTop":"60px"})
    elems = [html.Div([
        html.H5(data.get("title",""), style={"color":"#1a73e8","fontWeight":"700","fontSize":"15px","marginBottom":"4px"}),
        html.P(data.get("summary",""), style={"color":"#666","fontSize":"12px","marginBottom":"12px"})
    ])]
    for item in data.get("cards",[]):
        ct=item.get("type","item"); col=CARD_COLORS.get(ct,"#1a73e8"); icon=CARD_ICONS.get(ct,"📋")
        pri=item.get("priority",""); sts=item.get("status",""); pts=item.get("points")
        tags=item.get("tags",[]); acs=item.get("acceptance_criteria",[]); subs=item.get("sub_items",[])

        jira_info  = (jira_map or {}).get(item.get("id"))
        jira_badge = html.Div()
        if jira_info and jira_info.get("jira_key"):
            type_tag = f" [{jira_info['issue_type']}]" if jira_info.get("issue_type") else ""
            base_link = html.A(f"🔗 {jira_info['jira_key']}{type_tag}", href=jira_info.get("jira_url","#"),
                target="_blank", style={"fontSize":"10px","color":"#0052cc","marginLeft":"8px",
                "fontWeight":"700","textDecoration":"none"})
            drifted = jira_info.get("content_hash") and jira_info["content_hash"] != _card_content_hash(item)
            jira_badge = html.Span([base_link,
                html.Span(" ⚠️ changed since sync", style={"fontSize":"10px","color":"#e65100",
                    "fontWeight":"700","marginLeft":"4px"})]) if drifted else base_link
        elif jira_info and jira_info.get("status")=="error":
            jira_badge = html.Span("⚠️ sync failed", style={"fontSize":"10px","color":"#c62828","marginLeft":"8px"})

        elems.append(html.Div([
            html.Div([
                html.Span(f"{icon} {ct.upper()}", style={"fontSize":"10px","fontWeight":"700","color":"white","background":col,"padding":"2px 8px","borderRadius":"4px","marginRight":"8px"}),
                html.Span(item.get("id",""), style={"fontSize":"11px","color":"#888","flex":"1"}),
                html.Span(f"● {pri}", style={"fontSize":"11px","color":PRIORITY_COLORS.get(pri,"#666"),"fontWeight":"600","marginRight":"8px"}),
                html.Span(sts, style={"fontSize":"10px","color":"white","background":STATUS_COLORS.get(sts,"#888"),"padding":"2px 6px","borderRadius":"10px"}),
                html.Span(f" {pts}pts" if pts else "", style={"fontSize":"11px","color":"#888","marginLeft":"6px"}),
                jira_badge,
            ], style={"display":"flex","alignItems":"center","marginBottom":"8px"}),
            html.Div(item.get("title",""), style={"fontWeight":"600","fontSize":"13px","color":"#202124","marginBottom":"5px"}),
            html.Div(item.get("description",""), style={"fontSize":"12px","color":"#555","marginBottom":"8px","lineHeight":"1.5"}),
            html.Div([html.Span(t, style={"fontSize":"10px","background":"#e8f0fe","color":"#1a73e8","padding":"2px 7px","borderRadius":"10px","marginRight":"4px"}) for t in tags],
                style={"marginBottom":"6px","display":"flex","flexWrap":"wrap"}) if tags else html.Div(),
            html.Div([
                html.Div("Acceptance Criteria:", style={"fontSize":"11px","fontWeight":"700","color":"#666","marginTop":"6px"}),
                html.Div([html.Div([html.Span("✓ ",style={"color":"#2e7d32","fontWeight":"bold"}),html.Span(a,style={"fontSize":"12px","color":"#444"})],style={"marginBottom":"3px"}) for a in acs])
            ]) if acs else html.Div(),
            html.Div([
                html.Div("Implementation Tasks:", style={"fontSize":"11px","fontWeight":"700","color":"#666","marginTop":"6px"}),
                html.Div([html.Div([html.Span("▸ ",style={"color":col,"fontWeight":"bold"}),html.Span(s,style={"fontSize":"12px","color":"#444"})],style={"marginBottom":"3px"}) for s in subs])
            ]) if subs else html.Div(),
        ], style={"background":"white","border":"1px solid #e0e0e0","borderLeft":f"4px solid {col}","borderRadius":"8px","padding":"14px 16px","marginBottom":"10px","boxShadow":"0 1px 3px rgba(0,0,0,0.08)"}))
    return html.Div(elems, style={"padding":"16px 12px"})

# ════════════════════════════════════════════════════════════════
# SIDEBAR RENDERER
# ════════════════════════════════════════════════════════════════
def render_sidebar_tabs():
    projects = load_projects()
    archived = load_archived_chats()

    proj_sections = []
    for p in projects:
        chats   = load_chats_for_project(p["id"])
        chat_rows = []
        for c in chats:
            icon = {"ba":"📊","po":"🎯","pm":"🧭"}.get(c["role"], "📊")
            try: ds = datetime.fromisoformat(c["updated_at"]).strftime("%d %b")
            except: ds = ""
            chat_rows.append(html.Div([
                html.Div([
                    html.Span(f"{icon} ", style={"fontSize":"11px","flexShrink":0}),
                    html.Div(c["title"], style={"fontSize":"12px","fontWeight":"600","flex":"1",
                        "overflow":"hidden","textOverflow":"ellipsis","whiteSpace":"nowrap","color":"#333"}),
                ], id={"type":"sidebar-item","index":c["id"]}, n_clicks=0,
                   style={"display":"flex","alignItems":"center","flex":"1","minWidth":0,"cursor":"pointer"}),
                html.Div([
                    html.Span(ds, style={"fontSize":"10px","color":"#bbb","marginRight":"4px"}),
                    dbc.DropdownMenu(label="⋮", size="sm", color="light",
                        toggle_style={"padding":"0 4px","fontSize":"14px","lineHeight":"1"},
                        children=[
                            dbc.DropdownMenuItem("✏️ Rename",            id={"type":"sidebar-rename-btn","index":c["id"]},  n_clicks=0),
                            dbc.DropdownMenuItem("📁 Move to Project",    id={"type":"sidebar-move-btn","index":c["id"]},    n_clicks=0),
                            dbc.DropdownMenuItem("🗄️ Archive",            id={"type":"sidebar-archive-btn","index":c["id"]}, n_clicks=0),
                            dbc.DropdownMenuItem("🗑️ Delete",             id={"type":"sidebar-delete-btn","index":c["id"]},  n_clicks=0, style={"color":"#c62828"}),
                        ]),
                ], style={"display":"flex","alignItems":"center","flexShrink":0})
            ], style={"display":"flex","alignItems":"center","padding":"7px 10px","borderRadius":"6px",
                      "marginBottom":"3px","background":"white","border":"1px solid #eee","gap":"4px"}))

        is_collapsed = bool(p.get("is_collapsed"))
        proj_sections.append(html.Div([
            html.Div([
                (
                    html.Button(
                        f"▼ {'📂' if chats else '📁'} {p['name']}",
                        id={"type": "project-collapse-btn", "index": p["id"]},
                        n_clicks=0,
                        type="button",
                        style={
                            "fontSize": "12px",
                            "fontWeight": "600",
                            "color": "#333",
                            "flex": "1",
                            "cursor": "pointer",
                            "userSelect": "none",
                            "textAlign": "left",
                            "border": "0",
                            "background": "transparent",
                            "padding": "0",
                            "margin": "0",
                        },
                    )
                    if not is_collapsed
                    else html.Button(
                        f"▶ {'📂' if chats else '📁'} {p['name']}",
                        id={"type": "project-expand-btn", "index": p["id"]},
                        n_clicks=0,
                        type="button",
                        style={
                            "fontSize": "12px",
                            "fontWeight": "600",
                            "color": "#333",
                            "flex": "1",
                            "cursor": "pointer",
                            "userSelect": "none",
                            "textAlign": "left",
                            "border": "0",
                            "background": "transparent",
                            "padding": "0",
                            "margin": "0",
                        },
                    )
                ),
                html.Span(f"({len(chats)})", style={"fontSize":"10px","color":"#999","marginRight":"6px"}),
                html.Span("✏️", id={"type":"rename-project-btn","index":p["id"]}, n_clicks=0,
                    style={"cursor":"pointer","fontSize":"11px","opacity":"0.5","marginRight":"4px"},
                    title="Rename") if not p.get("is_default") else html.Div(),
                html.Span("🗑️", id={"type":"delete-project-btn","index":p["id"]}, n_clicks=0,
                    style={"cursor":"pointer","fontSize":"11px","opacity":"0.5"},
                    title="Delete") if not p.get("is_default") else html.Div(),
            ], style={"display":"flex","alignItems":"center","padding":"5px 8px",
                      "background":"#f0f4ff","borderRadius":"6px","marginBottom":"4px"}),
            html.Div(
                [] if is_collapsed else (
                    chat_rows if chat_rows else
                    [html.Div("No chats yet", style={"fontSize":"11px","color":"#bbb","padding":"4px 12px"})]
                ),
                style={"paddingLeft":"4px","display":"none" if is_collapsed else "block"}
            )
        ], style={"marginBottom":"12px"}))

    arch_rows = []
    for c in archived:
        arch_rows.append(html.Div([
            html.Div(c["title"], style={"fontSize":"12px","color":"#666","flex":"1",
                "overflow":"hidden","textOverflow":"ellipsis","whiteSpace":"nowrap"}),
            html.Span("↩️", id={"type":"sidebar-unarchive-btn","index":c["id"]}, n_clicks=0,
                style={"cursor":"pointer","fontSize":"14px","flexShrink":0}, title="Restore"),
        ], style={"display":"flex","alignItems":"center","padding":"7px 10px","borderRadius":"6px",
                  "marginBottom":"4px","background":"#f8f8f8","border":"1px solid #eee"}))

    # Point 3: No "Uncategorised" heading near the New Project button
    return dbc.Tabs([
        dbc.Tab(html.Div([
            html.Div([
                dbc.Button("Collapse All", id="collapse-all-projects-btn", color="light", size="sm", outline=True, className="me-1", style={"fontSize":"10px"}),
                dbc.Button("Expand All", id="expand-all-projects-btn", color="light", size="sm", outline=True, style={"fontSize":"10px"}),
            ], style={"display":"flex","marginBottom":"8px"}),
            html.Div(proj_sections),
            dbc.Button("＋ New Project", id="new-project-btn", color="light", size="sm",
                outline=True, style={"width":"100%","marginTop":"8px","fontSize":"12px"})
        ], style={"padding":"10px 8px"}), label="📁 Projects", tab_id="tp"),
        dbc.Tab(html.Div(
            arch_rows if arch_rows else
            [html.Div("No archived chats", style={"color":"#bbb","fontSize":"12px","padding":"12px","textAlign":"center"})],
            style={"padding":"10px 8px"}), label=f"🗄️ Archived ({len(archived)})", tab_id="ta"),
    ], active_tab="tp")

# ════════════════════════════════════════════════════════════════
# REQUIREMENTS DOCUMENT CONVERSION
# ════════════════════════════════════════════════════════════════

def build_document_conversion_prompt(source_artifact, target_type):
    """Build a semantic conversion prompt while preserving source facts."""
    source_type = resolve_artifact_document_type(source_artifact) or "BRD"
    target_type = (target_type or "").upper()
    target = DOCUMENT_TYPE_CONFIG.get(target_type)
    if not target:
        raise ValueError(f"Unsupported target document type: {target_type}")

    source = (source_artifact.get("doc_content") or "").strip()
    if not source:
        raise ValueError("The source artifact has no document content.")

    return f"""
You are a senior Business Analyst responsible for semantic document conversion.

SOURCE DOCUMENT TYPE: {source_type}
SOURCE DOCUMENT TITLE: {get_document_type_title(source_type)}
TARGET DOCUMENT TYPE: {target_type}
TARGET DOCUMENT TITLE: {target["title"]}

TARGET PURPOSE:
{target["description"]}

TARGET FOCUS:
{chr(10).join("- " + item for item in target["focus"])}

CONVERSION RULES:
1. Return ONLY the complete converted document in Markdown.
2. Preserve every relevant source requirement, business rule, decision,
   dependency, assumption, constraint, and traceability identifier that remains
   meaningful for the target document.
3. Transform the actual structure and terminology to fit the target type.
4. Do not merely rename the title or headings.
5. Do not invent unsupported requirements, dates, owners, regulations,
   commitments, metrics, interfaces, or commercial terms.
6. Where source information is insufficient for a target section, state
   "Not specified in source" rather than inventing facts.
7. Preserve requirement IDs and traceability references where possible.
8. Adapt technical or operational detail to the target document's purpose.
9. Preserve important business rules and acceptance criteria where meaningful.
10. Start with the exact target title:
    #{target["title"]}
11. Use a professional, implementation-ready document structure.

SOURCE DOCUMENT:
{source[:70000]}
""".strip()


def convert_workspace_artifact(source_artifact, target_type):
    """Create a new immutable artifact from semantic document conversion."""
    if not source_artifact:
        raise ValueError("Source workspace artifact was not found.")

    target_type = (target_type or "").upper()
    if target_type not in DOCUMENT_TYPE_CONFIG:
        raise ValueError(f"Unsupported target document type: {target_type}")

    source_type = resolve_artifact_document_type(source_artifact) or "BRD"
    if source_type == target_type:
        raise ValueError("Source and target document types must be different.")

    raw, ai_name, banner = AIProvider.call(
        [
            {
                "role": "system",
                "content": (
                    "You are a precise document conversion engine. "
                    "Return only the requested Markdown artifact. "
                    "Never invent unsupported facts."
                ),
            },
            {
                "role": "user",
                "content": build_document_conversion_prompt(
                    source_artifact,
                    target_type,
                ),
            },
        ],
        max_tokens=20000,
    )

    converted = (raw or "").strip()
    converted = re.sub(r"^```(?:markdown|md)?\s*", "", converted, flags=re.IGNORECASE)
    converted = re.sub(r"\s*```$", "", converted).strip()

    if not converted:
        raise ValueError("The AI returned an empty converted document.")

    title = get_document_type_title(target_type)
    if not re.search(
        rf"^#\s*{re.escape(title)}\s*$",
        converted,
        flags=re.IGNORECASE | re.MULTILINE,
    ):
        converted = f"# {title}\n\n{converted}"

    return {
        "doc_content": converted,
        "document_type": target_type,
        "source_document_type": source_type,
        "source_artifact_id": source_artifact.get("id"),
        "ai_name": ai_name or "",
        "banner": banner or "",
    }


# ════════════════════════════════════════════════════════════════
# EXPORT FUNCTIONS
# ════════════════════════════════════════════════════════════════
def scope_payload_to_markdown(content):
    """Convert the structured Scope Management payload into a trackable document."""
    try:
        payload = parse_scope_payload(content)
    except Exception:
        payload = None
    if not payload:
        return content or ""

    baseline = payload.get("scope_baseline") or {}
    lines = [
        f"# {payload.get('title', 'Scope Management')}",
        "",
        payload.get("summary", ""),
        "",
        "## 1. Scope Baseline",
        "",
        f"**Version:** {baseline.get('version', 'TBD')}",
        f"**Status:** {baseline.get('approval_status', 'Draft')}",
        f"**Acceptance Authority:** {baseline.get('acceptance_authority', 'TBD')}",
        "",
        baseline.get("scope_statement", ""),
        "",
        "### In Scope",
        "",
    ]
    lines += [f"- {x}" for x in baseline.get("in_scope", [])] or ["- None specified"]
    lines += ["", "### Out of Scope", ""]
    lines += [f"- {x}" for x in baseline.get("out_of_scope", [])] or ["- None specified"]
    lines += ["", "### Assumptions", ""]
    lines += [f"- {x}" for x in baseline.get("assumptions", [])] or ["- None specified"]
    lines += ["", "### Constraints", ""]
    lines += [f"- {x}" for x in baseline.get("constraints", [])] or ["- None specified"]
    lines += ["", "### Dependencies", ""]
    lines += [f"- {x}" for x in baseline.get("dependencies", [])] or ["- None specified"]

    lines += ["", "## 2. Work Breakdown Structure (WBS)", ""]
    lines += [
        "| WBS | Parent | Work Package | Deliverable | Acceptance Criteria |",
        "|---|---|---|---|---|",
    ]
    for item in payload.get("wbs", []):
        lines.append(
            f"| {item.get('id','')} | {item.get('parent_id') or '—'} | "
            f"{item.get('title','')} | {item.get('deliverable','')} | "
            f"{'; '.join(item.get('acceptance_criteria', []))} |"
        )

    lines += ["", "## 3. Requirements-to-Scope Traceability", ""]
    lines += [
        "| Requirement | WBS | Deliverable | Coverage |",
        "|---|---|---|---|",
    ]
    for item in payload.get("traceability", []):
        lines.append(
            f"| {item.get('requirement_id','')} | {item.get('wbs_id','')} | "
            f"{item.get('deliverable','')} | {item.get('coverage','')} |"
        )

    lines += ["", "## 4. JIRA Delivery Backlog", ""]
    lines += [
        "| Type | ID | Title | Priority | Status | Traceability |",
        "|---|---|---|---|---|---|",
    ]
    for item in payload.get("cards", []):
        lines.append(
            f"| {item.get('type','').upper()} | {item.get('id','')} | "
            f"{item.get('title','')} | {item.get('priority','')} | "
            f"{item.get('status','')} | {'; '.join(item.get('tags', []))} |"
        )
        if item.get("description"):
            lines += [f"", f"**{item.get('id','')} Description:** {item['description']}", ""]
        if item.get("acceptance_criteria"):
            lines.append("**Acceptance Criteria:**")
            lines.extend([f"- {x}" for x in item["acceptance_criteria"]])
            lines.append("")

    lines += ["## 5. Scope Change Register", ""]
    if payload.get("change_requests"):
        for item in payload["change_requests"]:
            lines += [
                f"### {item.get('id', 'CR-TBD')} — {item.get('requested_change', item.get('title',''))}",
                "",
                f"**Classification:** {item.get('classification','TBD')}",
                f"**Approval Status:** {item.get('approval_status', item.get('status','Proposed'))}",
                f"**Recommendation:** {item.get('recommendation','')}",
                "",
                f"- Baseline Impact: {item.get('baseline_impact','TBD')}",
                f"- Scope Impact: {item.get('scope_impact','TBD')}",
                f"- Schedule Impact: {item.get('schedule_impact','TBD')}",
                f"- Cost Impact: {item.get('cost_impact','TBD')}",
                f"- Risk Impact: {item.get('risk_impact','TBD')}",
            ]
    else:
        lines.append("No scope change requests recorded.")

    lines += ["", "## 6. Scope Creep Register", ""]
    if payload.get("scope_creep_findings"):
        for item in payload["scope_creep_findings"]:
            lines += [
                f"### {item.get('severity','TBD')} — {item.get('linked_jira_key','JIRA-TBD')}",
                "",
                f"**Observed Item:** {item.get('observed_item','')}",
                f"**Baseline Item:** {item.get('baseline_item','')}",
                f"**Reason Flagged:** {item.get('reason_flagged','')}",
                f"**Recommendation:** {item.get('recommendation','')}",
                "",
            ]
    else:
        lines.append("No scope-creep findings recorded.")

    lines += [
        "",
        "---",
        "",
        "This Scope Management document is the control record for the approved/proposed scope. "
        "JIRA work items are delivery records and must remain traceable to the Scope Baseline and WBS.",
    ]
    return "\n".join(lines)


def export_to_docx(content, mode):
    doc = Document()
    t = doc.add_heading(f"BA & PO Copilot — {mode.upper()}", level=1)
    t.runs[0].font.color.rgb = RGBColor(0x1a,0x73,0xe8)
    s = doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    s.runs[0].font.size = Pt(9); s.runs[0].font.color.rgb = RGBColor(0x88,0x88,0x88)
    doc.add_paragraph("")
    for line in content.split("\n"):
        line = line.strip()
        if not line: doc.add_paragraph(""); continue
        if line.startswith("### "): doc.add_heading(line[4:],level=3)
        elif line.startswith("## "): doc.add_heading(line[3:],level=2)
        elif line.startswith("# "):  doc.add_heading(line[2:],level=1)
        elif line.startswith(("- ","* ")): doc.add_paragraph(line[2:],style="List Bullet")
        elif re.match(r'^\d+\.\s',line): doc.add_paragraph(re.sub(r'^\d+\.\s','',line),style="List Number")
        elif "|" in line and not line.startswith("|-"):
            cells=[c.strip() for c in line.split("|") if c.strip()]
            if cells:
                try:
                    tb=doc.tables[-1]; r=tb.add_row()
                    for i,c in enumerate(cells):
                        if i<len(r.cells): r.cells[i].text=c
                except:
                    tb=doc.add_table(rows=1,cols=len(cells)); tb.style="Table Grid"
                    for i,c in enumerate(cells): tb.rows[0].cells[i].text=c
        else:
            p=doc.add_paragraph()
            for i,part in enumerate(re.split(r'\*\*(.*?)\*\*',line)):
                r=p.add_run(part); r.bold=(i%2==1)
    buf=io.BytesIO(); doc.save(buf); buf.seek(0)
    return buf.getvalue()

def export_to_pdf(content, mode):
    buf=io.BytesIO(); d=SimpleDocTemplate(buf,pagesize=letter); sty=getSampleStyleSheet()
    story=[Paragraph(f"BA & PO Copilot — {mode.upper()}",sty["Title"]),Spacer(1,12)]
    for line in content.split("\n"):
        line=line.strip()
        if not line: story.append(Spacer(1,6)); continue
        cl=re.sub(r'\*\*(.*?)\*\*',r'<b>\1</b>',line); cl=re.sub(r'^#+\s*','',cl)
        try: story.append(Paragraph(cl,sty["Normal"]))
        except: story.append(Paragraph(line,sty["Normal"]))
    d.build(story); buf.seek(0); return buf.getvalue()

def export_to_txt(content, mode):
    h=f"\n{'='*60}\nBA & PO Copilot — {mode.upper()}\n{'='*60}\n"
    f=f"\n{'='*60}\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n{'='*60}\n"
    return (h+content+f).encode('utf-8')

def export_to_markdown(content, mode):
    h=f"# BA & PO Copilot — {mode.upper()}\n\n**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n---\n\n"
    return (h+content).encode('utf-8')

# ── SESSION SAVE ──────────────────────────────────────────────────
def save_session_on_exit():
    if not session_log: return
    df=pd.DataFrame(session_log); stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
    try:
        mode_arg="a" if os.path.exists(SAVE_PATH) else "w"; kw={"engine":"openpyxl","mode":mode_arg}
        if mode_arg=="a": kw["if_sheet_exists"]="new"
        with pd.ExcelWriter(SAVE_PATH,**kw) as w:
            ba=df[df["role_context"]=="ba"]; po=df[df["role_context"]=="po"]
            if not ba.empty: ba.to_excel(w,sheet_name=f"BA_{stamp}"[:31],index=False)
            if not po.empty: po.to_excel(w,sheet_name=f"PO_{stamp}"[:31],index=False)
        print(f"✅ Saved → {SAVE_PATH}")
    except Exception as e: df.to_csv(f"BAPO_{stamp}.csv",index=False); print(f"⚠️ {e}")

atexit.register(save_session_on_exit)

# ════════════════════════════════════════════════════════════════
# DASH APP
# ════════════════════════════════════════════════════════════════
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.title = "BA, PO & PM AI Copilot"

app.index_string = '''
<!DOCTYPE html>
<html>
<head>
{%metas%}<title>{%title%}</title>{%favicon%}{%css%}
<link rel="stylesheet" href="https://unpkg.com/bpmn-js@17.11.1/dist/assets/diagram-js.css">
<link rel="stylesheet" href="https://unpkg.com/bpmn-js@17.11.1/dist/assets/bpmn-js.css">
<link rel="stylesheet" href="https://unpkg.com/bpmn-js@17.11.1/dist/assets/bpmn-font/css/bpmn-embedded.css">
<script src="https://unpkg.com/bpmn-js@17.11.1/dist/bpmn-navigated-viewer.development.js"></script>
<style>
* { box-sizing:border-box; }
html,body,#react-entry-point { width:100%; height:100%; margin:0; padding:0; overflow:auto; }
body { font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif; background:#f8f9fa; }
#react-entry-point > div { height:100%; }
.markdown-body { font-size:13px; line-height:1.7; color:#333; }
.markdown-body h1,.markdown-body h2,.markdown-body h3 { margin:16px 0 8px; font-weight:600; }
.markdown-body h1 { font-size:20px; border-bottom:2px solid #e0e0e0; padding-bottom:8px; }
.markdown-body h2 { font-size:16px; } .markdown-body h3 { font-size:14px; }
.markdown-body ul,.markdown-body ol { margin:8px 0; padding-left:24px; }
.markdown-body table { width:100%; border-collapse:collapse; margin:12px 0; }
.markdown-body table th,.markdown-body table td { border:1px solid #e0e0e0; padding:8px; text-align:left; }
.markdown-body table th { background:#f5f5f5; font-weight:600; }
.markdown-body pre { background:#f5f5f5; padding:12px; border-radius:6px; overflow-x:auto; }
.markdown-body code { background:#f5f5f5; padding:2px 6px; border-radius:4px; font-size:12px; }
.message-bubble { animation:slideIn .25s ease-out; }
@keyframes slideIn { from{opacity:0;transform:translateY(10px);}to{opacity:1;transform:translateY(0);} }
#chat-messages { overflow-y:scroll !important; scroll-behavior:smooth; }
#bpmn-canvas {
    background:#fafafa;
    overflow:auto !important;
    position:relative;
}
#bpmn-canvas .djs-container { overflow:visible !important; }
#bpmn-canvas .djs-scrollable { overflow:auto !important; }
#bpmn-canvas svg { margin: 0 !important; display: block; }
#bpmn-canvas .djs-container svg { width: auto !important; height: auto !important; min-width: 100%; min-height: 100%; }
.bjs-powered-by { display:none !important; }
</style>
</head>
<body>
{%app_entry%}
<footer>{%config%}{%scripts%}{%renderer%}</footer>
<script>
(function(){
  var ID="chat-messages",NEAR=120,DEB=50;
  function isNear(el){return(el.scrollHeight-el.scrollTop-el.clientHeight)<NEAR;}
  function scrollBot(el){if(!el)return;requestAnimationFrame(function(){el.scrollTop=el.scrollHeight;});}
  function setup(){
    var c=document.getElementById(ID);if(!c)return;
    var timer=null;
    function sched(f){if(timer)return;timer=setTimeout(function(){timer=null;if(f||isNear(c))scrollBot(c);},DEB);}
    new MutationObserver(function(ml){for(var i=0;i<ml.length;i++){if(ml[i].addedNodes&&ml[i].addedNodes.length){sched(false);break;}}}).observe(c,{childList:true,subtree:true});
    if('ResizeObserver' in window){var ro=new ResizeObserver(function(){sched(false);});ro.observe(c);}
    else{window.addEventListener('resize',function(){sched(false);},{passive:true});}
  }
  document.addEventListener('DOMContentLoaded',function(){setup();});
})();
</script>
</body>
</html>
'''

# ── WELCOME ──────────────────────────────────────────────────────
def make_welcome(role):
    msgs = {
        "ba": "BA mode active — select a mode and start your analysis.",
        "po": "PO mode active — select a mode to manage your product.",
        "pm": "PM mode active — select a mode to plan, track, or govern your project.",
    }
    msg = msgs.get(role, msgs["ba"])
    return html.Div([
        html.Div("👋 Hi Pranav!", style={"fontSize":"15px","fontWeight":"600","marginBottom":"4px"}),
        html.Div(msg, style={"fontSize":"13px","color":"#666"})
    ], style={"textAlign":"center","color":"#999","marginTop":"80px"})

# ── MODE BUTTONS ─────────────────────────────────────────────────
def ba_btns():
    return dbc.ButtonGroup([
        dbc.Button(f"{BA_ICONS[m]} {BA_LABELS[m]}", id=f"ba-btn-{m}", n_clicks=0, size="sm",
            color="primary" if m=="general" else "light",
            outline=m!="general", style={"fontSize":"12px"})
        for m in BA_MODES], style={"gap":"4px"})

def po_btns():
    return dbc.ButtonGroup([
        dbc.Button(f"{PO_ICONS[m]} {m}", id=f"po-btn-{m}", n_clicks=0, size="sm",
            color="primary" if m=="general" else "light",
            outline=m!="general", style={"fontSize":"12px"})
        for m in PO_MODES], style={"gap":"4px"})

def pm_btns():
    return dbc.ButtonGroup([
        dbc.Button(f"{PM_ICONS[m]} {PM_LABELS[m]}", id=f"pm-btn-{m}", n_clicks=0, size="sm",
            color="primary" if m=="general" else "light",
            outline=m!="general", style={"fontSize":"12px"})
        for m in PM_MODES], style={"gap":"4px"})

# Workspace column styles
WS_OPEN_STYLE = {
    "display":"flex","flexDirection":"column","padding":"0","background":"white",
    "borderLeft":"1px solid #e0e0e0","minHeight":"0","height":"100%","overflow":"hidden"
}
WS_CLOSED_STYLE = {"display":"none"}
CHAT_FULL_STYLE = {
    "display":"flex","flexDirection":"column","padding":"0","background":"#f8f9fa",
    "minHeight":"0","height":"100%","overflow":"hidden","width":"100%"
}
CHAT_HALF_STYLE = {**CHAT_FULL_STYLE, "width":"50%"}

WS_CONTENT_VISIBLE = {"flex":"1","overflowY":"auto","overflowX":"hidden","padding":"0",
                      "fontSize":"14px","lineHeight":"1.7","minHeight":"0","display":"block"}
WS_CONTENT_HIDDEN  = {**WS_CONTENT_VISIBLE,"display":"none"}
BPMN_CANVAS_VISIBLE = {"display":"block","width":"100%","flex":"1 1 auto","minHeight":"0"}
BPMN_CANVAS_HIDDEN  = {"display":"none"}

# ════════════════════════════════════════════════════════════════
# LAYOUT
# ════════════════════════════════════════════════════════════════
app.layout = dbc.Container(fluid=True,
    style={"height":"100vh","padding":"0","display":"flex","flexDirection":"column"}, children=[

    # ── Stores ───────────────────────────────────────────────────
    dcc.Store(id="current-role",       data="ba"),
    dcc.Store(id="current-mode",       data="general"),
    dcc.Store(id="pending-role",       data=None),
    dcc.Store(id="doc-content",        data=""),
    dcc.Store(id="doc-mode",           data="general"),
    dcc.Store(id="current-chat-id",    data=None),
    dcc.Store(id="process-submode",    data="text"),
    dcc.Store(id="bpmn-xml-store",     data=""),
    dcc.Store(id="bpmn-canvas-dummy",  data=None),
    dcc.Store(id="rename-chat-id",     data=None),
    dcc.Store(id="delete-chat-id",     data=None),
    dcc.Store(id="workspace-open",     data=False),
    dcc.Store(id="file-context-store", data=""),
    dcc.Store(id="ai-banner-store",    data=""),
    dcc.Store(id="meeting-file-text", data=""),
    dcc.Store(id="meeting-stories-payload", data=None),
    dcc.Store(id="jira-pending-cards", data=None),
    dcc.Store(id="scope-change-result", data=None),
    dcc.Store(id="dash-drill-store", data=None),
    dcc.Store(id="rename-project-id",  data=None),
    dcc.Store(id="move-chat-id",       data=None),
    dcc.Store(id="current-project-id", data=None),
    dcc.Store(id="new-project-startup-opened", data=True),
    dcc.Store(id="current-workspace-artifact-id", data=None),
    dcc.Store(id="remediation-confirm-artifact-id", data=None),
    dcc.Store(id="regulatory-scan-data", data=None),
    dcc.Store(id="regulatory-pending-remediation", data=None),
    dcc.Store(id="redteam-review-data", data=None),
    dcc.Store(id="conversion-source-artifact", data=None),
    dcc.Store(id="conversion-success-artifact-id", data=None),
    dcc.Store(id="redteam-remediation-request", data=None),
    dcc.Store(id="redteam-pending-remediation", data=None),
    dcc.Store(id="compliance-dashboard-data", data=None),
    dcc.Download(id="download-file"),

    # ── Modals ───────────────────────────────────────────────────
    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("Switch Role"), close_button=False),
        dbc.ModalBody("Are you sure? This will switch your active role and clear the current chat."),
        dbc.ModalFooter([
            dbc.Button("Yes, Switch", id="confirm-yes", color="primary",  className="me-2"),
            dbc.Button("Cancel",      id="confirm-no",  color="secondary", outline=True)
        ])
    ], id="switch-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("🌊 Change Ripple — Linked Artifacts"), close_button=False),
        dbc.ModalBody(html.Div(id="ripple-list-content")),
        dbc.ModalFooter([dbc.Button("Close", id="ripple-close-btn", color="secondary", outline=True)])
    ], id="ripple-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(
            dbc.ModalTitle("✓ Conversion Completed"),
            close_button=False,
        ),
        dbc.ModalBody(html.Div(id="conversion-success-content")),
        dbc.ModalFooter([
            dbc.Button("📖 Review Converted Document", id="conversion-review-btn", color="success", className="me-2"),
            dbc.Button("Close", id="conversion-success-close-btn", color="secondary", outline=True),
        ]),
    ], id="conversion-success-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(
            dbc.ModalTitle("🔄 Convert Requirements Document"),
            close_button=False,
        ),
        dbc.ModalBody([
            html.Div(id="conversion-source-summary", style={"marginBottom":"12px"}),
            dcc.Dropdown(
                id="conversion-target-select",
                options=[],
                placeholder="Select target document type...",
                style={"fontSize":"13px","marginBottom":"12px"},
            ),
            html.Div(
                [
                    html.Div("The conversion will:", style={"fontWeight":"700","marginBottom":"5px"}),
                    html.Div("✓ Preserve relevant source requirements"),
                    html.Div("✓ Transform the actual document structure"),
                    html.Div("✓ Apply target-specific sections and terminology"),
                    html.Div("✓ Preserve the original artifact unchanged"),
                    html.Div("✓ Create a new immutable workspace version"),
                ],
                style={"fontSize":"12px","color":"#555","padding":"10px","background":"#f8f9fa","borderRadius":"7px"},
            ),
        ]),
        dbc.ModalFooter([
            dbc.Button("Convert", id="conversion-confirm-btn", color="primary", className="me-2"),
            dbc.Button("Cancel", id="conversion-cancel-btn", color="secondary", outline=True),
        ]),
    ], id="conversion-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("🎭 Stakeholder Red-Team Review"), close_button=False),
        dbc.ModalBody([
            html.Div(
                "Pick one stakeholder or Select All to combine every stakeholder's concerns.",
                style={"fontSize":"13px","marginBottom":"10px"},
            ),
            dcc.Dropdown(
                id="redteam-persona-select",
                options=[
                    {"label": "👥 Select All Stakeholders", "value": "__all__"},
                    *[
                        {"label": p["label"], "value": k}
                        for k, p in REDTEAM_PERSONAS.items()
                    ],
                ],
                value=[],
                multi=True,
                placeholder="Select one or more stakeholder perspectives...",
                style={"marginBottom":"8px","fontSize":"13px"},
            ),
            dbc.Input(
                id="redteam-custom-stakeholder",
                placeholder="Optional custom stakeholder, e.g. Insurance Benefits Operations Manager",
                type="text",
                style={"fontSize":"12px","marginBottom":"10px"},
            ),
            dcc.Loading(
                html.Div(
                    id="redteam-results",
                    style={
                        "fontSize":"13px",
                        "maxHeight":"360px",
                        "overflowY":"auto",
                    },
                )
            ),
            dcc.Checklist(
                id="redteam-findings-checklist",
                options=[],
                value=[],
                inputStyle={"marginRight":"8px"},
                labelStyle={
                    "display":"block",
                    "padding":"8px 10px",
                    "marginBottom":"6px",
                    "border":"1px solid #eee",
                    "borderRadius":"7px",
                    "background":"#fafafa",
                },
                style={"marginTop":"10px"},
            ),
        ]),
        dbc.ModalFooter([
            dbc.Button(
                "🎭 Run Critique",
                id="redteam-run-btn",
                color="dark",
                className="me-2",
            ),
            dbc.Button(
                "✨ Apply All Concerns",
                id="redteam-apply-all-btn",
                color="primary",
                className="me-2",
                disabled=True,
            ),
            dbc.Button(
                "Apply Selected",
                id="redteam-apply-selected-btn",
                color="secondary",
                outline=True,
                className="me-2",
                disabled=True,
            ),
            dbc.Button(
                "📊 Dashboard",
                id="redteam-dashboard-btn",
                color="info",
                outline=True,
                className="me-2",
            ),
            dbc.Button(
                "Close",
                id="redteam-close-btn",
                color="secondary",
                outline=True,
            ),
        ])
    ], id="redteam-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(
            dbc.ModalTitle("✨ Red-Team Remediation Preview"),
            close_button=False,
        ),
        dbc.ModalBody(
            dcc.Loading(
                html.Div(id="redteam-diff-results")
            )
        ),
        dbc.ModalFooter([
            dbc.Button(
                "Apply Changes",
                id="redteam-commit-btn",
                color="success",
                className="me-2",
                disabled=True,
            ),
            dbc.Button(
                "Cancel",
                id="redteam-diff-close-btn",
                color="secondary",
                outline=True,
            ),
        ]),
    ], id="redteam-diff-modal", is_open=False, centered=True, size="xl"),

    # ════════════════════════════════════════════════════════════
    # REMEDIATION CONFIRMATION — SHARED BY REGULATORY + RED-TEAM
    # ════════════════════════════════════════════════════════════
    dbc.Modal([
        dbc.ModalHeader(
            dbc.ModalTitle("✅ Changes Applied Successfully"),
            close_button=False,
        ),
        dbc.ModalBody(
            html.Div(id="remediation-confirmation-content")
        ),
        dbc.ModalFooter([
            dbc.Button(
                "📖 Review Updated Document",
                id="remediation-confirm-open-btn",
                color="success",
                className="me-2",
            ),
            dbc.Button(
                "Close",
                id="remediation-confirm-close-btn",
                color="secondary",
                outline=True,
            ),
        ]),
    ], id="remediation-confirm-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("🎙️ Meeting → Artifacts Pipeline"), close_button=False),
        dbc.ModalBody([
            html.Div("Paste a transcript, or attach a file:",
                     style={"fontSize":"13px","marginBottom":"8px"}),
            dbc.Textarea(id="meeting-transcript-input", rows=8, placeholder="Paste transcript text here...",
                         style={"fontSize":"12px","marginBottom":"8px"}),
            dcc.Upload(id="meeting-file-upload",
                children=dbc.Button("📎 Attach transcript file", size="sm", color="light", outline=True),
                multiple=False),
            html.Div(id="meeting-file-badge", style={"fontSize":"11px","color":"#555","marginTop":"6px"}),
            dcc.Loading(html.Div(id="meeting-pipeline-results",
                        style={"fontSize":"13px","marginTop":"12px","maxHeight":"260px","overflowY":"auto"})),
        ]),
        dbc.ModalFooter([
            dbc.Button("🚀 Generate Artifacts", id="meeting-run-btn", color="dark", className="me-2"),
            dbc.Button("Close", id="meeting-close-btn", color="secondary", outline=True),
        ])
    ], id="meeting-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("Rename Chat"), close_button=False),
        dbc.ModalBody(dbc.Input(id="rename-input", type="text", placeholder="Chat title")),
        dbc.ModalFooter([
            dbc.Button("Save",   id="rename-save-btn",   color="primary", className="me-2"),
            dbc.Button("Cancel", id="rename-cancel-btn", color="secondary", outline=True)
        ])
    ], id="rename-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("Delete Chat"), close_button=False),
        dbc.ModalBody("Are you sure you want to delete this chat?"),
        dbc.ModalFooter([
            dbc.Button("Delete", id="delete-confirm-btn", color="danger",    className="me-2"),
            dbc.Button("Cancel", id="delete-cancel-btn",  color="secondary", outline=True)
        ])
    ], id="delete-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("New Project"), close_button=False),
        dbc.ModalBody(dbc.Input(id="new-project-input", type="text", placeholder="Project name")),
        dbc.ModalFooter([
            dbc.Button("Create", id="create-project-btn", color="primary",   className="me-2"),
            dbc.Button("Cancel", id="cancel-project-btn", color="secondary", outline=True)
        ])
    ], id="new-project-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("Rename Project"), close_button=False),
        dbc.ModalBody(dbc.Input(id="rename-project-input", type="text", placeholder="New name")),
        dbc.ModalFooter([
            dbc.Button("Save",   id="save-rename-project-btn",   color="primary",   className="me-2"),
            dbc.Button("Cancel", id="cancel-rename-project-btn", color="secondary", outline=True)
        ])
    ], id="rename-project-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("🛡️ Regulatory Radar"), close_button=False),
        dbc.ModalBody([
            html.Div(
                "AI-assisted informational scan — not legal advice. Consult compliance/legal counsel for authoritative review.",
                style={"fontSize":"11px","color":"#888","marginBottom":"10px","fontStyle":"italic"},
            ),
            dcc.Loading(html.Div(id="regulatory-scan-results")),
            dcc.Checklist(
                id="regulatory-findings-checklist",
                options=[],
                value=[],
                inputStyle={"marginRight": "8px"},
                labelStyle={
                    "display": "block",
                    "padding": "8px 10px",
                    "marginBottom": "6px",
                    "border": "1px solid #eee",
                    "borderRadius": "7px",
                    "background": "#fafafa",
                },
                style={"marginTop": "10px"},
            ),
        ]),
        dbc.ModalFooter([
            dbc.Button("🛡️ Run Scan", id="regulatory-run-btn", color="dark", className="me-2"),
            dbc.Button("✨ Apply All Findings", id="regulatory-apply-all-btn",
                        color="primary", className="me-2", disabled=True),
            dbc.Button("Apply Selected", id="regulatory-apply-selected-btn",
                        color="secondary", outline=True, className="me-2", disabled=True),
            dbc.Button("📊 Dashboard", id="regulatory-dashboard-btn",
                        color="info", outline=True, className="me-2"),
            dbc.Button("Close", id="regulatory-close-btn",
                        color="secondary", outline=True),
        ])
    ], id="regulatory-modal", is_open=False, centered=True, size="lg"),

    dbc.Modal([
        dbc.ModalHeader(
            dbc.ModalTitle("✨ Regulatory Remediation Preview"),
            close_button=False,
        ),
        dbc.ModalBody(dcc.Loading(html.Div(id="regulatory-diff-results"))),
        dbc.ModalFooter([
            dbc.Button("Apply Changes", id="regulatory-commit-btn",
                       color="success", className="me-2", disabled=True),
            dbc.Button("Cancel", id="regulatory-diff-close-btn",
                       color="secondary", outline=True),
        ]),
    ], id="regulatory-diff-modal", is_open=False, centered=True, size="xl"),

    dbc.Modal([
        dbc.ModalHeader(
            dbc.ModalTitle("📊 Compliance & Stakeholder Dashboard"),
            close_button=False,
        ),
        dbc.ModalBody(
            dcc.Loading(
                html.Div(
                    id="compliance-dashboard-results",
                    style={
                        "maxHeight": "620px",
                        "overflowY": "auto",
                        "fontSize": "13px",
                    },
                )
            )
        ),
        dbc.ModalFooter([
            dbc.Button(
                "🛡️ Run Regulatory Radar",
                id="dashboard-regulatory-btn",
                color="dark",
                outline=True,
                className="me-2",
            ),
            dbc.Button(
                "🎭 Run Red-Team",
                id="dashboard-redteam-btn",
                color="dark",
                outline=True,
                className="me-2",
            ),
            dbc.Button(
                "Close",
                id="compliance-dashboard-close-btn",
                color="secondary",
                outline=True,
            ),
        ]),
    ], id="compliance-dashboard-modal", is_open=False, centered=True, size="xl"),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("Move to Project"), close_button=False),
        dbc.ModalBody(dcc.Dropdown(id="move-project-select", placeholder="Select project...")),
        dbc.ModalFooter([
            dbc.Button("Move",   id="confirm-move-btn", color="primary",   className="me-2"),
            dbc.Button("Cancel", id="cancel-move-btn",  color="secondary", outline=True)
        ])
    ], id="move-chat-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("📌 Push to JIRA"), close_button=True),
        dbc.ModalBody([
            html.Div(id="jira-modal-summary", style={"fontSize":"13px","marginBottom":"10px"}),
            dcc.Dropdown(id="jira-space-select", placeholder="Select JIRA space...",
                         style={"marginBottom":"10px","fontSize":"13px"}),
            html.Div(id="jira-push-notification"),
            html.Div(id="jira-modal-results", style={"fontSize":"12px","maxHeight":"240px","overflowY":"auto"}),
        ]),
        dbc.ModalFooter([
            dbc.Button("🚀 Push to JIRA",   id="jira-push-confirm-btn", color="primary",   className="me-2"),
            dbc.Button("✏️ Refine Further", id="jira-push-cancel-btn",  color="secondary", outline=True),
            dbc.Button("✅ OK", id="jira-push-ok-btn", color="success", style={"display":"none"}),
        ])
    ], id="jira-push-modal", is_open=False, centered=True),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("🔀 Scope Change Request"), close_button=True),
        dbc.ModalBody([
            html.Div(
                "Describe the requested scope change. The analysis compares it with the current Scope Baseline and WBS.",
                style={"fontSize": "12px", "color": "#666", "marginBottom": "8px"},
            ),
            dbc.Textarea(
                id="scope-change-input",
                placeholder="Example: Add international payments to the approved project scope...",
                style={"fontSize": "13px"},
                rows=5,
            ),
            html.Div(id="scope-change-results", style={"marginTop": "12px"}),
        ]),
        dbc.ModalFooter([
            dbc.Button("Analyze Change", id="scope-change-analyze-btn", color="warning", className="me-2"),
            dbc.Button("Create JIRA Change Request", id="scope-change-push-jira-btn",
                       color="primary", className="me-2", disabled=True),
            dbc.Button("Close", id="scope-change-close-btn", color="secondary", outline=True),
        ]),
    ], id="scope-change-modal", is_open=False, centered=True, size="lg"),

    dbc.Modal([
        dbc.ModalHeader(dbc.ModalTitle("🔎 Scope Creep Detector"), close_button=True),
        dbc.ModalBody([
            html.Div(
                "Scans JIRA Scope Management issues against the approved Scope Baseline and WBS.",
                style={"fontSize": "12px", "color": "#666", "marginBottom": "8px"},
            ),
            dcc.Loading(html.Div(id="scope-creep-results")),
        ]),
        dbc.ModalFooter([
            dbc.Button("Run Scope Creep Scan", id="scope-creep-run-btn", color="danger", className="me-2"),
            dbc.Button("Close", id="scope-creep-close-btn", color="secondary", outline=True),
        ]),
    ], id="scope-creep-modal", is_open=False, centered=True, size="xl"),

    # ── History Offcanvas ─────────────────────────────────────────
    dbc.Offcanvas([
        html.Div(id="sidebar-content",
            style={"overflowY":"auto","height":"100%","paddingBottom":"16px"})
    ], id="history-offcanvas", title="📜 Chat History",
       placement="start", is_open=False, style={"width":"320px"}),

    # ── HEADER ───────────────────────────────────────────────────
    html.Div([
        dbc.Button("📜 History",  id="history-btn",  color="light", size="sm", outline=True, style={"marginRight":"8px"}),
        dbc.Button("➕ New Chat", id="new-chat-btn", color="light", size="sm", outline=True, style={"marginRight":"16px"}),
        dbc.Button("🎙️ Meeting → Artifacts", id="open-meeting-btn", color="light", size="sm", outline=True, style={"marginRight":"16px"}),
        dbc.Button("🌊 Ripple", id="open-ripple-btn", color="light", size="sm", outline=True, style={"marginRight":"16px"}),
        html.Span("🤖 BA, PO & PM AI Copilot", style={"fontSize":"18px","fontWeight":"bold","color":"#1a73e8"}),
        html.Span(id="ai-indicator", children="• Gemini",
                  style={"fontSize":"11px","color":"#888","marginLeft":"8px","marginRight":"32px"}),
        dbc.RadioItems(id="role-radio", inline=True, value="ba", options=[
            {"label":html.Span("📊 Business Analysis",style={"fontSize":"13px","marginRight":"12px"}),"value":"ba"},
            {"label":html.Span("🎯 Product Ownership",   style={"fontSize":"13px"}),                     "value":"po"},
            {"label":html.Span("🧭 Project Management", style={"fontSize":"13px"}),                     "value":"pm"},
        ])
    ], style={"display":"flex","alignItems":"center","padding":"12px 20px","background":"white","borderBottom":"1px solid #e0e0e0"}),

    # ── MAIN ROW ─────────────────────────────────────────────────
    dbc.Row(style={"flex":1,"display":"flex","margin":"0","minHeight":0,"overflow":"hidden"}, children=[

        # Chat column
        dbc.Col(width=12, id="chat-col", style=CHAT_FULL_STYLE, children=[

            html.Div([
                html.Div("Mode:", style={"fontSize":"11px","color":"#999","marginBottom":"8px","fontWeight":"600","textTransform":"uppercase"}),
                html.Div(id="ba-mode-bar", children=ba_btns(), style={"display":"flex","flexWrap":"wrap","gap":"6px"}),
                html.Div(id="po-mode-bar", children=po_btns(), style={"display":"none","flexWrap":"wrap","gap":"6px"}),
                html.Div(id="pm-mode-bar", children=pm_btns(), style={"display":"none","flexWrap":"wrap","gap":"6px"}),
                # Process sub-mode toggle
                html.Div(id="process-submode-bar", children=[
                    dbc.Button("📝 Text",         id="process-text-btn", size="sm", color="primary", outline=False, n_clicks=0, style={"fontSize":"11px"}),
                    dbc.Button("🔷 BPMN Diagram", id="process-bpmn-btn", size="sm", color="light",   outline=True,  n_clicks=0, style={"fontSize":"11px"}),
                ], style={"display":"none","gap":"6px","marginTop":"8px"}),
                # AI banner
                html.Div(id="ai-banner-div", style={"display":"none","marginTop":"6px"}, children=[
                    html.Span(id="ai-banner-text",
                        style={"fontSize":"11px","color":"#2e7d32","background":"#e8f5e9","padding":"3px 10px","borderRadius":"10px"})
                ]),
            ], style={"padding":"12px 16px","background":"white","borderBottom":"1px solid #e0e0e0"}),

            html.Div(id="chat-messages", style={
                "flex":"1 1 auto","overflowY":"auto","overflowX":"hidden","padding":"20px 24px",
                "display":"flex","flexDirection":"column","gap":"16px","minHeight":"0"
            }, children=[make_welcome("ba")]),

            html.Div([
                html.Div([
                    dcc.Upload(
                        id="file-upload",
                        children=dbc.Button("📎", size="sm", color="light", outline=True,
                                           title="Attach file",
                                           style={"fontSize":"15px","padding":"2px 8px"}),
                        multiple=False),
                    html.Span(id="file-badge", style={"fontSize":"11px","color":"#555","marginLeft":"8px","flex":"1"}),
                    dbc.Button("✕", id="clear-file-btn", size="sm", color="light", n_clicks=0,
                               style={"display":"none","padding":"2px 6px","fontSize":"11px"}),
                ], style={"display":"flex","alignItems":"center","marginBottom":"6px"}),
                dbc.Textarea(id="user-input",
                    placeholder="Ask anything — or attach a file 📎",
                    style={"resize":"none","fontSize":"13px","border":"1px solid #ddd","borderRadius":"8px","padding":"12px"}, rows=3),
                dbc.Row([
                    dbc.Col(dbc.Button("Send ➤", id="send-btn", color="primary", n_clicks=0,
                        style={"width":"100%","marginTop":"10px","fontWeight":"600","padding":"10px"}), width=9),
                    dbc.Col(dbc.Button("🔄 Reset", id="reset-btn", color="light", n_clicks=0,
                        style={"width":"100%","marginTop":"10px"}), width=3)
                ])
            ], style={"padding":"14px 16px","background":"white","borderTop":"1px solid #e0e0e0"}),
        ]),

        # Workspace column
        dbc.Col(width=6, id="workspace-col", style=WS_CLOSED_STYLE, children=[
            html.Div([
                html.Span("📋 Workspace", style={"fontWeight":"600","fontSize":"14px","color":"#333","flex":"1"}),
                html.Div([
                    dbc.DropdownMenu(label="⬇ Export", color="success", size="sm", className="me-2", children=[
                        dbc.DropdownMenuItem("Word (.docx)",     id="export-docx",     n_clicks=0),
                        dbc.DropdownMenuItem("PDF (.pdf)",       id="export-pdf",      n_clicks=0),
                        dbc.DropdownMenuItem("Text (.txt)",      id="export-txt",      n_clicks=0),
                        dbc.DropdownMenuItem("Markdown (.md)",   id="export-md",       n_clicks=0),
                        dbc.DropdownMenuItem(divider=True),
                        dbc.DropdownMenuItem("BPMN XML (.bpmn)", id="export-bpmn",     n_clicks=0),
                        dbc.DropdownMenuItem("Diagram (.svg)",   id="export-bpmn-svg", n_clicks=0),
                        dbc.DropdownMenuItem("Diagram (.png)",   id="export-bpmn-png", n_clicks=0),
                    ]),
                    dbc.Button("📋 Copy", id="copy-btn",            color="info",      size="sm", outline=True, className="me-2"),
                    dbc.Button("🎭 Red-Team", id="open-redteam-btn", color="dark", size="sm", outline=True, className="me-2"),
                    dbc.Button("🔄 Convert", id="open-conversion-btn", color="primary", size="sm", outline=True, className="me-2"),
                    dbc.Button("🛡️ Regulatory Radar", id="open-regulatory-btn", color="dark", size="sm", outline=True, className="me-2"),
                    dbc.Button("📊 Compliance Dashboard", id="open-compliance-dashboard-btn", color="info", size="sm", outline=True, className="me-2"),
                    dbc.Button("🚀 Push to JIRA", id="manual-push-jira-btn", color="primary", size="sm",
                               outline=True, className="me-2", style={"display":"none"}),
                    dbc.Button("🔀 Scope Change", id="scope-change-btn", color="warning", size="sm",
                               outline=True, className="me-2", style={"display":"none"}),
                    dbc.Button("🔎 Scope Creep", id="scope-creep-btn", color="danger", size="sm",
                               outline=True, className="me-2", style={"display":"none"}),
                    dbc.Button("✕",       id="close-workspace-btn", color="secondary", size="sm", outline=True),
                ], style={"display":"flex","gap":"4px"})
            ], style={"padding":"12px 20px","borderBottom":"1px solid #e0e0e0","display":"flex","alignItems":"center"}),
            html.Div(id="chart-area", style={"padding":"0 24px"}),
            html.Div(id="workspace-content", style=WS_CONTENT_VISIBLE, className="markdown-body"),
            html.Div(id="bpmn-canvas", style=BPMN_CANVAS_HIDDEN),
            dcc.Textarea(id="hidden-clipboard", style={"display":"none"}),
        ])
    ])
])

# ════════════════════════════════════════════════════════════════
# CALLBACKS
# ════════════════════════════════════════════════════════════════



# ── Quality verification and combined remediation scoring ──────────

QUALITY_SEVERITY_WEIGHTS = {
    "Critical": 10,
    "High": 7,
    "Medium": 4,
    "Low": 2,
}


def _quality_findings_burden(regulatory_data=None, redteam_data=None):
    """Calculate a severity-weighted finding burden."""
    burden = 0

    if isinstance(regulatory_data, dict):
        result = regulatory_data.get("result") or regulatory_data
        for item in _regulatory_findings(result):
            severity = item.get("severity") or item.get("risk_rating") or "Medium"
            burden += QUALITY_SEVERITY_WEIGHTS.get(
                str(severity).title(),
                QUALITY_SEVERITY_WEIGHTS["Medium"],
            )

    if isinstance(redteam_data, dict):
        for item in redteam_data.get("concerns") or []:
            severity = item.get("risk_rating") or "Medium"
            burden += QUALITY_SEVERITY_WEIGHTS.get(
                str(severity).title(),
                QUALITY_SEVERITY_WEIGHTS["Medium"],
            )

    return burden


def _combined_remediation_score(baseline_burden, current_burden):
    """Measure the percentage of baseline finding burden removed."""
    baseline = max(0, int(baseline_burden or 0))
    current = max(0, int(current_burden or 0))

    if baseline == 0:
        return 100.0 if current == 0 else 0.0

    return round(
        max(0.0, min(100.0, (1.0 - current / baseline) * 100.0)),
        1,
    )


def _current_quality_score(current_burden):
    """Convert residual severity burden into a simple 0-100 score."""
    burden = max(0, int(current_burden or 0))
    return round(max(0.0, 100.0 - min(100.0, burden * 5.0)), 1)


def _quality_record(artifact_id):
    if not artifact_id:
        return None

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(
            """
            SELECT *
            FROM artifact_quality_verifications
            WHERE artifact_id=?
            """,
            (artifact_id,),
        ).fetchone()

    return dict(row) if row else None


def _decode_quality_json(value):
    if not value:
        return None
    if isinstance(value, dict):
        return value
    try:
        return json.loads(value)
    except (TypeError, ValueError):
        return None


def _save_quality_record(
    artifact_id,
    baseline_regulatory,
    baseline_redteam,
    current_regulatory,
    current_redteam,
    source_artifact_id=None,
):
    if not artifact_id:
        return

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.execute(
            """
            INSERT INTO artifact_quality_verifications
                (
                    artifact_id,
                    source_artifact_id,
                    baseline_regulatory,
                    baseline_redteam,
                    current_regulatory,
                    current_redteam,
                    verified_at
                )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(artifact_id) DO UPDATE SET
                source_artifact_id=excluded.source_artifact_id,
                baseline_regulatory=excluded.baseline_regulatory,
                baseline_redteam=excluded.baseline_redteam,
                current_regulatory=excluded.current_regulatory,
                current_redteam=excluded.current_redteam,
                verified_at=excluded.verified_at
            """,
            (
                artifact_id,
                source_artifact_id,
                json.dumps(baseline_regulatory or {}),
                json.dumps(baseline_redteam or {}),
                json.dumps(current_regulatory or {}),
                json.dumps(current_redteam or {}),
                datetime.now().isoformat(),
            ),
        )
        conn.commit()


def _update_quality_current(
    artifact_id,
    regulatory_data=None,
    redteam_data=None,
):
    """Update one verified artifact's current scan without losing its baseline."""
    record = _quality_record(artifact_id)
    if not record:
        return

    baseline_regulatory = _decode_quality_json(
        record.get("baseline_regulatory")
    ) or {}
    baseline_redteam = _decode_quality_json(
        record.get("baseline_redteam")
    ) or {}
    current_regulatory = (
        regulatory_data
        if regulatory_data is not None
        else _decode_quality_json(
            record.get("current_regulatory")
        ) or {}
    )
    current_redteam = (
        redteam_data
        if redteam_data is not None
        else _decode_quality_json(
            record.get("current_redteam")
        ) or {}
    )

    _save_quality_record(
        artifact_id,
        baseline_regulatory,
        baseline_redteam,
        current_regulatory,
        current_redteam,
        record.get("source_artifact_id"),
    )


def _store_quality_snapshot(
    artifact_id,
    baseline_regulatory=None,
    baseline_redteam=None,
    current_regulatory=None,
    current_redteam=None,
    source_artifact_id=None,
):
    _save_quality_record(
        artifact_id,
        baseline_regulatory,
        baseline_redteam,
        current_regulatory,
        current_redteam,
        source_artifact_id,
    )

    baseline_burden = _quality_findings_burden(
        baseline_regulatory,
        baseline_redteam,
    )
    current_burden = _quality_findings_burden(
        current_regulatory,
        current_redteam,
    )

    return {
        "artifact_id": artifact_id,
        "source_artifact_id": source_artifact_id,
        "baseline_burden": baseline_burden,
        "current_burden": current_burden,
        "remediation_score": _combined_remediation_score(
            baseline_burden,
            current_burden,
        ),
        "current_quality_score": _current_quality_score(
            current_burden,
        ),
    }


def _run_redteam_reviews_for_artifact(artifact):
    """Run the document-aware stakeholder suite against one immutable artifact."""
    if not artifact:
        raise ValueError("No workspace artifact is available for Red-Team verification.")

    review_content = artifact.get("doc_content") or ""
    review_mode = artifact.get("mode") or "general"
    review_role = artifact.get("role") or "ba"
    document_type = resolve_artifact_document_type(artifact) or "BRD"
    persona_keys = (
        DOCUMENT_REDTEAM_PERSONAS.get("SOW", [])
        if review_role == "pm" and review_mode == "scope"
        else get_document_redteam_personas(document_type)
    )
    all_reviews = []

    for key in persona_keys:
        persona = REDTEAM_PERSONAS.get(key)
        if not persona:
            continue

        mode_focus = (
            REDTEAM_MODE_FOCUS.get(document_type.lower())
            or REDTEAM_MODE_FOCUS.get(review_mode, "")
        )
        system_prompt = REDTEAM_SYSTEM + "\n\nPersona: " + persona["prompt"]
        system_prompt += (
            "\n\nCurrent document type: "
            + document_type
            + "\n\nDocument purpose: "
            + DOCUMENT_TYPE_CONFIG.get(document_type, {}).get("description", "")
        )
        if mode_focus:
            system_prompt += "\n\nDocument-mode focus: " + mode_focus

        structured_prompt = """
Review the artifact below from the assigned stakeholder perspective.

Return ONLY valid JSON. Do not use markdown fences.

Schema:
{
  "persona": "<persona name>",
  "risk_rating": "Low|Medium|High",
  "risk_justification": "<one sentence>",
  "concerns": [
    {
      "title": "<short concern>",
      "detail": "<specific evidence-based concern>",
      "recommended_change": "<specific change that should be made>"
    }
  ],
  "missing_requirements": ["<missing requirement>"],
  "questions_for_author": ["<question>"]
}

Do not invent facts that are not supported by the artifact.
Each concern must be actionable and suitable for remediation.
""".strip()

        raw, ai_name, banner = AIProvider.call(
            [
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": (
                        structured_prompt
                        + "\n\nARTIFACT:\n\n"
                        + review_content[:60000]
                    ),
                },
            ],
            max_tokens=5000,
        )

        parsed = parse_po_json(raw) or {
            "persona": persona["label"],
            "risk_rating": "Medium",
            "risk_justification": "The AI response could not be parsed into structured findings.",
            "concerns": [{
                "title": "Review response",
                "detail": raw[:2000],
                "recommended_change": "Review the stakeholder critique manually.",
            }],
            "missing_requirements": [],
            "questions_for_author": [],
        }
        parsed["persona_key"] = key
        parsed["persona_label"] = persona["label"]
        parsed["ai_name"] = ai_name or ""
        parsed["banner"] = banner or ""
        all_reviews.append(parsed)

    concerns = []
    for review in all_reviews:
        for concern in review.get("concerns", []) or []:
            if not isinstance(concern, dict):
                continue
            concerns.append({
                "index": len(concerns),
                "persona_key": review.get("persona_key", ""),
                "persona": review.get("persona_label", review.get("persona", "Stakeholder")),
                "risk_rating": review.get("risk_rating", "Medium"),
                "title": concern.get("title", "Stakeholder concern"),
                "detail": concern.get("detail", ""),
                "recommended_change": concern.get("recommended_change", ""),
            })

    return {
        "artifact_id": artifact.get("id"),
        "role": review_role,
        "mode": review_mode,
        "document_type": document_type,
        "recommended_personas": persona_keys,
        "reviews": all_reviews,
        "concerns": concerns,
    }


def _verify_artifact_after_remediation(
    source_artifact,
    new_artifact,
    baseline_regulatory=None,
    baseline_redteam=None,
):
    """Run actual post-remediation Regulatory and Red-Team verification."""
    source_id = source_artifact.get("id") if source_artifact else None
    source_record = _quality_record(source_id)

    if baseline_regulatory is None and source_record:
        baseline_regulatory = (
            _decode_quality_json(source_record.get("baseline_regulatory"))
            or _decode_quality_json(source_record.get("current_regulatory"))
        )

    if baseline_redteam is None and source_record:
        baseline_redteam = (
            _decode_quality_json(source_record.get("baseline_redteam"))
            or _decode_quality_json(source_record.get("current_redteam"))
        )

    if baseline_regulatory is None:
        baseline_regulatory = {
            "artifact_id": source_id,
            "result": run_regulatory_scan(
                source_artifact.get("doc_content") or ""
            ),
        }

    if baseline_redteam is None:
        baseline_redteam = _run_redteam_reviews_for_artifact(source_artifact)

    current_regulatory = {
        "artifact_id": new_artifact.get("id"),
        "result": run_regulatory_scan(
            new_artifact.get("doc_content") or ""
        ),
    }

    current_redteam = _run_redteam_reviews_for_artifact(new_artifact)

    return _store_quality_snapshot(
        new_artifact.get("id"),
        baseline_regulatory,
        baseline_redteam,
        current_regulatory,
        current_redteam,
        source_artifact_id=source_id,
    )


# ── Compliance & stakeholder dashboard ────────────────────────────

def _dashboard_risk_color(risk):
    return {
        "Low": "#2e7d32",
        "Medium": "#e65100",
        "High": "#c62828",
        "Critical": "#8e0000",
    }.get(str(risk or "").title(), "#666")


def _build_dashboard_payload(artifact_id, regulatory_data, redteam_data):
    regulatory = (
        regulatory_data
        if isinstance(regulatory_data, dict)
        and regulatory_data.get("artifact_id") == artifact_id
        else None
    )
    redteam = (
        redteam_data
        if isinstance(redteam_data, dict)
        and redteam_data.get("artifact_id") == artifact_id
        else None
    )

    verification = _quality_record(artifact_id)
    if verification:
        regulatory = regulatory or _decode_quality_json(
            verification.get("current_regulatory")
        )
        redteam = redteam or _decode_quality_json(
            verification.get("current_redteam")
        )

    return {
        "artifact_id": artifact_id,
        "regulatory": regulatory,
        "redteam": redteam,
        "verification": verification,
    }


def _build_version_difference_graph(artifact, verification, baseline_burden, current_burden):
    """Build a stakeholder-friendly verified before/after burden comparison."""
    if not verification:
        return None

    source_artifact_id = verification.get("source_artifact_id")
    source_artifact = get_workspace_artifact(source_artifact_id) if source_artifact_id else None
    current_version = artifact.get("version", "?")
    before_version = (
        source_artifact.get("version", "?") if source_artifact else
        (int(current_version) - 1 if isinstance(current_version, int) and current_version > 1 else "Before")
    )

    baseline = max(0, int(baseline_burden or 0))
    current = max(0, int(current_burden or 0))
    delta = current - baseline
    pct = (abs(delta) / baseline * 100.0) if baseline > 0 else 0.0

    frame = pd.DataFrame([
        {"Version": f"V{before_version}", "Findings burden": baseline},
        {"Version": f"V{current_version}", "Findings burden": current},
    ])
    figure = px.bar(
        frame,
        x="Version",
        y="Findings burden",
        text="Findings burden",
        title="Version Difference — Verified Compliance Burden",
        labels={"Version": "Workspace version", "Findings burden": "Severity-weighted burden"},
    )
    figure.update_traces(
        textposition="outside",
        hovertemplate="Version %{x}<br>Verified burden: %{y}<extra></extra>",
    )
    figure.update_layout(
        height=285,
        margin={"l": 45, "r": 20, "t": 55, "b": 45},
        showlegend=False,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
    )
    figure.update_yaxes(rangemode="tozero")

    if delta < 0:
        headline = f"{abs(delta)} burden removed"
        detail = f"{pct:.1f}% reduction"
        color = "#2e7d32"
        icon = "🟢"
    elif delta > 0:
        headline = f"{delta} burden increase"
        detail = f"{pct:.1f}% increase"
        color = "#c62828"
        icon = "🔴"
    else:
        headline = "No burden change"
        detail = "0.0% change"
        color = "#666"
        icon = "⚪"

    return html.Div([
        html.Div("VERSION DIFFERENCE", style={"fontSize": "11px", "color": "#888", "fontWeight": "700", "textTransform": "uppercase", "marginBottom": "3px"}),
        html.Div([
            html.Strong(f"{icon} {headline}"),
            html.Span(f" · {detail}", style={"color": color, "fontWeight": "700"}),
        ], style={"fontSize": "13px", "marginBottom": "5px"}),
        dcc.Graph(figure=figure, config={"displayModeBar": False, "responsive": True}, style={"height": "285px"}),
        html.Div(
            f"Verified comparison: V{before_version} burden {baseline} → V{current_version} burden {current}.",
            style={"fontSize": "11px", "color": "#666", "marginTop": "-8px"},
        ),
    ], style={"padding": "14px", "background": "#fff", "border": "1px solid #dbeafe", "borderRadius": "8px", "marginBottom": "14px"})

def _finding_counts_by_source(regulatory_data=None, redteam_data=None):
    """Return actionable finding counts for each verification source."""
    regulatory_result = {}
    if isinstance(regulatory_data, dict):
        regulatory_result = regulatory_data.get("result") or regulatory_data

    regulatory_count = len(_regulatory_findings(regulatory_result))

    redteam_count = 0
    if isinstance(redteam_data, dict):
        redteam_count = len(redteam_data.get("concerns") or [])

    return {
        "Regulatory Radar": regulatory_count,
        "Red-Team": redteam_count,
        "Combined": regulatory_count + redteam_count,
    }


def _build_quality_source_comparison_graph(baseline_regulatory, baseline_redteam, current_regulatory, current_redteam):
    """Build the Regulatory Radar vs Red-Team before/after comparison."""
    before = _finding_counts_by_source(baseline_regulatory, baseline_redteam)
    after = _finding_counts_by_source(current_regulatory, current_redteam)

    frame = pd.DataFrame([
        {"Source": "Regulatory Radar", "Stage": "Before", "Findings": before["Regulatory Radar"]},
        {"Source": "Regulatory Radar", "Stage": "After", "Findings": after["Regulatory Radar"]},
        {"Source": "Red-Team", "Stage": "Before", "Findings": before["Red-Team"]},
        {"Source": "Red-Team", "Stage": "After", "Findings": after["Red-Team"]},
    ])
    figure = px.bar(
        frame, x="Source", y="Findings", color="Stage", barmode="group", text="Findings",
        title="Finding Reduction by Review Source",
        labels={"Source": "", "Findings": "Actionable findings", "Stage": ""},
    )
    figure.update_traces(textposition="outside", hovertemplate="%{x}<br>%{fullData.name}: %{y} finding(s)<extra></extra>")
    figure.update_layout(
        height=300, margin={"l": 45, "r": 20, "t": 55, "b": 45},
        legend={"orientation": "h", "y": 1.08, "x": 0},
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    )
    figure.update_yaxes(rangemode="tozero", dtick=1)

    total_before = before["Combined"]
    total_after = after["Combined"]
    delta = total_after - total_before
    pct = abs(delta) / total_before * 100.0 if total_before else 0.0
    if delta < 0:
        summary = f"🟢 {abs(delta)} actionable finding(s) removed · {pct:.1f}% combined reduction"
        color = "#2e7d32"
    elif delta > 0:
        summary = f"🔴 {delta} actionable finding(s) added · {pct:.1f}% combined increase"
        color = "#c62828"
    else:
        summary = "⚪ No change in combined actionable findings"
        color = "#666"

    source_rows = []
    for label in ("Regulatory Radar", "Red-Team"):
        d = after[label] - before[label]
        if d < 0:
            change = f"🟢 {abs(d)} fewer"
        elif d > 0:
            change = f"🔴 {d} more"
        else:
            change = "⚪ No change"
        source_rows.append(html.Div([
            html.Strong(label),
            html.Span(f"  {before[label]} → {after[label]}", style={"marginLeft": "8px"}),
            html.Span(f"  {change}", style={"marginLeft": "8px", "fontWeight": "700"}),
        ], style={"fontSize": "12px", "marginBottom": "4px"}))

    return html.Div([
        html.Div("FINDING REDUCTION BY REVIEW SOURCE", style={"fontSize": "11px", "color": "#888", "fontWeight": "700", "textTransform": "uppercase", "marginBottom": "3px"}),
        html.Div(summary, style={"fontSize": "13px", "fontWeight": "700", "color": color, "marginBottom": "7px"}),
        html.Div(source_rows, style={"marginBottom": "4px"}),
        dcc.Graph(figure=figure, config={"displayModeBar": False, "responsive": True}, style={"height": "300px"}),
        html.Div("Verified baseline versus the latest verified re-scan and Red-Team results for this exact artifact.", style={"fontSize": "11px", "color": "#666", "marginTop": "-8px"}),
    ], style={"padding": "14px", "background": "#fff", "border": "1px solid #dbeafe", "borderRadius": "8px", "marginBottom": "14px"})

def _get_artifact_version_history(artifact):
    """Load the immutable artifact chain for the current chat."""
    if not artifact:
        return []

    chat_id = artifact.get("chat_id")
    if not chat_id:
        return []

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT *
            FROM workspace_artifacts
            WHERE chat_id=?
            ORDER BY version ASC, created_at ASC
            """,
            (chat_id,),
        ).fetchall()

    return [dict(row) for row in rows]


def _version_burden_from_verification(record):
    """Resolve the best verified burden for one immutable artifact version."""
    if not record:
        return None

    current_regulatory = _decode_quality_json(
        record.get("current_regulatory")
    ) or {}
    current_redteam = _decode_quality_json(
        record.get("current_redteam")
    ) or {}

    return _quality_findings_burden(
        current_regulatory,
        current_redteam,
    )


def _build_version_history_timeline(artifact):
    """Build a verified version-by-version remediation timeline."""
    artifacts = _get_artifact_version_history(artifact)
    if not artifacts:
        return None

    records = {}
    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute("SELECT * FROM artifact_quality_verifications WHERE artifact_id IN (SELECT id FROM workspace_artifacts WHERE chat_id=?)", (artifact.get("chat_id"),)).fetchall()
    for row in rows:
        records[row["artifact_id"]] = dict(row)

    history = []
    for item in artifacts:
        record = records.get(item["id"])
        burden = _version_burden_from_verification(record)
        if burden is None:
            child_records = [v for v in records.values() if v.get("source_artifact_id") == item["id"]]
            if child_records:
                burden = _quality_findings_burden(
                    _decode_quality_json(child_records[0].get("baseline_regulatory")) or {},
                    _decode_quality_json(child_records[0].get("baseline_redteam")) or {},
                )
        history.append({"version": int(item.get("version") or 0), "burden": burden, "artifact_id": item["id"]})

    verified = [x for x in history if x["burden"] is not None]
    if len(verified) < 2:
        return html.Div([
            html.Div("VERSION HISTORY", style={"fontSize": "11px", "color": "#888", "fontWeight": "700", "textTransform": "uppercase"}),
            html.Div("A verified timeline will appear after at least two workspace versions have scan results.", style={"fontSize": "12px", "color": "#666", "marginTop": "5px"}),
        ], style={"padding": "14px", "background": "#fff", "border": "1px solid #e5e7eb", "borderRadius": "8px", "marginBottom": "14px"})

    frame = pd.DataFrame([{"Version": f"V{x['version']}", "Findings burden": x["burden"]} for x in verified])
    figure = px.line(frame, x="Version", y="Findings burden", markers=True, text="Findings burden", title="Verified Version History — Compliance Burden", labels={"Version": "Workspace version", "Findings burden": "Severity-weighted burden"})
    figure.update_traces(textposition="top center", hovertemplate="Version %{x}<br>Verified burden: %{y}<extra></extra>")
    figure.update_layout(height=300, margin={"l": 45, "r": 20, "t": 55, "b": 45}, showlegend=False, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    figure.update_yaxes(rangemode="tozero")

    latest, earliest = verified[-1], verified[0]
    delta = latest["burden"] - earliest["burden"]
    pct = abs(delta) / earliest["burden"] * 100.0 if earliest["burden"] > 0 else 0.0
    if delta < 0:
        summary = f"🟢 V{earliest['version']} → V{latest['version']} · {abs(delta)} burden removed · {pct:.1f}% reduction"
        color = "#2e7d32"
    elif delta > 0:
        summary = f"🔴 V{earliest['version']} → V{latest['version']} · {delta} burden increase · {pct:.1f}% increase"
        color = "#c62828"
    else:
        summary = f"⚪ V{earliest['version']} → V{latest['version']} · No burden change"
        color = "#666"

    table = html.Table([
        html.Thead(html.Tr([html.Th("Version"), html.Th("Burden"), html.Th("Change")])),
        html.Tbody([
            html.Tr([
                html.Td(f"V{x['version']}"), html.Td(x["burden"]),
                html.Td("—" if i == 0 else (f"+{x['burden']-verified[i-1]['burden']}" if x['burden'] > verified[i-1]['burden'] else str(x['burden']-verified[i-1]['burden'])))
            ]) for i, x in enumerate(verified[-8:])
        ])
    ], style={"width": "100%", "fontSize": "12px", "marginTop": "8px", "borderCollapse": "collapse"})
    return html.Div([
        html.Div("VERIFIED VERSION HISTORY", style={"fontSize": "11px", "color": "#888", "fontWeight": "700", "textTransform": "uppercase", "marginBottom": "3px"}),
        html.Div(summary, style={"fontSize": "13px", "fontWeight": "700", "color": color, "marginBottom": "5px"}),
        dcc.Graph(figure=figure, config={"displayModeBar": False, "responsive": True}, style={"height": "300px"}),
        table,
        html.Div("Only verified versions are plotted. Unscanned versions are excluded rather than estimated.", style={"fontSize": "11px", "color": "#666", "marginTop": "8px"}),
    ], style={"padding": "14px", "background": "#fff", "border": "1px solid #dbeafe", "borderRadius": "8px", "marginBottom": "14px"})

def _dashboard_severity_counts(regulatory_data=None, redteam_data=None):
    """Return severity counts across Regulatory Radar and Red-Team findings."""
    counts = {"Critical": 0, "High": 0, "Medium": 0, "Low": 0}
    if isinstance(regulatory_data, dict):
        result = regulatory_data.get("result") or regulatory_data
        for item in _regulatory_findings(result):
            severity = str(item.get("severity") or item.get("risk_rating") or "Medium").title()
            counts[severity if severity in counts else "Medium"] += 1
    if isinstance(redteam_data, dict):
        for item in redteam_data.get("concerns") or []:
            severity = str(item.get("risk_rating") or item.get("severity") or "Medium").title()
            counts[severity if severity in counts else "Medium"] += 1
    return counts


def _dashboard_kpi_card(label, value, subtitle, color="#1f3b64"):
    return html.Div([
        html.Div(label, style={"fontSize": "10px", "fontWeight": "700", "color": "#6b7280", "textTransform": "uppercase", "letterSpacing": "0.4px"}),
        html.Div(value, style={"fontSize": "27px", "fontWeight": "800", "color": color, "lineHeight": "1.15", "marginTop": "3px"}),
        html.Div(subtitle, style={"fontSize": "10px", "color": "#6b7280", "marginTop": "3px"}),
    ], style={"flex": "1 1 130px", "minWidth": "120px", "padding": "12px", "background": "#fff", "border": "1px solid #e5e7eb", "borderRadius": "8px"})


def _build_severity_breakdown(baseline_regulatory, baseline_redteam, current_regulatory, current_redteam):
    before = _dashboard_severity_counts(baseline_regulatory, baseline_redteam)
    after = _dashboard_severity_counts(current_regulatory, current_redteam)
    frame = pd.DataFrame([
        {"Severity": level, "Stage": "Before", "Findings": before[level]} for level in ("Critical", "High", "Medium", "Low")
    ] + [
        {"Severity": level, "Stage": "After", "Findings": after[level]} for level in ("Critical", "High", "Medium", "Low")
    ])
    figure = px.bar(frame, x="Severity", y="Findings", color="Stage", barmode="group", text="Findings", title="Severity Distribution — Before → After", labels={"Findings": "Findings", "Severity": ""})
    figure.update_traces(textposition="outside", hovertemplate="%{x}<br>%{fullData.name}: %{y} finding(s)<extra></extra>")
    figure.update_layout(height=280, margin={"l": 40, "r": 20, "t": 55, "b": 40}, legend={"orientation": "h", "y": 1.08, "x": 0}, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    figure.update_yaxes(rangemode="tozero", dtick=1)
    return html.Div([
        html.Div("SEVERITY DISTRIBUTION", style={"fontSize": "11px", "color": "#888", "fontWeight": "700", "textTransform": "uppercase", "marginBottom": "3px"}),
        dcc.Graph(figure=figure, config={"displayModeBar": False, "responsive": True}, style={"height": "280px"}),
        html.Div([
            html.Span(f"Critical {after['Critical']}"), html.Span(f"High {after['High']}", style={"marginLeft": "14px"}), html.Span(f"Medium {after['Medium']}", style={"marginLeft": "14px"}), html.Span(f"Low {after['Low']}", style={"marginLeft": "14px"}),
        ], style={"fontSize": "11px", "color": "#666"}),
    ], style={"padding": "14px", "background": "#fff", "border": "1px solid #dbeafe", "borderRadius": "8px", "marginBottom": "14px"})


def _render_compliance_dashboard(artifact, regulatory_data, redteam_data):
    """Render current regulatory and stakeholder findings for one artifact."""
    if not artifact:
        return html.Div(
            "⚠️ No workspace artifact is currently loaded.",
            style={"color": "#c62828"},
        )

    artifact_id = artifact.get("id")
    artifact_type = artifact.get("artifact_type") or "document"
    document_type = resolve_artifact_document_type(artifact)
    role = artifact.get("role") or "ba"
    mode = artifact.get("mode") or "general"
    display_mode = get_workspace_display_mode(role, mode, document_type)

    sections = [
        html.Div(
            [
                html.Div(
                    "CURRENT WORKSPACE ARTIFACT",
                    style={
                        "fontSize": "11px",
                        "color": "#888",
                        "textTransform": "uppercase",
                        "fontWeight": "700",
                    },
                ),
                html.Div(
                    f"{display_mode} · {artifact_type.upper()} · "
                    f"Version {artifact.get('version', '?')}",
                    style={
                        "fontSize": "17px",
                        "fontWeight": "700",
                        "marginTop": "3px",
                    },
                ),
                html.Div(
                    f"Artifact ID: {artifact_id}",
                    style={
                        "fontSize": "11px",
                        "color": "#999",
                        "marginTop": "2px",
                    },
                ),
            ],
            style={
                "padding": "12px",
                "background": "#f8f9fa",
                "borderRadius": "8px",
                "marginBottom": "14px",
            },
        )
    ]

    verification = _quality_record(artifact_id)
    baseline_burden = current_burden = 0
    baseline_regulatory = baseline_redteam = current_regulatory = current_redteam = {}

    if verification:
        baseline_regulatory = _decode_quality_json(verification.get("baseline_regulatory")) or {}
        baseline_redteam = _decode_quality_json(verification.get("baseline_redteam")) or {}
        current_regulatory = _decode_quality_json(verification.get("current_regulatory")) or {}
        current_redteam = _decode_quality_json(verification.get("current_redteam")) or {}
        baseline_burden = _quality_findings_burden(baseline_regulatory, baseline_redteam)
        current_burden = _quality_findings_burden(current_regulatory, current_redteam)

        before_counts = _finding_counts_by_source(baseline_regulatory, baseline_redteam)
        current_counts = _finding_counts_by_source(current_regulatory, current_redteam)
        remediation_score = _combined_remediation_score(baseline_burden, current_burden)
        burden_delta = current_burden - baseline_burden

        if current_burden == 0:
            status_text, status_color, status_bg = "CLEAR", "#2e7d32", "#f0fdf4"
        elif burden_delta < 0:
            status_text, status_color, status_bg = "IMPROVED · ACTION REQUIRED", "#2e7d32", "#f0fdf4"
        elif burden_delta > 0:
            status_text, status_color, status_bg = "ACTION REQUIRED · BURDEN INCREASED", "#c62828", "#fff5f5"
        else:
            status_text, status_color, status_bg = "ACTION REQUIRED · NO CHANGE", "#e65100", "#fff8f0"

        delta_text = (
            f"{abs(burden_delta)} burden removed" if burden_delta < 0 else
            f"{burden_delta} burden increase" if burden_delta > 0 else
            "No burden change"
        )
        delta_color = "#2e7d32" if burden_delta < 0 else "#c62828" if burden_delta > 0 else "#666"
        verified_at = verification.get("verified_at") or "Timestamp unavailable"

        score_sections = [
            html.Div([
                html.Div("CURRENT COMPLIANCE STATUS", style={"fontSize": "10px", "fontWeight": "700", "color": "#777", "textTransform": "uppercase"}),
                html.Div(status_text, style={"fontSize": "14px", "fontWeight": "800", "color": status_color, "marginTop": "3px"}),
                html.Div(f"Last verified: {verified_at}", style={"fontSize": "10px", "color": "#777", "marginTop": "2px"}),
            ], style={"padding": "10px 12px", "background": status_bg, "borderRadius": "7px", "marginBottom": "10px"}),
            html.Div([
                _dashboard_kpi_card("Remediation progress", f"{remediation_score:.1f}%", "verified burden reduction", "#2e7d32" if burden_delta < 0 else "#c62828" if burden_delta > 0 else "#666"),
                _dashboard_kpi_card("Current burden", str(current_burden), delta_text, delta_color),
                _dashboard_kpi_card("Regulatory findings", str(current_counts["Regulatory Radar"]), f"Before: {before_counts['Regulatory Radar']}"),
                _dashboard_kpi_card("Red-Team concerns", str(current_counts["Red-Team"]), f"Before: {before_counts['Red-Team']}"),
            ], style={"display": "flex", "gap": "8px", "flexWrap": "wrap"}),
            html.Div(
                "Severity-weighted burden combines Regulatory Radar and Red-Team findings. It is a comparison metric, not a legal/compliance certification score.",
                style={"fontSize": "10px", "color": "#666", "marginTop": "9px"},
            ),
        ]
    else:
        score_sections = [
            html.Div("CURRENT COMPLIANCE STATUS", style={"fontSize": "10px", "fontWeight": "700", "color": "#777", "textTransform": "uppercase"}),
            html.Div("NOT YET VERIFIED", style={"fontSize": "15px", "fontWeight": "800", "color": "#666", "marginTop": "3px"}),
            html.Div("Run and apply a Regulatory Radar or Red-Team remediation to establish a verified baseline/current comparison.", style={"fontSize": "11px", "color": "#777", "marginTop": "4px"}),
        ]

    sections.append(
        html.Div(
            score_sections,
            style={
                "padding": "14px",
                "background": "#f8fbff",
                "border": "1px solid #dbeafe",
                "borderRadius": "8px",
                "marginBottom": "14px",
            },
        )
    )

    version_difference_graph = _build_version_difference_graph(
        artifact,
        verification,
        baseline_burden if verification else 0,
        current_burden if verification else 0,
    )
    if version_difference_graph:
        sections.append(version_difference_graph)

    if verification:
        sections.append(
            _build_quality_source_comparison_graph(
                baseline_regulatory,
                baseline_redteam,
                current_regulatory,
                current_redteam,
            )
        )
        sections.append(
            _build_severity_breakdown(
                baseline_regulatory,
                baseline_redteam,
                current_regulatory,
                current_redteam,
            )
        )

    version_history_timeline = _build_version_history_timeline(artifact)
    if version_history_timeline:
        sections.append(version_history_timeline)

    if regulatory_data:
        result = regulatory_data.get("result") or {}
        findings = _regulatory_findings(result)
        risk = result.get("overall_risk", "Unknown")
        sections.append(
            html.Div(
                [
                    html.Div(
                        [
                            html.Span(
                                "🛡️ Regulatory Radar",
                                style={"fontWeight": "700", "fontSize": "15px"},
                            ),
                            html.Span(
                                f"Risk: {risk}",
                                style={
                                    "fontWeight": "700",
                                    "color": _dashboard_risk_color(risk),
                                    "marginLeft": "12px",
                                },
                            ),
                        ],
                        style={"marginBottom": "5px"},
                    ),
                    html.Div(
                        f"{len(findings)} actionable finding(s)",
                        style={
                            "fontSize": "12px",
                            "color": "#666",
                            "marginBottom": "8px",
                        },
                    ),
                    html.Div(
                        [
                            html.Div(
                                [
                                    html.Strong(item["regulation"]),
                                    html.Span(
                                        f": {item['gap']}",
                                        style={"marginLeft": "4px"},
                                    ),
                                ],
                                style={
                                    "padding": "7px 9px",
                                    "borderLeft": "3px solid #c62828",
                                    "background": "#fff8f8",
                                    "borderRadius": "5px",
                                    "marginBottom": "5px",
                                    "fontSize": "12px",
                                },
                            )
                            for item in findings[:20]
                        ]
                        or [
                            html.Div(
                                "✅ No actionable regulatory gaps.",
                                style={"color": "#2e7d32"},
                            )
                        ],
                    ),
                ],
                style={
                    "padding": "12px",
                    "border": "1px solid #eee",
                    "borderRadius": "8px",
                    "marginBottom": "12px",
                },
            )
        )
    else:
        sections.append(
            html.Div(
                [
                    html.Strong("🛡️ Regulatory Radar"),
                    html.Div(
                        "No Regulatory Radar scan is available for this workspace version.",
                        style={
                            "fontSize": "12px",
                            "color": "#888",
                            "marginTop": "4px",
                        },
                    ),
                ],
                style={
                    "padding": "12px",
                    "background": "#fafafa",
                    "borderRadius": "8px",
                    "marginBottom": "12px",
                },
            )
        )

    if redteam_data:
        reviews = redteam_data.get("reviews") or []
        concerns = redteam_data.get("concerns") or []
        priority = {
            "Critical": 4,
            "High": 3,
            "Medium": 2,
            "Low": 1,
        }
        overall_risk = (
            max(
                (
                    str(review.get("risk_rating", "Medium"))
                    for review in reviews
                ),
                key=lambda value: priority.get(value, 0),
            )
            if reviews
            else "Unknown"
        )

        persona_rows = []
        for review in reviews:
            persona_key = review.get("persona_key")
            persona = review.get(
                "persona_label",
                review.get("persona", "Stakeholder"),
            )
            risk = review.get("risk_rating", "Medium")
            persona_concerns = [
                item
                for item in concerns
                if item.get("persona_key") == persona_key
            ]
            persona_rows.append(
                html.Div(
                    [
                        html.Div(
                            [
                                html.Strong(persona),
                                html.Span(
                                    f" · {risk}",
                                    style={
                                        "color": _dashboard_risk_color(risk),
                                        "fontWeight": "700",
                                    },
                                ),
                            ],
                            style={"marginBottom": "4px"},
                        ),
                        html.Div(
                            f"{len(persona_concerns)} concern(s)",
                            style={
                                "fontSize": "11px",
                                "color": "#666",
                                "marginBottom": "5px",
                            },
                        ),
                        html.Ul(
                            [
                                html.Li(item.get("title", "Concern"))
                                for item in persona_concerns[:8]
                            ],
                            style={
                                "paddingLeft": "20px",
                                "marginBottom": "0",
                                "fontSize": "12px",
                            },
                        ),
                    ],
                    style={
                        "padding": "8px",
                        "border": "1px solid #eee",
                        "borderRadius": "6px",
                        "marginBottom": "6px",
                    },
                )
            )

        sections.append(
            html.Div(
                [
                    html.Div(
                        [
                            html.Span(
                                "🎭 Stakeholder Red-Team",
                                style={"fontWeight": "700", "fontSize": "15px"},
                            ),
                            html.Span(
                                f"Highest Risk: {overall_risk}",
                                style={
                                    "fontWeight": "700",
                                    "color": _dashboard_risk_color(overall_risk),
                                    "marginLeft": "12px",
                                },
                            ),
                        ],
                        style={"marginBottom": "5px"},
                    ),
                    html.Div(
                        f"{len(reviews)} stakeholder perspective(s) · "
                        f"{len(concerns)} concern(s)",
                        style={
                            "fontSize": "12px",
                            "color": "#666",
                            "marginBottom": "8px",
                        },
                    ),
                    *persona_rows,
                ],
                style={
                    "padding": "12px",
                    "border": "1px solid #eee",
                    "borderRadius": "8px",
                },
            )
        )
    else:
        sections.append(
            html.Div(
                [
                    html.Strong("🎭 Stakeholder Red-Team"),
                    html.Div(
                        "No Red-Team review is available for this workspace version.",
                        style={
                            "fontSize": "12px",
                            "color": "#888",
                            "marginTop": "4px",
                        },
                    ),
                ],
                style={
                    "padding": "12px",
                    "background": "#fafafa",
                    "borderRadius": "8px",
                },
            )
        )

    return sections


@app.callback(
    [
        Output("compliance-dashboard-modal", "is_open"),
        Output("compliance-dashboard-results", "children"),
        Output("compliance-dashboard-data", "data"),
    ],
    Input("open-compliance-dashboard-btn", "n_clicks"),
    [
        State("current-workspace-artifact-id", "data"),
        State("regulatory-scan-data", "data"),
        State("redteam-review-data", "data"),
    ],
    prevent_initial_call=True,
)
def open_compliance_dashboard(n, artifact_id, regulatory_data, redteam_data):
    if not n:
        raise dash.exceptions.PreventUpdate

    if not artifact_id:
        return (
            True,
            html.Div(
                [
                    html.Strong("📊 Compliance Dashboard"),
                    html.Div(
                        "No workspace artifact is active for the current mode. "
                        "Generate or open an artifact first.",
                        style={
                            "fontSize": "12px",
                            "color": "#888",
                            "marginTop": "6px",
                        },
                    ),
                ]
            ),
            {
                "artifact_id": None,
                "regulatory": None,
                "redteam": None,
                "verification": None,
            },
        )

    artifact = get_workspace_artifact(artifact_id)
    if not artifact:
        return (
            True,
            html.Div(
                "⚠️ The current workspace artifact no longer exists.",
                style={"color": "#c62828"},
            ),
            None,
        )

    # The exact artifact ID is authoritative. Never let a scan from another
    # version/mode populate this dashboard.
    payload = _build_dashboard_payload(
        artifact_id,
        regulatory_data,
        redteam_data,
    )

    return (
        True,
        _render_compliance_dashboard(
            artifact,
            payload["regulatory"],
            payload["redteam"],
        ),
        payload,
    )


@app.callback(
    [
        Output("compliance-dashboard-modal", "is_open", allow_duplicate=True),
        Output("compliance-dashboard-results", "children", allow_duplicate=True),
        Output("compliance-dashboard-data", "data", allow_duplicate=True),
    ],
    [
        Input("regulatory-dashboard-btn", "n_clicks"),
        Input("redteam-dashboard-btn", "n_clicks"),
    ],
    [
        State("current-workspace-artifact-id", "data"),
        State("regulatory-scan-data", "data"),
        State("redteam-review-data", "data"),
    ],
    prevent_initial_call=True,
)
def open_compliance_dashboard_from_review(
    regulatory_clicks,
    redteam_clicks,
    artifact_id,
    regulatory_data,
    redteam_data,
):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    artifact = get_workspace_artifact(artifact_id) if artifact_id else None
    payload = _build_dashboard_payload(
        artifact_id,
        regulatory_data,
        redteam_data,
    )

    return (
        True,
        _render_compliance_dashboard(
            artifact,
            payload["regulatory"],
            payload["redteam"],
        ),
        payload,
    )


@app.callback(
    [
        Output("compliance-dashboard-modal", "is_open", allow_duplicate=True),
        Output("compliance-dashboard-results", "children", allow_duplicate=True),
        Output("compliance-dashboard-data", "data", allow_duplicate=True),
    ],
    Input({"type": "artifact-dashboard-btn", "index": ALL}, "n_clicks"),
    prevent_initial_call=True,
)
def open_artifact_compliance_dashboard(clicks):
    if not clicks or not any(clicks):
        raise dash.exceptions.PreventUpdate

    ctx = callback_context
    triggered = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])
    artifact_id = triggered.get("index")
    artifact = get_workspace_artifact(artifact_id)

    if not artifact:
        return (
            True,
            html.Div(
                "⚠️ Workspace artifact was not found.",
                style={"color": "#c62828"},
            ),
            None,
        )

    payload = _build_dashboard_payload(
        artifact_id,
        None,
        None,
    )

    return (
        True,
        _render_compliance_dashboard(
            artifact,
            payload["regulatory"],
            payload["redteam"],
        ),
        payload,
    )


@app.callback(
    Output("compliance-dashboard-modal", "is_open", allow_duplicate=True),
    Input("compliance-dashboard-close-btn", "n_clicks"),
    prevent_initial_call=True,
)
def close_compliance_dashboard(n):
    if not n:
        raise dash.exceptions.PreventUpdate
    return False


@app.callback(
    [
        Output("compliance-dashboard-modal", "is_open", allow_duplicate=True),
        Output("redteam-modal", "is_open", allow_duplicate=True),
        Output("regulatory-modal", "is_open", allow_duplicate=True),
    ],
    [
        Input("dashboard-redteam-btn", "n_clicks"),
        Input("dashboard-regulatory-btn", "n_clicks"),
    ],
    prevent_initial_call=True,
)
def launch_review_from_dashboard(redteam_clicks, regulatory_clicks):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    trigger = ctx.triggered[0]["prop_id"].split(".")[0]

    if trigger == "dashboard-redteam-btn":
        return False, True, False

    if trigger == "dashboard-regulatory-btn":
        return False, False, True

    raise dash.exceptions.PreventUpdate


@app.callback(
    [
        Output("redteam-modal", "is_open", allow_duplicate=True),
        Output("redteam-results", "children", allow_duplicate=True),
        Output("redteam-review-data", "data", allow_duplicate=True),
        Output("redteam-findings-checklist", "options", allow_duplicate=True),
        Output("redteam-findings-checklist", "value", allow_duplicate=True),
        Output("redteam-apply-all-btn", "disabled", allow_duplicate=True),
        Output("redteam-apply-selected-btn", "disabled", allow_duplicate=True),
    ],
    Input({"type": "rerun-redteam-btn", "index": ALL}, "n_clicks"),
    prevent_initial_call=True,
)
def rerun_redteam_after_remediation(clicks):
    if not clicks or not any(clicks):
        raise dash.exceptions.PreventUpdate

    ctx = callback_context
    triggered = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])
    artifact_id = triggered.get("index")
    artifact = get_workspace_artifact(artifact_id)

    if not artifact:
        return (
            True,
            html.Div(
                "⚠️ Remediated workspace version was not found.",
                style={"color": "#c62828"},
            ),
            None,
            [],
            [],
            True,
            True,
        )

    return (
        True,
        html.Div(
            [
                html.Strong("🎭 Ready to re-run Red-Team"),
                html.Div(
                    "The current workspace has been updated. Select a "
                    "stakeholder or Select All Stakeholders, then click Run Critique.",
                    style={
                        "fontSize": "12px",
                        "color": "#666",
                        "marginTop": "4px",
                    },
                ),
                html.Div(
                    f"Artifact version: {artifact.get('version', '?')}",
                    style={
                        "fontSize": "11px",
                        "color": "#888",
                        "marginTop": "4px",
                    },
                ),
            ],
            style={
                "padding": "10px",
                "background": "#f3e5f5",
                "borderRadius": "7px",
            },
        ),
        None,
        [],
        [],
        True,
        True,
    )


@app.callback(
    [
        Output("regulatory-modal", "is_open", allow_duplicate=True),
        Output("regulatory-scan-results", "children", allow_duplicate=True),
        Output("regulatory-scan-data", "data", allow_duplicate=True),
        Output("regulatory-findings-checklist", "options", allow_duplicate=True),
        Output("regulatory-findings-checklist", "value", allow_duplicate=True),
        Output("regulatory-apply-all-btn", "disabled", allow_duplicate=True),
        Output("regulatory-apply-selected-btn", "disabled", allow_duplicate=True),
    ],
    Input({"type": "rescan-regulatory-btn", "index": ALL}, "n_clicks"),
    prevent_initial_call=True,
)
def rescan_regulatory_after_remediation(clicks):
    if not clicks or not any(clicks):
        raise dash.exceptions.PreventUpdate

    ctx = callback_context
    triggered = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])
    artifact_id = triggered.get("index")
    artifact = get_workspace_artifact(artifact_id)

    if not artifact:
        return (
            True,
            html.Div(
                "⚠️ Updated workspace version was not found.",
                style={"color": "#c62828"},
            ),
            None,
            [],
            [],
            True,
            True,
        )

    try:
        result = run_regulatory_scan(artifact.get("doc_content") or "")
    except Exception as exc:
        return (
            True,
            html.Div(
                f"❌ Regulatory re-scan failed: {str(exc)[:300]}",
                style={"color": "#c62828"},
            ),
            None,
            [],
            [],
            True,
            True,
        )

    findings = _regulatory_findings(result)
    payload = {
        "artifact_id": artifact_id,
        "result": result,
    }

    _update_quality_current(
        artifact_id,
        regulatory_data=payload,
    )

    return (
        True,
        render_regulatory_result(result),
        payload,
        _regulatory_checklist_options(result),
        [str(item["index"]) for item in findings],
        not bool(findings),
        not bool(findings),
    )


# ── AI Banner ─────────────────────────────────────────────────────
@app.callback(
    [Output("ai-banner-div","style"),
     Output("ai-banner-text","children"),
     Output("ai-indicator","children")],
    Input("ai-banner-store","data"),
    prevent_initial_call=False)
def show_ai_banner(banner):
    if banner:
        name = banner.split("to ")[-1].split(" —")[0] if "to " in banner else "AI"
        return {"display":"block","marginTop":"6px"}, banner, f"• {name}"
    return {"display":"none"}, "", "• Gemini"

# ── BPMN canvas toggle ────────────────────────────────────────────
@app.callback(
    [Output("bpmn-canvas","style"),
     Output("workspace-content","style")],
    Input("bpmn-xml-store","data"),
    prevent_initial_call=False)
def toggle_bpmn_canvas(xml):
    if xml: return BPMN_CANVAS_VISIBLE, WS_CONTENT_HIDDEN
    return BPMN_CANVAS_HIDDEN, WS_CONTENT_VISIBLE

# ── Process submode bar ────────────────────────────────────────────
@app.callback(
    Output("process-submode-bar","style"),
    Input("current-mode","data"),
    prevent_initial_call=False)
def toggle_submode_bar(mode):
    return {"display":"flex","gap":"6px","marginTop":"8px"} if mode=="process" else {"display":"none"}

@app.callback(
    [
        Output("conversion-modal", "is_open"),
        Output("conversion-source-summary", "children"),
        Output("conversion-target-select", "options"),
        Output("conversion-target-select", "value"),
        Output("conversion-source-artifact", "data"),
    ],
    Input("open-conversion-btn", "n_clicks"),
    [
        State("current-workspace-artifact-id", "data"),
        State("current-chat-id", "data"),
    ],
    prevent_initial_call=True,
)
def open_conversion_modal(n, artifact_id, chat_id):
    if not n:
        raise dash.exceptions.PreventUpdate

    artifact = get_workspace_artifact(artifact_id) if artifact_id else None
    if not artifact and chat_id:
        artifact = get_latest_workspace_artifact(chat_id)

    if not artifact or not artifact.get("doc_content"):
        return (
            True,
            html.Div("⚠️ No requirements document is currently open.", style={"color":"#c62828"}),
            [],
            None,
            None,
        )

    source_type = resolve_artifact_document_type(artifact) or "BRD"
    source_title = get_document_type_title(source_type)
    options = [
        {
            "label": f"{config['short_title']} — {config['title']}",
            "value": key,
        }
        for key, config in DOCUMENT_TYPE_CONFIG.items()
        if key != source_type
    ]

    summary = html.Div([
        html.Div("Source", style={"fontSize":"11px","fontWeight":"700","color":"#777","textTransform":"uppercase"}),
        html.Div(
            f"{source_type} · {source_title} · Version {artifact.get('version', '?')}",
            style={"fontWeight":"700","fontSize":"14px"},
        ),
        html.Div(
            "The original artifact will remain unchanged.",
            style={"fontSize":"12px","color":"#666","marginTop":"4px"},
        ),
    ])

    return True, summary, options, None, {
        "source_artifact_id": artifact.get("id"),
        "source_document_type": source_type,
        "source_version": artifact.get("version"),
    }


@app.callback(
    Output("conversion-modal", "is_open", allow_duplicate=True),
    Input("conversion-cancel-btn", "n_clicks"),
    prevent_initial_call=True,
)
def cancel_conversion(n):
    if not n:
        raise dash.exceptions.PreventUpdate
    return False


@app.callback(
    [
        Output("conversion-modal", "is_open", allow_duplicate=True),
        Output("conversion-source-artifact", "data", allow_duplicate=True),
        Output("conversion-source-summary", "children", allow_duplicate=True),
        Output("conversion-success-modal", "is_open", allow_duplicate=True),
        Output("conversion-success-content", "children", allow_duplicate=True),
        Output("conversion-success-artifact-id", "data", allow_duplicate=True),
        Output("chat-messages", "children", allow_duplicate=True),
        Output("workspace-content", "children", allow_duplicate=True),
        Output("chart-area", "children", allow_duplicate=True),
        Output("workspace-col", "style", allow_duplicate=True),
        Output("chat-col", "style", allow_duplicate=True),
        Output("doc-content", "data", allow_duplicate=True),
        Output("doc-mode", "data", allow_duplicate=True),
        Output("workspace-open", "data", allow_duplicate=True),
        Output("current-workspace-artifact-id", "data", allow_duplicate=True),
        Output("ai-banner-store", "data", allow_duplicate=True),
    ],
    Input("conversion-confirm-btn", "n_clicks"),
    [
        State("conversion-target-select", "value"),
        State("conversion-source-artifact", "data"),
        State("current-chat-id", "data"),
        State("chat-messages", "children"),
    ],
    prevent_initial_call=True,
)
def confirm_document_conversion(
    n,
    target_type,
    source_info,
    current_chat_id,
    chat_children,
):
    if not n:
        raise dash.exceptions.PreventUpdate

    source_id = (source_info or {}).get("source_artifact_id")
    source = get_workspace_artifact(source_id) if source_id else None
    if not source:
        return (
            False, None,
            html.Div("⚠️ Source workspace artifact no longer exists.", style={"color":"#c62828"}),
            False, dash.no_update, None,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update, dash.no_update,
        )

    if not target_type:
        return (
            True, source_info,
            html.Div("Select a target document type before converting.", style={"color":"#c62828"}),
            False, dash.no_update, None,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update, dash.no_update,
        )

    try:
        conversion = convert_workspace_artifact(source, target_type)
        chat_id = current_chat_id or source.get("chat_id")
        if not chat_id:
            raise ValueError("No chat is associated with the source artifact.")

        message_content = (
            f"🔄 Converted {conversion['source_document_type']} v{source.get('version', '?')} "
            f"to {target_type}."
        )
        message_id = save_message(
            chat_id,
            "assistant",
            message_content,
            source.get("mode") or "brd",
        )
        new_artifact_id = create_workspace_artifact(
            chat_id=chat_id,
            message_id=message_id,
            role=source.get("role") or "ba",
            mode=source.get("mode") or "brd",
            doc_content=conversion["doc_content"],
            document_type=target_type,
            process_submode=source.get("process_submode") or "text",
            bpmn_xml="",
            parent_artifact_id=source.get("id"),
        )
        new_artifact = get_workspace_artifact(new_artifact_id)
        if not new_artifact:
            raise ValueError("Converted artifact could not be stored.")

        state = workspace_artifact_to_state(new_artifact)
        ws_children, bpmn_xml, chart = render_workspace_artifact(new_artifact)
        WorkspaceState.save(
            chat_id,
            doc_content=state["doc_content"],
            doc_mode="brd",
            document_type=target_type,
            process_submode=state["process_submode"],
            bpmn_xml="",
            workspace_open=True,
        )

        role_label = {"ba":"📊 [BA]","po":"🎯 [PO]","pm":"🧭 [PM]"}.get(source.get("role"), "📊 [BA]")
        bubble = html.Div([
            html.Div(
                f"{role_label}·{target_type}",
                style={"fontWeight":"600","color":BA_COLORS.get("brd","#9c27b0"),"fontSize":"12px","marginBottom":"8px"},
            ),
            html.Div(
                f"🔄 {get_document_type_title(target_type)} v{new_artifact.get('version', '?')} created from "
                f"{get_document_type_title(conversion['source_document_type'])} v{source.get('version', '?')}.",
                style={"background":"#e8f0fe","padding":"10px 14px","borderRadius":"16px 16px 16px 4px","borderLeft":"3px solid #1a73e8","fontSize":"13px"},
            ),
        ], className="message-bubble")

        existing_chat = chat_children if isinstance(chat_children, list) else []
        confirmation = html.Div([
            html.Div(
                f"{get_document_type_title(target_type)} Version {new_artifact.get('version', '?')} created successfully.",
                style={"fontWeight":"700","fontSize":"14px","marginBottom":"8px"},
            ),
            html.Div(f"Source: {conversion['source_document_type']} Version {source.get('version', '?')}", style={"fontSize":"12px"}),
            html.Div(f"New Document: {target_type} Version {new_artifact.get('version', '?')}", style={"fontSize":"12px","marginBottom":"10px"}),
            html.Div(
                "The original document remains unchanged. The new artifact preserves parent lineage and is now open in Workspace.",
                style={"fontSize":"12px","color":"#666"},
            ),
        ])

        return (
            False, None,
            html.Div(f"Converted to {target_type}.", style={"color":"#2e7d32"}),
            True, confirmation, new_artifact_id,
            existing_chat + [bubble],
            ws_children,
            chart,
            WS_OPEN_STYLE,
            CHAT_HALF_STYLE,
            state["doc_content"],
            "brd",
            True,
            new_artifact_id,
            conversion.get("banner") or "",
        )
    except Exception as exc:
        return (
            True, source_info,
            html.Div([
                html.Div("❌ Conversion failed", style={"fontWeight":"700","color":"#c62828","marginBottom":"4px"}),
                html.Div(str(exc)[:1000], style={"fontSize":"12px","color":"#666","whiteSpace":"pre-wrap"}),
            ]),
            False, dash.no_update, None,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update, dash.no_update,
        )



def review_converted_document(n):
    if not n:
        raise dash.exceptions.PreventUpdate
    return False, True

@app.callback(Output("redteam-modal","is_open"),
    Input("open-redteam-btn","n_clicks"), prevent_initial_call=True)
def open_redteam_modal(n): return True

@app.callback(Output("redteam-modal","is_open",allow_duplicate=True),
    Input("redteam-close-btn","n_clicks"), prevent_initial_call=True)
def close_redteam_modal(n): return False

@app.callback(
    [
        Output("redteam-persona-select", "options", allow_duplicate=True),
        Output("redteam-persona-select", "value", allow_duplicate=True),
        Output("redteam-results", "children", allow_duplicate=True),
    ],
    Input("open-redteam-btn", "n_clicks"),
    [
        State("current-workspace-artifact-id", "data"),
        State("current-chat-id", "data"),
    ],
    prevent_initial_call=True,
)
def prepare_redteam_personas(n, artifact_id, chat_id):
    """Preselect the document-aware stakeholder suite when Red-Team opens."""
    if not n:
        raise dash.exceptions.PreventUpdate

    artifact = get_workspace_artifact(artifact_id) if artifact_id else None
    if not artifact and chat_id:
        artifact = get_latest_workspace_artifact(chat_id)

    document_type = resolve_artifact_document_type(artifact) if artifact else "BRD"
    recommended = get_document_redteam_personas(document_type)

    options = []
    for key in recommended:
        persona = REDTEAM_PERSONAS.get(key)
        if persona:
            options.append({
                "label": f"⭐ {persona['label']}",
                "value": key,
            })

    other_keys = [key for key in REDTEAM_PERSONAS if key not in recommended]
    options.extend(
        {"label": REDTEAM_PERSONAS[key]["label"], "value": key}
        for key in other_keys
    )
    options.insert(0, {"label": "👥 Select All Stakeholders", "value": "__all__"})

    return (
        options,
        recommended,
        html.Div(
            [
                html.Strong(f"Recommended for {document_type}"),
                html.Div(
                    "You can change the selection or add a custom stakeholder.",
                    style={"fontSize": "12px", "color": "#666", "marginTop": "4px"},
                ),
            ],
            style={"padding": "8px", "background": "#f8f5ff", "borderRadius": "7px"},
        ),
    )

@app.callback(
    [Output("ripple-modal","is_open"), Output("ripple-list-content","children"),
     Output("open-ripple-btn","children")],
    Input("open-ripple-btn","n_clicks"),
    prevent_initial_call=False)
def refresh_ripple_button(n):
    flags = load_unacknowledged_ripples()
    label = f"🌊 Ripple ({len(flags)})" if flags else "🌊 Ripple"
    if not n:
        return False, dash.no_update, label
    if not flags:
        return True, html.Div("No pending ripple flags — everything's in sync.", style={"color":"#888"}), label

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        titles = {r["id"]: r["title"] for r in conn.execute("SELECT id, title FROM chats").fetchall()}

    rows = []
    for f in flags:
        rows.append(html.Div([
            html.Div(f"📄 {titles.get(f['chat_id'], f['chat_id'])}", style={"fontWeight":"600","fontSize":"13px"}),
            html.Div(f["reason"], style={"fontSize":"12px","color":"#666","marginBottom":"6px"}),
            dbc.Button("✅ Mark Reviewed", id={"type":"ripple-ack-btn","index":f["id"]},
                       size="sm", color="light", outline=True, n_clicks=0)
        ], style={"borderBottom":"1px solid #eee","padding":"8px 0"}))
    return True, rows, label

@app.callback(Output("ripple-modal","is_open",allow_duplicate=True),
    Input("ripple-close-btn","n_clicks"), prevent_initial_call=True)
def close_ripple_modal(n): return False

@app.callback(
    Output("open-ripple-btn","children", allow_duplicate=True),
    Input({"type":"ripple-ack-btn","index":ALL},"n_clicks"),
    prevent_initial_call=True)
def ack_ripple(clicks):
    ctx = callback_context
    if not ctx.triggered or not any(clicks): raise dash.exceptions.PreventUpdate
    ripple_id = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"]
    acknowledge_ripple(ripple_id)
    flags = load_unacknowledged_ripples()
    return f"🌊 Ripple ({len(flags)})" if flags else "🌊 Ripple"

@app.callback(
    [Output("ripple-list-content","children", allow_duplicate=True),
     Output("open-ripple-btn","children", allow_duplicate=True)],
    Input({"type":"ripple-link-btn","index":ALL},"n_clicks"),
    [State({"type":"ripple-link-btn","index":ALL},"id"), State("current-chat-id","data")],
    prevent_initial_call=True)
def link_suggested_chat(clicks, ids, current_chat_id):
    ctx = callback_context
    if not ctx.triggered or not any(clicks) or not current_chat_id: raise dash.exceptions.PreventUpdate
    target_id = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"]
    link_artifacts(current_chat_id, target_id, relation="fuzzy-confirmed")
    return build_ripple_modal_content(current_chat_id)

@app.callback(
    [Output("ripple-list-content","children", allow_duplicate=True),
     Output("open-ripple-btn","children", allow_duplicate=True)],
    Input({"type":"ripple-dismiss-btn","index":ALL},"n_clicks"),
    [State({"type":"ripple-dismiss-btn","index":ALL},"id"), State("current-chat-id","data")],
    prevent_initial_call=True)
def dismiss_suggestion_cb(clicks, ids, current_chat_id):
    ctx = callback_context
    if not ctx.triggered or not any(clicks) or not current_chat_id: raise dash.exceptions.PreventUpdate
    target_id = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"]
    dismiss_suggestion(current_chat_id, target_id)
    return build_ripple_modal_content(current_chat_id)

@app.callback(
    [Output("ripple-list-content","children", allow_duplicate=True),
     Output("open-ripple-btn","children", allow_duplicate=True)],
    Input({"type":"ripple-unlink-btn","chat_a":ALL,"chat_b":ALL},"n_clicks"),
    [State({"type":"ripple-unlink-btn","chat_a":ALL,"chat_b":ALL},"id"), State("current-chat-id","data")],
    prevent_initial_call=True)
def unlink_chat(clicks, ids, current_chat_id):
    ctx = callback_context
    if not ctx.triggered or not any(clicks): raise dash.exceptions.PreventUpdate
    trig = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])
    unlink_artifacts(trig["chat_a"], trig["chat_b"])
    return build_ripple_modal_content(current_chat_id)

@app.callback(Output("ripple-modal","is_open",allow_duplicate=True),
    Input({"type":"sidebar-item","index":ALL},"n_clicks"), prevent_initial_call=True)
def close_ripple_on_navigate(clicks):
    if not any(clicks): raise dash.exceptions.PreventUpdate
    return False
    

@app.callback(
    [
        Output("redteam-results", "children"),
        Output("redteam-review-data", "data"),
        Output("redteam-findings-checklist", "options"),
        Output("redteam-findings-checklist", "value"),
        Output("redteam-apply-all-btn", "disabled"),
        Output("redteam-apply-selected-btn", "disabled"),
    ],
    Input("redteam-run-btn", "n_clicks"),
    [
        State("redteam-persona-select", "value"),
        State("redteam-custom-stakeholder", "value"),
        State("doc-content", "data"),
        State("doc-mode", "data"),
        State("current-workspace-artifact-id", "data"),
        State("current-chat-id", "data"),
    ],
    prevent_initial_call=True,
)
def run_redteam_critique(
    n,
    persona_keys,
    custom_stakeholder,
    doc_content,
    doc_mode,
    artifact_id,
    chat_id,
):
    if not n:
        raise dash.exceptions.PreventUpdate

    if not doc_content:
        return (
            html.Div("⚠️ No workspace artifact is loaded — generate an artifact first.", style={"color": "#c62828"}),
            None, [], [], True, True,
        )

    persona_keys = persona_keys or []
    if "__all__" in persona_keys:
        persona_keys = list(REDTEAM_PERSONAS.keys())

    if not persona_keys and not (custom_stakeholder or "").strip():
        if chat_id:
            latest = get_latest_workspace_artifact(chat_id)
            artifact_id = artifact_id or (latest["id"] if latest else None)
        artifact = get_workspace_artifact(artifact_id) if artifact_id else None
        inferred_type = resolve_artifact_document_type(artifact) if artifact else "BRD"
        persona_keys = get_document_redteam_personas(inferred_type)

    if not artifact_id and chat_id:
        latest = get_latest_workspace_artifact(chat_id)
        artifact_id = latest["id"] if latest else None

    artifact = get_workspace_artifact(artifact_id) if artifact_id else None
    if artifact:
        review_content = artifact.get("doc_content") or doc_content
        review_mode = artifact.get("mode") or doc_mode or "general"
        review_role = artifact.get("role") or "ba"
        document_type = resolve_artifact_document_type(artifact) or "BRD"
    else:
        review_content = doc_content
        review_mode = doc_mode or "general"
        review_role = "ba"
        document_type = "BRD"

    all_reviews = []
    custom_persona = build_custom_redteam_persona(custom_stakeholder)

    if custom_persona:
        persona_specs = [
            (key, REDTEAM_PERSONAS.get(key))
            for key in persona_keys
            if key in REDTEAM_PERSONAS
        ]
        persona_specs.append(("custom", custom_persona))
    else:
        persona_specs = [
            (key, REDTEAM_PERSONAS.get(key))
            for key in persona_keys
            if key in REDTEAM_PERSONAS
        ]

    if not persona_specs:
        persona_specs = [
            (key, REDTEAM_PERSONAS[key])
            for key in get_document_redteam_personas(document_type)
            if key in REDTEAM_PERSONAS
        ]

    try:
        for key, persona in persona_specs:
            mode_focus = (
            REDTEAM_MODE_FOCUS.get(document_type.lower())
            or REDTEAM_MODE_FOCUS.get(review_mode, "")
        )
            system_prompt = (
                REDTEAM_SYSTEM
                + "\n\nPersona: "
                + persona["prompt"]
                + "\n\nCurrent document type: "
                + document_type
                + "\n\nDocument purpose: "
                + DOCUMENT_TYPE_CONFIG.get(document_type, {}).get("description", "")
            )
            if mode_focus:
                system_prompt += "\n\nDocument-mode focus: " + mode_focus

            structured_prompt = """
Review the artifact below from the assigned stakeholder perspective.

Return ONLY valid JSON. Do not use markdown fences.

Schema:
{
  "persona": "<persona name>",
  "risk_rating": "Low|Medium|High",
  "risk_justification": "<one sentence>",
  "concerns": [
    {
      "title": "<short concern>",
      "detail": "<specific evidence-based concern>",
      "recommended_change": "<specific change that should be made>"
    }
  ],
  "missing_requirements": ["<missing requirement>"],
  "questions_for_author": ["<question>"]
}

Do not invent facts that are not supported by the artifact.
Each concern must be actionable and suitable for remediation.
""".strip()

            raw, ai_name, banner = AIProvider.call(
                [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": structured_prompt + "\n\nARTIFACT:\n\n" + review_content[:30000]},
                ],
                max_tokens=5000,
            )
            parsed = parse_po_json(raw) or {
                "persona": persona["label"],
                "risk_rating": "Medium",
                "risk_justification": "The AI response could not be parsed into structured findings.",
                "concerns": [{
                    "title": "Review response",
                    "detail": raw[:2000],
                    "recommended_change": "Review the stakeholder critique manually.",
                }],
                "missing_requirements": [],
                "questions_for_author": [],
            }
            parsed["persona_key"] = key
            parsed["persona_label"] = persona["label"]
            parsed["ai_name"] = ai_name or ""
            parsed["banner"] = banner or ""
            all_reviews.append(parsed)
    except Exception as exc:
        return (
            html.Div(f"❌ Red-Team review failed: {str(exc)[:300]}", style={"color": "#c62828"}),
            None, [], [], True, True,
        )

    concerns = []
    for review in all_reviews:
        for concern in review.get("concerns", []) or []:
            if not isinstance(concern, dict):
                continue
            concerns.append({
                "index": len(concerns),
                "persona_key": review.get("persona_key", ""),
                "persona": review.get("persona_label", review.get("persona", "Stakeholder")),
                "risk_rating": review.get("risk_rating", "Medium"),
                "title": concern.get("title") or "Stakeholder concern",
                "detail": concern.get("detail") or "",
                "recommended_change": concern.get("recommended_change") or "",
            })

    review_data = {
        "artifact_id": artifact_id,
        "role": review_role,
        "mode": review_mode,
        "document_type": document_type,
        "selected_personas": [key for key, _ in persona_specs],
        "custom_stakeholder": (custom_stakeholder or "").strip(),
        "recommended_personas": get_document_redteam_personas(document_type),
        "reviews": all_reviews,
        "concerns": concerns,
    }

    _update_quality_current(artifact_id, redteam_data=review_data)

    result_rows = []
    if len(all_reviews) > 1:
        result_rows.append(html.Div([
            html.Strong(f"👥 {len(all_reviews)} stakeholder perspectives combined"),
            html.Div(
                f"Document type: {document_type}. All concerns below come from the selected perspectives.",
                style={"fontSize":"12px","color":"#666"},
            ),
        ], style={"padding":"10px","background":"#f1f3f4","borderRadius":"8px","marginBottom":"10px"}))

    for review in all_reviews:
        risk = review.get("risk_rating", "Medium")
        risk_color = {"Low":"#2e7d32","Medium":"#e65100","High":"#c62828"}.get(risk, "#666")
        section = [
            html.Div(review.get("persona_label", review.get("persona", "Stakeholder")),
                     style={"fontWeight":"700","fontSize":"14px","marginBottom":"4px"}),
            html.Div(f"Risk: {risk}", style={"fontWeight":"600","color":risk_color,"fontSize":"12px","marginBottom":"4px"}),
            html.Div(review.get("risk_justification", ""), style={"fontSize":"12px","color":"#666","marginBottom":"8px"}),
        ]
        for concern in review.get("concerns", []) or []:
            section.append(html.Div([
                html.Strong(concern.get("title", "Concern")),
                html.Div(concern.get("detail", ""), style={"fontSize":"12px","marginTop":"3px"}),
                html.Div("Recommended change: " + concern.get("recommended_change", ""),
                         style={"fontSize":"12px","color":"#555","marginTop":"4px"}),
            ], style={"padding":"8px","borderLeft":"3px solid #6a1b9a","background":"#fafafa","borderRadius":"5px","marginBottom":"6px"}))
        result_rows.append(html.Div(section, style={"marginBottom":"12px"}))

    options = [
        {
            "label": f"[{item['risk_rating']}] {item['persona']} — {item['title']}",
            "value": str(item["index"]),
        }
        for item in concerns
    ]

    return (
        html.Div(result_rows),
        review_data,
        options,
        [str(item["index"]) for item in concerns],
        not bool(concerns),
        not bool(concerns),
    )


@app.callback(
    [
        Output(
            "redteam-diff-modal",
            "is_open",
            allow_duplicate=True,
        ),
        Output(
            "redteam-diff-results",
            "children",
            allow_duplicate=True,
        ),
        Output(
            "redteam-remediation-request",
            "data",
        ),
    ],
    [
        Input(
            "redteam-apply-all-btn",
            "n_clicks",
        ),
        Input(
            "redteam-apply-selected-btn",
            "n_clicks",
        ),
    ],
    [
        State(
            "redteam-review-data",
            "data",
        ),
        State(
            "redteam-findings-checklist",
            "value",
        ),
        State(
            "current-workspace-artifact-id",
            "data",
        ),
    ],
    prevent_initial_call=True,
)
def request_redteam_remediation(
    apply_all_clicks,
    apply_selected_clicks,
    review_data,
    selected_values,
    artifact_id,
):
    """Open the remediation preview immediately and queue the AI work."""
    ctx = callback_context

    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    if not review_data:
        return (
            True,
            html.Div(
                "⚠️ Run a Red-Team review first.",
                style={"color": "#c62828"},
            ),
            None,
        )

    scanned_artifact_id = review_data.get(
        "artifact_id"
    )

    if (
        not scanned_artifact_id
        or scanned_artifact_id != artifact_id
    ):
        return (
            True,
            html.Div(
                "⚠️ The workspace changed after this Red-Team review. "
                "Run the review again on the current version.",
                style={"color": "#c62828"},
            ),
            None,
        )

    artifact = get_workspace_artifact(
        scanned_artifact_id
    )

    if not artifact:
        return (
            True,
            html.Div(
                "⚠️ The reviewed workspace version was not found.",
                style={"color": "#c62828"},
            ),
            None,
        )

    concerns = review_data.get(
        "concerns",
        [],
    )

    triggered_id = getattr(
        ctx,
        "triggered_id",
        None,
    )

    if triggered_id is None:
        prop_id = ctx.triggered[0]["prop_id"]
        triggered_id = prop_id.rsplit(
            ".",
            1,
        )[0]

    if triggered_id == "redteam-apply-all-btn":
        selected = concerns
    else:
        selected_indexes = {
            int(value)
            for value in (selected_values or [])
            if str(value).isdigit()
        }

        selected = [
            item
            for item in concerns
            if item.get("index") in selected_indexes
        ]

    if not selected:
        return (
            True,
            html.Div(
                "⚠️ Select at least one stakeholder concern.",
                style={"color": "#c62828"},
            ),
            None,
        )

    request = {
        "source_artifact_id": scanned_artifact_id,
        "chat_id": artifact.get("chat_id"),
        "role": artifact.get("role") or "ba",
        "mode": artifact.get("mode") or "general",
        "document_type": artifact.get("document_type") or "",
        "process_submode": (
            artifact.get("process_submode")
            or "text"
        ),
        "artifact_type": artifact.get("artifact_type") or "document",
        "selected_concerns": selected,
        "baseline_redteam_review": review_data,
    }

    loading_ui = html.Div(
        [
            html.Div(
                "✨ Preparing Red-Team remediation…",
                style={
                    "fontWeight": "700",
                    "marginBottom": "8px",
                },
            ),
            html.Div(
                (
                    f"{len(selected)} stakeholder concern(s) selected. "
                    "The original workspace will not be changed."
                ),
                style={
                    "fontSize": "12px",
                    "color": "#666",
                },
            ),
        ],
        style={
            "padding": "16px",
            "background": "#f3e5f5",
            "borderRadius": "8px",
        },
    )

    return (
        True,
        loading_ui,
        request,
    )


@app.callback(
    [
        Output(
            "redteam-diff-results",
            "children",
            allow_duplicate=True,
        ),
        Output(
            "redteam-pending-remediation",
            "data",
            allow_duplicate=True,
        ),
        Output(
            "redteam-commit-btn",
            "disabled",
            allow_duplicate=True,
        ),
    ],
    Input(
        "redteam-remediation-request",
        "data",
    ),
    prevent_initial_call=True,
)
def build_redteam_remediation_preview(request):
    """Generate the AI remediation after the preview modal is already open."""
    if not request:
        raise dash.exceptions.PreventUpdate

    artifact = get_workspace_artifact(
        request.get("source_artifact_id")
    )

    if not artifact:
        return (
            html.Div(
                "⚠️ The reviewed workspace version was not found.",
                style={"color": "#c62828"},
            ),
            None,
            True,
        )

    concerns = request.get(
        "selected_concerns",
        [],
    )

    if not concerns:
        return (
            html.Div(
                "⚠️ No stakeholder concerns were selected.",
                style={"color": "#c62828"},
            ),
            None,
            True,
        )

    try:
        preview = remediate_redteam_artifact(
            artifact,
            concerns,
        )
    except Exception as exc:
        return (
            html.Div(
                [
                    html.Div(
                        "❌ Red-Team remediation preview failed.",
                        style={
                            "fontWeight": "700",
                            "color": "#c62828",
                            "marginBottom": "6px",
                        },
                    ),
                    html.Div(
                        str(exc)[:1000],
                        style={
                            "fontSize": "12px",
                            "color": "#666",
                            "whiteSpace": "pre-wrap",
                        },
                    ),
                ]
            ),
            None,
            True,
        )

    pending = {
        "source_artifact_id": request.get(
            "source_artifact_id"
        ),
        "chat_id": request.get(
            "chat_id"
        ),
        "role": request.get(
            "role"
        ) or "ba",
        "mode": request.get(
            "mode"
        ) or "general",
        "document_type": request.get(
            "document_type"
        ) or "",
        "process_submode": request.get(
            "process_submode"
        ) or "text",
        "artifact_type": preview.get(
            "artifact_type"
        ) or "document",
        "original_content": artifact.get(
            "doc_content"
        ) or "",
        "updated_content": preview.get(
            "doc_content"
        ) or "",
        "updated_bpmn_xml": preview.get(
            "bpmn_xml"
        ) or "",
        "concerns": concerns,
        "baseline_regulatory_result": None,
        "baseline_redteam_review": request.get(
            "baseline_redteam_review"
        ),
        "ai_name": preview.get(
            "ai_name"
        ) or "",
        "banner": preview.get(
            "banner"
        ) or "",
    }

    preview_ui = html.Div(
        [
            html.Div(
                f"{len(concerns)} stakeholder concern(s) selected.",
                style={
                    "fontWeight": "700",
                    "marginBottom": "8px",
                },
            ),
            html.Div(
                (
                    "Nothing has been changed yet. "
                    "Apply Changes creates a new immutable workspace version."
                ),
                style={
                    "fontSize": "12px",
                    "color": "#666",
                    "marginBottom": "12px",
                },
            ),
            _render_remediation_preview(
                pending["original_content"],
                pending["updated_content"],
                pending["artifact_type"],
            ),
        ]
    )

    return (
        preview_ui,
        pending,
        False,
    )


@app.callback(
    Output(
        "redteam-diff-modal",
        "is_open",
        allow_duplicate=True,
    ),
    Input(
        "redteam-diff-close-btn",
        "n_clicks",
    ),
    prevent_initial_call=True,
)
def close_redteam_diff_modal(n):
    if not n:
        raise dash.exceptions.PreventUpdate

    return False


@app.callback(
    [
        Output(
            "redteam-diff-modal",
            "is_open",
            allow_duplicate=True,
        ),
        Output(
            "redteam-pending-remediation",
            "data",
            allow_duplicate=True,
        ),
        Output(
            "chat-messages",
            "children",
            allow_duplicate=True,
        ),
        Output(
            "workspace-content",
            "children",
            allow_duplicate=True,
        ),
        Output(
            "workspace-col",
            "style",
            allow_duplicate=True,
        ),
        Output(
            "chat-col",
            "style",
            allow_duplicate=True,
        ),
        Output(
            "doc-content",
            "data",
            allow_duplicate=True,
        ),
        Output(
            "doc-mode",
            "data",
            allow_duplicate=True,
        ),
        Output(
            "bpmn-xml-store",
            "data",
            allow_duplicate=True,
        ),
        Output(
            "workspace-open",
            "data",
            allow_duplicate=True,
        ),
        Output(
            "current-workspace-artifact-id",
            "data",
            allow_duplicate=True,
        ),
        Output(
            "ai-banner-store",
            "data",
            allow_duplicate=True,
        ),
        Output(
            "remediation-confirm-modal",
            "is_open",
            allow_duplicate=True,
        ),
        Output(
            "remediation-confirmation-content",
            "children",
            allow_duplicate=True,
        ),
        Output(
            "remediation-confirm-artifact-id",
            "data",
            allow_duplicate=True,
        ),
    ],
    Input(
        "redteam-commit-btn",
        "n_clicks",
    ),
    [
        State(
            "redteam-pending-remediation",
            "data",
        ),
        State(
            "chat-messages",
            "children",
        ),
        State(
            "current-chat-id",
            "data",
        ),
    ],
    prevent_initial_call=True,
)
def commit_redteam_remediation(
    n,
    pending,
    chat_children,
    current_chat_id,
):
    if not n or not pending:
        raise dash.exceptions.PreventUpdate

    source = get_workspace_artifact(
        pending.get("source_artifact_id")
    )

    if not source:
        return (
            False,
            None,
            dash.no_update,
            dash.no_update,
            dash.no_update,
            dash.no_update,
            dash.no_update,
            dash.no_update,
            dash.no_update,
            dash.no_update,
            dash.no_update,
            "⚠️ Original workspace version no longer exists.",
            False,
            dash.no_update,
            dash.no_update,
        )

    chat_id = (
        pending.get("chat_id")
        or current_chat_id
    )
    mode = pending.get("mode") or "general"
    role = pending.get("role") or "ba"

    summary = "\n".join(
        (
            f"- {item.get('persona', 'Stakeholder')}: "
            f"{item.get('title', 'Concern')}"
        )
        for item in pending.get(
            "concerns",
            [],
        )
    )

    message_content = (
        "✨ Red-Team remediation applied to workspace "
        f"version {source.get('version', '?')}.\n\n"
        "Stakeholder concerns addressed:\n"
        f"{summary}"
    )

    message_id = save_message(
        chat_id,
        "assistant",
        message_content,
        mode,
    )

    new_artifact_id = create_workspace_artifact(
        chat_id=chat_id,
        message_id=message_id,
        role=role,
        mode=mode,
        doc_content=pending.get(
            "updated_content",
            "",
        ),
        document_type=pending.get(
            "document_type",
            "",
        ),
        process_submode=pending.get(
            "process_submode",
            "text",
        ),
        bpmn_xml=pending.get(
            "updated_bpmn_xml",
            "",
        ),
    )

    new_artifact = get_workspace_artifact(
        new_artifact_id
    )

    if not new_artifact:
        raise dash.exceptions.PreventUpdate

    state = workspace_artifact_to_state(
        new_artifact
    )

    ws_children, bpmn_xml, chart = (
        render_workspace_artifact(
            new_artifact
        )
    )

    WorkspaceState.save(
        chat_id,
        doc_content=state["doc_content"],
        doc_mode=state["doc_mode"],
        document_type=(
            state["document_type"]
            or "BRD"
        ),
        process_submode=state["process_submode"],
        bpmn_xml=state["bpmn_xml"],
        workspace_open=True,
    )

    role_color_maps = {
        "ba": BA_COLORS,
        "po": PO_COLORS,
        "pm": PM_COLORS,
    }

    role_labels = {
        "ba": "📊 [BA]",
        "po": "🎯 [PO]",
        "pm": "🧭 [PM]",
    }

    color = role_color_maps.get(
        role,
        BA_COLORS,
    ).get(
        mode,
        "#1a73e8",
    )

    label = role_labels.get(
        role,
        "📊 [BA]",
    )

    display_mode = get_workspace_display_mode(
        role,
        mode,
        pending.get(
            "document_type",
            "",
        ),
    )

    bubble = html.Div(
        [
            html.Div(
                html.Span(
                    f"{label}·{display_mode}",
                    style={
                        "fontWeight": "600",
                        "color": color,
                        "fontSize": "12px",
                    },
                ),
                style={
                    "marginBottom": "8px",
                },
            ),
            html.Div(
                [
                    html.Span(
                        "🎭 Stakeholder remediation applied",
                        style={
                            "fontWeight": "600",
                        },
                    ),
                    html.Br(),
                    html.Span(
                        f"{len(pending.get('concerns', []))} concern(s) "
                        "incorporated into a new workspace version.",
                        style={
                            "fontSize": "12px",
                        },
                    ),
                ],
                style={
                    "background": "#f3e5f5",
                    "padding": "10px 14px",
                    "borderRadius": "16px 16px 16px 4px",
                    "borderLeft": "3px solid #6a1b9a",
                    "fontSize": "13px",
                },
            ),
            html.Div(
                [
                    dbc.Button(
                        "📖 Open Remediated Workspace",
                        id={
                            "type": "open-workspace-btn",
                            "index": new_artifact_id,
                        },
                        color="success",
                        size="sm",
                        n_clicks=0,
                        className="me-2",
                    ),
                    dbc.Button(
                        "🎭 Re-run Red-Team",
                        id={
                            "type": "rerun-redteam-btn",
                            "index": new_artifact_id,
                        },
                        color="dark",
                        size="sm",
                        outline=True,
                        n_clicks=0,
                    ),
                    dbc.Button(
                        "🛡️ Re-scan Regulatory Radar",
                        id={
                            "type": "rescan-regulatory-btn",
                            "index": new_artifact_id,
                        },
                        color="dark",
                        size="sm",
                        outline=True,
                        n_clicks=0,
                        className="ms-2",
                    ),
                    dbc.Button(
                        "📊 Compliance Dashboard",
                        id={
                            "type": "artifact-dashboard-btn",
                            "index": new_artifact_id,
                        },
                        color="info",
                        size="sm",
                        outline=True,
                        n_clicks=0,
                        className="ms-2",
                    ),
                ],
                style={"marginTop": "10px"},
            ),
        ],
        style={
            "textAlign": "left",
        },
        className="message-bubble",
    )

    if not isinstance(
        chat_children,
        list,
    ):
        chat_children = []

    maybe_refresh_chat_summary(
        chat_id,
        role,
        mode,
    )

    return (
        False,
        None,
        chat_children + [bubble],
        ws_children,
        WS_OPEN_STYLE,
        CHAT_HALF_STYLE,
        state["doc_content"],
        state["doc_mode"],
        bpmn_xml,
        True,
        new_artifact_id,
        pending.get(
            "banner",
            "",
        ),
        True,
        html.Div(
            [
                html.Div(
                    "Your Red-Team changes have been committed as a new immutable workspace version.",
                    style={"fontWeight": "600", "fontSize": "14px", "marginBottom": "10px"},
                ),
                html.Div(
                    [
                        html.Span(
                            f"{len(pending.get('concerns', []))} stakeholder concern(s) incorporated.",
                            style={"fontSize": "13px"},
                        ),
                        html.Br(),
                        html.Span(
                            f"Workspace version: {new_artifact.get('version', '?')}",
                            style={"fontSize": "12px", "color": "#666"},
                        ),
                    ],
                ),
                html.Div(
                    "The updated document is already loaded in the workspace. You can close this confirmation or open the exact updated version for review.",
                    style={
                        "fontSize": "12px",
                        "color": "#666",
                        "marginTop": "12px",
                        "padding": "10px 12px",
                        "background": "#f7f9fc",
                        "borderRadius": "8px",
                    },
                ),
            ]
        ),
        new_artifact_id,
    )


# ════════════════════════════════════════════════════════════════
# REGULATORY RADAR — SCAN, PREVIEW, SELECTIVE REMEDIATION
# ════════════════════════════════════════════════════════════════

def _regulatory_findings(result):
    """Return actionable regulatory findings with stable indexes."""
    findings = []
    for regulation in (result or {}).get("regulations", []):
        if not regulation.get("applicable"):
            continue
        for gap in regulation.get("gaps", []):
            if not gap:
                continue
            findings.append({
                "index": len(findings),
                "regulation": regulation.get("name", "Regulatory requirement"),
                "authority": regulation.get("authority", ""),
                "gap": gap,
            })
    return findings


def _regulatory_checklist_options(result):
    """Build checklist options for the current scan."""
    options = []
    for finding in _regulatory_findings(result):
        label = finding["regulation"]
        if finding["authority"]:
            label += f" ({finding['authority']})"
        label += f": {finding['gap']}"
        options.append({
            "label": label,
            "value": str(finding["index"]),
        })
    return options


def render_regulatory_result(result):
    """Render scan findings with individual selection controls."""
    if not result:
        return html.Div(
            "❌ Could not parse scan results — try again.",
            style={"color": "#c62828"},
        )

    risk = result.get("overall_risk", "Medium")
    risk_color = {
        "Low": "#2e7d32",
        "Medium": "#e65100",
        "High": "#c62828",
    }.get(risk, "#666")

    findings = _regulatory_findings(result)

    rows = [
        html.Div([
            html.Div(
                f"Domain: {result.get('detected_domain', '?')} · "
                f"Jurisdiction: {result.get('detected_jurisdiction', '?')}",
                style={"fontSize": "12px", "color": "#666", "marginBottom": "4px"},
            ),
            html.Span(
                f"Overall Risk: {risk}",
                style={"fontWeight": "700", "color": risk_color, "fontSize": "14px"},
            ),
            html.Div(
                result.get("risk_rationale", ""),
                style={"fontSize": "12px", "color": "#666"},
            ),
        ], style={"marginBottom": "14px", "paddingBottom": "10px",
                  "borderBottom": "1px solid #eee"}),
    ]

    if not findings:
        rows.append(html.Div(
            "✅ No actionable regulatory gaps were identified.",
            style={"padding": "12px", "background": "#e8f5e9",
                   "borderRadius": "8px", "color": "#2e7d32",
                   "fontWeight": "600"},
        ))
        return rows

    rows.append(html.Div(
        "Select the findings you want the AI to incorporate. "
        "Apply All Findings includes every actionable gap.",
        style={"fontSize": "12px", "color": "#666", "marginBottom": "8px"},
    ))

    options = []
    for finding in findings:
        label = finding["regulation"]
        if finding["authority"]:
            label += f" ({finding['authority']})"
        label += f": {finding['gap']}"
        options.append({
            "label": html.Div(
                label,
                style={"fontSize": "12px", "fontWeight": "600"},
            ),
            "value": str(finding["index"]),
        })

    return rows


@app.callback(
    [
        Output("regulatory-modal", "is_open"),
        Output("regulatory-scan-results", "children", allow_duplicate=True),
        Output("regulatory-scan-data", "data", allow_duplicate=True),
        Output("regulatory-apply-all-btn", "disabled", allow_duplicate=True),
        Output("regulatory-apply-selected-btn", "disabled", allow_duplicate=True),
        Output("regulatory-findings-checklist", "options", allow_duplicate=True),
        Output("regulatory-findings-checklist", "value", allow_duplicate=True),
    ],
    Input("open-regulatory-btn", "n_clicks"),
    [
        State("doc-content", "data"),
        State("current-workspace-artifact-id", "data"),
    ],
    prevent_initial_call=True,
)
def open_regulatory_modal(n, doc_content, artifact_id):
    """Open Regulatory Radar without performing a hidden duplicate scan."""
    if not n:
        raise dash.exceptions.PreventUpdate

    if not doc_content:
        return (
            True,
            html.Div(
                "⚠️ No workspace artifact is loaded.",
                style={"color": "#c62828"},
            ),
            None,
            True,
            True,
            [],
            [],
        )

    return (
        True,
        html.Div(
            [
                html.Div(
                    "🛡️ Regulatory Radar is ready.",
                    style={
                        "fontWeight": "700",
                        "marginBottom": "5px",
                    },
                ),
                html.Div(
                    "Click Run Scan to analyze the currently open "
                    "workspace version. The scan is tied to that exact "
                    "artifact version.",
                    style={
                        "fontSize": "12px",
                        "color": "#666",
                    },
                ),
            ],
            style={
                "padding": "12px",
                "background": "#f7f9fc",
                "borderRadius": "8px",
            },
        ),
        None,
        True,
        True,
        [],
        [],
    )


@app.callback(
    [
        Output("regulatory-scan-results", "children", allow_duplicate=False),
        Output("regulatory-scan-data", "data", allow_duplicate=True),
        Output("regulatory-apply-all-btn", "disabled", allow_duplicate=True),
        Output("regulatory-apply-selected-btn", "disabled", allow_duplicate=True),
        Output("regulatory-findings-checklist", "options", allow_duplicate=True),
        Output("regulatory-findings-checklist", "value", allow_duplicate=True),
    ],
    Input("regulatory-run-btn", "n_clicks"),
    [
        State("doc-content", "data"),
        State("current-workspace-artifact-id", "data"),
    ],
    prevent_initial_call=True,
)
def run_regulatory_scan_cb(n, doc_content, artifact_id):
    if not n:
        raise dash.exceptions.PreventUpdate
    if not doc_content:
        return (
            html.Div("⚠️ No workspace artifact is loaded.",
                     style={"color": "#c62828"}),
            None,
            True,
            True,
            [],
            [],
        )

    try:
        result = run_regulatory_scan(doc_content)
    except Exception as exc:
        return (
            html.Div(
                f"❌ Regulatory scan failed: {str(exc)[:300]}",
                style={"color": "#c62828"},
            ),
            None,
            True,
            True,
            [],
            [],
        )

    findings = _regulatory_findings(result)
    scan_payload = {
        "artifact_id": artifact_id,
        "result": result,
    }

    return (
        render_regulatory_result(result),
        scan_payload,
        not bool(findings),
        not bool(findings),
        _regulatory_checklist_options(result),
        [str(item["index"]) for item in findings],
    )


@app.callback(
    Output("regulatory-modal", "is_open", allow_duplicate=True),
    Input("regulatory-close-btn", "n_clicks"),
    prevent_initial_call=True,
)
def close_regulatory_modal(n):
    if not n:
        raise dash.exceptions.PreventUpdate
    return False


def _artifact_remediation_prompt(artifact, findings):
    """Build an artifact-aware remediation prompt."""
    artifact_type = artifact.get("artifact_type") or "document"
    role = artifact.get("role") or "ba"
    mode = artifact.get("mode") or "general"
    document_type = artifact.get("document_type") or ""

    findings_text = "\n".join(
        f"{idx + 1}. {item['regulation']}: {item['gap']}"
        for idx, item in enumerate(findings)
    )
    source = artifact.get("doc_content") or ""

    if artifact_type == "bpmn":
        output_instruction = """
Return ONLY valid JSON representing the complete corrected BPMN model.
Preserve the existing process and add only the regulatory controls required
by the selected findings. Return the complete model, not a patch.
"""
    elif artifact_type == "cards":
        output_instruction = """
Return ONLY valid JSON in the application's existing cards format.
Preserve existing IDs/content where possible. Add or update stories, epics,
acceptance criteria, controls, dependencies, or tasks only where needed.
Return the complete structured artifact, not a patch.
"""
    else:
        output_instruction = """
Return ONLY the complete revised artifact in Markdown.
Preserve the existing structure, identifiers, terminology, and valid content.
Add missing controls to the most appropriate sections. Do not remove valid
requirements just to shorten the document.
"""

    return f"""
You are a senior Business Analyst and regulatory remediation specialist.

Artifact role: {role}
Application mode: {mode}
Document type: {document_type or artifact_type}

ORIGINAL ARTIFACT:
{source[:30000]}

SELECTED REGULATORY FINDINGS:
{findings_text}

TASK:
Refactor this exact artifact to address ONLY the selected findings.
Preserve valid existing content. Make the remediation concrete, testable,
traceable, and appropriate to the artifact type.
Do not claim legal approval or guaranteed compliance.
Do not fabricate regulatory clauses or citations.
{output_instruction}
""".strip()


def _validate_remediated_document_integrity(original, updated):
    """Reject obviously truncated Markdown before it can be committed."""
    original = original or ""
    updated = updated or ""

    if not original.strip() or not updated.strip():
        raise ValueError("The remediation returned an empty document.")

    original_lines = [
        line.strip()
        for line in original.splitlines()
        if line.strip()
    ]
    updated_lines = [
        line.strip()
        for line in updated.splitlines()
        if line.strip()
    ]

    if len(original) >= 5000:
        minimum_length = int(len(original) * 0.70)
        if len(updated) < minimum_length:
            raise ValueError(
                "The AI remediation appears truncated: the revised document "
                f"is {len(updated):,} characters versus "
                f"{len(original):,} before remediation."
            )

    original_headings = sum(
        1 for line in original_lines if re.match(r"^#{1,6}\s+", line)
    )
    updated_headings = sum(
        1 for line in updated_lines if re.match(r"^#{1,6}\s+", line)
    )

    if original_headings >= 4 and updated_headings < max(
        1,
        int(original_headings * 0.70),
    ):
        raise ValueError(
            "The AI remediation appears structurally incomplete: too many "
            "document headings were lost."
        )
# ════════════════════════════════════════════════════════════════
# CHUNKED DOCUMENT REMEDIATION
# ════════════════════════════════════════════════════════════════
def _split_markdown_into_chunks(text, max_chars=9000):
    """Split Markdown at #/## headings (fence-aware) into ~max_chars chunks."""
    sections, current, in_fence = [], [], False
    for line in (text or "").splitlines(keepends=True):
        if line.lstrip().startswith("```"):
            in_fence = not in_fence
        if not in_fence and current and re.match(r"^#{1,2}\s+\S", line):
            sections.append("".join(current))
            current = []
        current.append(line)
    if current:
        sections.append("".join(current))

    chunks, buf = [], ""
    for section in sections:
        if buf and len(buf) + len(section) > max_chars:
            chunks.append(buf)
            buf = ""
        buf += section
    if buf:
        chunks.append(buf)
    return chunks or [text or ""]


def _strip_code_fence(text):
    value = (text or "").strip()
    value = re.sub(r"^```(?:markdown|md|json)?\s*\n?", "", value, flags=re.IGNORECASE)
    value = re.sub(r"\n?```\s*$", "", value)
    return value.strip()


_ADDRESSED_RE = re.compile(r"<!--\s*ADDRESSED:\s*([0-9,\s]*)-->", re.IGNORECASE)


def _remediate_document_in_chunks(artifact, findings):
    """Remediate a large document section-by-section so nothing is truncated."""
    source = artifact.get("doc_content") or ""
    if not source.strip():
        raise ValueError("The source artifact has no content to remediate.")

    numbered = "\n".join(
        f"{i + 1}. {f['regulation']}: {f['gap']}" for i, f in enumerate(findings)
    )
    state = {"ai_name": "", "banner": ""}

    def ask(system, user, max_tokens):
        last_exc = None
        for _ in range(2):  # one retry
            try:
                raw, ai_name, banner = AIProvider.call(
                    [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
                    max_tokens=max_tokens,
                )
                state["ai_name"] = ai_name or state["ai_name"]
                state["banner"] = state["banner"] or (banner or "")
                return raw
            except Exception as exc:
                last_exc = exc
        raise last_exc

    # ── Structured JSON documents (e.g. Scope Management) ──────────
    if parse_po_json(source) is not None:
        raw = ask(
            "You are an expert artifact remediation engine. Return only valid JSON.",
            f"Revise this JSON artifact to address ONLY the findings below. "
            f"Preserve the schema, all existing keys, IDs and valid content. "
            f"Return the COMPLETE JSON object, not a patch.\n\n"
            f"FINDINGS:\n{numbered}\n\nARTIFACT:\n{source}",
            20000,
        )
        cleaned = _strip_code_fence(raw)
        if parse_po_json(cleaned) is None:
            raise ValueError("The AI returned invalid JSON for the remediated artifact.")
        return {"doc_content": cleaned, "bpmn_xml": "", "artifact_type": "document",
                "ai_name": state["ai_name"], "banner": state["banner"]}

    # ── Markdown documents: section-by-section ─────────────────────
    system = (
        "You are a senior Business Analyst performing artifact remediation. "
        "You revise one section group of a larger document at a time."
    )
    chunks = _split_markdown_into_chunks(source)
    revised, addressed = [], set()

    for idx, chunk in enumerate(chunks, start=1):
        prompt = f"""
You are revising part {idx} of {len(chunks)} of a larger document.

FINDINGS TO ADDRESS (numbered):
{numbered}

RULES:
- Apply ONLY the findings that naturally belong in this part. Add concrete,
  testable, traceable controls/requirements where they fit.
- Preserve all existing headings, IDs, terminology, tables and valid content.
  Do not shorten, summarise or drop anything.
- Do not invent regulatory clause numbers, dates, owners or commitments.
- If no finding belongs in this part, return it EXACTLY unchanged.
- Return ONLY the complete revised Markdown for this part, no commentary.
- On the very last line output: <!-- ADDRESSED: n,n --> listing the finding
  numbers you incorporated here (or <!-- ADDRESSED: --> if none).

DOCUMENT PART:
{chunk}
""".strip()

        result = None
        for _ in range(2):
            out = _strip_code_fence(ask(system, prompt, 12000))
            match = _ADDRESSED_RE.search(out)
            ids = set()
            if match:
                ids = {int(x) for x in re.findall(r"\d+", match.group(1))}
                out = _ADDRESSED_RE.sub("", out).strip()
            # Reject truncated/degenerate chunk output
            if len(out) >= int(len(chunk) * 0.85):
                result, addressed = out, addressed | ids
                break
        revised.append((result if result is not None else chunk.strip()))

    body = "\n\n".join(part.strip("\n") for part in revised).strip() + "\n"

    # ── Findings no section absorbed → append new sections ─────────
    missing = [f for i, f in enumerate(findings, start=1) if i not in addressed]
    if missing:
        missing_text = "\n".join(f"- {f['regulation']}: {f['gap']}" for f in missing)
        extra = _strip_code_fence(ask(
            system,
            "These findings were not incorporated into any existing section. "
            "Write NEW Markdown sections (each starting with '## ') that address "
            "them with concrete, testable requirements/controls. Return only the "
            "new sections, not the existing document. Do not fabricate citations.\n\n"
            f"FINDINGS:\n{missing_text}\n\nDOCUMENT CONTEXT (start):\n{source[:4000]}",
            4000,
        ))
        if extra:
            body = body.rstrip() + "\n\n" + extra.strip() + "\n"

    return {"doc_content": body, "bpmn_xml": "", "artifact_type": "document",
            "ai_name": state["ai_name"], "banner": state["banner"]}


def remediate_redteam_artifact(artifact, concerns):
    """Red-Team remediation: reuse the same engine with concerns as findings."""
    if not artifact or not concerns:
        raise ValueError("No workspace artifact or stakeholder concerns selected.")

    findings = [{
        "index": i,
        "regulation": f"{c.get('persona', 'Stakeholder')} — {c.get('title', 'Concern')}",
        "authority": "",
        "gap": (
            f"{c.get('detail', '')} "
            f"Recommended change: {c.get('recommended_change', '')}"
        ).strip(),
    } for i, c in enumerate(concerns)]

    return remediate_workspace_artifact(artifact, findings)
    

def remediate_workspace_artifact(artifact, findings):
    """Generate a non-destructive remediation preview."""
    if not artifact or not findings:
        raise ValueError("No workspace artifact or regulatory findings selected.")

    prompt = _artifact_remediation_prompt(artifact, findings)

    raw, ai_name, banner = AIProvider.call(
        [
            {
                "role": "system",
                "content": (
                    "You are an expert artifact remediation engine. "
                    "Follow the requested output format exactly."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        max_tokens=12000,
    )

    artifact_type = artifact.get("artifact_type") or "document"

    if artifact_type == "bpmn":
        parsed = parse_po_json(raw)
        if not parsed:
            raise ValueError("AI returned invalid BPMN JSON.")
        bpmn_xml = build_bpmn_xml(parsed)
        if not bpmn_xml:
            raise ValueError("The corrected BPMN could not be converted to XML.")
        return {
            "doc_content": raw,
            "bpmn_xml": bpmn_xml,
            "artifact_type": "bpmn",
            "ai_name": ai_name,
            "banner": banner or "",
        }

    if artifact_type == "cards":
        parsed = parse_po_json(raw)
        if not parsed:
            raise ValueError("AI returned invalid cards JSON.")
        return {
            "doc_content": raw,
            "bpmn_xml": "",
            "artifact_type": "cards",
            "ai_name": ai_name,
            "banner": banner or "",
        }

    preview = _remediate_document_in_chunks(
        artifact,
        findings,
    )

    if preview.get("artifact_type") == "document":
        _validate_remediated_document_integrity(
            artifact.get("doc_content") or "",
            preview.get("doc_content") or "",
        )

    return preview


def _build_unified_diff(original, updated, artifact_type):
    """Build the complete textual diff for the remediation preview."""
    if artifact_type in {"bpmn", "cards"}:
        original_obj = parse_po_json(original) or {}
        updated_obj = parse_po_json(updated) or {}
        before = json.dumps(
            original_obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ).splitlines()
        after = json.dumps(
            updated_obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        ).splitlines()
    else:
        before = (original or "").splitlines()
        after = (updated or "").splitlines()

    diff = list(
        difflib.unified_diff(
            before,
            after,
            fromfile="Original",
            tofile="Remediated",
            lineterm="",
            n=2,
        )
    )

    return (
        "\n".join(diff)
        if diff
        else "No textual changes were detected."
    )


def _render_remediation_preview(original, updated, artifact_type):
    """Render complete Before/After artifacts without character truncation."""
    original = original or ""
    updated = updated or ""
    diff_text = _build_unified_diff(
        original,
        updated,
        artifact_type,
    )

    if artifact_type in {"bpmn", "cards"}:
        before_obj = parse_po_json(original) or {}
        after_obj = parse_po_json(updated) or {}
        before_text = json.dumps(
            before_obj,
            indent=2,
            ensure_ascii=False,
        )
        after_text = json.dumps(
            after_obj,
            indent=2,
            ensure_ascii=False,
        )

        content = dbc.Row([
            dbc.Col([
                html.H6("Before", className="fw-bold"),
                dcc.Markdown(
                    "```json\n" + before_text + "\n```",
                    style={
                        "fontSize": "11px",
                        "maxHeight": "600px",
                        "overflowY": "auto",
                        "border": "1px solid #eee",
                        "borderRadius": "8px",
                        "padding": "12px",
                    },
                ),
            ], md=6),
            dbc.Col([
                html.H6("After", className="fw-bold"),
                dcc.Markdown(
                    "```json\n" + after_text + "\n```",
                    style={
                        "fontSize": "11px",
                        "maxHeight": "600px",
                        "overflowY": "auto",
                        "border": "1px solid #c8e6c9",
                        "borderRadius": "8px",
                        "padding": "12px",
                        "background": "#f8fff8",
                    },
                ),
            ], md=6),
        ])
    else:
        content = dbc.Row([
            dbc.Col([
                html.H6("Before", className="fw-bold"),
                dcc.Markdown(
                    original,
                    className="markdown-body",
                    style={
                        "fontSize": "12px",
                        "border": "1px solid #eee",
                        "borderRadius": "8px",
                        "padding": "12px",
                        "maxHeight": "600px",
                        "overflowY": "auto",
                    },
                ),
            ], md=6),
            dbc.Col([
                html.H6("After", className="fw-bold"),
                dcc.Markdown(
                    updated,
                    className="markdown-body",
                    style={
                        "fontSize": "12px",
                        "border": "1px solid #c8e6c9",
                        "borderRadius": "8px",
                        "padding": "12px",
                        "maxHeight": "600px",
                        "overflowY": "auto",
                        "background": "#f8fff8",
                    },
                ),
            ], md=6),
        ])

    return html.Div([
        html.Div(
            [
                html.Strong("Change Diff"),
                html.Span(
                    f" · Before: {len(original):,} chars"
                    f" · After: {len(updated):,} chars",
                    style={
                        "fontSize": "11px",
                        "color": "#666",
                        "fontWeight": "400",
                    },
                ),
            ],
            style={"marginBottom": "8px"},
        ),
        html.Pre(
            diff_text,
            style={
                "fontSize": "11px",
                "whiteSpace": "pre-wrap",
                "maxHeight": "420px",
                "overflowY": "auto",
                "background": "#f7f7f7",
                "border": "1px solid #e5e5e5",
                "borderRadius": "8px",
                "padding": "12px",
                "marginBottom": "16px",
            },
        ),
        content,
    ])


@app.callback(
    [
        Output("regulatory-diff-modal", "is_open"),
        Output("regulatory-diff-results", "children"),
        Output("regulatory-pending-remediation", "data"),
        Output("regulatory-commit-btn", "disabled"),
    ],
    [
        Input("regulatory-apply-all-btn", "n_clicks"),
        Input("regulatory-apply-selected-btn", "n_clicks"),
    ],
    [
        State("regulatory-scan-data", "data"),
        State("regulatory-findings-checklist", "value"),
        State("current-workspace-artifact-id", "data"),
    ],
    prevent_initial_call=True,
)
def preview_regulatory_remediation(
    apply_all_clicks,
    apply_selected_clicks,
    scan_data,
    selected_values,
    artifact_id,
):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    if not scan_data:
        return (
            True,
            html.Div("⚠️ No regulatory scan is available.",
                     style={"color": "#c62828"}),
            None,
            True,
        )

    scanned_artifact_id = scan_data.get("artifact_id")
    scan_result = scan_data.get("result") or {}

    if not scanned_artifact_id or scanned_artifact_id != artifact_id:
        return (
            True,
            html.Div(
                "⚠️ The workspace changed after this scan. Run Regulatory Radar again on the current version.",
                style={"color": "#c62828"},
            ),
            None,
            True,
        )

    artifact = get_workspace_artifact(scanned_artifact_id)
    if not artifact:
        return (
            True,
            html.Div("⚠️ Scanned workspace version was not found.",
                     style={"color": "#c62828"}),
            None,
            True,
        )

    findings = _regulatory_findings(scan_result)
    trigger = ctx.triggered[0]["prop_id"].split(".")[0]

    if trigger == "regulatory-apply-all-btn":
        selected = findings
    else:
        selected_indexes = {
            int(value) for value in (selected_values or [])
            if str(value).isdigit()
        }
        selected = [
            item for item in findings
            if item["index"] in selected_indexes
        ]

    if not selected:
        return (
            True,
            html.Div("⚠️ Select at least one regulatory finding.",
                     style={"color": "#c62828"}),
            None,
            True,
        )

    try:
        preview = remediate_workspace_artifact(
            artifact,
            selected,
        )
    except Exception as exc:
        return (
            True,
            html.Div(
                f"❌ Remediation preview failed: {str(exc)[:400]}",
                style={"color": "#c62828"},
            ),
            None,
            True,
        )

    pending = {
        "source_artifact_id": artifact_id,
        "chat_id": artifact["chat_id"],
        "role": artifact["role"],
        "mode": artifact["mode"],
        "document_type": artifact.get("document_type") or "",
        "process_submode": artifact.get("process_submode") or "text",
        "artifact_type": preview["artifact_type"],
        "original_content": artifact.get("doc_content") or "",
        "updated_content": preview["doc_content"],
        "updated_bpmn_xml": preview.get("bpmn_xml") or "",
        "findings": selected,
        "baseline_regulatory_result": scan_result,
        "baseline_redteam_review": None,
        "ai_name": preview.get("ai_name") or "",
        "banner": preview.get("banner") or "",
    }

    preview_ui = html.Div([
        html.Div(
            f"{len(selected)} regulatory finding(s) selected.",
            style={"fontWeight": "700", "marginBottom": "8px"},
        ),
        html.Div(
            "Nothing has been changed yet. Apply Changes creates a new "
            "immutable workspace version.",
            style={"fontSize": "12px", "color": "#666", "marginBottom": "12px"},
        ),
        _render_remediation_preview(
            pending["original_content"],
            pending["updated_content"],
            pending["artifact_type"],
        ),
    ])

    return True, preview_ui, pending, False


@app.callback(
    Output("regulatory-diff-modal", "is_open", allow_duplicate=True),
    Input("regulatory-diff-close-btn", "n_clicks"),
    prevent_initial_call=True,
)
def close_regulatory_diff_modal(n):
    if not n:
        raise dash.exceptions.PreventUpdate
    return False


@app.callback(
    [
        Output("regulatory-diff-modal", "is_open", allow_duplicate=True),
        Output("regulatory-pending-remediation", "data", allow_duplicate=True),
        Output("chat-messages", "children", allow_duplicate=True),
        Output("workspace-content", "children", allow_duplicate=True),
        Output("workspace-col", "style", allow_duplicate=True),
        Output("chat-col", "style", allow_duplicate=True),
        Output("doc-content", "data", allow_duplicate=True),
        Output("doc-mode", "data", allow_duplicate=True),
        Output("bpmn-xml-store", "data", allow_duplicate=True),
        Output("workspace-open", "data", allow_duplicate=True),
        Output("current-workspace-artifact-id", "data", allow_duplicate=True),
        Output("ai-banner-store", "data", allow_duplicate=True),
        Output("remediation-confirm-modal", "is_open", allow_duplicate=True),
        Output("remediation-confirmation-content", "children", allow_duplicate=True),
        Output("remediation-confirm-artifact-id", "data", allow_duplicate=True),
    ],
    Input("regulatory-commit-btn", "n_clicks"),
    [
        State("regulatory-pending-remediation", "data"),
        State("chat-messages", "children"),
        State("current-chat-id", "data"),
    ],
    prevent_initial_call=True,
)
def commit_regulatory_remediation(n, pending, chat_children, current_chat_id):
    if not n or not pending:
        raise dash.exceptions.PreventUpdate

    source = get_workspace_artifact(
        pending.get("source_artifact_id")
    )
    if not source:
        return (
            False, None, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, "⚠️ Original workspace version no longer exists.",
            False, dash.no_update, dash.no_update,
        )

    chat_id = pending.get("chat_id") or current_chat_id
    mode = pending.get("mode") or "general"
    role = pending.get("role") or "ba"

    summary = "\n".join(
        f"- {item['regulation']}: {item['gap']}"
        for item in pending.get("findings", [])
    )

    message_content = (
        f"✨ Regulatory remediation applied to workspace version "
        f"{source.get('version', '?')}.\n\n"
        f"Findings addressed:\n{summary}"
    )

    message_id = save_message(
        chat_id,
        "assistant",
        message_content,
        mode,
    )

    new_artifact_id = create_workspace_artifact(
        chat_id=chat_id,
        message_id=message_id,
        role=role,
        mode=mode,
        doc_content=pending.get("updated_content") or "",
        document_type=pending.get("document_type") or "",
        process_submode=pending.get("process_submode") or "text",
        bpmn_xml=pending.get("updated_bpmn_xml") or "",
    )

    new_artifact = get_workspace_artifact(
        new_artifact_id
    )
    if not new_artifact:
        raise dash.exceptions.PreventUpdate

    verification_error = ""
    quality_snapshot = None
    try:
        quality_snapshot = _verify_artifact_after_remediation(
            source,
            new_artifact,
            baseline_regulatory=(
                {
                    "artifact_id": source.get("id"),
                    "result": pending.get("baseline_regulatory_result"),
                }
                if pending.get("baseline_regulatory_result")
                else None
            ),
            baseline_redteam=pending.get("baseline_redteam_review"),
        )
    except Exception as exc:
        verification_error = str(exc)[:300]

    state = workspace_artifact_to_state(new_artifact)
    ws_children, bpmn_xml, chart = render_workspace_artifact(new_artifact)

    WorkspaceState.save(
        chat_id,
        doc_content=state["doc_content"],
        doc_mode=state["doc_mode"],
        document_type=state["document_type"] or "BRD",
        process_submode=state["process_submode"],
        bpmn_xml=state["bpmn_xml"],
        workspace_open=True,
    )

    role_color_maps = {
        "ba": BA_COLORS,
        "po": PO_COLORS,
        "pm": PM_COLORS,
    }
    role_labels = {
        "ba": "📊 [BA]",
        "po": "🎯 [PO]",
        "pm": "🧭 [PM]",
    }

    color = role_color_maps.get(role, BA_COLORS).get(
        mode,
        "#1a73e8",
    )
    label = role_labels.get(role, "📊 [BA]")
    display_mode = get_workspace_display_mode(
        role,
        mode,
        pending.get("document_type") or "",
    )

    bubble = html.Div([
        html.Div(
            html.Span(
                f"{label}·{display_mode}",
                style={"fontWeight": "600", "color": color, "fontSize": "12px"},
            ),
            style={"marginBottom": "8px"},
        ),
        html.Div(
            [
                html.Span("✨ Regulatory remediation applied",
                          style={"fontWeight": "600"}),
                html.Br(),
                html.Span(
                    f"{len(pending.get('findings', []))} finding(s) "
                    "incorporated into a new workspace version."
                    + (
                        f" Verified remediation score: "
                        f"{quality_snapshot['remediation_score']:.1f}%."
                        if quality_snapshot
                        else (
                            f" Automatic verification could not complete: "
                            f"{verification_error}"
                            if verification_error
                            else ""
                        )
                    ),
                    style={"fontSize": "12px"},
                ),
            ],
            style={
                "background": "#e8f5e9",
                "padding": "10px 14px",
                "borderRadius": "16px 16px 16px 4px",
                "borderLeft": "3px solid #2e7d32",
                "fontSize": "13px",
            },
        ),
        html.Div(
            [
                dbc.Button(
                    "📖 Open Updated Workspace",
                    id={
                        "type": "open-workspace-btn",
                        "index": new_artifact_id,
                    },
                    color="success",
                    size="sm",
                    n_clicks=0,
                    className="me-2",
                ),
                dbc.Button(
                    "🛡️ Re-scan Regulatory Radar",
                    id={
                        "type": "rescan-regulatory-btn",
                        "index": new_artifact_id,
                    },
                    color="dark",
                    size="sm",
                    outline=True,
                    n_clicks=0,
                ),
                    dbc.Button(
                        "🎭 Re-run Red-Team",
                        id={
                            "type": "rerun-redteam-btn",
                            "index": new_artifact_id,
                        },
                        color="dark",
                        size="sm",
                        outline=True,
                        n_clicks=0,
                        className="ms-2",
                    ),
                    dbc.Button(
                        "📊 Compliance Dashboard",
                        id={
                            "type": "artifact-dashboard-btn",
                            "index": new_artifact_id,
                        },
                        color="info",
                        size="sm",
                        outline=True,
                        n_clicks=0,
                        className="ms-2",
                    ),
            ],
            style={"marginTop": "10px"},
        ),
    ], style={"textAlign": "left"}, className="message-bubble")

    if not isinstance(chat_children, list):
        chat_children = []

    maybe_refresh_chat_summary(chat_id, role, mode)

    return (
        False,
        None,
        chat_children + [bubble],
        ws_children,
        WS_OPEN_STYLE,
        CHAT_HALF_STYLE,
        state["doc_content"],
        state["doc_mode"],
        bpmn_xml,
        True,
        new_artifact_id,
        pending.get("banner") or "",
        True,
        html.Div(
            [
                html.Div(
                    "Your Regulatory Radar changes have been committed as a new immutable workspace version.",
                    style={"fontWeight": "600", "fontSize": "14px", "marginBottom": "10px"},
                ),
                html.Div(
                    [
                        html.Span(
                            f"{len(pending.get('findings', []))} regulatory finding(s) incorporated.",
                            style={"fontSize": "13px"},
                        ),
                        html.Br(),
                        html.Span(
                            f"Workspace version: {new_artifact.get('version', '?')}",
                            style={"fontSize": "12px", "color": "#666"},
                        ),
                    ],
                ),
                (
                    html.Div(
                        f"Automatic verification score: {quality_snapshot['remediation_score']:.1f}%.",
                        style={
                            "fontSize": "12px",
                            "color": "#2e7d32",
                            "marginTop": "10px",
                            "fontWeight": "600",
                        },
                    )
                    if quality_snapshot
                    else html.Div(
                        "The document was committed, but automatic verification could not complete.",
                        style={
                            "fontSize": "12px",
                            "color": "#8a6d1d",
                            "marginTop": "10px",
                        },
                    )
                ),
                html.Div(
                    "The updated document is already loaded in the workspace. You can close this confirmation or open the exact updated version for review.",
                    style={
                        "fontSize": "12px",
                        "color": "#666",
                        "marginTop": "12px",
                        "padding": "10px 12px",
                        "background": "#f7f9fc",
                        "borderRadius": "8px",
                    },
                ),
            ]
        ),
        new_artifact_id,
    )


# ── Process submode toggle (buttons + store restore) ──────────────
@app.callback(
    [Output("process-submode","data",    allow_duplicate=True),
     Output("process-text-btn","color"), Output("process-text-btn","outline"),
     Output("process-bpmn-btn","color"), Output("process-bpmn-btn","outline")],
    [Input("process-text-btn","n_clicks"),
     Input("process-bpmn-btn","n_clicks"),
     Input("process-submode","data")],
    prevent_initial_call=True)
def sync_process_submode(text_n, bpmn_n, stored):
    ctx = callback_context
    if not ctx.triggered: raise dash.exceptions.PreventUpdate
    trigger = ctx.triggered[0]["prop_id"].split(".")[0]
    if trigger == "process-bpmn-btn": new = "bpmn"
    elif trigger == "process-text-btn": new = "text"
    else: new = stored or "text"
    if new == "bpmn": return new,"light",True,"primary",False
    return new,"primary",False,"light",True

@app.callback(Output("meeting-modal","is_open"),
    Input("open-meeting-btn","n_clicks"), prevent_initial_call=True)
def open_meeting_modal(n): return True

@app.callback(Output("meeting-modal","is_open",allow_duplicate=True),
    Input("meeting-close-btn","n_clicks"), prevent_initial_call=True)
def close_meeting_modal(n): return False

@app.callback(
    [Output("meeting-file-text","data"), Output("meeting-file-badge","children")],
    Input("meeting-file-upload","contents"),
    State("meeting-file-upload","filename"), prevent_initial_call=True)
def handle_meeting_file(contents, filename):
    if not contents or not filename: raise dash.exceptions.PreventUpdate
    text, badge_label = extract_file_text(filename, contents)
    return text, f"📎 {badge_label}"

@app.callback(
    [Output("meeting-pipeline-results","children"),
     Output("meeting-stories-payload","data")],
    Input("meeting-run-btn","n_clicks"),
    [State("meeting-transcript-input","value"), State("meeting-file-text","data")],
    prevent_initial_call=True)
def run_meeting_pipeline_cb(n, pasted_text, file_text):
    if not n: raise dash.exceptions.PreventUpdate
    transcript = (pasted_text or "").strip() or (file_text or "").strip()
    if not transcript:
        return html.Div("⚠️ Paste a transcript or attach a file first.", style={"color":"#c62828"}), None

    result = run_meeting_pipeline(transcript)
    if result.get("error"):
        return html.Div(f"❌ {result['error']}", style={"color":"#c62828"}), None

    ex = result["extracted"]
    rows = [
        html.Div(f"📌 {ex.get('meeting_title','Meeting')}", style={"fontWeight":"700","marginBottom":"6px"}),
        html.Div(ex.get("requirements_summary",""), style={"fontSize":"12px","color":"#555","marginBottom":"10px"}),
    ]
    stories_payload = None
    for a in result["artifacts"]:
        if a["status"] == "created":
            rows.append(html.Div([
                html.Span("✅ "),
                html.Span(f"{a['type']} generated — "),
                html.Span("open it from 📜 History to view/push it.",
                           style={"color":"#888","fontSize":"11px"})
            ], style={"marginBottom":"4px"}))
            if a["type"] == "Stories" and a.get("cards"):
                stories_payload = {"chat_id": a["chat_id"], "cards": a["cards"]}
        else:
            rows.append(html.Div(f"⚠️ {a['type']} — {a.get('error','failed')}",
                        style={"color":"#c62828","marginBottom":"4px"}))

    if ex.get("open_questions"):
        rows.append(html.Div("Open questions surfaced:", style={"fontWeight":"600","marginTop":"10px","fontSize":"12px"}))
        rows += [html.Div(f"▸ {q}", style={"fontSize":"12px","color":"#666"}) for q in ex["open_questions"]]

    if stories_payload:
        rows.append(html.Div(
            dbc.Button("🚀 Push Stories to JIRA",
                       id={"type":"meeting-push-stories-btn","index":str(uuid.uuid4())},
                       color="primary", size="sm", n_clicks=0),
            style={"marginTop":"10px"}))

    return rows, stories_payload
    
# ── Role radio → modal ────────────────────────────────────────────
@app.callback(
    [Output("switch-modal","is_open"), Output("pending-role","data"), Output("role-radio","value")],
    Input("role-radio","value"), State("current-role","data"), prevent_initial_call=True)
def request_switch(selected, current):
    if selected==current: raise dash.exceptions.PreventUpdate
    return True, selected, current

# ── Confirm YES ───────────────────────────────────────────────────
@app.callback(
    [
        Output("current-role","data"),
        Output("current-mode","data"),
        Output("chat-messages","children"),
        Output("ba-mode-bar","style"),
        Output("po-mode-bar","style"),
        Output("pm-mode-bar","style"),
        Output("workspace-col","style"),
        Output("chat-col","style"),
        Output("switch-modal","is_open",allow_duplicate=True),
        Output("role-radio","value",allow_duplicate=True),
        Output("current-workspace-artifact-id","data",allow_duplicate=True),
        Output("regulatory-scan-data","data",allow_duplicate=True),
        Output("regulatory-pending-remediation","data",allow_duplicate=True),
        Output("redteam-review-data","data",allow_duplicate=True),
        Output("redteam-remediation-request","data",allow_duplicate=True),
        Output("redteam-pending-remediation","data",allow_duplicate=True),
        Output("compliance-dashboard-data","data",allow_duplicate=True),
    ],
    Input("confirm-yes","n_clicks"),
    [State("pending-role","data"), State("chat-col","style")],
    prevent_initial_call=True,
)
def confirm_switch(n, new_role, chat_style):
    if not new_role:
        raise dash.exceptions.PreventUpdate

    hidden = {"display":"none"}
    visible = {"display":"flex","flexWrap":"wrap","gap":"6px"}
    ba_st = visible if new_role == "ba" else hidden
    po_st = visible if new_role == "po" else hidden
    pm_st = visible if new_role == "pm" else hidden

    return (
        new_role,
        "general",
        [make_welcome(new_role)],
        ba_st,
        po_st,
        pm_st,
        WS_CLOSED_STYLE,
        CHAT_FULL_STYLE,
        False,
        new_role,
        None,
        None,
        None,
        None,
        None,
        None,
        None,
    )

# ── Confirm NO ───────────────────────────────────────────────────
@app.callback(
    [Output("switch-modal","is_open",allow_duplicate=True),
     Output("role-radio","value",allow_duplicate=True)],
    Input("confirm-no","n_clicks"), State("current-role","data"), prevent_initial_call=True)
def cancel_switch(n, current): return False, current

# ── BA mode buttons ───────────────────────────────────────────────
@app.callback(
    [
        Output("current-mode","data",allow_duplicate=True),
        Output("current-workspace-artifact-id","data",allow_duplicate=True),
        Output("regulatory-scan-data","data",allow_duplicate=True),
        Output("regulatory-pending-remediation","data",allow_duplicate=True),
        Output("redteam-review-data","data",allow_duplicate=True),
        Output("redteam-remediation-request","data",allow_duplicate=True),
        Output("redteam-pending-remediation","data",allow_duplicate=True),
        Output("compliance-dashboard-data","data",allow_duplicate=True),
        Output("workspace-open","data",allow_duplicate=True),
        Output("workspace-col","style",allow_duplicate=True),
        Output("chat-col","style",allow_duplicate=True),
    ] +
    [Output(f"ba-btn-{m}","color") for m in BA_MODES] +
    [Output(f"ba-btn-{m}","outline") for m in BA_MODES],
    [Input(f"ba-btn-{m}","n_clicks") for m in BA_MODES],
    prevent_initial_call=True,
)
def switch_ba_mode(*args):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    clicked = ctx.triggered[0]["prop_id"].split(".")[0].replace("ba-btn-","")

    return (
        [clicked, None, None, None, None, None, None, None, False,
         WS_CLOSED_STYLE, CHAT_FULL_STYLE]
        + ["primary" if m == clicked else "light" for m in BA_MODES]
        + [m != clicked for m in BA_MODES]
    )

@app.callback(
    [
        Output("manual-push-jira-btn","style"),
        Output("scope-change-btn","style"),
        Output("scope-creep-btn","style"),
    ],
    [Input("current-role","data"), Input("current-mode","data")],
    prevent_initial_call=False,
)
def toggle_scope_and_jira_controls(role, mode):
    jira_modes = {
        ("ba", "stories"),
        ("po", "backlog"),
        ("po", "prioritize"),
        ("pm", "raid"),
        ("pm", "changelog"),
        ("pm", "scope"),
    }
    if (role, mode) == ("pm", "scope"):
        visible = {"display": "inline-block", "fontSize": "12px"}
        return visible, visible, visible
    if (role, mode) in jira_modes:
        return {"display": "inline-block", "fontSize": "12px"}, {"display": "none"}, {"display": "none"}
    return {"display": "none"}, {"display": "none"}, {"display": "none"}

@app.callback(
    [Output("jira-pending-cards","data", allow_duplicate=True),
     Output("jira-push-modal","is_open", allow_duplicate=True)],
    Input("manual-push-jira-btn","n_clicks"),
    [State("jira-pending-cards","data"),
     State("doc-content","data"), State("doc-mode","data"), State("current-chat-id","data")],
    prevent_initial_call=True)
def manual_open_jira_push(n, pending, doc_content, doc_mode, chat_id):
    if not n: raise dash.exceptions.PreventUpdate
    if pending and pending.get("cards"):
        return dash.no_update, True
    JIRA_CAPABLE_DOC_MODES = {"stories", "backlog", "prioritize", "raid", "changelog", "scope"}
    if doc_mode in JIRA_CAPABLE_DOC_MODES and doc_content:
        parsed = parse_scope_payload(doc_content) if doc_mode == "scope" else parse_po_json(doc_content)
        cards = parsed.get("cards", []) if parsed else []
        if cards:
            return {"chat_id": chat_id, "mode": doc_mode, "cards": cards}, True
    raise dash.exceptions.PreventUpdate

# ── PO mode buttons ───────────────────────────────────────────────
@app.callback(
    [
        Output("current-mode","data",allow_duplicate=True),
        Output("current-workspace-artifact-id","data",allow_duplicate=True),
        Output("regulatory-scan-data","data",allow_duplicate=True),
        Output("regulatory-pending-remediation","data",allow_duplicate=True),
        Output("redteam-review-data","data",allow_duplicate=True),
        Output("redteam-remediation-request","data",allow_duplicate=True),
        Output("redteam-pending-remediation","data",allow_duplicate=True),
        Output("compliance-dashboard-data","data",allow_duplicate=True),
        Output("workspace-open","data",allow_duplicate=True),
        Output("workspace-col","style",allow_duplicate=True),
        Output("chat-col","style",allow_duplicate=True),
    ] +
    [Output(f"po-btn-{m}","color") for m in PO_MODES] +
    [Output(f"po-btn-{m}","outline") for m in PO_MODES],
    [Input(f"po-btn-{m}","n_clicks") for m in PO_MODES],
    prevent_initial_call=True,
)
def switch_po_mode(*args):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    clicked = ctx.triggered[0]["prop_id"].split(".")[0].replace("po-btn-","")

    return (
        [clicked, None, None, None, None, None, None, None, False,
         WS_CLOSED_STYLE, CHAT_FULL_STYLE]
        + ["primary" if m == clicked else "light" for m in PO_MODES]
        + [m != clicked for m in PO_MODES]
    )

@app.callback(
    [
        Output("current-mode","data",allow_duplicate=True),
        Output("current-workspace-artifact-id","data",allow_duplicate=True),
        Output("regulatory-scan-data","data",allow_duplicate=True),
        Output("regulatory-pending-remediation","data",allow_duplicate=True),
        Output("redteam-review-data","data",allow_duplicate=True),
        Output("redteam-remediation-request","data",allow_duplicate=True),
        Output("redteam-pending-remediation","data",allow_duplicate=True),
        Output("compliance-dashboard-data","data",allow_duplicate=True),
        Output("workspace-open","data",allow_duplicate=True),
        Output("workspace-col","style",allow_duplicate=True),
        Output("chat-col","style",allow_duplicate=True),
    ] +
    [Output(f"pm-btn-{m}","color") for m in PM_MODES] +
    [Output(f"pm-btn-{m}","outline") for m in PM_MODES],
    [Input(f"pm-btn-{m}","n_clicks") for m in PM_MODES],
    prevent_initial_call=True,
)
def switch_pm_mode(*args):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    clicked = ctx.triggered[0]["prop_id"].split(".")[0].replace("pm-btn-","")

    return (
        [clicked, None, None, None, None, None, None, None, False,
         WS_CLOSED_STYLE, CHAT_FULL_STYLE]
        + ["primary" if m == clicked else "light" for m in PM_MODES]
        + [m != clicked for m in PM_MODES]
    )


@app.callback(
    [
        Output("workspace-content", "children", allow_duplicate=True),
        Output("workspace-open", "data", allow_duplicate=True),
        Output("workspace-col", "style", allow_duplicate=True),
        Output("chat-col", "style", allow_duplicate=True),
    ],
    Input("current-mode", "data"),
    [
        State("current-role", "data"),
        State("doc-mode", "data"),
        State("doc-content", "data"),
        State("current-chat-id", "data"),
    ],
    prevent_initial_call=True,
)
def show_scope_management_dashboard(mode, role, doc_mode, doc_content, chat_id):
    if role != "pm" or mode != "scope":
        raise dash.exceptions.PreventUpdate

    if doc_mode == "scope" and doc_content:
        parsed = parse_scope_payload(doc_content)
        if parsed:
            return (
                render_scope_management(parsed, jira_map=load_jira_sync(chat_id) if chat_id else {}),
                True, WS_SPLIT_STYLE, CHAT_SPLIT_STYLE,
            )

    return render_scope_management_dashboard(), True, WS_SPLIT_STYLE, CHAT_SPLIT_STYLE


# ── Scope Management controls ─────────────────────────────────────
@app.callback(
    Output("scope-change-modal", "is_open", allow_duplicate=True),
    Input("scope-change-btn", "n_clicks"),
    prevent_initial_call=True,
)
def open_scope_change_modal(n_clicks):
    if not n_clicks:
        raise dash.exceptions.PreventUpdate
    return True


@app.callback(
    [
        Output("scope-change-results", "children"),
        Output("scope-change-result", "data"),
        Output("scope-change-push-jira-btn", "disabled"),
        Output("doc-content", "data", allow_duplicate=True),
        Output("workspace-content", "children", allow_duplicate=True),
    ],
    Input("scope-change-analyze-btn", "n_clicks"),
    [
        State("scope-change-input", "value"),
        State("doc-content", "data"),
        State("current-chat-id", "data"),
    ],
    prevent_initial_call=True,
)
def analyze_scope_change_request(n_clicks, request_text, doc_content, chat_id):
    if not n_clicks or not request_text or not request_text.strip():
        raise dash.exceptions.PreventUpdate

    scope_payload = parse_scope_payload(doc_content)
    if not scope_payload:
        return (
            html.Div(
                "No Scope Management baseline is available for this workspace.",
                style={"color": "#c62828"},
            ),
            None,
            True,
            dash.no_update,
            dash.no_update,
        )

    analysis = scope_change_request_analysis(scope_payload, request_text.strip())
    result = html.Div([
        html.H6(
            f"{analysis['classification']} · Baseline {analysis['baseline_version']}",
            style={"fontWeight": "700"},
        ),
        html.P(analysis["recommendation"], style={"fontSize": "12px"}),
        html.Div([
            html.Div(
                f"Out-of-scope matches: {len(analysis['out_of_scope_matches'])}",
                style={"fontSize": "11px"},
            ),
            html.Div(
                f"WBS matches: {len(analysis['wbs_matches'])}",
                style={"fontSize": "11px"},
            ),
        ]),
        html.Hr(),
        html.Div(
            f"JIRA-ready change request: {analysis['card']['id']}",
            style={"fontSize": "11px", "fontWeight": "700"},
        ),
    ])

    change_record = dict(analysis)
    change_record["requested_change"] = request_text.strip()
    change_record["approval_status"] = "Proposed"
    change_record["status"] = "Proposed"

    updated_payload = dict(scope_payload)
    updated_payload["change_requests"] = list(scope_payload.get("change_requests") or []) + [change_record]
    updated_doc_content = json.dumps(updated_payload, ensure_ascii=False, indent=2)

    return (
        result,
        analysis,
        False,
        updated_doc_content,
        render_scope_management(
            updated_payload,
            jira_map=load_jira_sync(chat_id) if chat_id else {},
        ),
    )


@app.callback(
    [
        Output("jira-pending-cards", "data", allow_duplicate=True),
        Output("jira-push-modal", "is_open", allow_duplicate=True),
        Output("scope-change-modal", "is_open", allow_duplicate=True),
    ],
    Input("scope-change-push-jira-btn", "n_clicks"),
    [
        State("scope-change-result", "data"),
        State("current-chat-id", "data"),
    ],
    prevent_initial_call=True,
)
def push_scope_change_to_jira(n_clicks, result, chat_id):
    if not n_clicks or not result or not result.get("card"):
        raise dash.exceptions.PreventUpdate
    return {
        "chat_id": chat_id,
        "mode": "scope",
        "cards": [result["card"]],
    }, True, False


@app.callback(
    Output("scope-change-modal", "is_open", allow_duplicate=True),
    [
        Input("scope-change-close-btn", "n_clicks"),
        Input("scope-change-modal", "is_open"),
    ],
    prevent_initial_call=True,
)
def close_scope_change_modal(n_clicks, is_open):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate
    trigger = ctx.triggered[0]["prop_id"].split(".")[0]
    if trigger == "scope-change-modal" and is_open is False:
        return False
    return False


@app.callback(
    Output("scope-creep-modal", "is_open", allow_duplicate=True),
    Input("scope-creep-btn", "n_clicks"),
    prevent_initial_call=True,
)
def open_scope_creep_modal(n_clicks):
    if not n_clicks:
        raise dash.exceptions.PreventUpdate
    return True


@app.callback(
    [
        Output("scope-creep-results", "children"),
        Output("doc-content", "data", allow_duplicate=True),
        Output("workspace-content", "children", allow_duplicate=True),
    ],
    Input("scope-creep-run-btn", "n_clicks"),
    [
        State("doc-content", "data"),
        State("current-chat-id", "data"),
    ],
    prevent_initial_call=True,
)
def run_scope_creep_scan(n_clicks, doc_content, chat_id):
    if not n_clicks:
        raise dash.exceptions.PreventUpdate

    scope_payload = parse_scope_payload(doc_content)
    if not scope_payload:
        return (
            html.Div("No Scope Management baseline is available.", style={"color": "#c62828"}),
            dash.no_update,
            dash.no_update,
        )

    context = fetch_scope_jira_issues(scope_payload)
    if context.get("error"):
        return (
            html.Div(f"⚠️ {context['error']}", style={"color": "#c62828", "fontSize": "12px"}),
            dash.no_update,
            dash.no_update,
        )

    findings = detect_scope_creep(scope_payload, context)
    updated_payload = dict(scope_payload)
    updated_payload["scope_creep_findings"] = findings
    updated_doc_content = json.dumps(updated_payload, ensure_ascii=False, indent=2)

    if not findings:
        results = html.Div([
            html.H6("✅ No scope-creep candidates detected."),
            html.Div(
                f"Scanned {len(context.get('issues', []))} JIRA Scope Management issues "
                f"against baseline {scope_payload.get('scope_baseline', {}).get('version', 'TBD')}.",
                style={"fontSize": "12px", "color": "#666"},
            ),
        ])
    else:
        results = html.Div([
            html.H6(
                f"⚠️ {len(findings)} scope-creep candidate(s) detected.",
                style={"fontWeight": "700", "color": "#c62828"},
            ),
            html.Div([
                html.Div([
                    html.Span(
                        f"{item['severity']} · {item['linked_jira_key']}",
                        style={"fontWeight": "700", "fontSize": "11px"},
                    ),
                    html.Div(item["observed_item"], style={"fontSize": "12px"}),
                    html.Div(item["reason_flagged"], style={"fontSize": "11px", "color": "#666"}),
                    html.Div(
                        f"Recommendation: {item['recommendation']}",
                        style={"fontSize": "11px", "marginTop": "3px"},
                    ),
                ], style={
                    "border": "1px solid #eee",
                    "borderLeft": "4px solid #c62828",
                    "borderRadius": "6px",
                    "padding": "10px",
                    "marginBottom": "8px",
                }) for item in findings
            ]),
        ])

    return (
        results,
        updated_doc_content,
        render_scope_management(
            updated_payload,
            jira_map=load_jira_sync(chat_id) if chat_id else {},
        ),
    )


@app.callback(
    Output("scope-creep-modal", "is_open", allow_duplicate=True),
    [
        Input("scope-creep-close-btn", "n_clicks"),
        Input("scope-creep-modal", "is_open"),
    ],
    prevent_initial_call=True,
)
def close_scope_creep_modal(n_clicks, is_open):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate
    return False


# ── File upload ────────────────────────────────────────────────────
@app.callback(
    [Output("file-context-store","data"),
     Output("file-badge","children"),
     Output("clear-file-btn","style")],
    Input("file-upload","contents"),
    State("file-upload","filename"), prevent_initial_call=True)
def handle_file_upload(contents, filename):
    if not contents or not filename: raise dash.exceptions.PreventUpdate
    text, badge_label = extract_file_text(filename, contents)
    badge = html.Span(badge_label, style={"background":"#e8f0fe","padding":"2px 8px","borderRadius":"10px","fontSize":"11px"})
    return text, badge, {"display":"inline-block","padding":"2px 6px","fontSize":"11px"}

@app.callback(
    [Output("file-context-store","data",allow_duplicate=True),
     Output("file-badge","children",allow_duplicate=True),
     Output("clear-file-btn","style",allow_duplicate=True)],
    Input("clear-file-btn","n_clicks"), prevent_initial_call=True)
def clear_file(n): return "", "", {"display":"none"}

@app.callback(
    [Output("jira-pending-cards","data", allow_duplicate=True),
     Output("jira-push-modal","is_open", allow_duplicate=True),
     Output("meeting-modal","is_open", allow_duplicate=True)],
    Input({"type":"meeting-push-stories-btn","index":ALL},"n_clicks"),
    State("meeting-stories-payload","data"),
    prevent_initial_call=True)
def push_meeting_stories_to_jira(clicks, payload):
    if not any(clicks) or not payload or not payload.get("cards"):
        raise dash.exceptions.PreventUpdate
    return {"chat_id": payload["chat_id"], "mode": "stories", "cards": payload["cards"]}, True, False

# ════════════════════════════════════════════════════════════════
# SEND MESSAGE — 14 outputs, verified alignment
# ════════════════════════════════════════════════════════════════
@app.callback(
    [Output("chat-messages",    "children"),
     Output("workspace-content","children"),
     Output("workspace-col",    "style",    allow_duplicate=True),
     Output("chat-col",         "style",    allow_duplicate=True),
     Output("doc-content",      "data"),
     Output("doc-mode",         "data"),
     Output("user-input",       "value"),
     Output("chart-area",       "children"),
     Output("current-chat-id",  "data"),
     Output("bpmn-xml-store",   "data",     allow_duplicate=True),
     Output("ai-banner-store",  "data"),
     Output("file-badge",       "children", allow_duplicate=True),
     Output("clear-file-btn",   "style",    allow_duplicate=True),
     Output("workspace-open",   "data"),
     Output("current-workspace-artifact-id", "data", allow_duplicate=True),
     Output("jira-push-modal",   "is_open", allow_duplicate=True),
     Output("jira-pending-cards","data",    allow_duplicate=True)],
    Input("send-btn","n_clicks"),
    [State("user-input",         "value"),
     State("current-role",       "data"),
     State("current-mode",       "data"),
     State("chat-messages",      "children"),
     State("workspace-col",      "style"),
     State("chat-col",           "style"),
     State("current-chat-id",    "data"),
     State("process-submode",    "data"),
     State("file-context-store", "data")],
    prevent_initial_call=True)
def send_message(n, message, role, mode, chat_children,
                 ws_style, chat_style, chat_id, process_submode, file_context):

    if not message or not message.strip(): raise dash.exceptions.PreventUpdate

    msg = message.strip()
    ROLE_COLOR_MAPS = {"ba":BA_COLORS,"po":PO_COLORS,"pm":PM_COLORS}
    ROLE_LABELS     = {"ba":"📊 [BA]","po":"🎯 [PO]","pm":"🧭 [PM]"}
    color = ROLE_COLOR_MAPS.get(role, BA_COLORS).get(mode,"#1a73e8")
    label = ROLE_LABELS.get(role, "📊 [BA]")
    if not isinstance(chat_children, list): chat_children=[]
    _NO_BADGE = ("", {"display":"none"})

    if not chat_id:
        chat_id = create_chat(role=role, mode=mode, title=auto_title(msg))
    else:
        if not load_messages(chat_id):
            rename_chat(chat_id, auto_title(msg))
    save_message(chat_id, "user", msg, mode)
    update_chat_context(chat_id, role, mode, process_submode)

    document_type = (
        resolve_requirements_document_type(chat_id, msg)
        if role == "ba" and mode == "brd"
        else "BRD"
    )

    display_mode = get_workspace_display_mode(
        role,
        mode,
        document_type,
    )

    session_log.append({"timestamp":datetime.now().isoformat(),
                        "role_context":role,"mode":mode,"speaker":"user","message":msg})

    user_bubble = html.Div([
        html.Div(msg, style={"background":"#1a73e8","color":"white","padding":"10px 14px",
            "borderRadius":"16px 16px 4px 16px","display":"inline-block","maxWidth":"85%","fontSize":"13px"}),
        html.Div("You", style={"fontSize":"11px","color":"#bbb","marginTop":"4px","textAlign":"right"})
    ], style={"textAlign":"right"}, className="message-bubble")
    new_chat = chat_children + [user_bubble]

    def ret(chat,ws_ch,ws_st,ch_st,doc,dmode,inp,chart,cid,bxml,banner,badge,badge_st,ws_open,artifact_id,jopen,jpending):
        return (chat,ws_ch,ws_st,ch_st,doc,dmode,inp,chart,cid,bxml,banner,badge,badge_st,ws_open,artifact_id,jopen,jpending)

    try:
        is_bpmn = (role=="ba" and mode=="process" and process_submode=="bpmn")

        if is_bpmn:
            bpmn_json_raw, ai_name, banner = AIProvider.bpmn(msg)
            bpmn_data = parse_po_json(bpmn_json_raw)
            bpmn_xml  = build_bpmn_xml(bpmn_data) if bpmn_data else ""
            bpmn_ok   = bool(bpmn_xml)

            WorkspaceState.save(chat_id,
                doc_content=bpmn_json_raw, doc_mode=mode,
                process_submode="bpmn", bpmn_xml=bpmn_xml, workspace_open=bpmn_ok)
            update_chat_context(chat_id, role, mode, "bpmn")
            bpmn_message_content = (
                f"[BPMN] {bpmn_json_raw[:300]}…"
                if bpmn_ok
                else f"[BPMN-FAILED] {bpmn_json_raw[:300]}…"
            )
            bpmn_message_id = save_message(
                chat_id,
                "assistant",
                bpmn_message_content,
                mode,
            )
            bpmn_artifact_id = (
                create_workspace_artifact(
                    chat_id=chat_id,
                    message_id=bpmn_message_id,
                    role=role,
                    mode=mode,
                    doc_content=bpmn_json_raw,
                    document_type="",
                    process_submode="bpmn",
                    bpmn_xml=bpmn_xml,
                )
                if bpmn_ok
                else None
            )
            maybe_refresh_chat_summary(chat_id, role, mode)
            session_log.append({"timestamp":datetime.now().isoformat(),
                                "role_context":role,"mode":mode,"speaker":"assistant",
                                "message":"[BPMN generated]" if bpmn_ok else "[BPMN generation failed]"})

            if not bpmn_ok:
                bot_bubble = html.Div([
                    html.Div(html.Span("📊 [BA]·process·BPMN",
                        style={"fontWeight":"600","color":"#c62828","fontSize":"12px"}),style={"marginBottom":"8px"}),
                    html.Div("⚠️ Couldn't parse a valid BPMN model from the AI response — please try again.",
                        style={"background":"#ffebee","padding":"10px 14px","borderRadius":"16px 16px 16px 4px",
                               "borderLeft":"3px solid #c62828","fontSize":"13px","display":"inline-block",
                               "maxWidth":"85%","color":"#c62828"})
                ], style={"textAlign":"left"}, className="message-bubble")

                return ret(new_chat+[bot_bubble], dash.no_update, dash.no_update, dash.no_update,
                           bpmn_json_raw, mode, "", [], chat_id, "",
                           banner or "", *_NO_BADGE, False, bpmn_artifact_id,
                           dash.no_update, dash.no_update)

            bot_bubble = html.Div([
                html.Div(html.Span("📊 [BA]·process·BPMN",
                    style={"fontWeight":"600","color":"#00695c","fontSize":"12px"}),style={"marginBottom":"8px"}),
                html.Div("🔷 BPMN diagram — open Workspace to view",
                    style={"background":"#e8f5e9","padding":"10px 14px","borderRadius":"16px 16px 16px 4px",
                           "borderLeft":"3px solid #00695c","fontSize":"13px","display":"inline-block",
                           "maxWidth":"85%","color":"#1b5e20"}),
                html.Div(dbc.Button("📖 Open Workspace",
                    id={"type":"open-workspace-btn","index":bpmn_artifact_id},
                    color="success",size="sm",n_clicks=0),style={"marginTop":"10px"})
            ], style={"textAlign":"left"}, className="message-bubble")

            ws_ch = html.Div([
                html.Div("🔷 BPMN Diagram",style={"fontWeight":"700","color":"#00695c","fontSize":"14px","marginBottom":"6px"}),
                html.Div("Rendered in canvas — use Export to download",style={"fontSize":"12px","color":"#888"})
            ], style={"padding":"24px 32px","textAlign":"center"})

            return ret(new_chat+[bot_bubble], ws_ch, WS_OPEN_STYLE, CHAT_HALF_STYLE,
                       bpmn_json_raw, mode, "", [], chat_id, bpmn_xml,
                       banner or "", *_NO_BADGE, True, bpmn_artifact_id,
                       dash.no_update, dash.no_update)

        reply, ai_name, banner = AIProvider.build_messages_and_call(
            msg,
            role,
            mode,
            file_context or "",
            chat_id=chat_id,
        )
        open_ws = is_doc_request(role, mode, msg)

        assistant_message_id = save_message(chat_id, "assistant", reply, mode)
        maybe_refresh_chat_summary(chat_id, role, mode)
        session_log.append({"timestamp":datetime.now().isoformat(),
                            "role_context":role,"mode":mode,"speaker":"assistant","message":reply})

        if open_ws:
            WorkspaceState.save(
                chat_id,
                doc_content=reply,
                doc_mode=mode,
                document_type=document_type,
                process_submode=process_submode,
                bpmn_xml="",
                workspace_open=True,
            )

            artifact_id = create_workspace_artifact(
                chat_id=chat_id,
                message_id=assistant_message_id,
                role=role,
                mode=mode,
                doc_content=reply,
                document_type=document_type,
                process_submode=process_submode,
                bpmn_xml="",
            )

            check_and_flag_ripple(chat_id, reply)

            is_structured = is_structured_card_mode(role, mode)
            parsed_json  = parse_po_json(reply) if is_structured else None
            parsed_cards = parsed_json.get("cards", []) if parsed_json else None

            chart = build_workspace_chart(
                reply,
                role,
                mode,
            )

            jira_modal_open = dash.no_update
            jira_pending = dash.no_update
            jira_note = ""

            JIRA_TRIGGER_MODES = {
                ("ba", "stories"),
                ("po", "backlog"),
                ("po", "prioritize"),
                ("pm", "raid"),
                ("pm", "changelog"),
                ("pm", "scope"),
            }

            if (role, mode) in JIRA_TRIGGER_MODES and parsed_cards:
                jira_modal_open = True
                jira_pending = {
                    "chat_id": chat_id,
                    "mode": mode,
                    "cards": parsed_cards,
                }
                prior_sync = load_jira_sync(chat_id)
                ws_content = render_jira_cards(
                    parsed_json,
                    jira_map=prior_sync,
                )

            elif parsed_json:
                if role == "pm" and mode == "scope":
                    ws_content = render_scope_management(
                        parsed_json,
                        jira_map=load_jira_sync(chat_id) if chat_id else {},
                    )
                else:
                    ws_content = render_jira_cards(parsed_json)

            else:
                ws_content = build_workspace_document(
                    reply,
                    role,
                    mode,
                    document_type=document_type,
                )

            bot_bubble = html.Div([
                html.Div(html.Span(f"{label}·{display_mode}",
                    style={"fontWeight":"600","color":color,"fontSize":"12px"}),style={"marginBottom":"8px"}),
                html.Div(f"✅ {'Cards' if parsed_json else 'Document'} ready — Preview in Workspace →{jira_note}",
                    style={"background":"#f1f3f4","padding":"10px 14px","borderRadius":"16px 16px 16px 4px","borderLeft":f"3px solid {color}","fontSize":"13px"}),
                html.Div(dbc.Button("📖 Open Workspace",
                    id={"type":"open-workspace-btn","index":artifact_id},
                    color="primary",size="sm",n_clicks=0),style={"marginTop":"10px"})
            ], style={"textAlign":"left"}, className="message-bubble")

            return ret(new_chat+[bot_bubble], ws_content, WS_OPEN_STYLE, CHAT_HALF_STYLE,
                       reply, mode, "", chart, chat_id, "",
                       banner or "", *_NO_BADGE, True, artifact_id,
                       jira_modal_open, jira_pending)

        else:
            existing_ws = WorkspaceState.load(chat_id)
            if existing_ws["doc_content"] or existing_ws["bpmn_xml"]:
                pass
            else:
                WorkspaceState.save(
                    chat_id,
                    doc_content=reply,
                    doc_mode=mode,
                    document_type=document_type,
                    process_submode=process_submode,
                    bpmn_xml="",
                    workspace_open=False,
                )

            bot_bubble = html.Div([
                html.Div(html.Span(f"{label}·{display_mode}",
                    style={"fontWeight":"600","color":color,"fontSize":"12px"}),style={"marginBottom":"6px"}),
                dcc.Markdown(reply,style={"background":"#f1f3f4","padding":"10px 14px",
                    "borderRadius":"16px 16px 16px 4px","borderLeft":f"3px solid {color}",
                    "fontSize":"13px","display":"inline-block","maxWidth":"85%"},className="markdown-body")
            ], style={"textAlign":"left"}, className="message-bubble")

            return ret(new_chat+[bot_bubble], dash.no_update, dash.no_update, dash.no_update,
                       reply, mode, "", dash.no_update, chat_id, "",
                       banner or "", *_NO_BADGE, False, None,
                       dash.no_update, dash.no_update)

    except Exception as e:
        if chat_id: WorkspaceState.save(chat_id, workspace_open=False)
        err = html.Div(f"❌ {str(e)[:120]}",
                       style={"background":"#ffebee","color":"#c62828","padding":"10px","borderRadius":"8px","fontSize":"13px"})
        return ret(new_chat+[err], dash.no_update, dash.no_update, dash.no_update,
                   dash.no_update, dash.no_update, "", dash.no_update, chat_id,
                   dash.no_update, "", *_NO_BADGE, False, dash.no_update,
                   dash.no_update, dash.no_update)

# ── RESET ─────────────────────────────────────────────────────────
@app.callback(
    [
        Output("chat-messages", "children", allow_duplicate=True),
        Output("workspace-content", "children", allow_duplicate=True),
        Output("workspace-col", "style", allow_duplicate=True),
        Output("chat-col", "style", allow_duplicate=True),
        Output("doc-content", "data", allow_duplicate=True),
        Output("doc-mode", "data", allow_duplicate=True),
        Output("bpmn-xml-store", "data", allow_duplicate=True),
        Output("chart-area", "children", allow_duplicate=True),
        Output("user-input", "value", allow_duplicate=True),
    ],
    Input("reset-btn", "n_clicks"),
    [
        State("current-role", "data"),
        State("current-mode", "data"),
        State("process-submode", "data"),
        State("current-chat-id", "data"),
    ],
    prevent_initial_call=True,
)
def reset_chat(n, role, mode, submode, chat_id):
    """Clear the current chat's persisted workspace and reset the UI."""
    if not n:
        raise dash.exceptions.PreventUpdate

    if chat_id:
        clear_chat_summary(chat_id)
        WorkspaceState.save(
            chat_id,
            doc_content="",
            doc_mode="general",
            document_type="BRD",
            process_submode=submode or "text",
            bpmn_xml="",
            workspace_open=False,
        )

    msg = html.Div(
        f"🔄 [{role.upper()}·{mode}] conversation cleared",
        style={
            "textAlign": "center",
            "color": "#ff9800",
            "fontSize": "12px",
            "padding": "6px",
        },
    )

    return (
        [msg],
        html.Div(),
        WS_CLOSED_STYLE,
        CHAT_FULL_STYLE,
        "",
        "general",
        "",
        [],
        "",
    )

@app.callback(
    [
        Output("current-workspace-artifact-id", "data", allow_duplicate=True),
        Output("regulatory-scan-data", "data", allow_duplicate=True),
        Output("regulatory-pending-remediation", "data", allow_duplicate=True),
    ],
    Input("reset-btn", "n_clicks"),
    prevent_initial_call=True,
)
def clear_regulatory_workspace_context(n):
    if not n:
        raise dash.exceptions.PreventUpdate
    return None, None, None


# ── NEW CHAT ──────────────────────────────────────────────────────
@app.callback(
    [Output("chat-messages","children",allow_duplicate=True),
     Output("workspace-col","style",   allow_duplicate=True),
     Output("chat-col","style",        allow_duplicate=True),
     Output("current-chat-id","data",  allow_duplicate=True)],
    Input("new-chat-btn","n_clicks"),
    [State("current-role","data"), State("chat-col","style")], prevent_initial_call=True)
def new_chat(n, role, chat_style):
    chat_id = create_chat(role=role)
    return [make_welcome(role)], WS_CLOSED_STYLE, CHAT_FULL_STYLE, chat_id

# ── JIRA push modal ─────────────────────────────────────────────
@app.callback(
    [Output("jira-modal-summary","children"),
     Output("jira-space-select","options"),
     Output("jira-space-select","value"),
     Output("jira-push-confirm-btn","style", allow_duplicate=True),
     Output("jira-push-cancel-btn","style", allow_duplicate=True),
     Output("jira-push-ok-btn","style", allow_duplicate=True),
     Output("jira-push-notification","children", allow_duplicate=True),
     Output("jira-modal-results","children", allow_duplicate=True)],
    Input("jira-push-modal","is_open"),
    State("jira-pending-cards","data"),
    prevent_initial_call=True)
def show_jira_modal_summary(is_open, pending):
    if not is_open or not pending:
        raise dash.exceptions.PreventUpdate
    cards = pending.get("cards", [])

    type_counts = {}
    for c in cards:
        t = c.get("type","item")
        type_counts[t] = type_counts.get(t, 0) + 1
    desc = ", ".join(f"{v} {k}{'s' if v != 1 else ''}" for k, v in type_counts.items()) or "0 items"

    spaces  = jira_fetch_projects()
    options = [{"label": f"{s['name']} ({s['key']})", "value": s["key"]} for s in spaces]

    combined = " ".join(f"{c.get('title','')} {c.get('description','')}" for c in cards)
    matched  = match_jira_space(combined, spaces)
    default_key = matched["key"] if matched else (spaces[0]["key"] if spaces else None)

    summary = (f"Found {desc}."
               + (f" Best match: '{matched['name']}'." if matched else " Pick a space below."))
    return summary, options, default_key, {}, {}, {"display":"none"}, "", ""

@app.callback(
    [Output("jira-modal-results","children"),
     Output("jira-push-modal","is_open",allow_duplicate=True),
     Output("jira-push-confirm-btn","style", allow_duplicate=True),
     Output("jira-push-cancel-btn","style", allow_duplicate=True),
     Output("jira-push-ok-btn","style", allow_duplicate=True),
     Output("jira-push-notification","children", allow_duplicate=True)],
    Input("jira-push-confirm-btn","n_clicks"),
    [State("jira-pending-cards","data"), State("jira-space-select","value")],
    prevent_initial_call=True)
def confirm_jira_push(n, pending, space_key):
    if not n or not pending:
        raise dash.exceptions.PreventUpdate
    chat_id = pending.get("chat_id")
    cards   = pending.get("cards", [])
    results = push_cards_to_jira(chat_id, cards, pending.get("mode","stories"), project_key=space_key)

    rows = []
    for r in results:
        if r["status"] == "error":
            rows.append(html.Div(f"⚠️ {r['title'][:60]} — {r.get('error','failed')}",
                style={"color":"#c62828","marginBottom":"4px"}))
            continue
        epic_tag = html.Span(" 🔗Epic", style={"color":"#9c27b0","fontSize":"10px"}) \
                   if r.get("epic_link") and r["epic_link"] != "unsupported" else html.Span()
        rows.append(html.Div([
            html.Span("✅ " if r["status"]=="created" else "🔄 "),
            html.A(f"{r['jira_key']}", href=r["jira_url"], target="_blank",
                   style={"color":"#0052cc","fontWeight":"600","marginRight":"6px"}),
            html.Span(f"[{r.get('issue_type','?')}] ", style={"color":"#888","fontSize":"11px"}),
            html.Span(r["title"][:60]),
            epic_tag,
        ], style={"marginBottom":"2px"}))
        for st in r.get("subtasks", []):
            if st["status"] == "error":
                rows.append(html.Div(f"⚠️ {st['title'][:50]} — {st.get('error','failed')}",
                    style={"color":"#c62828","fontSize":"11px","marginBottom":"2px","paddingLeft":"20px"}))
            else:
                rows.append(html.Div([
                    html.Span("↳ "),
                    html.A(st["jira_key"], href=st["jira_url"], target="_blank",
                           style={"color":"#0052cc","fontWeight":"600","marginRight":"6px"}),
                    html.Span(st["title"][:50])
                ], style={"fontSize":"11px","color":"#666","marginBottom":"2px","paddingLeft":"20px"}))

    ok     = sum(1 for r in results if r["status"] in ("created","updated"))
    sub_ok = sum(1 for r in results for st in r.get("subtasks",[]) if st["status"] in ("created","updated"))
    fail   = (sum(1 for r in results if r["status"]=="error")
              + sum(1 for r in results for st in r.get("subtasks",[]) if st["status"]=="error"))

    summary_line = f"Done — {ok} synced"
    if sub_ok: summary_line += f", {sub_ok} sub-tasks"
    if fail:   summary_line += f", {fail} failed"
    rows.insert(0, html.Div(summary_line, style={"fontWeight":"700","marginBottom":"8px"}))

    notif_ok = fail == 0
    notification = html.Div([
        html.Div(
            f"{'✅' if notif_ok else '⚠️'} Pushed — Status: {'OK' if notif_ok else f'{fail} failed'}",
            style={"fontSize":"12px","fontWeight":"600","padding":"6px 10px","borderRadius":"6px",
                   "background": "#e8f5e9" if notif_ok else "#ffebee",
                   "color": "#2e7d32" if notif_ok else "#c62828"}),
        html.Div("Data synced — you can close this tab.",
                  style={"fontSize":"11px","color":"#888","marginTop":"4px"}) if notif_ok else html.Div(),
    ], style={"marginBottom":"8px"})

    hide, show_ok = {"display":"none"}, {"display":"inline-block"}
    return rows, True, hide, hide, show_ok, notification

@app.callback(Output("jira-push-modal","is_open",allow_duplicate=True),
    [Input("jira-push-cancel-btn","n_clicks"),
     Input("jira-push-ok-btn","n_clicks"),
     Input("jira-push-modal","is_open")],
    prevent_initial_call=True)
def close_jira_modal(cancel_n, ok_n, header_x_state):
    ctx = callback_context
    if not ctx.triggered: raise dash.exceptions.PreventUpdate
    trigger = ctx.triggered[0]["prop_id"].split(".")[0]
    if trigger == "jira-push-modal":
        # Fires when the header ✕ (dbc's built-in close) sets is_open False on its own
        if header_x_state is False:
            return False
        raise dash.exceptions.PreventUpdate
    return False

# ── SYNC LATEST WORKSPACE ARTIFACT FOR CHAT RESTORE ───────────────
@app.callback(
    Output("current-workspace-artifact-id", "data", allow_duplicate=True),
    Input("current-chat-id", "data"),
    prevent_initial_call=True,
)
def sync_latest_workspace_artifact(chat_id):
    if not chat_id:
        raise dash.exceptions.PreventUpdate

    artifact = get_latest_workspace_artifact(chat_id)
    return artifact["id"] if artifact else None


# ════════════════════════════════════════════════════════════════
# REMEDIATION CONFIRMATION — REVIEW OR CLOSE
# ════════════════════════════════════════════════════════════════
@app.callback(
    Output("remediation-confirm-modal", "is_open", allow_duplicate=True),
    Input("remediation-confirm-close-btn", "n_clicks"),
    prevent_initial_call=True,
)
def close_remediation_confirmation(n):
    if not n:
        raise dash.exceptions.PreventUpdate
    return False


@app.callback(
    [
        Output("remediation-confirm-modal", "is_open", allow_duplicate=True),
        Output("workspace-col", "style", allow_duplicate=True),
        Output("chat-col", "style", allow_duplicate=True),
        Output("workspace-content", "children", allow_duplicate=True),
        Output("doc-content", "data", allow_duplicate=True),
        Output("doc-mode", "data", allow_duplicate=True),
        Output("bpmn-xml-store", "data", allow_duplicate=True),
        Output("chart-area", "children", allow_duplicate=True),
        Output("workspace-open", "data", allow_duplicate=True),
        Output("current-workspace-artifact-id", "data", allow_duplicate=True),
    ],
    Input("remediation-confirm-open-btn", "n_clicks"),
    State("remediation-confirm-artifact-id", "data"),
    prevent_initial_call=True,
)
def open_remediation_confirmation_workspace(n, artifact_id):
    if not n or not artifact_id:
        raise dash.exceptions.PreventUpdate

    artifact = get_workspace_artifact(artifact_id)
    if not artifact:
        raise dash.exceptions.PreventUpdate

    state = workspace_artifact_to_state(artifact)
    ws_children, bpmn_xml, chart = render_workspace_artifact(artifact)

    WorkspaceState.save(
        artifact["chat_id"],
        doc_content=state["doc_content"],
        doc_mode=state["doc_mode"],
        document_type=state["document_type"] or "BRD",
        process_submode=state["process_submode"],
        bpmn_xml=state["bpmn_xml"],
        workspace_open=True,
    )

    return (
        False,
        WS_OPEN_STYLE,
        CHAT_HALF_STYLE,
        ws_children,
        state["doc_content"],
        state["doc_mode"],
        bpmn_xml,
        chart,
        True,
        artifact_id,
    )


# ── CLOSE WORKSPACE ───────────────────────────────────────────────
@app.callback(
    [Output("workspace-col","style",allow_duplicate=True),
     Output("chat-col","style",     allow_duplicate=True)],
    Input("close-workspace-btn","n_clicks"),
    [State("workspace-col","style"), State("chat-col","style")], prevent_initial_call=True)
def close_workspace(n, ws, chat):
    return WS_CLOSED_STYLE, CHAT_FULL_STYLE

# ── REOPEN EXACT WORKSPACE VERSION ───────────────────────────────
@app.callback(
    [
        Output("workspace-col", "style", allow_duplicate=True),
        Output("chat-col", "style", allow_duplicate=True),
        Output("workspace-content", "children", allow_duplicate=True),
        Output("doc-content", "data", allow_duplicate=True),
        Output("doc-mode", "data", allow_duplicate=True),
        Output("bpmn-xml-store", "data", allow_duplicate=True),
        Output("chart-area", "children", allow_duplicate=True),
        Output("workspace-open", "data", allow_duplicate=True),
        Output("current-workspace-artifact-id", "data", allow_duplicate=True),
    ],
    Input({"type": "open-workspace-btn", "index": ALL}, "n_clicks"),
    State({"type": "open-workspace-btn", "index": ALL}, "id"),
    prevent_initial_call=True,
)
def reopen_workspace(clicks, ids):
    """Open the exact immutable artifact attached to the clicked message."""
    ctx = callback_context
    if not ctx.triggered or not any(clicks):
        raise dash.exceptions.PreventUpdate

    triggered = json.loads(ctx.triggered[0]["prop_id"].split(".")[0])
    artifact_id = triggered.get("index")
    artifact = get_workspace_artifact(artifact_id)

    if not artifact:
        raise dash.exceptions.PreventUpdate

    state = workspace_artifact_to_state(artifact)
    ws_children, bpmn_xml, chart = render_workspace_artifact(artifact)

    # Keep the selected version in the legacy workspace state so close/reopen
    # behaves naturally, while AI document-type resolution still uses the
    # immutable latest artifact.
    WorkspaceState.save(
        artifact["chat_id"],
        doc_content=state["doc_content"],
        doc_mode=state["doc_mode"],
        document_type=state["document_type"] or "BRD",
        process_submode=state["process_submode"],
        bpmn_xml=state["bpmn_xml"],
        workspace_open=True,
    )

    return (
        WS_OPEN_STYLE,
        CHAT_HALF_STYLE,
        ws_children,
        state["doc_content"],
        state["doc_mode"],
        bpmn_xml,
        chart,
        True,
        artifact_id,
    )

# ════════════════════════════════════════════════════════════════
# LOAD CHAT FROM SIDEBAR — 63 outputs, verified alignment
# ════════════════════════════════════════════════════════════════
# Path: app.py

@app.callback(
    [
        Output("chat-messages", "children", allow_duplicate=True),
        Output("history-offcanvas", "is_open", allow_duplicate=True),
        Output("current-chat-id", "data", allow_duplicate=True),
        Output("current-role", "data", allow_duplicate=True),
        Output("current-mode", "data", allow_duplicate=True),
        Output("process-submode", "data", allow_duplicate=True),
        Output("workspace-open", "data", allow_duplicate=True),
        Output("doc-content", "data", allow_duplicate=True),
        Output("doc-mode", "data", allow_duplicate=True),
        Output("bpmn-xml-store", "data", allow_duplicate=True),
        Output("role-radio", "value", allow_duplicate=True),
        Output("ba-mode-bar", "style", allow_duplicate=True),
        Output("po-mode-bar", "style", allow_duplicate=True),
        Output("pm-mode-bar", "style", allow_duplicate=True),
        Output("workspace-content", "children", allow_duplicate=True),
        Output("chart-area", "children", allow_duplicate=True),
        Output("workspace-col", "style", allow_duplicate=True),
        Output("chat-col", "style", allow_duplicate=True),
    ]
    + [
        Output(f"ba-btn-{m}", "color", allow_duplicate=True)
        for m in BA_MODES
    ]
    + [
        Output(f"ba-btn-{m}", "outline", allow_duplicate=True)
        for m in BA_MODES
    ]
    + [
        Output(f"po-btn-{m}", "color", allow_duplicate=True)
        for m in PO_MODES
    ]
    + [
        Output(f"po-btn-{m}", "outline", allow_duplicate=True)
        for m in PO_MODES
    ]
    + [
        Output(f"pm-btn-{m}", "color", allow_duplicate=True)
        for m in PM_MODES
    ]
    + [
        Output(f"pm-btn-{m}", "outline", allow_duplicate=True)
        for m in PM_MODES
    ],
    Input({"type": "sidebar-item", "index": ALL}, "n_clicks"),
    State({"type": "sidebar-item", "index": ALL}, "id"),
    prevent_initial_call=True,
)
def load_chat_from_sidebar(n_clicks_list, ids):
    ctx = callback_context

    if not ctx.triggered or not any(n_clicks_list):
        raise dash.exceptions.PreventUpdate

    triggered_id = json.loads(
        ctx.triggered[0]["prop_id"].split(".")[0]
    )
    chat_id = triggered_id["index"]

    with closing(sqlite3.connect(DB_PATH)) as conn:
        conn.row_factory = sqlite3.Row
        chat_row = conn.execute(
            """
            SELECT role, mode, active_submode
            FROM chats
            WHERE id = ?
            """,
            (chat_id,),
        ).fetchone()

    if not chat_row:
        raise dash.exceptions.PreventUpdate

    role = chat_row["role"] or "ba"
    mode = chat_row["mode"] or "general"
    submode = chat_row["active_submode"] or "text"

    ws = WorkspaceState.load(chat_id)

    workspace_open = ws["workspace_open"]
    doc_content = ws["doc_content"]
    doc_mode = ws["doc_mode"]
    document_type = (ws.get("document_type") or "BRD").upper()
    bpmn_xml = ws["bpmn_xml"]

    ws_children, bpmn_override, chart_children = WorkspaceState.build_content(
        ws,
        role,
        chat_id,
    )

    if bpmn_override:
        bpmn_xml = bpmn_override

    role_color_maps = {
        "ba": BA_COLORS,
        "po": PO_COLORS,
        "pm": PM_COLORS,
    }

    role_labels = {
        "ba": "📊 [BA]",
        "po": "🎯 [PO]",
        "pm": "🧭 [PM]",
    }

    role_colors = role_color_maps.get(role, BA_COLORS)
    role_label = role_labels.get(role, "📊 [BA]")

    # Any non-general mode is an artifact mode. The workspace owns the
    # full artifact; chat only shows a compact preview message.

    msgs = load_messages(chat_id)
    bubbles = []

    if not msgs:
        bubbles = [make_welcome(role)]
    else:
        last_user_msg = ""

        for message in msgs:
            content = message["content"]
            msg_mode = message.get("mode", "general")
            color = role_colors.get(msg_mode, "#1a73e8")

            if message["role"] == "user":
                last_user_msg = content

                bubbles.append(
                    html.Div(
                        [
                            html.Div(
                                content,
                                style={
                                    "background": "#1a73e8",
                                    "color": "white",
                                    "padding": "10px 14px",
                                    "borderRadius": "16px 16px 4px 16px",
                                    "display": "inline-block",
                                    "maxWidth": "85%",
                                    "fontSize": "13px",
                                },
                            ),
                            html.Div(
                                "You",
                                style={
                                    "fontSize": "11px",
                                    "color": "#bbb",
                                    "marginTop": "4px",
                                    "textAlign": "right",
                                },
                            ),
                        ],
                        style={"textAlign": "right"},
                        className="message-bubble",
                    )
                )

                continue

            is_bpmn_msg = content.startswith("[BPMN]")
            artifact = get_workspace_artifact_by_message_id(message.get("id"))
            artifact_id = artifact["id"] if artifact else None
            artifact_document_type = (
                resolve_artifact_document_type(artifact)
                if artifact
                else ""
            )
            display_mode = get_workspace_display_mode(
                role,
                msg_mode,
                artifact_document_type,
            )

            had_doc = (
                is_bpmn_msg
                or msg_mode != "general"
            )

            open_btn = (
                html.Div(
                    dbc.Button(
                        "📖 Open Workspace",
                        id={
                            "type": "open-workspace-btn",
                            "index": artifact_id,
                        },
                        color="success" if is_bpmn_msg else "primary",
                        size="sm",
                        n_clicks=0,
                    ),
                    style={"marginTop": "10px"},
                )
                if had_doc and artifact_id
                else html.Div()
            )

            if is_bpmn_msg:
                # BPMN is represented as a workspace artifact.
                bubbles.append(
                    html.Div(
                        [
                            html.Div(
                                html.Span(
                                    f"📊 [BA]·{msg_mode}·BPMN",
                                    style={
                                        "fontWeight": "600",
                                        "color": "#00695c",
                                        "fontSize": "12px",
                                    },
                                ),
                                style={"marginBottom": "6px"},
                            ),
                            html.Div(
                                "🔷 BPMN diagram — open Workspace to view",
                                style={
                                    "background": "#e8f5e9",
                                    "padding": "10px 14px",
                                    "borderRadius": "16px 16px 16px 4px",
                                    "borderLeft": "3px solid #00695c",
                                    "fontSize": "13px",
                                    "display": "inline-block",
                                    "maxWidth": "85%",
                                    "color": "#1b5e20",
                                },
                            ),
                            open_btn,
                        ],
                        style={"textAlign": "left"},
                        className="message-bubble",
                    )
                )

            elif (
                is_structured_card_mode(role, msg_mode)
                and parse_po_json(content) is not None
            ):
                # Structured cards are workspace artifacts.
                bubbles.append(
                    html.Div(
                        [
                            html.Div(
                                html.Span(
                                    f"{role_label}·{display_mode}",
                                    style={
                                        "fontWeight": "600",
                                        "color": color,
                                        "fontSize": "12px",
                                    },
                                ),
                                style={"marginBottom": "8px"},
                            ),
                            html.Div(
                                "✅ Cards ready — Preview in Workspace →",
                                style={
                                    "background": "#f1f3f4",
                                    "padding": "10px 14px",
                                    "borderRadius": "16px 16px 16px 4px",
                                    "borderLeft": f"3px solid {color}",
                                    "fontSize": "13px",
                                },
                            ),
                            open_btn,
                        ],
                        style={"textAlign": "left"},
                        className="message-bubble",
                    )
                )

            elif msg_mode != "general":
                # All non-general text artifacts use a compact chat preview.
                bubbles.append(
                    html.Div(
                        [
                            html.Div(
                                html.Span(
                                    f"{role_label}·{display_mode}",
                                    style={
                                        "fontWeight": "600",
                                        "color": color,
                                        "fontSize": "12px",
                                    },
                                ),
                                style={"marginBottom": "8px"},
                            ),
                            html.Div(
                                (
                                    "✅ Cards ready — Preview in Workspace →"
                                    if is_structured_card_mode(role, msg_mode)
                                    else "📄 Document ready — Preview in Workspace →"
                                ),
                                style={
                                    "background": "#f1f3f4",
                                    "padding": "10px 14px",
                                    "borderRadius": "16px 16px 16px 4px",
                                    "borderLeft": f"3px solid {color}",
                                    "fontSize": "13px",
                                },
                            ),
                            open_btn,
                        ],
                        style={"textAlign": "left"},
                        className="message-bubble",
                    )
                )

            else:
                # Normal conversational responses remain visible in chat.
                bubbles.append(
                    html.Div(
                        [
                            html.Div(
                                html.Span(
                                    f"{role_label}·{display_mode}",
                                    style={
                                        "fontWeight": "600",
                                        "color": color,
                                        "fontSize": "12px",
                                    },
                                ),
                                style={"marginBottom": "6px"},
                            ),
                            dcc.Markdown(
                                content[:3000],
                                style={
                                    "background": "#f1f3f4",
                                    "padding": "10px 14px",
                                    "borderRadius": "16px 16px 16px 4px",
                                    "borderLeft": f"3px solid {color}",
                                    "fontSize": "13px",
                                    "display": "inline-block",
                                    "maxWidth": "85%",
                                },
                                className="markdown-body",
                            ),
                            open_btn,
                        ],
                        style={"textAlign": "left"},
                        className="message-bubble",
                    )
                )

    ws_col_style = (
        WS_OPEN_STYLE
        if workspace_open
        else WS_CLOSED_STYLE
    )

    chat_col_style = (
        CHAT_HALF_STYLE
        if workspace_open
        else CHAT_FULL_STYLE
    )

    hidden = {"display": "none"}
    visible = {
        "display": "flex",
        "flexWrap": "wrap",
        "gap": "6px",
    }

    ba_style = visible if role == "ba" else hidden
    po_style = visible if role == "po" else hidden
    pm_style = visible if role == "pm" else hidden

    ba_active = [m == mode for m in BA_MODES]
    po_active = [m == mode for m in PO_MODES]
    pm_active = [m == mode for m in PM_MODES]

    return (
        bubbles,
        False,
        chat_id,
        role,
        mode,
        submode,
        workspace_open,
        doc_content,
        doc_mode,
        bpmn_xml,
        role,
        ba_style,
        po_style,
        pm_style,
        ws_children,
        chart_children,
        ws_col_style,
        chat_col_style,
        *["primary" if active else "light" for active in ba_active],
        *[not active for active in ba_active],
        *["primary" if active else "light" for active in po_active],
        *[not active for active in po_active],
        *["primary" if active else "light" for active in pm_active],
        *[not active for active in pm_active],
    )

# ── EXPORT ───────────────────────────────────────────────────────
@app.callback(
    Output("download-file","data"),
    [Input("export-docx","n_clicks"), Input("export-pdf","n_clicks"),
     Input("export-txt","n_clicks"),  Input("export-md","n_clicks"),
     Input("export-bpmn","n_clicks")],
    [State("doc-content","data"), State("doc-mode","data"),
     State("bpmn-xml-store","data")], prevent_initial_call=True)
def handle_export(d,p,t,m,bpmn_n, content, mode, bpmn_xml):
    if not content and not bpmn_xml:
        raise dash.exceptions.PreventUpdate
    ctx = callback_context
    which = ctx.triggered[0]["prop_id"].split(".")[0] if ctx.triggered else None
    stamp = datetime.now().strftime('%Y%m%d_%H%M')
    export_content = scope_payload_to_markdown(content) if mode == "scope" else content
    if which=="export-docx":
        return dcc.send_bytes(export_to_docx(export_content, mode), filename=f"BAPO_{mode}_{stamp}.docx")
    if which=="export-pdf":
        return dcc.send_bytes(export_to_pdf(export_content, mode), filename=f"BAPO_{mode}_{stamp}.pdf")
    if which=="export-txt":
        return dcc.send_bytes(export_to_txt(export_content, mode), filename=f"BAPO_{mode}_{stamp}.txt")
    if which=="export-md":
        return dcc.send_bytes(export_to_markdown(export_content, mode), filename=f"BAPO_{mode}_{stamp}.md")
    if which=="export-bpmn": return dcc.send_bytes((bpmn_xml or "").encode("utf-8"), filename=f"BAPO_bpmn_{stamp}.bpmn")

# ── HISTORY SIDEBAR ───────────────────────────────────────────────
@app.callback(
    [Output("history-offcanvas","is_open",allow_duplicate=True),
     Output("sidebar-content","children")],
    Input("history-btn","n_clicks"), State("history-offcanvas","is_open"), prevent_initial_call=True)
def toggle_history(n, is_open):
    return not is_open, render_sidebar_tabs()

# ── DELETE CHAT ───────────────────────────────────────────────────
@app.callback(
    [Output("delete-modal","is_open"), Output("delete-chat-id","data")],
    Input({"type":"sidebar-delete-btn","index":ALL},"n_clicks"),
    State({"type":"sidebar-delete-btn","index":ALL},"id"), prevent_initial_call=True)
def open_delete_modal(clicks, ids):
    ctx=callback_context
    if not ctx.triggered or not any(clicks): raise dash.exceptions.PreventUpdate
    return True, json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"]

@app.callback(
    [Output("delete-modal","is_open",   allow_duplicate=True),
     Output("sidebar-content","children",allow_duplicate=True),
     Output("chat-messages","children", allow_duplicate=True),
     Output("current-chat-id","data",   allow_duplicate=True),
     Output("workspace-col","style",    allow_duplicate=True),
     Output("chat-col","style",         allow_duplicate=True),
     Output("doc-content","data",       allow_duplicate=True),
     Output("bpmn-xml-store","data",    allow_duplicate=True),
     Output("chart-area","children",    allow_duplicate=True),
     Output("workspace-open","data",    allow_duplicate=True)],
    Input("delete-confirm-btn","n_clicks"),
    [State("delete-chat-id","data"), State("current-chat-id","data"), State("current-role","data")],
    prevent_initial_call=True)
def confirm_delete(n, chat_id, current_id, role):
    if not chat_id: raise dash.exceptions.PreventUpdate
    delete_chat(chat_id)
    if chat_id == current_id:
        return (False, render_sidebar_tabs(), [make_welcome(role)], None,
                WS_CLOSED_STYLE, CHAT_FULL_STYLE, "", "", [], False)
    return (False, render_sidebar_tabs(), dash.no_update, dash.no_update,
            dash.no_update, dash.no_update, dash.no_update, dash.no_update,
            dash.no_update, dash.no_update)

@app.callback(Output("delete-modal","is_open",allow_duplicate=True),
    Input("delete-cancel-btn","n_clicks"), prevent_initial_call=True)
def cancel_delete(n): return False

# ── RENAME CHAT ───────────────────────────────────────────────────
@app.callback(
    [Output("rename-modal","is_open"), Output("rename-chat-id","data"), Output("rename-input","value")],
    Input({"type":"sidebar-rename-btn","index":ALL},"n_clicks"),
    State({"type":"sidebar-rename-btn","index":ALL},"id"), prevent_initial_call=True)
def open_rename_modal(clicks, ids):
    ctx=callback_context
    if not ctx.triggered or not any(clicks): raise dash.exceptions.PreventUpdate
    cid=json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"]
    chats=load_chats(); title=next((c["title"] for c in chats if c["id"]==cid),"")
    return True, cid, title

@app.callback(
    [Output("rename-modal","is_open",allow_duplicate=True),
     Output("sidebar-content","children",allow_duplicate=True)],
    Input("rename-save-btn","n_clicks"),
    [State("rename-chat-id","data"), State("rename-input","value")], prevent_initial_call=True)
def save_rename(n, cid, title):
    if cid and title: rename_chat(cid, title.strip())
    return False, render_sidebar_tabs()

@app.callback(Output("rename-modal","is_open",allow_duplicate=True),
    Input("rename-cancel-btn","n_clicks"), prevent_initial_call=True)
def cancel_rename(n): return False

# ── ARCHIVE / UNARCHIVE ───────────────────────────────────────────
@app.callback(
    Output("sidebar-content","children",allow_duplicate=True),
    Input({"type":"sidebar-archive-btn","index":ALL},"n_clicks"),
    State({"type":"sidebar-archive-btn","index":ALL},"id"), prevent_initial_call=True)
def do_archive(clicks, ids):
    ctx=callback_context
    if not ctx.triggered or not any(clicks): raise dash.exceptions.PreventUpdate
    archive_chat(json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"])
    return render_sidebar_tabs()

@app.callback(
    Output("sidebar-content","children",allow_duplicate=True),
    Input({"type":"sidebar-unarchive-btn","index":ALL},"n_clicks"),
    State({"type":"sidebar-unarchive-btn","index":ALL},"id"), prevent_initial_call=True)
def do_unarchive(clicks, ids):
    ctx=callback_context
    if not ctx.triggered or not any(clicks): raise dash.exceptions.PreventUpdate
    unarchive_chat(json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"])
    return render_sidebar_tabs()

# ── PROJECT COLLAPSE / EXPAND ─────────────────────────────────────
@app.callback(
    Output("sidebar-content", "children", allow_duplicate=True),
    Input({"type": "project-collapse-btn", "index": ALL}, "n_clicks"),
    prevent_initial_call=True,
)
def collapse_single_project(_clicks):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    trigger = ctx.triggered[0]
    if not trigger.get("value"):
        raise dash.exceptions.PreventUpdate

    try:
        triggered_id = json.loads(trigger["prop_id"].split(".")[0])
    except (TypeError, ValueError, json.JSONDecodeError):
        raise dash.exceptions.PreventUpdate

    project_id = triggered_id.get("index")
    if not project_id:
        raise dash.exceptions.PreventUpdate

    project = next(
        (item for item in load_projects() if item["id"] == project_id),
        None,
    )
    if not project or project.get("is_collapsed"):
        raise dash.exceptions.PreventUpdate

    set_project_collapsed(project_id, True)
    return render_sidebar_tabs()


@app.callback(
    Output("sidebar-content", "children", allow_duplicate=True),
    Input({"type": "project-expand-btn", "index": ALL}, "n_clicks"),
    prevent_initial_call=True,
)
def expand_single_project(_clicks):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    trigger = ctx.triggered[0]
    if not trigger.get("value"):
        raise dash.exceptions.PreventUpdate

    try:
        triggered_id = json.loads(trigger["prop_id"].split(".")[0])
    except (TypeError, ValueError, json.JSONDecodeError):
        raise dash.exceptions.PreventUpdate

    project_id = triggered_id.get("index")
    if not project_id:
        raise dash.exceptions.PreventUpdate

    project = next(
        (item for item in load_projects() if item["id"] == project_id),
        None,
    )
    if not project or not project.get("is_collapsed"):
        raise dash.exceptions.PreventUpdate

    set_project_collapsed(project_id, False)
    return render_sidebar_tabs()


@app.callback(
    Output("sidebar-content", "children", allow_duplicate=True),
    Input("collapse-all-projects-btn", "n_clicks"),
    prevent_initial_call=True,
)
def collapse_all_projects(_n_clicks):
    if not _n_clicks:
        raise dash.exceptions.PreventUpdate

    set_all_projects_collapsed(True)
    return render_sidebar_tabs()


@app.callback(
    Output("sidebar-content", "children", allow_duplicate=True),
    Input("expand-all-projects-btn", "n_clicks"),
    prevent_initial_call=True,
)
def expand_all_projects(_n_clicks):
    if not _n_clicks:
        raise dash.exceptions.PreventUpdate

    set_all_projects_collapsed(False)
    return render_sidebar_tabs()

# ── NEW PROJECT ───────────────────────────────────────────────────
@app.callback(
    [
        Output("new-project-modal", "is_open"),
        Output("new-project-startup-opened", "data"),
    ],
    Input("new-project-startup-opened", "data"),
    prevent_initial_call=False,
)
def open_new_project_on_startup(startup_opened):
    if startup_opened:
        return True, False
    raise dash.exceptions.PreventUpdate


@app.callback(
    Output("new-project-modal", "is_open", allow_duplicate=True),
    Input("new-project-btn", "n_clicks"),
    prevent_initial_call=True,
)
def open_new_project_manually(n_clicks):
    if n_clicks:
        return True
    raise dash.exceptions.PreventUpdate

@app.callback(
    [Output("new-project-modal","is_open",allow_duplicate=True),
     Output("sidebar-content","children",allow_duplicate=True),
     Output("new-project-input","value")],
    [Input("create-project-btn","n_clicks"), Input("cancel-project-btn","n_clicks")],
    State("new-project-input","value"), prevent_initial_call=True)
def handle_new_project(create_n, cancel_n, name):
    ctx = callback_context
    if not ctx.triggered:
        raise dash.exceptions.PreventUpdate

    triggered_id = ctx.triggered[0]["prop_id"].split(".")[0]

    if triggered_id == "create-project-btn" and name and name.strip():
        create_project(name.strip())
        return False, render_sidebar_tabs(), ""

    if triggered_id == "cancel-project-btn":
        return False, dash.no_update, ""

    raise dash.exceptions.PreventUpdate

# ── RENAME PROJECT ────────────────────────────────────────────────
@app.callback(
    [Output("rename-project-modal","is_open"),
     Output("rename-project-id","data"),
     Output("rename-project-input","value")],
    Input({"type":"rename-project-btn","index":ALL},"n_clicks"),
    State({"type":"rename-project-btn","index":ALL},"id"), prevent_initial_call=True)
def open_rename_project(clicks, ids):
    ctx=callback_context
    if not ctx.triggered or not any(clicks): raise dash.exceptions.PreventUpdate
    pid=json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"]
    projects=load_projects(); name=next((p["name"] for p in projects if p["id"]==pid),"")
    return True, pid, name

@app.callback(
    [Output("rename-project-modal","is_open",allow_duplicate=True),
     Output("sidebar-content","children",allow_duplicate=True)],
    [Input("save-rename-project-btn","n_clicks"), Input("cancel-rename-project-btn","n_clicks")],
    [State("rename-project-id","data"), State("rename-project-input","value")],
    prevent_initial_call=True)
def save_rename_project(save_n, cancel_n, pid, name):
    ctx=callback_context
    if not ctx.triggered: raise dash.exceptions.PreventUpdate
    triggered_id = ctx.triggered[0]["prop_id"].split(".")[0]
    if triggered_id == "save-rename-project-btn" and pid and name:
        rename_project(pid, name)
        return False, render_sidebar_tabs()
    if triggered_id == "cancel-rename-project-btn":
        return False, render_sidebar_tabs()
    return False, dash.no_update

# ── DELETE PROJECT ────────────────────────────────────────────────
@app.callback(
    Output("sidebar-content","children",allow_duplicate=True),
    Input({"type":"delete-project-btn","index":ALL},"n_clicks"),
    State({"type":"delete-project-btn","index":ALL},"id"), prevent_initial_call=True)
def do_delete_project(clicks, ids):
    ctx=callback_context
    if not ctx.triggered or not any(clicks): raise dash.exceptions.PreventUpdate
    delete_project(json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"])
    return render_sidebar_tabs()

# ── MOVE CHAT ─────────────────────────────────────────────────────
@app.callback(
    [Output("move-chat-modal","is_open"),
     Output("move-chat-id","data"),
     Output("move-project-select","options")],
    Input({"type":"sidebar-move-btn","index":ALL},"n_clicks"),
    State({"type":"sidebar-move-btn","index":ALL},"id"), prevent_initial_call=True)
def open_move_modal(clicks, ids):
    ctx=callback_context
    if not ctx.triggered or not any(clicks): raise dash.exceptions.PreventUpdate
    cid=json.loads(ctx.triggered[0]["prop_id"].split(".")[0])["index"]
    opts=[{"label":p["name"],"value":p["id"]} for p in load_projects()]
    return True, cid, opts

@app.callback(
    [Output("move-chat-modal","is_open",allow_duplicate=True),
     Output("sidebar-content","children",allow_duplicate=True)],
    [Input("confirm-move-btn","n_clicks"), Input("cancel-move-btn","n_clicks")],
    [State("move-chat-id","data"), State("move-project-select","value")],
    prevent_initial_call=True)
def confirm_move(confirm_n, cancel_n, chat_id, pid):
    ctx=callback_context
    if not ctx.triggered: raise dash.exceptions.PreventUpdate
    triggered_id = ctx.triggered[0]["prop_id"].split(".")[0]
    if triggered_id == "confirm-move-btn" and chat_id and pid:
        assign_chat_project(chat_id, pid)
        return False, render_sidebar_tabs()
    if triggered_id == "cancel-move-btn":
        return False, render_sidebar_tabs()
    return False, dash.no_update

# ── COPY ──────────────────────────────────────────────────────────
app.clientside_callback(
    """async function(n_clicks){
        if(!n_clicks){return window.dash_clientside.no_update;}
        var ws=document.getElementById("workspace-content");
        if(!ws){return "";}
        var text=ws.innerText||ws.textContent||"";
        if(!text.trim()){alert("Nothing to copy.");return "";}
        try{
            await navigator.clipboard.writeText(text);
            var btn=document.getElementById("copy-btn");
            if(btn){var o=btn.innerText;btn.innerText="✅ Copied!";setTimeout(function(){btn.innerText=o;},1500);}
        }catch(err){alert("Unable to copy.");}
        return "";
    }""",
    Output("hidden-clipboard","value"),
    Input("copy-btn","n_clicks"), prevent_initial_call=True)

# ── BPMN VIEWER ───────────────────────────────────────────────────
app.clientside_callback(
    """function(xml){
        if(!xml||typeof BpmnJS==='undefined'){return window.dash_clientside.no_update;}
        var el=document.getElementById('bpmn-canvas');
        if(!el){return window.dash_clientside.no_update;}

        function doFit(){
            if(!window.__bpmnViewer) return;
            var canvas=window.__bpmnViewer.get('canvas');
            var rect=el.getBoundingClientRect();
            if(Number.isFinite(rect.width)&&Number.isFinite(rect.height)&&rect.width>=20&&rect.height>=20){
                canvas.resized();
                canvas.zoom('fit-viewport','auto');
            }
        }

        var attempt=0;
        function renderWhenReady(){
            var rect=el&&el.getBoundingClientRect();
            if(!el||!rect||!Number.isFinite(rect.width)||!Number.isFinite(rect.height)||rect.width<20||rect.height<20){
                if(attempt++<20){setTimeout(renderWhenReady,50);}
                return;
            }
            if(!window.__bpmnViewer){window.__bpmnViewer=new BpmnJS({container:'#bpmn-canvas'});}
            window.__bpmnViewer.importXML(xml).then(function(){
                requestAnimationFrame(function(){
                    doFit();
                    // Re-fit again shortly after, in case the panel is still
                    // mid-transition (e.g. just opened via Open Workspace).
                    setTimeout(doFit, 150);
                    setTimeout(doFit, 400);
                });

                // Keep the diagram centered if the panel is resized later
                // (window resize, sidebar toggle, etc.)
                if(window.__bpmnResizeObs){window.__bpmnResizeObs.disconnect();}
                if('ResizeObserver' in window){
                    window.__bpmnResizeObs=new ResizeObserver(function(){doFit();});
                    window.__bpmnResizeObs.observe(el);
                }
            }).catch(function(err){
                el.innerHTML='<div style="padding:20px;color:#c62828;font-size:13px;">⚠️ Unable to render BPMN: '+err.message+'</div>';
            });
        }
        setTimeout(renderWhenReady,0);
        return window.dash_clientside.no_update;
    }""",
    Output("bpmn-canvas-dummy","data"),
    Input("bpmn-xml-store","data"), prevent_initial_call=True)

# ── BPMN SVG / PNG EXPORT ─────────────────────────────────────────
app.clientside_callback(
    """function(svgN,pngN){
        var ctx=window.dash_clientside.callback_context;
        if(!ctx.triggered.length||!window.__bpmnViewer){return window.dash_clientside.no_update;}
        var trig=ctx.triggered[0].prop_id.split('.')[0];
        window.__bpmnViewer.saveSVG().then(function(r){
            var svg=r.svg;
            if(trig==='export-bpmn-svg'){
                var a=document.createElement('a');
                a.href=URL.createObjectURL(new Blob([svg],{type:'image/svg+xml'}));
                a.download='bpmn_diagram.svg';a.click();
            }else if(trig==='export-bpmn-png'){
                var img=new Image();
                var url=URL.createObjectURL(new Blob([svg],{type:'image/svg+xml;charset=utf-8'}));
                img.onload=function(){
                    var c=document.createElement('canvas');
                    c.width=img.width*2;c.height=img.height*2;
                    var ctx2=c.getContext('2d');
                    ctx2.fillStyle='#ffffff';ctx2.fillRect(0,0,c.width,c.height);
                    ctx2.drawImage(img,0,0,c.width,c.height);
                    URL.revokeObjectURL(url);
                    c.toBlob(function(blob){var a=document.createElement('a');a.href=URL.createObjectURL(blob);a.download='bpmn_diagram.png';a.click();});
                };img.src=url;
            }
        }).catch(function(e){console.error(e);});
        return window.dash_clientside.no_update;
    }""",
    Output("bpmn-canvas-dummy","data",allow_duplicate=True),
    [Input("export-bpmn-svg","n_clicks"), Input("export-bpmn-png","n_clicks")],
    prevent_initial_call=True)

# ════════════════════════════════════════════════════════════════
# STARTUP
# ════════════════════════════════════════════════════════════════
def open_browser():
    time.sleep(1.5)
    webbrowser.open("http://127.0.0.1:8051")

if __name__ == "__main__":
    threading.Thread(target=open_browser, daemon=True).start()
    try:
        app.run(debug=False, port=8051)
    finally:
        save_session_on_exit()